# NB-04_sintesis_reajustes

Process Flow SAS: **Síntesis_Reajustes** — `PFD-9xOO9Nj6dXtUnawt`

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text
import pyreadstat

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-04_sintesis_reajustes.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S2_01_Inicio

Construye la base BD_CTSI cargando el cierre de presíntesis y aplica reglas de reclasificación de contragentes e instrumentos financieros (AF.29/32/41/42/5/7) por año/trimestre, imputando ajustes de SIFMI, dividendos de hogares, préstamos de corto plazo y bonos de largo plazo en patrimonio separado

*confianza: medium · verificador: revise · SAS: PROC SQL: CREATE TABLE desde archivo externo + >50 UPDATE/DELETE de reclasificación + PROC APPEND FORCE con WORK temporales*

In [ ]:
# ========= S2_01_Inicio =========
# TRAE TABLA PRINCIPAL DESDE PRESÍNTESIS PARA COMENZAR EL PROCESO
# M-001: ruta original hardcodeada '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/BD_CTSI_CIERRE.sas7bdat' -> relativa al workspace
ruta_bd_ctsi_cierre = Path("sasdata") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "02_PRE_SINTESIS" / "BD_CTSI_CIERRE.sas7bdat"
bd_ctsi_cierre, _ = pyreadstat.read_sas7bdat(str(ruta_bd_ctsi_cierre))

# COMPRIME TABLA PRINCIPAL (COMPRESS=YES no tiene equivalente físico; se conserva el mismo contenido)
# AGREGA COLUMNA DE PROC A BASE BD_CTSI
bd_ctsi_cierre["PROC"] = pd.Series(dtype="object")
# alter FUENTE char(12) -> se normaliza como string
bd_ctsi_cierre["FUENTE"] = bd_ctsi_cierre["FUENTE"].astype("object")

with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI"))
    _log("DELETE previo TABLAS.dbo.BD_CTSI", res.rowcount)
bd_ctsi_cierre.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("BD_CTSI cargada desde presíntesis", bd_ctsi_cierre)


In [ ]:
# ELIMINA AÑO 2002 DE LA BASE
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE AÑO = 2002"))
    _log("DELETE año 2002", res.rowcount)

# ELIMINA STOCKS QUE NO SON DE LA CUENTA CORRESPONDIENTE
sql_del_stocks = """
DELETE FROM TABLAS.dbo.BD_CTSI
WHERE C_CUENTA IN ('Bce Final','Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen')
  AND C_SCN IN ('D.41','K.1','P.2','P.51','P.11','D.42','D.5','B.9','B.90','B.10.2','B.10.3','P.52','D.62','D.75','D.1')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_del_stocks))
    _log("DELETE stocks no correspondientes", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES 36912 a 36 en todos los sectores porque es lo mismo y la síntesis está con 36. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
# cierre 2021: se deja contragente 36912 tal cual ya que ahora existe ese sector. Se cambia sector 363 a 36912, ya que no tenemos sector 363
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '36912' WHERE C_CAGENTE IN ('363')"))
    _log("UPDATE C_CAGENTE 363->36912", res.rowcount)

# CAMBIA CONTRAGENTES 7 a 53 en todos los sectores porque es lo mismo. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53' WHERE C_CAGENTE IN ('7','7.1','7.2')"))
    _log("UPDATE C_CAGENTE 7->53", res.rowcount)

# CAMBIA CONTRAGENTES 512 a 511 en todos los sectores porque es lo mismo. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_CAGENTE IN ('512')"))
    _log("UPDATE C_CAGENTE 512->511", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES NACIONALES EN EMISIONES DE TITULO DE HOLDING Y CASAS MATRICES A 51
sql_1 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_CAGENTE NOT IN ('6') AND SECTOR IN (37,36907) AND C_SCN IN ('AF.31','AF.32') AND C_ENTRADA = 'H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_1))
    _log("UPDATE holding/casas matrices -> 53", res.rowcount)

# CAMBIA CONTRAGENTES NACIONALES EN EMISIONES DE TITULO DE EMPRESAS A 53
sql_2 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_CAGENTE NOT IN ('6','54') AND SECTOR IN (51021,5101) AND C_SCN IN ('AF.31','AF.32') AND C_ENTRADA = 'H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_2))
    _log("UPDATE empresas -> 53", res.rowcount)

# CAMBIA CONTRAGENTES en inversiones en af31 con CA 6 EN FONDOS MUTUOS A AF32
sql_3 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.32', N_SCN = 'Valores distintos de acciones a largo plazo'
WHERE C_CAGENTE = '6' AND SECTOR IN (3390101,3390102) AND C_SCN IN ('AF.31') AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_3))
    _log("UPDATE AF31->AF32 fondos mutuos", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF41 ACTIVO DEL RM DESDE VACIO A 53
# SAS: C_CAGENTE='' captura blanco real y NULL de un JOIN previo -> IS NULL OR = ''
sql_4 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE (C_CAGENTE IS NULL OR C_CAGENTE = '') AND SECTOR = 6 AND C_SCN = 'AF.41' AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_4))
    _log("UPDATE AF41 activo RM vacio -> 53", res.rowcount)

# CAMBIA CONTRAGENTES EN AF7 ACTIVO DE RESTO DE EMPRESAS CON CONTRAGENTE HOGARES
sql_5 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_CAGENTE = '511' AND SECTOR = 51022 AND C_SCN = 'AF.7' AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_5))
    _log("UPDATE AF7 activo resto empresas", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF5 ACTIVO DE FONDOS DE INVERSIÓN DESDE 0 A 53 (CIERRE 2021)
# SAS: C_CAGENTE in ('0','') es comparación exacta contra '0' y contra blanco; '' cubre blanco real y NULL de outer join -> IS NULL OR = ''
sql_6 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE (C_CAGENTE = '0' OR C_CAGENTE IS NULL OR C_CAGENTE = '') AND SECTOR = 339011 AND C_SCN = 'AF.5' AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_6))
    _log("UPDATE AF5 activo fondos inversion 0/vacio -> 53", res.rowcount)

# CAMBIA CONTRAGENTES EN AF5 ACTIVO DE FONDOS DE INVERSIÓN DESDE 339012 CORRESPONDIENTE A FONDOS DE INVERSIÓN PRIVADOS A CA 36912 (CIERRE 2021)
sql_7 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '36912'
WHERE C_CAGENTE IN ('339012') AND SECTOR = 339011 AND C_SCN = 'AF.5' AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_7))
    _log("UPDATE AF5 activo 339012 -> 36912", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (41,42,412 Y 413)
sql_gob_s1 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53' WHERE C_CAGENTE IN ('S1','S19') AND SECTOR IN (41,42,412,413)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s1))
    _log("UPDATE gobierno S1/S19 -> 53", res.rowcount)

sql_gob_s11 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51' WHERE C_CAGENTE = 'S11' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s11))
    _log("UPDATE gobierno S11 -> 51", res.rowcount)

sql_gob_s12 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '3' WHERE C_CAGENTE = 'S12' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s12))
    _log("UPDATE gobierno S12 -> 3", res.rowcount)

sql_gob_s121 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '31' WHERE C_CAGENTE = 'S121' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s121))
    _log("UPDATE gobierno S121 -> 31", res.rowcount)

sql_gob_s122 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321' WHERE C_CAGENTE = 'S122' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s122))
    _log("UPDATE gobierno S122 -> 321", res.rowcount)

sql_gob_s123 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '331' WHERE C_CAGENTE = 'S123' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s123))
    _log("UPDATE gobierno S123 -> 331", res.rowcount)

sql_gob_s13 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '4' WHERE C_CAGENTE = 'S13' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s13))
    _log("UPDATE gobierno S13 -> 4", res.rowcount)

sql_gob_s131 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '41' WHERE C_CAGENTE = 'S131' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s131))
    _log("UPDATE gobierno S131 -> 41", res.rowcount)

sql_gob_s14 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_CAGENTE = 'S14' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s14))
    _log("UPDATE gobierno S14 -> 511", res.rowcount)

sql_gob_s2 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '6' WHERE C_CAGENTE = 'S2' AND SECTOR IN (41,42,412,413)"
with engine.begin() as conn:
    res = conn.execute(text(sql_gob_s2))
    _log("UPDATE gobierno S2 -> 6", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE INSTRUMENTOS EN DEPÓSITOS AF29 ENTRE AFP Y FONDOS DE PENSIONES
sql_af29_1 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.612', N_SCN = 'Reserva de fondos de pensiones'
WHERE C_CAGENTE = '34' AND SECTOR = 361 AND C_SCN = 'AF.29' AND C_ENTRADA = 'D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af29_1))
    _log("UPDATE AF29 34/361 D -> AF.612", res.rowcount)

sql_af29_2 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.612', N_SCN = 'Reserva de fondos de pensiones'
WHERE C_CAGENTE = '361' AND SECTOR = 34 AND C_SCN = 'AF.29' AND C_ENTRADA = 'H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af29_2))
    _log("UPDATE AF29 361/34 H -> AF.612", res.rowcount)

# CAMBIA CONTRAGENTE 339011P
sql_339011p = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '339011' WHERE C_CAGENTE = '339011_p'"
with engine.begin() as conn:
    res = conn.execute(text(sql_339011p))
    _log("UPDATE 339011_p -> 339011", res.rowcount)


In [ ]:
sql_af41_36909 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE SECTOR = 36909 AND C_SCN = 'AF.41' AND C_ENTRADA = 'D' AND C_CAGENTE IN ('321','36904')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af41_36909))
    _log("UPDATE AF41 36909 (1)", res.rowcount)

# repetido idéntico en el SAS original
with engine.begin() as conn:
    res = conn.execute(text(sql_af41_36909))
    _log("UPDATE AF41 36909 (2, repetido en SAS)", res.rowcount)

sql_af5_512 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND C_CAGENTE = '512'"
with engine.begin() as conn:
    res = conn.execute(text(sql_af5_512))
    _log("UPDATE AF5 512 -> 511", res.rowcount)

# SAS: C_CAGENTE='' captura blanco real y NULL de outer join -> IS NULL OR = ''
sql_af5_352_vacio = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND (C_CAGENTE IS NULL OR C_CAGENTE = '') AND SECTOR = 352
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af5_352_vacio))
    _log("UPDATE AF5 352 vacio -> 53", res.rowcount)

sql_af5_339011_ = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '339011'
WHERE C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND C_CAGENTE = '339011_' AND SECTOR = 352
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af5_339011_))
    _log("UPDATE AF5 339011_ -> 339011", res.rowcount)

# CIERRE 2021: PARA DEJAR BIEN CONTRAGENTE EN CUOTAS DE FONDOS
sql_af522_cuotas = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '3390102'
WHERE C_SCN = 'AF.522' AND C_ENTRADA = 'D' AND C_CAGENTE = '339012' AND SECTOR = 339011
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af522_cuotas))
    _log("UPDATE AF522 cuotas fondos", res.rowcount)

# CIERRE 2022. CAMBIA C_CAGENTE EN AF7 ACTIVO DE BANCOS CON HOGARES A RESTO, YA QUE NO DEBE PASAR A SER PASIVO DE LOS HOGARES
sql_af7_321 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_SCN = 'AF.7' AND C_ENTRADA = 'D' AND C_CAGENTE = '511' AND SECTOR = 321
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af7_321))
    _log("UPDATE AF7 bancos-hogares -> resto", res.rowcount)


In [ ]:
# CIERRE 2023q2. CAMBIA C_CAGENTE EN AF41 ACTIVO DE FONDOS DE INVERSIÓN CON SEGUROS A REST
sql_af41_fi_seg = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_SCN = 'AF.41' AND C_ENTRADA = 'D' AND C_CAGENTE IN ('352','339011') AND SECTOR = 339011
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af41_fi_seg))
    _log("UPDATE AF41 FI/seguros -> 53", res.rowcount)

# CIERRE 2023q2. CAMBIA C_CAGENTE EN AF29 ACTIVO DE FONDOS DE INVERSIÓN CON RM A BANCOS
sql_af29_fi_rm = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321'
WHERE C_SCN = 'AF.29' AND C_ENTRADA = 'D' AND C_CAGENTE IN ('6') AND SECTOR = 339011 AND AÑO >= 2023
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af29_fi_rm))
    _log("UPDATE AF29 FI/RM -> bancos", res.rowcount)

# CIERRE 2023q3. CAMBIA C_CAGENTE EN af32 PASIVO DE FONDOS DE INVERSIÓN CON RM A RESTO DE LA ECONOMÍA
sql_af32_fi_rm = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53'
WHERE C_SCN = 'AF.32' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('6') AND SECTOR = 339011 AND AÑO >= 2023
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af32_fi_rm))
    _log("UPDATE AF32 pasivo FI/RM -> resto", res.rowcount)

# CIERRE 2023q4. CAMBIA C_CAGENTE EN af41 PASIVO DE SOCIEDADES CAUTIVAS CON GOBIERNO A BANCOS
sql_af41_cautivas = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321'
WHERE C_SCN = 'AF.41' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('41') AND SECTOR = 37
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af41_cautivas))
    _log("UPDATE AF41 cautivas/gobierno -> bancos", res.rowcount)

# CIERRE 2025q4. CAMBIA af42 ACTIVO EN SECTOR 37 A af5
sql_af42_37 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.5', C_CAGENTE = '2', N_SCN = 'Acciones y otras participaciones de capital'
WHERE C_SCN = 'AF.42' AND C_ENTRADA = 'D' AND C_CAGENTE IN ('321') AND SECTOR = 37
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af42_37))
    _log("UPDATE AF42 sector37 -> AF5", res.rowcount)


In [ ]:
# IMPUTA AJUSTES INICIALES DE SIFMI, DIVIDENDOS DE HOGARES (PROC APPEND FORCE: alinea por nombre, columnas de apoyo descartadas)
cols_rp_hh = pd.read_sql(text("SELECT TOP 0 * FROM TABLAS.dbo.BD_CTSI"), engine).columns
for tabla_origen in ["RP_HH", "SIFMI", "AJ_VARIOS", "FBCF_SF"]:
    cols_comunes = pd.read_sql(text(f"SELECT TOP 0 * FROM TABLAS.dbo.{tabla_origen}"), engine).columns
    cols = [c for c in cols_rp_hh if c in cols_comunes]
    cols_sql = ', '.join(cols)
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_sql}) SELECT {cols_sql} FROM TABLAS.dbo.{tabla_origen}"
    with engine.begin() as conn:
        res = conn.execute(text(sql_append))
        _log(f"APPEND BD_CTSI <- {tabla_origen}", res.rowcount)

sql_proc_fuentes = """
UPDATE TABLAS.dbo.BD_CTSI SET PROC = '0071'
WHERE FUENTE IN ('DI_RP_H','DI_Aj_SIFMI','AJ_DEP','AJ_K_RESTO')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_proc_fuentes))
    _log("UPDATE PROC 0071 por FUENTE", res.rowcount)

# IMPUTA PTMOS DE CORTO PLAZO EN HOGARES POR USO DE TARJETAS COMERCIALES
cols_ajptmos = pd.read_sql(text("SELECT TOP 0 * FROM TABLAS.dbo.BD_CTSI_AJPTMOS_CP"), engine).columns
cols = [c for c in cols_rp_hh if c in cols_ajptmos]
cols_sql = ', '.join(cols)
sql_append_ajptmos = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_sql}) SELECT {cols_sql} FROM TABLAS.dbo.BD_CTSI_AJPTMOS_CP"
with engine.begin() as conn:
    res = conn.execute(text(sql_append_ajptmos))
    _log("APPEND BD_CTSI <- BD_CTSI_AJPTMOS_CP", res.rowcount)


In [ ]:
# RECLASIFICA PARTICIPACIONES DE LOS FONDOS DE INVERSIÓN COMO INSTRUMENTO AF.522 PARA DIFERENCIARLO DE LO CORRESPONDIENTE A FONDOS MUTUOS
# EN TEORÍA SOLO EL MERCADO MONETARIO DEBE QUEDAR EN AF.521. LOS FFMM DE LP DEBEN TAMBIÉN QUEDAR COMO AF.522
# TODO LO QUE ES NO MONEY MARKET LO DEJA EN INST AF.522
sql_af521 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.522', N_SCN = 'Participaciones emitidas por fondos de inversión'
WHERE C_SCN = 'AF.521' AND C_ENTRADA = 'H' AND SECTOR IN (339011,3390102)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af521))
    _log("UPDATE AF521->AF522 no money market", res.rowcount)

# ACTUALIZACIONES DE REGISTROS
sql_af41_42_creditos = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.7', N_SCN = 'Créditos comerciales', PROC = '0002'
WHERE C_SCN IN ('AF.41','AF.42') AND C_ENTRADA = 'D' AND SECTOR IN (334,369011)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af41_42_creditos))
    _log("UPDATE AF41/42 -> AF7 creditos comerciales", res.rowcount)

sql_d29_produccion = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'D.21'
WHERE C_SCN = 'D.29' AND C_CUENTA = 'Producción' AND SECTOR = 41
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_d29_produccion))
    _log("UPDATE D.29 -> D.21 produccion", res.rowcount)

sql_d_511 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_SCN IN ('D.1','D.61','D.62','D.8')"
with engine.begin() as conn:
    res = conn.execute(text(sql_d_511))
    _log("UPDATE C_CAGENTE 511 por D.x", res.rowcount)

sql_d_41 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '41' WHERE C_SCN IN ('D.21','D.29','D.5')"
with engine.begin() as conn:
    res = conn.execute(text(sql_d_41))
    _log("UPDATE C_CAGENTE 41 por D.x", res.rowcount)

sql_d_53_sector41 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53' WHERE C_SCN IN ('D.21','D.29','D.5') AND SECTOR = 41"
with engine.begin() as conn:
    res = conn.execute(text(sql_d_53_sector41))
    _log("UPDATE C_CAGENTE 53 por D.x sector41", res.rowcount)

sql_331_ci = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51', PROC = '0007Z'
WHERE C_CAGENTE = '331' AND SECTOR = 41 AND C_SCN = 'AF.42' AND FUENTE = 'CI'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_331_ci))
    _log("UPDATE 331->51 CI", res.rowcount)


In [ ]:
# session_sql: WORK nace de la BD, se materializa como #tmp por work_conn (server-side)
# IMPUTA AF42 EN ACTIVO DE GOB CENTRAL CON CA 321 USANDO INFO DEL PASIVO DE BCOS COMERCIALES CON CA CORFO(331). AJUSTA EN GOB CONTRA SNF
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_331"))
sql_af42_41_331 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0007x' AS PROC
INTO #af42_41_331
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 321 AND T1.C_CAGENTE = '331' AND T1.C_ENTRADA = 'H'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_41_331))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_snf"))
sql_af42_41_snf = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '51' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO * -1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0007y' AS PROC
INTO #af42_41_snf
FROM #af42_41_331
"""
work_conn.execute(text(sql_af42_41_snf))

cols_af42 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42}) SELECT {cols_af42} FROM #af42_41_331"))
_log("APPEND BD_CTSI <- #af42_41_331", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42}) SELECT {cols_af42} FROM #af42_41_snf"))
_log("APPEND BD_CTSI <- #af42_41_snf", res.rowcount)

for t in ["#af42_41_331", "#af42_41_snf"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZACIONES DE REGISTROS
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE C_CUENTA = 'Var Balance'"))
    _log("DELETE Var Balance", res.rowcount)

# SAS: C_CAGENTE="" asigna string vacío (no NULL); se mantiene como asignación literal
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '' WHERE C_SCN IN ('K.1','P.51')"))
    _log("UPDATE C_CAGENTE vacio K.1/P.51", res.rowcount)

sql_af2_af22 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.22', N_SCN = 'Depósitos', PROC = '0030'
WHERE C_SCN = 'AF.2'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af2_af22))
    _log("UPDATE AF.2 -> AF.22", res.rowcount)

sql_af1_af42 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.42', N_SCN = 'Préstamos a largo plazo', PROC = '0032', FUENTE = 'PS'
WHERE C_SCN = 'AF.1' AND C_ENTRADA = 'D' AND SECTOR = 6
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af1_af42))
    _log("UPDATE AF.1 -> AF.42", res.rowcount)

sql_af21_31 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '31'
WHERE SECTOR IN (41,42) AND C_SCN = 'AF.21' AND C_CAGENTE IN ('3','321')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_af21_31))
    _log("UPDATE AF21 41/42 -> 31", res.rowcount)


In [ ]:
# AJUSTA BONOS DE LARGO PLAZO DE BANCOS CON PATRIMONIO SEPARADO USANDO INFO DCV. AJUSTA EN BONOS DE LP DE BANCOS CON EMPRESAS
# SELECCIONA ACTIVO AF32 INFORMADA POR BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_bcos"))
sql_af32_bcos = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0015' AS PROC
INTO #af32_bcos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.C_CAGENTE = '332' AND T1.SECTOR = 321
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_bcos))

# DCV AF32 TENENCIA DE BCOS: fuente es archivo externo, no tabla BD -> se lee con pyreadstat y se sube como #tmp
# M-001: ruta original hardcodeada -> relativa al workspace
ruta_dcv = Path("sasdata") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "05_DCV" / "Data" / "DCVRES" / "base_af3_total_v2.sas7bdat"
dcv_raw, _ = pyreadstat.read_sas7bdat(str(ruta_dcv))
dcv_raw.columns = [c.upper() for c in dcv_raw.columns]
dcv_f = dcv_raw[
    (dcv_raw["C_SI_EMISOR"] == "3324")
    & (dcv_raw["C_INSTRUMENTO_SCN"] == "AF.32")
    & (dcv_raw["C_SI_TENEDOR"].isin(["321"]))
    & (dcv_raw["AÑO"] > 2002)
    & (~dcv_raw["VARIABLE"].isin(["Valor Par Indice", "Valor Par MM$"]))
].copy()
mapa_cuenta = {
    "Cuenta Financiera": "Financiera",
    "Cuenta de Revalorización": "Rec Precio",
    "Saldo Final": "Bce Final",
    "Saldo Inicial": "Bce Inicio",
    "Cuenta Volumen": "Rec Volumen",
}
dcv_f["C_CUENTA_MAP"] = dcv_f["C_CUENTA"].map(mapa_cuenta)
dcv_f["SECTOR_NUM"] = pd.to_numeric(dcv_f["C_SI_TENEDOR"], errors="coerce")
af32_d_dcv = (
    dcv_f.groupby(["AÑO", "TRIMESTRE", "C_SI_TENEDOR", "C_SI_EMISOR", "C_CUENTA_MAP", "C_INSTRUMENTO_SCN"], dropna=False)["DATO"]
    .sum()
    .reset_index()
)
af32_d_dcv = af32_d_dcv.rename(columns={"TRIMESTRE": "TRIM", "C_CUENTA_MAP": "C_CUENTA", "C_INSTRUMENTO_SCN": "C_SCN"})
af32_d_dcv["MONEDA"] = "P"
af32_d_dcv["SECTOR"] = pd.to_numeric(af32_d_dcv["C_SI_TENEDOR"], errors="coerce")
af32_d_dcv["C_CAGENTE"] = "332"
af32_d_dcv["C_ENTRADA"] = "D"
af32_d_dcv["N_SCN"] = "Valores distintos de acciones a largo plazo"
af32_d_dcv["FUENTE"] = "PS"
af32_d_dcv["PROC"] = "0015"
af32_d_dcv = af32_d_dcv[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
af32_d_dcv.to_sql("#af32_d_dcv", work_conn, if_exists="replace", index=False)

# JUNTA TABLAS PARA HACER RESTA (PROC APPEND FORCE)
cols_af32 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO #af32_bcos ({cols_af32}) SELECT {cols_af32} FROM #af32_d_dcv"))
_log("APPEND #af32_bcos <- #af32_d_dcv", res.rowcount)


In [ ]:
# AGRUPA DIF DCV-CI
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
sql_af32_dif = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN,
       N_SCN,
       FUENTE, PROC
INTO #af32_dif
FROM #af32_bcos
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, C_CAGENTE, FUENTE, SECTOR, PROC
"""
work_conn.execute(text(sql_af32_dif))

# AJUSTE POR IMPUTACIÓN ANTERIOR EN CRED COM DE BANCOS CON SECTOR 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_emp"))
sql_af32_emp = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       (DATO * -1) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       FUENTE, '0015b' AS PROC
INTO #af32_emp
FROM #af32_dif
"""
work_conn.execute(text(sql_af32_emp))

# ANEXA AJUSTE BONOS LP EN AUX / ANEXA AJUSTE cred com EN AUX
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_dif"))
_log("APPEND BD_CTSI <- #af32_dif", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_emp"))
_log("APPEND BD_CTSI <- #af32_emp", res.rowcount)

for t in ["#af32_bcos", "#af32_d_dcv", "#af32_dif", "#af32_emp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTACIONES EN PATRIMONIO SEPARADO
# 1. IMPUTA BONOS DE LARGO PLAZO EN PASIVO DE PAT SEPARADO USANDO INFO DEL ACTIVO DE CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af32_3324"))
sql_af32_3324 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       3324 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0017' AS PROC
INTO #af32_3324
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CAGENTE IN ('332','3323','3324') AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_3324))

# 2. IMPUTA PTMOS DE LARGO PLAZO EN ACTIVO DE PAT SEPARADO USANDO 2/3 DE LO IMPUTADO EN BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_3324"))
sql_af42_3324 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '51022' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) * 2 / 3 AS DATO,
       'AF.42' AS C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       FUENTE,
       '0018' AS PROC
INTO #af42_3324
FROM #af32_3324
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, FUENTE
"""
work_conn.execute(text(sql_af42_3324))

# 3. IMPUTA CRED COMERCIALES EN ACTIVO DE PAT SEPARADO USANDO 1/3 DE LO IMPUTADO EN BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_3324"))
sql_af7_3324 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '51022' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) / 3 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       FUENTE,
       '0020' AS PROC
INTO #af7_3324
FROM #af32_3324
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, FUENTE
"""
work_conn.execute(text(sql_af7_3324))

cols_3324 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_3324}) SELECT {cols_3324} FROM #af32_3324"))
_log("APPEND BD_CTSI <- #af32_3324", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_3324}) SELECT {cols_3324} FROM #af42_3324"))
_log("APPEND BD_CTSI <- #af42_3324", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_3324}) SELECT {cols_3324} FROM #af7_3324"))
_log("APPEND BD_CTSI <- #af7_3324", res.rowcount)

for t in ["#af32_3324", "#af42_3324", "#af7_3324"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE RECONOCIMIENTO EN PASIVO DEL GOB CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af321_41"))
sql_af321_41 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0021' AS PROC
INTO #af321_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (C_SCN = 'AF.321')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af321_41))

cols_af321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af321}) SELECT {cols_af321} FROM #af321_41"))
_log("APPEND BD_CTSI <- #af321_41", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af321_41"))


## S2_02_Din&Dep

Reclasifica y ajusta partidas de instrumentos financieros (oro monetario, títulos, depósitos) entre pares de sectores institucionales para que activo y pasivo cuadren, respetando el dato de la contraparte designada como fuente de verdad en cada regla y registrando la diferencia como ajuste en la contraparte / Concilia depósitos y ajustes de conciliación (AF.29, AF.5, AF.32) entre sectores contrapartida (resto economía, RM, mineras, banco central, bancos comerciales, gobierno, FP/AFP), reclasifica contragentes puntuales y acumula los ajustes resultantes en la base consolidada

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE / UPDATE / PROC APPEND masivo sobre TABLAS.BD_CTSI (conciliación cruzada de instrumentos financieros AF.1/AF.21/AF.22/AF.29/AF.31/AF.32 entre sectores institucionales) + PROC SQL: series de CREATE TABLE (agregaciones AF.29/AF.5/AF.32 entre sectores contrapartida), PROC SQL UPDATE de reclasificación de contragente y PROC APPEND acumulativos hacia TABLAS.BD_CTSI*

In [ ]:
# ========= S2_02_Din&Dep =========
# COMPRIME TABLA PRINCIPAL
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi_compress"))
work_conn.execute(text("SELECT * INTO #bd_ctsi_compress FROM TABLAS.dbo.BD_CTSI"))
work_conn.execute(text("TRUNCATE TABLE TABLAS.dbo.BD_CTSI"))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #bd_ctsi_compress"))
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi_compress"))
_log("COMPRESS TABLAS.BD_CTSI", res.rowcount)


In [ ]:
# CAMBIA COD DE INSTRUMENTO DESDE AF.22 A AF.29 EN EL PASIVO DEL RESTO DEL MUNDO CON CONTRAENTE HOGARES
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos'
    WHERE SECTOR = 6 AND C_SCN = 'AF.29' AND C_ENTRADA = 'H' AND C_CAGENTE = '511'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF.22->AF.29 CA511)", res.rowcount)


In [ ]:
# CIERRE 2021: EN FONDOS DE INVERSIÓN ACTIVO AF41 CON CA BANCOS, CAMBIA INSTRUMENTO A AF.29
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos'
    WHERE SECTOR = 339011 AND C_SCN = 'AF.41' AND C_ENTRADA = 'D' AND C_CAGENTE = '321'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF.41->AF.29 sector339011)", res.rowcount)


In [ ]:
# CIERRE 2021: EN HODINGS Y CASAS MATRICES CAMBIA CONTRAGENTE DE AF21 DESDE 321 A 31
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '31'
    WHERE SECTOR IN (37, 36907) AND C_SCN = 'AF.21' AND C_ENTRADA = 'D' AND C_CAGENTE = '321'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF.21 CA321->31 holdings)", res.rowcount)


In [ ]:
# ORO MONETARIO ENTRE BCENTRAL-ACTIVO Y RM-PASIVO, MANDA DATO DE BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af1_31_6"))
sql_af1_31_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.1' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0037' AS PROC
INTO #af1_31_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.1' AND T1.FUENTE = 'CI')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.1' AND T1.FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af1_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af1_31_6_v1"))
sql_af1_31_6_v1 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0038' AS PROC
INTO #af1_31_6_v1
FROM #af1_31_6 T1
"""
work_conn.execute(text(sql_af1_31_6_v1))


In [ ]:
# PROC APPEND: append tal cual, re-ejecutar duplica igual que el SAS original
cols_af1_31_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af1_31_6}) SELECT {cols_af1_31_6} FROM #af1_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF1_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af1_31_6}) SELECT {cols_af1_31_6} FROM #af1_31_6_v1"))
_log("APPEND TABLAS.BD_CTSI (AF1_31_6_V1)", res.rowcount)
for t in ["#af1_31_6", "#af1_31_6_v1"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y FONDOS DE INVERSIÓN-ACTIVO. RESPETA DATO DE LOS FONDOS Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN PASIVO DEL RM CON CA 33901 (CIERRE 2021)
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_fi"))
sql_af31_6_fi = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '33901' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d1' AS PROC
INTO #af31_6_fi
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (339011, 3390101, 3390102) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('339011', '33901') AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_fi))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_fi_aj"))
sql_af31_6_fi_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_fi_aj
FROM #af31_6_fi T1
"""
work_conn.execute(text(sql_af31_6_fi_aj))


In [ ]:
cols_af31_6_fi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_6_fi}) SELECT {cols_af31_6_fi} FROM #af31_6_fi"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_FI)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_6_fi}) SELECT {cols_af31_6_fi} FROM #af31_6_fi_aj"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_FI_AJ)", res.rowcount)
for t in ["#af31_6_fi", "#af31_6_fi_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN ACTIVO AF29 DE GOBIERNO DESDE 3 A 321
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321'
    WHERE C_CAGENTE = '3' AND SECTOR = 41 AND C_SCN = 'AF.29' AND C_ENTRADA = 'D'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF29 CA3->321 gob)", res.rowcount)


In [ ]:
# TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL GOBIERNO Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN PASIVO DEL RM CON CA 41
# CIERRE 2021: SE CAMBIA REGLA PARA RESPETAR EL DATO DEL RM, POR ENDE SE AJUSTE EN ACTIVO AF32 DEL GOB CON CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_41"))
sql_af31_6_41 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d1' AS PROC
INTO #af31_6_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (41) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('41') AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_41_aj"))
sql_af31_6_41_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_41_aj
FROM #af31_6_41 T1
"""
work_conn.execute(text(sql_af31_6_41_aj))


In [ ]:
cols_af31_6_41 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_6_41}) SELECT {cols_af31_6_41} FROM #af31_6_41"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_6_41}) SELECT {cols_af31_6_41} FROM #af31_6_41_aj"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_41_AJ)", res.rowcount)
for t in ["#af31_6_41", "#af31_6_41_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y BANCOS-SEGUROS-EMPRESAS-ACTIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN ACTIVO DE LOS SECTORES CON CA RM
# selecciona pasivo del rm con contragentes distintos a gobierno y fondos de inversión
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_sect"))
sql_af31_6_sect = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       (CASE WHEN T1.C_CAGENTE = '53' THEN 51022 ELSE TRY_CAST(T1.C_CAGENTE AS int) END) AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_sect
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CAGENTE NOT IN ('41', '33901') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CAGENTE = '53' THEN 51022 ELSE TRY_CAST(T1.C_CAGENTE AS int) END),
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_sect))


In [ ]:
# selecciona activo af31 de los sectores (excepto gobierno y fondos de inversiones) con resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #af31_sect_6"))
sql_af31_sect_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_sect_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR NOT IN (41, 4, 42, 412, 339011) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_sect_6))


In [ ]:
# DATA step: concatena AF31_6_sect con AF31_SECT_6
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_sect_all"))
work_conn.execute(text("""
    SELECT * INTO #af31_6_sect_all FROM #af31_6_sect
    UNION ALL
    SELECT * FROM #af31_sect_6
"""))


In [ ]:
# calcula delta a imputar en sectores
work_conn.execute(text("DROP TABLE IF EXISTS #af31_sect_6_imp"))
sql_af31_sect_6_imp = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af31_sect_6_imp
FROM #af31_6_sect_all T1
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC, T1.MONEDA
"""
work_conn.execute(text(sql_af31_sect_6_imp))


In [ ]:
# ajusta en activo af32 de los sectores con contragente RM
work_conn.execute(text("DROP TABLE IF EXISTS #af32_sect_6_aj"))
sql_af32_sect_6_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af32_sect_6_aj
FROM #af31_sect_6_imp T1
"""
work_conn.execute(text(sql_af32_sect_6_aj))


In [ ]:
cols_af31_sect_6_imp = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_sect_6_imp}) SELECT {cols_af31_sect_6_imp} FROM #af31_sect_6_imp"))
_log("APPEND TABLAS.BD_CTSI (AF31_SECT_6_IMP)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_sect_6_imp}) SELECT {cols_af31_sect_6_imp} FROM #af32_sect_6_aj"))
_log("APPEND TABLAS.BD_CTSI (AF32_SECT_6_AJ)", res.rowcount)
for t in ["#af31_sect_6_imp", "#af32_sect_6_aj", "#af31_6_sect", "#af31_sect_6", "#af31_6_sect_all"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# TÍTULOS AF32 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL RM Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.29 EN GOB CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_41_2"))
sql_af32_6_41_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d' AS PROC
INTO #af32_6_41_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_6_41_2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_6"))
sql_af29_41_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0051d3' AS PROC
INTO #af29_41_6
FROM #af32_6_41_2 T1
"""
work_conn.execute(text(sql_af29_41_6))


In [ ]:
cols_af32_6_41_2 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_6_41_2}) SELECT {cols_af32_6_41_2} FROM #af32_6_41_2"))
_log("APPEND TABLAS.BD_CTSI (AF32_6_41_2)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_6_41_2}) SELECT {cols_af32_6_41_2} FROM #af29_41_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_6 primer bloque)", res.rowcount)
for t in ["#af32_6_41_2", "#af29_41_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPÓSITOS AF22 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL GOB Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.29 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af22_41_6"))
sql_af22_41_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '41' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0042_04' AS PROC
INTO #af22_41_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22' AND T1.FUENTE = 'CI')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22' AND T1.FUENTE = 'CI')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_41_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_6_b"))
sql_af29_41_6_b = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0042_05' AS PROC
INTO #af29_41_6_b
FROM #af22_41_6 T1
"""
work_conn.execute(text(sql_af29_41_6_b))


In [ ]:
cols_af22_41_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_41_6}) SELECT {cols_af22_41_6} FROM #af22_41_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_41_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_41_6}) SELECT {cols_af22_41_6} FROM #af29_41_6_b"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_6 segundo bloque)", res.rowcount)
for t in ["#af22_41_6", "#af29_41_6_b"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPÓSITOS AF29 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL RM Y LUEGO SE AJUSTA IMPUTACIÓIN CONTRA AF.29 EN GOB CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_41"))
sql_af29_6_41 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051' AS PROC
INTO #af29_6_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_321"))
sql_af29_41_321 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0052' AS PROC
INTO #af29_41_321
FROM #af29_6_41 T1
"""
work_conn.execute(text(sql_af29_41_321))


In [ ]:
cols_af29_6_41 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_41}) SELECT {cols_af29_6_41} FROM #af29_6_41"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_41}) SELECT {cols_af29_6_41} FROM #af29_41_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_321)", res.rowcount)
for t in ["#af29_6_41", "#af29_41_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DINERO AF21 ENTRE RESTO DEL MUNDO-PASIVO Y BCOS/BCO CENTRAL-ACTIVO. RESPETA DATO DEL BCOS Y BCO CENTRAL Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.22 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af21_31_321_6"))
sql_af21_31_321_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0055' AS PROC
INTO #af21_31_321_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (321, 31, 336, 36912) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.21')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR, T1.DATO
"""
work_conn.execute(text(sql_af21_31_321_6))


In [ ]:
# cierre 2021: para no dejar saldo negativo en pasivo del rm con contragentes 336 y 36912 en af22, el ajuste en estos sectores se resta de ca 53
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_321_6"))
sql_af22_31_321_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       (CASE WHEN T1.C_CAGENTE IN ('336', '36912') THEN '53' ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.22' AS C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0056' AS PROC
INTO #af22_31_321_6
FROM #af21_31_321_6 T1
"""
work_conn.execute(text(sql_af22_31_321_6))


In [ ]:
cols_af21_31_321_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_31_321_6}) SELECT {cols_af21_31_321_6} FROM #af21_31_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF21_31_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_31_321_6}) SELECT {cols_af21_31_321_6} FROM #af22_31_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_321_6)", res.rowcount)
for t in ["#af21_31_321_6", "#af22_31_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ANULA AF21 EN EL ACTIVO DE SECTORES 5111 CON BCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af21_5111"))
sql_af21_5111 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0053' AS PROC
INTO #af21_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (5111) AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.21')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af21_5111))


In [ ]:
cols_af21_5111 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_5111}) SELECT {cols_af21_5111} FROM #af21_5111"))
_log("APPEND TABLAS.BD_CTSI (AF21_5111)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af21_5111"))


In [ ]:
# IMPUTA AF21 EN HOGARES COMO LA DIF ENTRE ACTIVO Y PASIVO DEL INSTRUMENTO
work_conn.execute(text("DROP TABLE IF EXISTS #af21_511"))
sql_af21_511 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0057' AS PROC
INTO #af21_511
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_SCN = 'AF.21')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af21_511))


In [ ]:
cols_af21_511 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_511}) SELECT {cols_af21_511} FROM #af21_511"))
_log("APPEND TABLAS.BD_CTSI (AF21_511)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af21_511"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN AF22 DEL GOBIERNO
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321'
    WHERE C_CAGENTE IN ('3', '41') AND SECTOR IN (41, 42) AND C_SCN = 'AF.22'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF22 CA gobierno->321)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y BCOS COMERCIALES-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_321"))
sql_af22_31_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 321 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0062' AS PROC
INTO #af22_31_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 321 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_321"))
sql_af29_31_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0063' AS PROC
INTO #af29_31_321
FROM #af22_31_321 T1
"""
work_conn.execute(text(sql_af29_31_321))


In [ ]:
cols_af22_31_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_321}) SELECT {cols_af22_31_321} FROM #af22_31_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_321}) SELECT {cols_af22_31_321} FROM #af29_31_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_31_321)", res.rowcount)
for t in ["#af22_31_321", "#af29_31_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN AF22 BCO CENTRAL-PASIVO
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '41'
    WHERE SECTOR = 31 AND C_ENTRADA = 'H' AND C_SCN = 'AF.22' AND C_CAGENTE IN ('5101', '53')
"""))
_log("UPDATE TABLAS.BD_CTSI (AF22 bco central CA->41)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF22 CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_41"))
sql_af22_31_41 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 41 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0067' AS PROC
INTO #af22_31_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 41 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_31_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af22_321_41"))
sql_af22_321_41 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0068' AS PROC
INTO #af22_321_41
FROM #af22_31_41 T1
"""
work_conn.execute(text(sql_af22_321_41))


In [ ]:
cols_af22_31_41 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_41}) SELECT {cols_af22_31_41} FROM #af22_31_41"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_41}) SELECT {cols_af22_31_41} FROM #af22_321_41"))
_log("APPEND TABLAS.BD_CTSI (AF22_321_41)", res.rowcount)
for t in ["#af22_31_41", "#af22_321_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y RM-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF71 CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_6"))
sql_af22_31_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0072' AS PROC
INTO #af22_31_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_31_6))


In [ ]:
# El SAS re-aplica sobre WORK.AF22_31_6 el mismo WHERE compuesto que generó esa tabla (fix del verificador)
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0073' AS PROC
INTO #af71_31_6
FROM #af22_31_6 T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.SECTOR, T1.C_CAGENTE, T1.DATO
"""
work_conn.execute(text(sql_af71_31_6))


In [ ]:
cols_af22_31_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_6}) SELECT {cols_af22_31_6} FROM #af22_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_6}) SELECT {cols_af22_31_6} FROM #af71_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF71_31_6)", res.rowcount)
for t in ["#af22_31_6", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCOS COMERCIALES-ACTIVO Y RM-PASIVO. RESPETA DATO DE BCOS Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_321_6"))
sql_af22_321_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '321' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0082' AS PROC
INTO #af22_321_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 321 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_321_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_6"))
sql_af29_321_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0083' AS PROC
INTO #af29_321_6
FROM #af22_321_6 T1
"""
work_conn.execute(text(sql_af29_321_6))


In [ ]:
cols_af22_321_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_321_6}) SELECT {cols_af22_321_6} FROM #af22_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_321_6}) SELECT {cols_af22_321_6} FROM #af29_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_321_6)", res.rowcount)
for t in ["#af22_321_6", "#af29_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE FONDO DE PENSIONES-ACTIVO Y RM-PASIVO. RESPETA DATO DE RM Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_34_6"))
sql_af22_34_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       34 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR IN (34, 341) THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0087' AS PROC
INTO #af22_34_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (341, 34) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '34' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_34_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_34_6"))
sql_af29_34_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       '0088' AS PROC
INTO #af29_34_6
FROM #af22_34_6 T1
"""
work_conn.execute(text(sql_af29_34_6))


In [ ]:
cols_af22_34_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_34_6}) SELECT {cols_af22_34_6} FROM #af22_34_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_34_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_34_6}) SELECT {cols_af22_34_6} FROM #af29_34_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_34_6)", res.rowcount)
for t in ["#af22_34_6", "#af29_34_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF2 Y AF22 EN RESTO DEL MUNDO CON CA 33901/351 PARA CAMBIAR INST A AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_6_33901"))
sql_af22_6_33901 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0089' AS PROC
INTO #af22_6_33901
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('33901/351', '351', '33901') AND T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.22', 'AF.2'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_6_33901))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_33901"))
sql_af29_6_33901 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0089' AS PROC
INTO #af29_6_33901
FROM #af22_6_33901 T1
"""
work_conn.execute(text(sql_af29_6_33901))


In [ ]:
cols_af22_6_33901 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_33901}) SELECT {cols_af22_6_33901} FROM #af22_6_33901"))
_log("APPEND TABLAS.BD_CTSI (AF22_6_33901)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_33901}) SELECT {cols_af22_6_33901} FROM #af29_6_33901"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_33901)", res.rowcount)
for t in ["#af22_6_33901", "#af29_6_33901"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF22 EN 5101 (SOC PUB)-ACTIVO CON RM-PASIVO. RESPETA DATO DE RM Y AJUSTA CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5101_6"))
sql_af22_5101_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       5101 AS SECTOR,
       51 AS C_SI_publ,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0091' AS PROC
INTO #af22_5101_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '5101' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_5101_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5101_321"))
sql_af22_5101_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_SI_publ,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0090' AS PROC
INTO #af22_5101_321
FROM #af22_5101_6 T1
"""
work_conn.execute(text(sql_af22_5101_321))


In [ ]:
cols_af22_5101_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_SI_publ, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI (MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC) SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC FROM #af22_5101_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_5101_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI (MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC) SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC FROM #af22_5101_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_5101_321)", res.rowcount)
for t in ["#af22_5101_6", "#af22_5101_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN EMPRESAS (51021) QUE REPORTAN EN DÓLARES
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '6', FUENTE = 'PS', PROC = '0092'
    WHERE MONEDA = 'D' AND SECTOR = 51021 AND C_CAGENTE = '321'
"""))
_log("UPDATE TABLAS.BD_CTSI (51021 CA->6 USD)", res.rowcount)


In [ ]:
# ACTUALIZA INSTRUMENTO EN EMPRESAS MINERAS (51022 MONEDA DOLARES) DESDE AF22 A AF29
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos', PROC = '0092b'
    WHERE MONEDA = 'D' AND SECTOR = 51022 AND C_CAGENTE = '6' AND C_SCN = 'AF.22' AND C_ENTRADA = 'D'
"""))
_log("UPDATE TABLAS.BD_CTSI (51022 AF22->AF29 USD)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE 51021-ACTIVO Y RM-PASIVO. RESPETA DATO DE 51021 Y AJUSTA IMPUTACIÓN CONTRA AF29 CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af22_53_6"))
sql_af22_53_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '53' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0096' AS PROC
INTO #af22_53_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 51021 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '53' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_53_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_53_6"))
sql_af29_53_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0097' AS PROC
INTO #af29_53_6
FROM #af22_53_6 T1
"""
work_conn.execute(text(sql_af29_53_6))


In [ ]:
cols_af22_53_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_53_6}) SELECT {cols_af22_53_6} FROM #af22_53_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_53_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_53_6}) SELECT {cols_af22_53_6} FROM #af29_53_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_53_6)", res.rowcount)
for t in ["#af22_53_6", "#af29_53_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF22 DE SECTOR 42-ACTIVO CON 321-PASIVO. RESPETA DATO DEL SECTOR 321, AJUSTA CONTRA AF71
work_conn.execute(text("DROP TABLE IF EXISTS #af22_42_321"))
sql_af22_42_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       42 AS SECTOR,
       '321' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio' THEN 'Rec Precio Reaj' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 42 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0113' AS PROC
INTO #af22_42_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 42 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.C_CAGENTE = '42' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio' THEN 'Rec Precio Reaj' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_42_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_42_321"))
sql_af71_42_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0114' AS PROC
INTO #af71_42_321
FROM #af22_42_321 T1
"""
work_conn.execute(text(sql_af71_42_321))


In [ ]:
cols_af22_42_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_42_321}) SELECT {cols_af22_42_321} FROM #af22_42_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_42_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_42_321}) SELECT {cols_af22_42_321} FROM #af71_42_321"))
_log("APPEND TABLAS.BD_CTSI (AF71_42_321)", res.rowcount)
for t in ["#af22_42_321", "#af71_42_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA EN BCO COMERCIALES EL INST AF.1 A AF.22 DEL ACTIVO Y CONTRAGENTE DESDE 53 A 321
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321', C_SCN = 'AF.22', N_SCN = 'Depósitos', PROC = '0114b', FUENTE = 'PS'
    WHERE C_CAGENTE = '53' AND SECTOR = 321 AND C_SCN = 'AF.1' AND C_ENTRADA = 'D'
"""))
_log("UPDATE TABLAS.BD_CTSI (AF.1->AF.22 bco comerciales)", res.rowcount)


In [ ]:
# ANULA AF22 DEL ACTIVO DE SECTOR 5111 CON CA 321. POR CIERRE 2020 ANULA TMB AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5111_321"))
sql_af22_5111_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0115' AS PROC
INTO #af22_5111_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.22', 'AF.29'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_5111_321))


In [ ]:
cols_af22_5111_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_5111_321}) SELECT {cols_af22_5111_321} FROM #af22_5111_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_5111_321)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5111_321"))


In [ ]:
# CIERRE 2020. ANULA AF31,AF32,AF34, AF521, AF522 DEL ACTIVO Y PASIVO DE SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af_5111_321"))
sql_af_5111_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0115' AS PROC
INTO #af_5111_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN IN ('AF.31', 'AF.32', 'AF.34', 'AF.521', 'AF.522'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af_5111_321))


In [ ]:
cols_af_5111_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af_5111_321}) SELECT {cols_af_5111_321} FROM #af_5111_321"))
_log("APPEND TABLAS.BD_CTSI (AF_5111_321)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af_5111_321"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_6_31"))
sql_af22_6_31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0115d' AS PROC
INTO #af22_6_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_6_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_31_a"))
sql_af29_6_31_a = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       '0115e' AS PROC
INTO #af29_6_31_a
FROM #af22_6_31 T1
"""
work_conn.execute(text(sql_af29_6_31_a))


In [ ]:
cols_af22_6_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_31}) SELECT {cols_af22_6_31} FROM #af22_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF22_6_31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_31}) SELECT {cols_af22_6_31} FROM #af29_6_31_a"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_31 primer bloque)", res.rowcount)
for t in ["#af22_6_31", "#af29_6_31_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA CERTIFICADOS DE DEPÓSITOS ENTRE BCOS COMERCIALES Y RESTO DEL MUNDO. LOS RECLASIFICA COMO TÍTULOS DE LARGO PLAZO
# Fuente es un archivo .sas7bdat externo (ruta absoluta hardcodeada) -> se relativiza per M-001
cd_path = Path("insumos") / "t_aj_cd_321_6_af32.sas7bdat"
cd_321_6_raw, _meta = pyreadstat.read_sas7bdat(str(cd_path))
cd_321_6_raw.columns = [c.upper() for c in cd_321_6_raw.columns]
cd_321_6 = pd.DataFrame({
    "MONEDA": "P",
    "AÑO": cd_321_6_raw["AÑO"],
    "TRIM": cd_321_6_raw["TRIM"],
    "SECTOR": cd_321_6_raw["C_SI_EMISOR"],
    "C_CAGENTE": cd_321_6_raw["C_SI_TENEDOR"].astype(str).str.strip(),
    "C_CUENTA": cd_321_6_raw["C_CUENTA"],
    "C_ENTRADA": cd_321_6_raw["ENTRADA"],
    "DATO": cd_321_6_raw["DATO"] * -1,
    "C_SCN": "AF.29",
    "N_SCN": "Otros depósitos",
    "FUENTE": "DI",
    "PROC": "0119a",
})
# CIERRE 2021: SE INCORPORA PARA NO ALTERAR REAJUSTABILIDAD DE LA CUENTA YG
cd_321_6.loc[cd_321_6["C_CUENTA"] == "Rec Precio Reaj", "C_CUENTA"] = "Rec Precio"


In [ ]:
# CIERRE 2021: SE CAMBIA IMPUTACIÓN A TÍTULOS DE CORTO PLAZO
cd_321_6_2 = cd_321_6.copy()
cd_321_6_2["DATO"] = cd_321_6["DATO"] * -1
cd_321_6_2["C_SCN"] = "AF.31"
cd_321_6_2["N_SCN"] = "Valores distintos de acciones a corto plazo"
cd_321_6_2["PROC"] = "0119b"


In [ ]:
# el WORK nació en pandas (archivo externo): se sube a la sesión para el append server-side
cd_321_6.to_sql("#cd_321_6", work_conn, if_exists="replace", index=False)
cd_321_6_2.to_sql("#cd_321_6_2", work_conn, if_exists="replace", index=False)
cols_cd_321_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_cd_321_6}) SELECT {cols_cd_321_6} FROM #cd_321_6"))
_log("APPEND TABLAS.BD_CTSI (CD_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_cd_321_6}) SELECT {cols_cd_321_6} FROM #cd_321_6_2"))
_log("APPEND TABLAS.BD_CTSI (CD_321_6_2)", res.rowcount)
for t in ["#cd_321_6", "#cd_321_6_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 6 Y AJUSTA IMPUTACIÓN CONTRA AF71
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_31"))
sql_af29_6_31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       31 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 31 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0122' AS PROC
INTO #af29_6_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_6_31"))
sql_af71_6_31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0123' AS PROC
INTO #af71_6_31
FROM #af29_6_31 T1
"""
work_conn.execute(text(sql_af71_6_31))


In [ ]:
cols_af29_6_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_31}) SELECT {cols_af29_6_31} FROM #af29_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_31 segundo bloque)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_31}) SELECT {cols_af29_6_31} FROM #af71_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF71_6_31)", res.rowcount)
for t in ["#af29_6_31", "#af71_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE BCO COMERCIAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 6 Y AJUSTA IMPUTACIÓN CONTRA AF32
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_321"))
sql_af29_6_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 321 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0127' AS PROC
INTO #af29_6_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 321 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_321"))
sql_af32_6_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0128' AS PROC
INTO #af32_6_321
FROM #af29_6_321 T1
"""
work_conn.execute(text(sql_af32_6_321))


In [ ]:
cols_af29_6_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_321}) SELECT {cols_af29_6_321} FROM #af29_6_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_321}) SELECT {cols_af29_6_321} FROM #af32_6_321"))
_log("APPEND TABLAS.BD_CTSI (AF32_6_321)", res.rowcount)
for t in ["#af29_6_321", "#af32_6_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# OTROS DEPÓSITOS AF29 ENTRE FFMM Y FONDOS DE INVERSIÓN (3390102+339011+3390102)-ACTIVO Y RM-PASIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN EN FFMM MM CON CONTRAGENTE 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_ffmm_6"))
sql_af29_ffmm_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       3390101 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0129' AS PROC
INTO #af29_ffmm_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '33901' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR IN (3390102, 339011, 3390101) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_ffmm_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_ffmm_321"))
sql_af29_ffmm_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0130' AS PROC
INTO #af29_ffmm_321
FROM #af29_ffmm_6 T1
"""
work_conn.execute(text(sql_af29_ffmm_321))


In [ ]:
cols_af29_ffmm_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_ffmm_6}) SELECT {cols_af29_ffmm_6} FROM #af29_ffmm_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_FFMM_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_ffmm_6}) SELECT {cols_af29_ffmm_6} FROM #af29_ffmm_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_FFMM_321)", res.rowcount)
for t in ["#af29_ffmm_6", "#af29_ffmm_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# OTROS DEPÓSITOS AF29 ENTRE CIAS DE SEGURO DE VIDA-ACTIVO Y RM-PASIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN EN CIAS DE SEGURO EN BONOS DE LP CON CONTRAGENTE 6
work_conn.execute(text("DROP TABLE IF EXISTS #af29_351_6"))
sql_af29_351_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0129a' AS PROC
INTO #af29_351_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '351' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 351 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_351_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_351_321"))
sql_af29_351_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0130b' AS PROC
INTO #af29_351_321
FROM #af29_351_6 T1
"""
work_conn.execute(text(sql_af29_351_321))


In [ ]:
cols_af29_351_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_351_6}) SELECT {cols_af29_351_6} FROM #af29_351_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_351_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_351_6}) SELECT {cols_af29_351_6} FROM #af29_351_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_351_321)", res.rowcount)
for t in ["#af29_351_6", "#af29_351_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_resto_6"))
# DEPOSITOS AF29 ENTRE RESTO DE LA ECONOMÍA-ACTIVO Y RM-PASIVO.EN RM RESTA LO QUE DEBE QUEDAR CON SECTOR 51021 Y EL RESTO LO ASIGNA A HOGARES(10%) Y OTRAS SOCIEDADES(90%)
sql_af29_resto_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       36912 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0129c' AS PROC
INTO #af29_resto_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 6 AND T1.C_CAGENTE = '33901/351' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
   OR ((SUBSTRING(LEFT(CAST(t1.SECTOR AS varchar(8)), 8), 1, 2) IN ('33', '36', '37') OR T1.SECTOR IN (411))
       AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29' AND T1.C_CAGENTE = '6'
       AND T1.SECTOR NOT IN (3390101, 3390102, 33901, 339011, 339))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN
"""
work_conn.execute(text(sql_af29_resto_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_resto_321"))
sql_af29_resto_321 = """
SELECT t1.MONEDA, t1.[AÑO], T1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       t1.FUENTE,
       '0130c' AS PROC
INTO #af29_resto_321
FROM #af29_resto_6 t1
"""
work_conn.execute(text(sql_af29_resto_321))


In [ ]:
# APPEND server-side: FORCE alinea por nombre, columnas explícitas
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_resto_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_RESTO_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_resto_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_RESTO_321)", res.rowcount)


In [ ]:
for t in ["#af29_resto_6", "#af29_resto_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_511"))
# DEPOSITOS AF29 ENTRE RESTO DE LA ECONOMÍA-ACTIVO Y RM-PASIVO.EN RM RESTA LO QUE DEBE QUEDAR CON SECTOR 51021 Y EL RESTO LO ASIGNA A HOGARES(10%) Y OTRAS SOCIEDADES(90%)
# CIERRE 2021: CAMBIA REC PRECIO REAJ A REC PRECIO
sql_af29_6_511 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 51021 THEN T1.DATO * -1 ELSE T1.DATO END) * 0.1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0132' AS PROC
INTO #af29_6_511
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 51021 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (t1.SECTOR = 6 AND T1.C_CAGENTE = '53' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY t1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END,
         t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_511))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_51022"))
# ESTIMADO MINERAS
sql_af29_6_51022 = """
SELECT 'D' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 51021 THEN T1.DATO * -1 ELSE T1.DATO END) * 0.9 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0133' AS PROC
INTO #af29_6_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 51021 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (t1.SECTOR = 6 AND T1.C_CAGENTE = '53' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY t1.[AÑO], T1.TRIM,
         t1.C_CUENTA,
         t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_51022))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_mineras"))
# DATO EFECTIVO DE MINERAS PARA COMPARAR CON ESTIMACIÓN ANTERIOR
sql_af29_mineras = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0133' AS PROC
INTO #af29_mineras
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 51022 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29' AND T1.MONEDA = 'D')
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_mineras))


In [ ]:
# JUNTA AMBAS TABLAS DE MINERAS PARA CALCULAR DELTA A IMPUTAR EN AF29 (append server-side dentro de la sesión)
cols_af29_6_51022 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO #af29_6_51022 ({cols_af29_6_51022}) SELECT {cols_af29_6_51022} FROM #af29_mineras"))
_log("APPEND #af29_6_51022 (AF29_MINERAS)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #delta_mineras"))
# CALCULA DELTA MINERAS A IMPUTAR EN AF29 CON CA6
sql_delta_mineras = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       t1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #delta_mineras
FROM #af29_6_51022 t1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_delta_mineras))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #delta_321"))
# AJUSTE EN AF29 DE SECTOR MINERAS CON CA 321
sql_delta_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       t1.N_SCN,
       T1.FUENTE,
       '0133b' AS PROC
INTO #delta_321
FROM #delta_mineras t1
"""
work_conn.execute(text(sql_delta_321))


In [ ]:
# APPEND server-side hacia TABLAS.dbo.BD_CTSI
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_6_511"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_6_511)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #delta_mineras"))
_log("APPEND TABLAS.dbo.BD_CTSI (DELTA_MINERAS)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #delta_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (DELTA_321)", res.rowcount)


In [ ]:
for t in ["#af29_6_51022", "#af29_6_511", "#delta_321", "#delta_mineras", "#af29_mineras"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_351_6"))
# AJUSTA AF5 DEL RM CON CA 351 CON AF5+AF521 REPORTADO POR EL SECTOR 351 CON CA 6. MANDA EL DATO DEL RM, DIFERENCIA LA IMPUTA EN AF.5 DEL SECTOR 351 CON CA 6
# CONTRA AJUSTE LO HACE EN ACTIVO AF32 DEL SECTOR 351 CON CONTRAGENTE 6
sql_af5_351_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'AF.5' AS C_SCN,
       'Acciones y otras participaciones de capital' AS N_SCN,
       'PS' AS FUENTE,
       '0129d' AS PROC
INTO #af5_351_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 6 AND T1.C_CAGENTE = '351' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5')
   OR (t1.SECTOR = 351 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.5', 'AF.522'))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_af5_351_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_6"))
sql_af32_351_6 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       t1.FUENTE,
       '0130d' AS PROC
INTO #af32_351_6
FROM #af5_351_6 t1
"""
work_conn.execute(text(sql_af32_351_6))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_351_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF5_351_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_351_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF32_351_6)", res.rowcount)
for t in ["#af32_351_6", "#af5_351_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA EN FP EL INST AF.29 DEL ACTIVO CONTRAGENTE DESDE 6 A 321
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321' WHERE C_CAGENTE = '6' AND SECTOR = 34 AND C_SCN = 'AF.29'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (FP AF.29 6->321)", res.rowcount)


In [ ]:
# ACTUALIZA EN CCAF EL INST AF.29 DEL ACTIVO CONTRAGENTE DESDE 9 A 511
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_CAGENTE = '9' AND SECTOR = 411 AND C_SCN = 'AF.29'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (CCAF AF.29 9->511)", res.rowcount)


In [ ]:
# ACTUALIZA EN COOP EL INST AF.29 DEL PASIVO CONTRAGENTE DESDE 321 A 511
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '511' WHERE C_CAGENTE = '321' AND SECTOR = 322 AND C_SCN = 'AF.29' AND C_ENTRADA = 'H'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (COOP AF.29 321->511)", res.rowcount)


In [ ]:
# ACTUALIZA EN GOB GRAL EL INST AF.29 CONTRAGENTE DESDE 3 A 321
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321', PROC = '0137' WHERE C_CAGENTE = '3' AND SECTOR = 41 AND C_SCN = 'AF.29'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (GOB GRAL AF.29 3->321)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_321"))
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y BCOS COM-ACTIVO.RESPETA DATO DE SECTOR 31 Y AJUSTA CONTRA AF.29 CONTRAGENTE 321
sql_af29_31_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 321 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0146' AS PROC
INTO #af29_31_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 321 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (t1.SECTOR = 31 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_321"))
sql_af29_321_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0147' AS PROC
INTO #af29_321_321
FROM #af29_31_321 t1
"""
work_conn.execute(text(sql_af29_321_321))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_31_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_321_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_321_321)", res.rowcount)
for t in ["#af29_31_321", "#af29_321_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_41"))
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y GOB-ACTIVO.RESPETA DATO DE SECTOR 31 Y AJUSTA CONTRA AF.29 CONTRAGENTE 321
sql_af29_31_41 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0150' AS PROC
INTO #af29_31_41
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 31 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_31_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_41"))
sql_af29_321_41 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0151' AS PROC
INTO #af29_321_41
FROM #af29_31_41 t1
"""
work_conn.execute(text(sql_af29_321_41))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_41"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_31_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_321_41"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_321_41)", res.rowcount)
for t in ["#af29_31_41", "#af29_321_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_6"))
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y RM-ACTIVO.RESPETA DATO DE SECTOR 31 Y AJUSTA CONTRA AF.71
sql_af29_31_6 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0155' AS PROC
INTO #af29_31_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (t1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0156' AS PROC
INTO #af71_31_6
FROM #af29_31_6 t1
"""
work_conn.execute(text(sql_af71_31_6))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_31_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF71_31_6)", res.rowcount)
for t in ["#af29_31_6", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_34_361"))
# DEPOSITOS AF29 ENTRE FP-PASIVO Y AFP-ACTIVO.RESPETA DATO DE SECTOR 34 Y AJUSTA CONTRA AF.29 CONTRAGENTE 321
sql_af29_34_361 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       361 AS SECTOR,
       '34' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 361 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0156i' AS PROC
INTO #af29_34_361
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR = 361 AND T1.C_CAGENTE = '34' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.612')
   OR (t1.SECTOR = 34 AND T1.C_CAGENTE = '361' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.612')
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_34_361))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_361_321"))
sql_af29_361_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       '0156j' AS PROC
INTO #af29_361_321
FROM #af29_34_361 t1
"""
work_conn.execute(text(sql_af29_361_321))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_34_361"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_34_361)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_361_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF29_361_321)", res.rowcount)
for t in ["#af29_34_361", "#af29_361_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_03_Reservas&Capital

Reasigna reservas de fondos de pensiones (AF.62) del sector 352 a los sectores 321/51022/41/511 con ratios fijos, corrige contragente 412, e imputa valor de mercado y valor libro de inversiones AF.5 entre holdings, casas matrices y empresas supervisadas usando tablas externas de ratios

*confianza: low · verificador: revise · SAS: PROC SQL secuencial con UPDATE + CREATE TABLE agregadas + PROC APPEND FORCE sobre TABLAS.BD_CTSI, con joins externos a archivos .sas7bdat de ratios*

In [ ]:
# ========= S2_03_Reservas&Capital =========
# COMPRIME TABLA PRINCIPAL (session_sql: se materializa BD_CTSI como #tmp para operar en la sesión)
# El resto del nodo opera secuencialmente SIEMPRE releyendo TABLAS.BD_CTSI vía #bd_ctsi tras cada APPEND,
# igual que el SAS que relee la tabla base en cada FROM TABLAS.BD_CTSI.
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN RESERVA DE FONDO DE PENSIONES DEL SECTOR 412 DE CA 9 A 511
res = work_conn.execute(
    text("UPDATE #bd_ctsi SET C_CAGENTE=:cagente, PROC=:proc WHERE C_CAGENTE=:cagente_old AND SECTOR=:sector AND C_SCN=:c_scn"),
    {"cagente": "511", "proc": "0160", "cagente_old": "9", "sector": 412, "c_scn": "AF.612"},
)
_log("UPDATE #bd_ctsi C_CAGENTE 511", res.rowcount)


In [ ]:
# Vuelca el update a la tabla base para que el resto del nodo (que re-lee TABLAS.BD_CTSI en cada bloque) vea el cambio
work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:cagente, PROC=:proc WHERE C_CAGENTE=:cagente_old AND SECTOR=:sector AND C_SCN=:c_scn"),
    {"cagente": "511", "proc": "0160", "cagente_old": "9", "sector": 412, "c_scn": "AF.612"})


In [ ]:
# AF62 DEL PASIVO DEL SECTOR 352 LO ASIGNA AL ACTIVO DE LOS SECTORES 321(20%),51022(45%),41(2%) Y 511(33%)
# ratios hardcodeados en el SAS original (0.2, 0.45, 0.02, 0.33) sin documentación de origen (ver nota de analista)
repartos_af62_352 = [
    (321, 0.2, "0162"),
    (51022, 0.45, "0163"),
    (41, 0.02, "0164"),
    (511, 0.33, "0165"),
]
tablas_af62_352 = {}
for sector_dest, ratio, cod_proc in repartos_af62_352:
    sql_af62 = """
    SELECT MONEDA, [AÑO], TRIM,
           :sector_dest AS SECTOR,
           '352' AS C_CAGENTE,
           C_CUENTA,
           'D' AS C_ENTRADA,
           SUM(DATO * :ratio) AS DATO,
           C_SCN, N_SCN,
           'PS' AS FUENTE,
           :cod_proc AS PROC
    FROM #bd_ctsi
    WHERE C_SCN = 'AF.62'
    GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
    """
    tablas_af62_352[sector_dest] = pd.read_sql(
        text(sql_af62), work_conn, params={"sector_dest": sector_dest, "ratio": ratio, "cod_proc": cod_proc}
    )
af62_352_321 = tablas_af62_352[321]
af62_352_51022 = tablas_af62_352[51022]
af62_352_41 = tablas_af62_352[41]
af62_352_511 = tablas_af62_352[511]
_log("af62_352_321", af62_352_321)


In [ ]:
# APPEND server-side de los 4 repartos a TABLAS.BD_CTSI (PROC APPEND FORCE): acumula, re-ejecutar duplica igual que el SAS original
for sector_dest, df_rep in tablas_af62_352.items():
    df_rep.to_sql("#tmp_af62_352", work_conn, if_exists="replace", index=False)
    cols_af62 = ", ".join(df_rep.columns)
    res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af62}) SELECT {cols_af62} FROM #tmp_af62_352"))
    _log(f"APPEND TABLAS.BD_CTSI desde af62_352_{sector_dest}", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #tmp_af62_352"))


In [ ]:
# refresca #bd_ctsi para reflejar los APPEND anteriores antes de operar el siguiente bloque
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# AF62 CONTRA ACTIVO DE SECTORES 41 Y 321 EN INST AF71 CON CA 352 DEBIDO A IMPUTACIONES ANTERIORES
sql_af62_41_321 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0166' AS PROC
FROM #bd_ctsi
WHERE C_SCN = 'AF.62' AND SECTOR IN (41, 321) AND C_ENTRADA = 'D' AND FUENTE = 'PS'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_ENTRADA, SECTOR, C_CAGENTE
"""
af62_41_321 = pd.read_sql(text(sql_af62_41_321), work_conn)
_log("af62_41_321", af62_41_321)


In [ ]:
af62_41_321.to_sql("#tmp_af62_41_321", work_conn, if_exists="replace", index=False)
cols_af62_41_321 = ", ".join(af62_41_321.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af62_41_321}) SELECT {cols_af62_41_321} FROM #tmp_af62_41_321"))
_log("APPEND TABLAS.BD_CTSI desde af62_41_321", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #tmp_af62_41_321"))


In [ ]:
# refresca #bd_ctsi tras el append de af62_41_321
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON BANCOS
# tabla externa de ratios: no versionada en la base -- se declara el hueco explícitamente (ver nota de analista)
raise NotImplementedError(
    "ratio_inv_hc_c.sas7bdat (/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/36907_37_HOLDING/) "
    "no está migrado a una tabla del catálogo de conexiones ni a un archivo del workspace: "
    "sin esa fuente de ratios no se puede calcular VM_AF5_HC_D"
)


In [ ]:
# REC PRECIO POR AJUSTAR A MERCADO (depende de VM_AF5_HC_D, bloqueado arriba)
raise NotImplementedError(
    "VM_AF5_HC_RP depende de WORK.VM_AF5_HC_D, que no se pudo calcular por falta de ratio_inv_hc_c.sas7bdat"
)


In [ ]:
# APPEND de VM_AF5_HC_D y VM_AF5_HC_RP a TABLAS.BD_CTSI: bloqueado, ambas fuentes no existen
raise NotImplementedError(
    "APPEND de VM_AF5_HC_D/VM_AF5_HC_RP a TABLAS.BD_CTSI bloqueado: los DataFrames no se pudieron generar"
)


In [ ]:
# refresca #bd_ctsi (el SAS re-lee la base tras el DROP de VM_AF5_HC_D/RP, aunque el append previo no corrió)
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# IMPUTA INVERSIONES EN AF5 DE HOLDINGS Y CASAS MATRICES ENTRE EL SECTOR USANDO INFO DEL PATRIMONIO A VALOR LIBRO
# PATRIMONIO VM DE 37 Y 36907 CON CA 37 Y 36907
sql_vm_af5_hc_p = """
SELECT 'P' AS MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN,
       SUM(DATO) AS DATO
FROM #bd_ctsi
WHERE C_SCN = 'AF.5' AND SECTOR IN (37, 36907) AND C_CAGENTE IN ('37', '36907') AND C_ENTRADA = 'H'
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
vm_af5_hc_p = pd.read_sql(text(sql_vm_af5_hc_p), work_conn)
_log("vm_af5_hc_p", vm_af5_hc_p)


In [ ]:
# lleva vm anterior a valor libro en balances -- tabla externa de ratios no versionada
raise NotImplementedError(
    "ratio_pat_hc_c.sas7bdat (/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/36907_37_HOLDING/) "
    "no está migrado a una tabla del catálogo ni a un archivo del workspace: "
    "sin esa fuente de ratios no se puede calcular VL_AF5_HC_P"
)


In [ ]:
# RESTA EFECTO VALOR LIBRO DE RP -- depende de VL_AF5_HC_P, bloqueado arriba
raise NotImplementedError(
    "RP_AF5_HC_P depende de WORK.VL_AF5_HC_P, que no se pudo calcular por falta de ratio_pat_hc_c.sas7bdat"
)


In [ ]:
# JUNTA BASES PARA CALCULAR DIFERENCIAL A VALOR LIBRO A IMPUTAR EN ACTIVO DE 37 Y 36907 -- bloqueado
raise NotImplementedError(
    "WORK.VM_AF5_HC_P (concat con VL_AF5_HC_P y RP_AF5_HC_P) bloqueado por la misma dependencia faltante"
)


In [ ]:
# AGRUPA TABLA ANTERIOR CON BALANCES A VALOR LIBRO
# el SAS usa INPUT(C_CAGENTE, BEST5.) para SECTOR y LEFT(PUT(SECTOR, CHAR5.)) para C_CAGENTE:
# swap numérico/texto con formato fijo de 5 caracteres justificado a la izquierda -- bloqueado por dependencia previa
raise NotImplementedError(
    "WORK.AG_AF5_HC_P bloqueado: depende de WORK.VM_AF5_HC_P (no calculable, ver bloque anterior)"
)


In [ ]:
# INV AF5 DE 37 Y 36907 CON CA 37 Y 36907
sql_inv_af5_hc_hc = """
SELECT 'P' AS MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN,
       SUM(DATO) * -1 AS DATO
FROM #bd_ctsi
WHERE C_SCN = 'AF.5' AND SECTOR IN (37, 36907) AND C_CAGENTE IN ('37', '36907') AND C_ENTRADA = 'D'
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
inv_af5_hc_hc = pd.read_sql(text(sql_inv_af5_hc_hc), work_conn)
_log("inv_af5_hc_hc", inv_af5_hc_hc)


In [ ]:
# CALCULA DELTA DE INVERSIONES A IMPUTAR EN 37 Y 36907 -- depende de AG_AF5_HC_P, bloqueado
raise NotImplementedError(
    "WORK.AG_AF5_HC_P (concat con INV_AF5_HC_HC) bloqueado: AG_AF5_HC_P no se pudo calcular"
)


In [ ]:
# IMP_AF5_HC_HC -- depende del bloque anterior
raise NotImplementedError("WORK.IMP_AF5_HC_HC bloqueado: depende de WORK.AG_AF5_HC_P, no calculable")


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN 37 Y 36907 CON CA 2 -- depende de IMP_AF5_HC_HC
raise NotImplementedError("WORK.IMP_AF5_HC_2 bloqueado: depende de WORK.IMP_AF5_HC_HC, no calculable")


In [ ]:
# APPEND de IMP_AF5_HC_HC e IMP_AF5_HC_2 a TABLAS.BD_CTSI -- bloqueado, ambas fuentes no existen
raise NotImplementedError(
    "APPEND de IMP_AF5_HC_HC/IMP_AF5_HC_2 a TABLAS.BD_CTSI bloqueado: los DataFrames no se pudieron generar"
)


In [ ]:
# refresca #bd_ctsi (el SAS re-lee la base tras el DROP de las tablas de holdings, aunque el append previo no corrió)
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON HOLDINGS Y CASAS MATRICES
# join usa T1.SECTOR=INPUT(T2.C_CAGENTE,BEST5.) y T1.C_CAGENTE=LEFT(PUT(T2.SECTOR,CHAR6.)) -- swap numérico/texto
# con formato fijo CHAR6 de SAS; bloqueado por la misma tabla externa de ratios no versionada
raise NotImplementedError(
    "ratio_pat_hc_c.sas7bdat no está migrado a una tabla del catálogo ni a un archivo del workspace: "
    "sin esa fuente no se puede calcular VM_AF5_HC_INTRA (el join con formato CHAR6/BEST5 de SAS tampoco "
    "se puede replicar sin conocer el contenido real de la tabla de ratios)"
)


In [ ]:
# REC PRECIO POR AJUSTAR A MERCADO -- depende de VM_AF5_HC_INTRA, bloqueado
raise NotImplementedError("WORK.RP_AF5_HC_INTRA bloqueado: depende de WORK.VM_AF5_HC_INTRA, no calculable")


In [ ]:
# APPEND de VM_AF5_HC_INTRA y RP_AF5_HC_INTRA a TABLAS.BD_CTSI -- bloqueado
raise NotImplementedError(
    "APPEND de VM_AF5_HC_INTRA/RP_AF5_HC_INTRA a TABLAS.BD_CTSI bloqueado: los DataFrames no se pudieron generar"
)


In [ ]:
# refresca #bd_ctsi antes del bloque de empresas supervisadas
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# IMPUTA INVERSIONES EN AF5 DE HOLDINGS/CASAS MATRICES CON EMPRESAS SUPERVISADAS -- PATRIMONIO VM DE 51021/5101 CON CA 37 Y 36907
sql_vm_af5_hc_emp = """
SELECT 'P' AS MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN,
       SUM(DATO) AS DATO
FROM #bd_ctsi
WHERE C_SCN = 'AF.5' AND SECTOR IN (51021, 5101) AND C_CAGENTE IN ('37', '36907') AND C_ENTRADA = 'H'
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
vm_af5_hc_emp = pd.read_sql(text(sql_vm_af5_hc_emp), work_conn)
_log("vm_af5_hc_emp", vm_af5_hc_emp)


In [ ]:
# lleva vm anterior a valor libro en balances -- tabla externa de ratios AF5 emp/holding no versionada
raise NotImplementedError(
    "ratio_AF5_emp_hold_n.sas7bdat (/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/S11_EMPRESAS/) "
    "no está migrado a una tabla del catálogo ni a un archivo del workspace: "
    "sin esa fuente de ratios no se puede calcular VL_AF5_HC_EMP"
)


In [ ]:
# RESTA EFECTO VALOR LIBRO DE RP -- depende de VL_AF5_HC_EMP, bloqueado
raise NotImplementedError("WORK.RP_AF5_HC_EMP bloqueado: depende de WORK.VL_AF5_HC_EMP, no calculable")


In [ ]:
# JUNTA BASES (VM_AF5_HC_EMP + VL_AF5_HC_EMP + RP_AF5_HC_EMP) -- bloqueado
raise NotImplementedError(
    "WORK.VM_AF5_HC_EMP (concat con VL_AF5_HC_EMP y RP_AF5_HC_EMP) bloqueado por dependencia faltante"
)


In [ ]:
# AGRUPA TABLA ANTERIOR CON BALANCES A VALOR LIBRO -- bloqueado
raise NotImplementedError("WORK.AG_AF5_HC_EMP bloqueado: depende de WORK.VM_AF5_HC_EMP, no calculable")


In [ ]:
# INV AF5 DE 37 Y 36907 CON CA 51021/5101
sql_inv_af5_hc_emp = """
SELECT 'P' AS MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN,
       SUM(DATO) * -1 AS DATO
FROM #bd_ctsi
WHERE C_SCN = 'AF.5' AND SECTOR IN (37, 36907) AND C_CAGENTE IN ('51021', '5101') AND C_ENTRADA = 'D'
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
inv_af5_hc_emp = pd.read_sql(text(sql_inv_af5_hc_emp), work_conn)
_log("inv_af5_hc_emp", inv_af5_hc_emp)


In [ ]:
# CALCULA DELTA DE INVERSIONES A IMPUTAR -- depende de AG_AF5_HC_EMP, bloqueado
raise NotImplementedError(
    "WORK.AG_AF5_HC_EMP (concat con INV_AF5_HC_EMP) bloqueado: AG_AF5_HC_EMP no se pudo calcular"
)


In [ ]:
# IMP_AF5_HC_EMP -- depende del bloque anterior
raise NotImplementedError("WORK.IMP_AF5_HC_EMP bloqueado: depende de WORK.AG_AF5_HC_EMP, no calculable")


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN 37 Y 36907 CON CA 2 -- depende de IMP_AF5_HC_EMP
raise NotImplementedError("WORK.IMP_AF5_HC_2 (rama EMP) bloqueado: depende de WORK.IMP_AF5_HC_EMP, no calculable")


In [ ]:
# APPEND de IMP_AF5_HC_EMP e IMP_AF5_HC_2 a TABLAS.BD_CTSI -- bloqueado
raise NotImplementedError(
    "APPEND de IMP_AF5_HC_EMP/IMP_AF5_HC_2 a TABLAS.BD_CTSI bloqueado: los DataFrames no se pudieron generar"
)


In [ ]:
# refresca #bd_ctsi antes del bloque final VM/RP con empresas supervisadas
work_conn.execute(text("DROP TABLE IF EXISTS #bd_ctsi"))
work_conn.execute(text("SELECT * INTO #bd_ctsi FROM TABLAS.dbo.BD_CTSI"))


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON EMPRESAS SUPERVISADAS
# CIERRE 2021: FALTABA INGRESAR COMO FILTRO EL C_SCN -- filtro agregado en el SAS original, se preserva
# join usa T1.SECTOR=INPUT(T2.C_CAGENTE,BEST5.) y T1.C_CAGENTE=LEFT(PUT(T2.SECTOR,CHAR6.)) -- misma tabla
# externa de ratios AF5 emp/holding, no versionada
raise NotImplementedError(
    "ratio_AF5_emp_hold_n.sas7bdat no está migrado a una tabla del catálogo ni a un archivo del workspace: "
    "sin esa fuente no se puede calcular VM_AF5_HC_ES (el join con formato CHAR6/BEST5 de SAS tampoco "
    "se puede replicar sin conocer el contenido real de la tabla de ratios)"
)


In [ ]:
# REC PRECIO POR AJUSTAR A MERCADO -- depende de VM_AF5_HC_ES, bloqueado
raise NotImplementedError("WORK.RP_AF5_HC_ES bloqueado: depende de WORK.VM_AF5_HC_ES, no calculable")


In [ ]:
# APPEND de VM_AF5_HC_ES y RP_AF5_HC_ES a TABLAS.BD_CTSI -- bloqueado, cierre del nodo
raise NotImplementedError(
    "APPEND de VM_AF5_HC_ES/RP_AF5_HC_ES a TABLAS.BD_CTSI bloqueado: los DataFrames no se pudieron generar"
)


## S2_04_HH&Dep

Reclasifica el patrimonio (AF.5) entre empresas y hogares y ajusta depósitos (AF.22/AF.29/AF.7) imputando la contraparte, incluyendo el reparto en tercios del sector 36912 y el balance de activo/pasivo del sector 51022

*confianza: medium · verificador: approve · SAS: PROC SQL: UPDATE/DELETE condicionales + CREATE TABLE con agregación + PROC APPEND FORCE sobre tabla acumulativa*

In [ ]:
# ========= S2_04_HH&Dep =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES no tiene equivalente en SQL Server; se documenta y se omite)
_log("nota", "DATA step COMPRESS=YES es una opción de almacenamiento SAS sin equivalente en SQL Server; no se traduce")


In [ ]:
# CAMBIA CONTRAGENTE 511 EN EL PATRIMONIO DE EMPRESAS SUPERVISADAS A 53 PARA RECALCULAR LA PARTE QUE VA A HOGARES EN BASE AL % DEL TOTAL DEL PAT DE EMPRESAS
with engine.begin() as conn:
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = '53', PROC = '0014'
        WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 51021
    """))
    _log("UPDATE BD_CTSI C_CAGENTE 511->53 (sector 51021)", res.rowcount)


In [ ]:
# CIERRE 2020. ELIMINA PASIVO Y ACTIVO AF7 DE EMPRESAS HOGARES CON CONTRAGENTE HOGARES
with engine.begin() as conn:
    res = conn.execute(text("""
        DELETE FROM TABLAS.dbo.BD_CTSI
        WHERE SECTOR = 5111 AND C_SCN = 'AF.7' AND C_CAGENTE = '511'
    """))
    _log("DELETE BD_CTSI AF.7 sector 5111 ca 511", res.rowcount)


In [ ]:
# CIERRE 2021: ELIMINA CONTRAGENTE HOGARES EN PATRIMONIO DE RESTO DE EMPRESAS SECTOR 51022, PORQUE LE GENERA MUCHA VOLATILIDAD AL SECTOR
with engine.begin() as conn:
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = '53', PROC = '0014a'
        WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 51022
    """))
    _log("UPDATE BD_CTSI C_CAGENTE 511->53 (sector 51022)", res.rowcount)


In [ ]:
# CIERRE 2021: EN EMPRESAS HOGARES (SECTOR 5111) DEJA TODO EL PATRIMONIO PARA HOGARES (LO DEJA EN CA 53 PARA QUE LO IMPUTE PORQUE ASÍ ESTABA ANTES
with engine.begin() as conn:
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = '53', PROC = '0014b'
        WHERE C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 5111
    """))
    _log("UPDATE BD_CTSI C_CAGENTE ->53 (sector 5111, todo AF.5 H)", res.rowcount)


In [ ]:
# CIERRE 2021: AJUSTA PATRIMONIO DE SECTOR 36912 ASIGNADO A HOGARES EN 1/3 YA QUE ES MUY VOLÁTIL Y DISTORSIONA LA CTA FINANCIERA. LA DIFERENCIA SE LLEVA A CONTRAGENTE 36912 y 53
# calcula patrimonio ajustado a hogares (queda en CA 511 original / 3)
sql_pat_36912_hh = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) / 3.0 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0014d' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE = '511' AND C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 36912
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
pat_36912_hh = pd.read_sql(text(sql_pat_36912_hh), engine)
_log("pat_36912_hh", pat_36912_hh)


In [ ]:
# deja 1/3 en ca 36912
sql_pat_36912_36 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '36912' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * 1.0 / 3 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0014e' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE = '511' AND C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 36912
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
pat_36912_36 = pd.read_sql(text(sql_pat_36912_36), engine)
_log("pat_36912_36", pat_36912_36)


In [ ]:
# deja 1/3 en ca 53
sql_pat_36912_53 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * 1.0 / 3 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0014e' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE = '511' AND C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 36912
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
pat_36912_53 = pd.read_sql(text(sql_pat_36912_53), engine)
_log("pat_36912_53", pat_36912_53)


In [ ]:
# elimina patrimonio inicial asignado a hogares (reversa el 100% original)
sql_pat_36912_hh_elimina = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0014f' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE = '511' AND C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 36912
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
pat_36912_hh_elimina = pd.read_sql(text(sql_pat_36912_hh_elimina), engine)
_log("pat_36912_hh_elimina", pat_36912_hh_elimina)


In [ ]:
# APPEND de los 4 resultados a BD_CTSI (acumulativo, replica proc append force): re-ejecutar duplica, igual que el SAS original
pat_36912_hh.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde pat_36912_hh", len(pat_36912_hh))
pat_36912_36.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde pat_36912_36", len(pat_36912_36))
pat_36912_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde pat_36912_53", len(pat_36912_53))
pat_36912_hh_elimina.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde pat_36912_hh_elimina", len(pat_36912_hh_elimina))


In [ ]:
# el DROP TABLE de los work datasets SAS no aplica: son DataFrames en memoria (pat_36912_hh, pat_36912_36, pat_36912_53, pat_36912_hh_elimina), no hay objeto de BD que borrar


In [ ]:
# IMPUTA ACTIVO DE HOGARES CON INFO DE CONTRAPARTIDA
sql_activo_hh = """
SELECT MONEDA, [AÑO], TRIM,
       511 AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(11))) AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0015' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE IN ('5', '511', 'S14') AND C_ENTRADA = 'H'
      AND C_CUENTA IN ('Bce Final', 'Financiera', 'Bce Inicio', 'Rec Precio', 'Rec Precio Reaj', 'Rec Volumen')
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR
"""
activo_hh = pd.read_sql(text(sql_activo_hh), engine)
# ajusta el contragente calculado 5111 a 51022 solo para AF.5
activo_hh.loc[(activo_hh["C_CAGENTE"] == "5111") & (activo_hh["C_SCN"] == "AF.5"), "C_CAGENTE"] = "51022"
_log("activo_hh", activo_hh)


In [ ]:
activo_hh.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde activo_hh", len(activo_hh))


In [ ]:
# IMPUTA PASIVO DE HOGARES CON INFO DE CONTRAPARTIDA
sql_pasivo_hh = """
SELECT MONEDA, [AÑO], TRIM,
       511 AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(11))) AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0016' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE C_CAGENTE IN ('5', '511', 'S14') AND C_ENTRADA = 'D'
      AND C_CUENTA IN ('Bce Final', 'Financiera', 'Bce Inicio', 'Rec Precio', 'Rec Precio Reaj', 'Rec Volumen')
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR
"""
pasivo_hh = pd.read_sql(text(sql_pasivo_hh), engine)
_log("pasivo_hh", pasivo_hh)


In [ ]:
pasivo_hh.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde pasivo_hh", len(pasivo_hh))


In [ ]:
# DEPOSITOS AF29 ENTRE BCOS-PASIVO Y MUNICIPALIDADES-ACTIVO. RESPETA DATO DE SECTOR 321
sql_af29_42_321 = """
SELECT MONEDA, [AÑO], TRIM,
       42 AS SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0142' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE SECTOR = 321 AND C_CAGENTE = '42' AND C_ENTRADA = 'H' AND C_SCN = 'AF.29'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
"""
af29_42_321 = pd.read_sql(text(sql_af29_42_321), engine)
_log("af29_42_321", af29_42_321)


In [ ]:
af29_42_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde af29_42_321", len(af29_42_321))


In [ ]:
# AJUSTA TOTAL DE DEPÓSITOS BANCARIOS IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL ACTIVO DEL SECTOR 51022 CON CONTRAGENTE 321
sql_af22_total = """
SELECT 'P' AS MONEDA, [AÑO], TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       CASE WHEN SECTOR = 321 AND C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
            WHEN SECTOR <> 321 AND C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
            ELSE C_CUENTA END AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA = 'D' THEN DATO * -1 ELSE DATO END) AS DATO,
       C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0118' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'D' AND C_SCN = 'AF.22' AND C_CAGENTE = '321')
   OR (C_ENTRADA = 'H' AND C_SCN = 'AF.22' AND SECTOR = 321)
GROUP BY [AÑO], TRIM,
         CASE WHEN SECTOR = 321 AND C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
              WHEN SECTOR <> 321 AND C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
              ELSE C_CUENTA END,
         C_ENTRADA, C_SCN
"""
af22_total = pd.read_sql(text(sql_af22_total), engine)
_log("af22_total", af22_total)


In [ ]:
# CIERRE 2020. EL AJUSTE ANTERIOR SE LLEVA A ACTIVO AF29 SECTOR 51022 CA 321 PARA EQUILIBRAR EL BALANCE
af29_aj = af22_total.copy()
af29_aj["DATO"] = af29_aj["DATO"] * -1
af29_aj["C_SCN"] = "AF.29"
af29_aj["N_SCN"] = "Otros depósitos"
af29_aj = af29_aj[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af29_aj", af29_aj)


In [ ]:
af22_total.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde af22_total", len(af22_total))
af29_aj.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde af29_aj", len(af29_aj))


In [ ]:
# AJUSTA TOTAL DE OTROS DEPÓSITOS IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL ACTIVO DEL SECTOR 51022 CON CONTRAGENTE 321
sql_af29_total = """
SELECT 'P' AS MONEDA, [AÑO], TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA = 'D' THEN DATO * -1 ELSE DATO END) AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0159' AS PROC
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'D' AND C_SCN = 'AF.29') OR (C_ENTRADA = 'H' AND C_SCN = 'AF.29')
GROUP BY [AÑO], TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
af29_total = pd.read_sql(text(sql_af29_total), engine)
_log("af29_total", af29_total)


In [ ]:
# CIERRE 2020. AJUSTE ANTERIOR SE IMPUTA EN ACTIVO AF7 SECTOR 51022 CA 53, PARA EQUILIBRAR BALANCE
af7_ajuste = af29_total.copy()
af7_ajuste["C_CAGENTE"] = "53"
af7_ajuste["DATO"] = af7_ajuste["DATO"] * -1
af7_ajuste["C_SCN"] = "AF.7"
af7_ajuste["N_SCN"] = "Créditos comerciales"
af7_ajuste = af7_ajuste[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af7_ajuste", af7_ajuste)


In [ ]:
af29_total.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde af29_total", len(af29_total))
af7_ajuste.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.BD_CTSI desde af7_ajuste", len(af7_ajuste))


## S2_05_Ptmos

Reconcilia préstamos de corto y largo plazo (AF.41/AF.42) entre pares de sectores institucionales, respetando el dato del sector 'fuente de verdad' de cada par e imputando el diferencial en el sector contrapartida (mayormente 51022/321), acumulando los ajustes en la tabla maestra de instrumentos financieros / Concilia y ajusta préstamos de corto y largo plazo (AF.41/AF.42) entre sectores institucionales (RM, gobierno, bancos, empresas, resto del mundo) imputando el diferencial en la contraparte convenida (321, 51022, 53, etc.) y acumula cada ajuste en la base de conciliación TABLAS.BD_CTSI / Concilia préstamos de largo plazo (AF.41/AF.42) entre sectores institucionales, reclasifica contrapartidas del sector 5111 hacia 51022, imputa fondos de pensiones en pasivo de empresas privadas y ajusta el diferencial activo-pasivo total en el sector 51022/321, acumulando cada resultado en BD_CTSI

*confianza: medium · verificador: revise · SAS: PROC SQL UPDATE/CREATE TABLE + PROC DATASETS APPEND, tablas temporales de sesión (#tmp) encadenadas sobre TABLAS.BD_CTSI + PROC SQL (múltiples CREATE TABLE con SUM/GROUP BY sobre TABLAS.BD_CTSI) + PROC DATASETS APPEND FORCE + UPDATE, tramo 2/3 + PROC SQL (SUM/GROUP BY, imputación de diferencial) + PROC DATASETS APPEND FORCE + UPDATE, con tablas temporales de sesión*

In [ ]:
# ========= S2_05_Ptmos =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con
# DELETE FROM + INSERT deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# CIERRE 2021: CAMBIOS EN ALGUNOS CONTRAGENTE DE PRESTAMO PASIVOS DE EMPRESAS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE SECTOR IN (51021,5101) AND C_SCN='AF.41' AND C_ENTRADA='H' AND C_CAGENTE IN ('511','53','31')
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (51021/5101 H)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.42', N_SCN='Préstamos a largo plazo'
WHERE SECTOR IN (51021,5101) AND C_SCN='AF.41' AND C_ENTRADA='H' AND C_CAGENTE IN ('351','352','41')
"""))
_log("UPDATE BD_CTSI AF.41->AF.42 (51021/5101 H)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE SECTOR IN (51021,5101) AND C_SCN='AF.42' AND C_ENTRADA='H' AND C_CAGENTE IN ('511')
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (51021/5101 AF.42 H)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE SECTOR IN (37) AND C_SCN='AF.42' AND C_ENTRADA='H' AND C_CAGENTE IN ('33222')
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (37 AF.42 H)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.42', N_SCN='Préstamos a largo plazo'
WHERE SECTOR IN (37) AND C_SCN='AF.41' AND C_ENTRADA='H' AND C_CAGENTE IN ('351','352','35')
"""))
_log("UPDATE BD_CTSI AF.41->AF.42 (37 H)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE SECTOR IN (51021,5101) AND C_SCN='AF.42' AND C_ENTRADA='H' AND C_CAGENTE IN ('31')
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (51021/5101 AF.42 H, ca 31)", res.rowcount)


In [ ]:
# CIERRE 2022: CAMBIOS EN ALGUNOS CONTRAGENTE DE PRESTAMOS PASIVOS DE AUXILIARES
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE SECTOR IN (37) AND C_SCN='AF.41' AND C_ENTRADA='H' AND C_CAGENTE IN ('2')
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (37 AF.41 H, ca 2)", res.rowcount)


In [ ]:
# AJUSTA PTMO DE CP ENTRE FONDOS DE PENSIONES-ACTIVO Y RM-PASIVO. RESPETA DATO DE RM Y AJUSTA EN AF.5 ACTIVO DE FP CON CONTRAGENTE RM
# cierre 2022q3. incorpora ajuste en FP por cambios en los datos de la balanza
work_conn.execute(text("DROP TABLE IF EXISTS #af41_34_6"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 34 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR=34 THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0179a' AS PROC
INTO #af41_34_6
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=34 AND C_CAGENTE='6')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE='34')
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af5_34_6"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.5' AS C_SCN, 'Acciones y otras participaciones de capital' AS N_SCN,
       'PS' AS FUENTE, '0180a' AS PROC
INTO #af5_34_6
FROM #af41_34_6
"""))

cols_af41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_34_6"))
_log("APPEND BD_CTSI af41_34_6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af5_34_6"))
_log("APPEND BD_CTSI af5_34_6", res.rowcount)

for t in ["#af41_34_6", "#af5_34_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO DE CP ENTRE BCO CENTRAL-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE BCO CENTRAL Y AJUSTA EN PTMO DE LP DE SECTOR 321 CON CA 31
# cierre 2021. no se cambia rec precio reaj a rec precio sino que se concilian con cuentas originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_31"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 321 AS SECTOR, '31' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR=321 THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0179' AS PROC
INTO #af41_321_31
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=31 AND C_CAGENTE='321')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=321 AND C_CAGENTE='31')
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_31"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       DATO*-1 AS DATO, 'AF.42' AS C_SCN, 'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0180' AS PROC
INTO #af42_321_31
FROM #af41_321_31
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321_31"))
_log("APPEND BD_CTSI af41_321_31", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_321_31"))
_log("APPEND BD_CTSI af42_321_31", res.rowcount)

for t in ["#af41_321_31", "#af42_321_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO DE CP ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DE BCO CENTRAL Y AJUSTA EN PTMO DE LP DE SECTOR 6 CON CA 321
# cierre 2021. no se cambia rec precio reaj a rec precio sino que se concilian con cuentas originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_31"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 6 AS SECTOR, '31' AS C_CAGENTE,
       CASE WHEN SECTOR=31 AND C_CUENTA IN ('Rec Volumen') THEN 'Rec Precio' ELSE C_CUENTA END AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR=6 THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0180d' AS PROC
INTO #af41_6_31
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=31 AND C_CAGENTE='6')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE='31')
GROUP BY MONEDA, AÑO, TRIM,
         CASE WHEN SECTOR=31 AND C_CUENTA IN ('Rec Volumen') THEN 'Rec Precio' ELSE C_CUENTA END,
         C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_31"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.42' AS C_SCN, 'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0180e' AS PROC
INTO #af42_6_31
FROM #af41_6_31
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_31"))
_log("APPEND BD_CTSI af41_6_31", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_6_31"))
_log("APPEND BD_CTSI af42_6_31", res.rowcount)

for t in ["#af41_6_31", "#af42_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR 412 DE CA 9 A 321
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321'
WHERE C_CAGENTE='9' AND SECTOR=412 AND C_SCN='AF.41'
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (412 ca 9)", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA EN FONDOS DE INVERISÓN PTMOS DE LARGO PLAZO OTORGADO A RESTO DEL MUNDO A PTMOS DE CORTO PLAZO
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.41', N_SCN='Préstamos a corto plazo', PROC='0210a'
WHERE C_CAGENTE='6' AND SECTOR=339011 AND C_SCN='AF.42' AND C_ENTRADA='D'
"""))
_log("UPDATE BD_CTSI AF.42->AF.41 (339011 D)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA PTMOS DE CP DE FONDOS DE INVERSIÓN ACTIVO CON CA RM EN EL PASIVO DEL RM. AJUSTA EN PASIVO DEL RM CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_fi"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 6 AS SECTOR, '33901' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR=6 THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0210b' AS PROC
INTO #af41_6_fi
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR IN (339011,3390102) AND C_CAGENTE='6')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE IN ('33901','33901/351'))
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_53"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0210c' AS PROC
INTO #af41_6_53
FROM #af41_6_fi
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_fi"))
_log("APPEND BD_CTSI af41_6_fi", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_53"))
_log("APPEND BD_CTSI af41_6_53", res.rowcount)

for t in ["#af41_6_fi", "#af41_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA PTMOS DE CP DE FONDOS DE INVERSIÓN-OFIS-AUX ACTIVO CON CA OFIS Y AUXILIARES EN EL PASIVO DE ESOS SECTORES. AJUSTA EN PASIVO DE OFIS Y AUX CON CA 321
# SELECCIONA ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_a"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM,
       TRY_CAST(C_CAGENTE AS int) AS SECTOR,
       CAST(SECTOR AS varchar(8)) AS C_CAGENTE,
       C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0210d' AS PROC
INTO #af41_oa_fi_a
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='D' AND C_SCN='AF.41'
  AND (SECTOR IN (339011,3390102,411,37) OR LEFT(CAST(SECTOR AS varchar(8)),2)='36' OR LEFT(CAST(SECTOR AS varchar(8)),2)='33')
  AND (LEFT(C_CAGENTE,2)='36' OR LEFT(C_CAGENTE,2)='33' OR C_CAGENTE IN ('411','37'))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, TRY_CAST(C_CAGENTE AS int), CAST(SECTOR AS varchar(8))
"""))

# SELECCIONA PASIVO. EN CASO QUE ALGUNO DE ESTOS SECTORES INFORME PTMOS CON FONDOS DE INVERSIÓN, ASÍ SOLO DE IMPUTA LA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_b"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0210d' AS PROC
INTO #af41_oa_fi_b
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='H' AND C_SCN='AF.41'
  AND (LEFT(CAST(SECTOR AS varchar(8)),2)='36' OR LEFT(CAST(SECTOR AS varchar(8)),2)='33' OR SECTOR IN (411,37))
  AND (LEFT(C_CAGENTE,2)='36' OR LEFT(C_CAGENTE,2)='33' OR C_CAGENTE IN ('411','37'))
GROUP BY AÑO, TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi"))
cols_oa_fi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
work_conn.execute(text(f"""
SELECT {cols_oa_fi} INTO #af41_oa_fi FROM #af41_oa_fi_a
UNION ALL
SELECT {cols_oa_fi} FROM #af41_oa_fi_b
"""))

# IMPUTA DIFERENCIAL DE PTMOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_def"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_oa_fi_def
FROM #af41_oa_fi
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

# AJUSTA EN CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_321"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_oa_321
FROM #af41_oa_fi_def
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_oa_fi_def"))
_log("APPEND BD_CTSI af41_oa_fi_def", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_oa_321"))
_log("APPEND BD_CTSI af41_oa_321", res.rowcount)

for t in ["#af41_oa_321", "#af41_oa_fi_def", "#af41_oa_fi", "#af41_oa_fi_a", "#af41_oa_fi_b"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR RM DE CA 53 A 321
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321', PROC='0210'
WHERE C_CAGENTE='53' AND SECTOR=6 AND C_SCN='AF.41' AND C_ENTRADA='H'
"""))
_log("UPDATE BD_CTSI C_CAGENTE=321 (sector 6, ca 53 H)", res.rowcount)


In [ ]:
# AJUSTA TOTAL DE PTMOS DE CP IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL PASIVO DEL SECTOR 51022 CON CONTRAGENTE 321 Y AJUSTANDO CONTRA PTMO DE LP EN EL MISMO SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af41_total"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, '321' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0170' AS PROC
INTO #af41_total
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41') OR (C_ENTRADA='H' AND C_SCN='AF.41')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_aj"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.42' AS C_SCN, 'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0171' AS PROC
INTO #af41_aj
FROM #af41_total
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_total"))
_log("APPEND BD_CTSI af41_total", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_aj"))
_log("APPEND BD_CTSI af41_aj", res.rowcount)

for t in ["#af41_total", "#af41_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR RM DE CA 41 A 53
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='53'
WHERE C_CAGENTE='41' AND SECTOR=6 AND C_SCN='AF.41'
"""))
_log("UPDATE BD_CTSI C_CAGENTE=53 (sector 6, ca 41)", res.rowcount)


In [ ]:
# AJUSTA DE PTMOS DE CP entre RM-ACTIVO Y 5101-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 5101 CON CONTRAGENTE 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_5101"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 5101 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0197' AS PROC
INTO #af41_6_5101
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE='5101')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=5101 AND C_CAGENTE='6')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_5101"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0198' AS PROC
INTO #af41_321_5101
FROM #af41_6_5101
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_5101"))
_log("APPEND BD_CTSI af41_6_5101", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321_5101"))
_log("APPEND BD_CTSI af41_321_5101", res.rowcount)

for t in ["#af41_6_5101", "#af41_321_5101"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# NOTA: bloque AF41_6_36 / AF41_6_53 (0197a/0198a) está COMENTADO en el SAS original — no se traduce

# AJUSTA DE PTMOS DE CP entre RM-ACTIVO Y 51021/51022 MINERAS-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 51021 CON CONTRAGENTE 321.
# CIERRE 2020. SE AGREGA CONTRAGENTE 51021 YA QUE ES NUEVO EN LA CTA DEL RM
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51021 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0202' AS PROC
INTO #af41_6_51021
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE IN ('53','51021'))
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=51021 AND C_CAGENTE='6')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=51022 AND MONEDA='D' AND C_CAGENTE='6')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_51021"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0203' AS PROC
INTO #af41_321_51021
FROM #af41_6_51021
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_51021"))
_log("APPEND BD_CTSI af41_6_51021", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321_51021"))
_log("APPEND BD_CTSI af41_321_51021", res.rowcount)

for t in ["#af41_6_51021", "#af41_321_51021"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP RM-ACTIVO CON CA 31 USANDO INFO DE LO REPORTADO EN PASIVO DEL SECTOR 31 CON CA 6. AJUSTA CONTRA AF7
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_31_b"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 6 AS SECTOR, '31' AS C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0205' AS PROC
INTO #af41_6_31_b
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=31 AND C_CAGENTE='6')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af71_6_31"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.71' AS C_SCN, 'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE, '0206' AS PROC
INTO #af71_6_31
FROM #af41_6_31_b
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_31_b"))
_log("APPEND BD_CTSI af41_6_31 (0205)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af71_6_31"))
_log("APPEND BD_CTSI af71_6_31", res.rowcount)

af71_6_31 = pd.read_sql(text("SELECT * FROM #af71_6_31"), work_conn)
_log("af71_6_31", af71_6_31)

for t in ["#af41_6_31_b", "#af71_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP EN PASIVO DE 51022 CON CA FMNM Y OFIS USANDO INFO DE LO REPORTADO EN ACTIVO DE ESOS SECTORES
# CIERRE 2021: SE MODIFICA PARA QUE SOLO CONSIDERE CONTRAGENTES DE EMPRESAS Y BANCOS (PARA QUE QUEDEN EN EMPRESAS),
# YA QUE POR CAMBIO EN FI TMB HAY PTMOS A OFIS Y AUX. sin ca 334 poque se cambia a auxiliares
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_a"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, CAST(SECTOR AS varchar(11)) AS C_CAGENTE,
       C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208' AS PROC
INTO #af41_51022_a
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='D' AND C_SCN='AF.41'
  AND LEFT(CAST(SECTOR AS varchar(8)),2)='33' AND SECTOR NOT IN (334,3390101)
  AND C_CAGENTE IN ('53','51021','5101','51022','5102','9')
GROUP BY AÑO, TRIM, C_CUENTA, CAST(SECTOR AS varchar(11)), C_SCN, N_SCN
"""))

# CIERRE 2021: INCORPORA EL PASIVO DE EMPRESAS CON OFIS PARA QUE SE RESTE A LA CONSULTA ANTERIOR Y SE IMPUTE LA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_2"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208' AS PROC
INTO #af41_51022_2
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR IN (51021,51022,5101)
  AND LEFT(C_CAGENTE,2)='33' AND C_CAGENTE NOT IN ('3390101','334')
GROUP BY AÑO, TRIM, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022"))
cols_51022 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
work_conn.execute(text(f"""
SELECT {cols_51022} INTO #af41_51022 FROM #af41_51022_a
UNION ALL
SELECT {cols_51022} FROM #af41_51022_2
"""))

# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_def"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_51022_def
FROM #af41_51022
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

# AJUSTA EN PTMOS DE CP DE SECTOR 51022 CONTRAGENTE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_321"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_51_321
FROM #af41_51022_def
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_51_321"))
_log("APPEND BD_CTSI af41_51_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_51022_def"))
_log("APPEND BD_CTSI af41_51022_def", res.rowcount)

for t in ["#af41_51022", "#af41_51022_2", "#af41_51022_def", "#af41_51_321", "#af41_51022_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA DE PTMOS DE CP EN PASIVO DE EMPRESAS CONTRAGENTE AUXILIARES USANDO LO REPORTADO EN EL ACTIVO DE AUXILIARES CON CONTRAGENTE EMPRESAS.
# AJUSTA EN PTMOS DE EMPRESAS CON BANCOS
# ACTIVO DE AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51_a"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, CAST(SECTOR AS varchar(5)) AS C_CAGENTE,
       C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208a' AS PROC
INTO #af41_36_51_a
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='D' AND C_SCN='AF.41' AND LEFT(CAST(SECTOR AS varchar(8)),2)='36'
  AND C_CAGENTE IN ('53','51021','5101','51022','5102','9','51')
GROUP BY AÑO, TRIM, C_CUENTA, CAST(SECTOR AS varchar(5)), C_ENTRADA, C_SCN, N_SCN
"""))

# PASIVO DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_36"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208a' AS PROC
INTO #af41_51_36
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR IN (51021,51022,5101) AND LEFT(C_CAGENTE,2)='36'
GROUP BY AÑO, TRIM, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51"))
work_conn.execute(text(f"""
SELECT {cols_51022} INTO #af41_36_51 FROM #af41_36_51_a
UNION ALL
SELECT {cols_51022} FROM #af41_51_36
"""))

# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51_ag"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_36_51_ag
FROM #af41_36_51
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

# GENERA IMPUTACIÓN EN PASIVO DE EMPRESAS CON CA BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_321_b"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_51_321_b
FROM #af41_36_51_ag
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_36_51_ag"))
_log("APPEND BD_CTSI af41_36_51_ag", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_51_321_b"))
_log("APPEND BD_CTSI af41_51_321 (0208a)", res.rowcount)

for t in ["#af41_36_51_ag", "#af41_51_321_b", "#af41_36_51", "#af41_51_36", "#af41_36_51_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA DE PTMOS DE CP EN ACTIVO DE EMPRESAS USANDO LO REPORTADO EN EL PASIVO DE EMPRESAS CON CONTRAGENTE EMPRESAS.
# AJUSTA EN CREDITOS COMERCIALES DEL SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_a"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, '51022' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208' AS PROC
INTO #af41_51_a
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='D' AND C_SCN='AF.41' AND LEFT(CAST(SECTOR AS varchar(8)),2)='51'
  AND SECTOR NOT IN (511,5111) AND C_CAGENTE IN ('53','51021','5101','51022','5102','9','51')
GROUP BY AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""))

# PASIVO DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_2"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, TRY_CAST(C_CAGENTE AS int) AS SECTOR, '51022' AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0208' AS PROC
INTO #af41_51_2
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR IN (51021,51022,5101)
  AND LEFT(C_CAGENTE,2) IN ('51','53','9','2') AND C_CAGENTE NOT IN ('511')
GROUP BY AÑO, TRIM, TRY_CAST(C_CAGENTE AS int), C_CUENTA, C_SCN, N_SCN
"""))

res = work_conn.execute(text("UPDATE #af41_51_2 SET SECTOR=51022 WHERE SECTOR IN (5102,2)"))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_51"))
work_conn.execute(text(f"""
SELECT {cols_51022} INTO #af41_51 FROM #af41_51_a
UNION ALL
SELECT {cols_51022} FROM #af41_51_2
"""))

# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_def"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af41_51_def
FROM #af41_51
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""))

# AJUSTA EN AF.7 DEL SECTOR RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, 'AF.7' AS C_SCN, 'Créditos comerciales' AS N_SCN,
       FUENTE, PROC
INTO #af7_51
FROM #af41_51_def
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, FUENTE, PROC
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_51_def"))
_log("APPEND BD_CTSI af41_51_def", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af7_51"))
_log("APPEND BD_CTSI af7_51", res.rowcount)

af7_51 = pd.read_sql(text("SELECT * FROM #af7_51"), work_conn)
_log("af7_51", af7_51)

for t in ["#af41_51_def", "#af7_51", "#af41_51", "#af41_51_2", "#af41_51_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA DE PTMOS DE CP entre BCOS-ACTIVO Y SECTORES-PASIVO. RESPETA DATO DE BCOS E IMPUTA DIF EN 51022 CON CONTRAGENTE 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORGINALES Y ADEMAS AGREGA CONTRAGENTE 322, YA QUE EMPRESAS TIENE ESE CONTRAGENTE
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, '321' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0184' AS PROC
INTO #af41_321
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=321)
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND C_CAGENTE IN ('321','322'))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_51022"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.42' AS C_SCN, 'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0185' AS PROC
INTO #af42_321_51022
FROM #af41_321
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321"))
_log("APPEND BD_CTSI af41_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_321_51022"))
_log("APPEND BD_CTSI af42_321_51022", res.rowcount)

for t in ["#af41_321", "#af42_321_51022"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PTMOS DE CP DEL ACTIVO DE GOB CON CA BCOS Y PASIVO DE BCOS CON GOB. RESPETA DATO DE GOBIERNO E IMPUTA EN BCOS. AJUSTA IMPUTACIÓN EN PASIVO DE BCOS CON CA 53
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_41"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 321 AS SECTOR, '41' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0187a' AS PROC
INTO #af41_321_41
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=41 AND C_CAGENTE='321')
   OR (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=321 AND C_CAGENTE='41')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_53"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, 321 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0187b' AS PROC
INTO #af41_321_53
FROM #af41_321_41
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321_41"))
_log("APPEND BD_CTSI af41_321_41", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321_53"))
_log("APPEND BD_CTSI af41_321_53", res.rowcount)

for t in ["#af41_321_41", "#af41_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP ENTRE BCOS-PASIVO CON CONTRAGENTE RM, USANDO INFO DEL PASIVO DE BCOS CON CONTRAGENTES 53 Y 36901
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321p"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO)*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0187' AS PROC
INTO #af41_321p
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=321 AND C_CAGENTE IN ('53','36901')
GROUP BY AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af41_321p_6"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, '6' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0188' AS PROC
INTO #af41_321p_6
FROM #af41_321p
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321p"))
_log("APPEND BD_CTSI af41_321p", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_321p_6"))
_log("APPEND BD_CTSI af41_321p_6", res.rowcount)

for t in ["#af41_321p", "#af41_321p_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE CP ENTRE RM-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE RM E AJUSTA CONTRA PTMOS DE LP DEL SECTOR 321 CON CA RM
# CIERRE 2021: NO SE CAMBIA CUENTA REC PRECIO A REC PRECIO REAJ
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_321"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 321 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0192' AS PROC
INTO #af41_6_321
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.41' AND SECTOR=321 AND C_CAGENTE='6')
   OR (C_ENTRADA='D' AND C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE='321')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_321"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.42' AS C_SCN, 'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0193' AS PROC
INTO #af42_6_321
FROM #af41_6_321
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af41_6_321"))
_log("APPEND BD_CTSI af41_6_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_6_321"))
_log("APPEND BD_CTSI af42_6_321 (0193)", res.rowcount)

for t in ["#af41_6_321", "#af42_6_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACIÓN PTMOS DE LP, RESPETANDO DATO DEL TOTAL ACTIVO E IMPUTANDO DIFERENCIA EN PASIVO DEL SECTOR 51022 CON CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_all"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, '321' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, 'Préstamos a largo plazo' AS N_SCN, 'PS' AS FUENTE, '0175' AS PROC
INTO #af42_all
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.42') OR (C_ENTRADA='D' AND C_SCN='AF.42')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN
"""))

# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51022 CON CA 53. REGLA NUEVA CIERRE 2020
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
work_conn.execute(text("""
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.7' AS C_SCN, 'Créditos comerciales' AS N_SCN,
       FUENTE, PROC
INTO #af7_ajuste
FROM #af42_all
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_all"))
_log("APPEND BD_CTSI af42_all", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af7_ajuste"))
_log("APPEND BD_CTSI af7_ajuste (0175)", res.rowcount)

af7_ajuste = pd.read_sql(text("SELECT * FROM #af7_ajuste"), work_conn)
_log("af7_ajuste", af7_ajuste)

for t in ["#af42_all", "#af7_ajuste"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# NOTA: bloque comentado en el SAS original (cambio C_SCN a AF.42 para sector 31 ca 41, inst AF.32->AF.42) — no se traduce

# CAMBIA CONTRAGENTE DE DEUDA SUBORDINADA ACTIVO DEL BANCO CENTRAL
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='37', PROC='0214a'
WHERE C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='321' AND C_ENTRADA='D'
"""))
_log("UPDATE BD_CTSI C_CAGENTE=37 (sector 31, AF.42 D)", res.rowcount)


In [ ]:
# AJUSTA PTMOS DE LP ENTRE BCO CENTRAL-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE BCENTRAL Y AJUSTA CONTRA PTMOS DE LP DEL SECTOR 321 CON CA RM
# CIERRE 2021. NO SE HACE CAMBIO DE NOMBRE DE CUENTA PARA CONCILIAR ASI QUEDA IGUAL A BANCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af42_31_321"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 321 AS SECTOR, '31' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0214' AS PROC
INTO #af42_31_321
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.42' AND SECTOR=321 AND C_CAGENTE='31')
   OR (C_ENTRADA='D' AND C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='321')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_6"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, '6' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0215' AS PROC
INTO #af42_321_6
FROM #af42_31_321
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_31_321"))
_log("APPEND BD_CTSI af42_31_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_321_6"))
_log("APPEND BD_CTSI af42_321_6 (0215)", res.rowcount)

for t in ["#af42_31_321", "#af42_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE BCO CENTRAL-ACTIVO Y HOLDING-PASIVO. RESPETA DATO DE BCENTRAL Y AJUSTA CONTRA PTMOS DE LP DEL SECTOR 37 CON CA 321
# CIERRE 2021. NO SE HACE CAMBIO DE NOMBRE DE CUENTA PARA CONCILIAR ASI QUEDA IGUAL A BANCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af42_31_37"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 37 AS SECTOR, '31' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0214b' AS PROC
INTO #af42_31_37
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.42' AND SECTOR=37 AND C_CAGENTE='31')
   OR (C_ENTRADA='D' AND C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='37')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_37_321"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0215b' AS PROC
INTO #af42_37_321
FROM #af42_31_37
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_31_37"))
_log("APPEND BD_CTSI af42_31_37", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_37_321"))
_log("APPEND BD_CTSI af42_37_321", res.rowcount)

for t in ["#af42_31_37", "#af42_37_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA AJ DE CONCILIACION DEL SECTOR 321 CON CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_321_b"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 321 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0220' AS PROC
INTO #af42_6_321_b
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.42' AND SECTOR=321 AND C_CAGENTE='6')
   OR (C_ENTRADA='D' AND C_SCN='AF.42' AND SECTOR=6 AND C_CAGENTE='321')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #af71_321_6"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO, 'AF.71' AS C_SCN, 'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE, '0221' AS PROC
INTO #af71_321_6
FROM #af42_6_321_b
"""))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af42_6_321_b"))
_log("APPEND BD_CTSI af42_6_321 (0220)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41}) SELECT {cols_af41} FROM #af71_321_6"))
_log("APPEND BD_CTSI af71_321_6", res.rowcount)

for t in ["#af42_6_321_b", "#af71_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y GOBIERNO-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA AJ DE CONCILIACION DEL SECTOR 41 CON CA RM
# CIERRE 2021. NO REALIZA CAMBIO DE REC PRECIO A REC PRECIO REAJ, DADO QUE RM SLO TIENE REC PRECIO Y ESTE SECTOR MANDA
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_41"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, AÑO, TRIM, 41 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA='H' THEN DATO*-1 ELSE DATO END) AS DATO,
       C_SCN, N_SCN, 'PS' AS FUENTE, '0225' AS PROC
INTO #af42_6_41
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA='H' AND C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE='6')
   OR (C_ENTRADA='D' AND C_SCN='AF.42' AND SECTOR=6 AND C_CAGENTE='41')
GROUP BY AÑO, TRIM, C_CUENTA, SECTOR, C_CAGENTE, C_SCN, N_SCN
"""))

af42_6_41 = pd.read_sql(text("SELECT * FROM #af42_6_41"), work_conn)
_log("af42_6_41", af42_6_41)
# Nota: este tramo corta aquí; el DROP de #af42_6_41 y su uso posterior (AF42_6_53) se manejan en el tramo siguiente del notebook


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y EMPRESAS PUBLICAS-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA PTMOS DE LP DEL SECTOR 5101 CON CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. TMB INCORPORA SACAR DATO DE PASIVO DE 5101 CON RM PARA IMPUTAR SOLO EL NETO Y AJUSTA EN PTMOS CON CA BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_5101"))
sql_af42_6_5101 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       5101 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0230' AS PROC
INTO #af42_6_5101
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE='5101')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=5101 AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_6_5101))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_5101"))
sql_af42_321_5101 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0229' AS PROC
INTO #af42_321_5101
FROM #af42_6_5101
"""
work_conn.execute(text(sql_af42_321_5101))


In [ ]:
# APPEND server-side de ambas tablas generadas contra la base y limpieza de las #tmp intermedias
cols_af42_6_5101 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_5101}) SELECT {cols_af42_6_5101} FROM #af42_6_5101"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_6_5101)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_5101}) SELECT {cols_af42_6_5101} FROM #af42_321_5101"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_321_5101)", res.rowcount)
for t in ["#af42_6_5101", "#af42_321_5101"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PTMOS DE LP EN ACTIVO DEL RM CON CONTRAGENTE 37, USANDO INFO DEL PASIVO DEL SECTOR 37 CON CA 6. AJUSTA IMPUTACIÓN EN ACTIVO DEL RM CON CONTRAGENTE 51022
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES Y ADEMÁS IMPUTA EL NETO, YA QUE AHORA RM DICE TENER ACTIVO CON EL SECTOR 37
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_37"))
sql_af42_6_37 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0231' AS PROC
INTO #af42_6_37
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=37 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE='37')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_6_37))


In [ ]:
# AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_53"))
sql_af42_6_53 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0232' AS PROC
INTO #af42_6_53
FROM #af42_6_37
"""
work_conn.execute(text(sql_af42_6_53))


In [ ]:
cols_af42_6_37 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_37}) SELECT {cols_af42_6_37} FROM #af42_6_37"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_6_37)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_37}) SELECT {cols_af42_6_37} FROM #af42_6_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_6_53)", res.rowcount)
for t in ["#af42_6_53", "#af42_6_37"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PTMOS DE CP Y LP EN PASIVO DE OFIS CON CONTRAGENTE RM, USANDO INFO DEL ACTIVO DEL SECTOR 6 CON CA 33901...cambia cod de ca debido a apertura de monto por si desde la cuenta del RM
# CIERRE 2021: NO CAMBIA CONTRAGENTE EN SECTOR 36
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='33211' WHERE C_CAGENTE IN ('33212') AND SECTOR=6 AND C_SCN='AF.41' AND C_ENTRADA='D'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (33212->33211)", res.rowcount)


In [ ]:
# AJUSTA RABOFINANCE
# IMPUTA RABO EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af4_rabo"))
sql_af4_rabo = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '33211' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0233k' AS PROC
INTO #af4_rabo
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=33211 AND T1.C_CAGENTE='6' AND T1.FUENTE='CI'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af4_rabo))


In [ ]:
# AJUSTA DEL RM CON CA36
work_conn.execute(text("DROP TABLE IF EXISTS #af4_36"))
sql_af4_36 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '36' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO)*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0233l' AS PROC
INTO #af4_36
FROM #af4_rabo
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, SECTOR
"""
work_conn.execute(text(sql_af4_36))


In [ ]:
cols_af4_rabo = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af4_rabo}) SELECT {cols_af4_rabo} FROM #af4_rabo"))
_log("APPEND TABLAS.dbo.BD_CTSI (af4_rabo)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af4_rabo}) SELECT {cols_af4_rabo} FROM #af4_36"))
_log("APPEND TABLAS.dbo.BD_CTSI (af4_36)", res.rowcount)
for t in ["#af4_rabo", "#af4_36"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# SELECCIONA DATO DEL RM. CIERRE 2021: NO SE CAMBIA CUENTA REC PRECIO A REC PRECIO REAJ CUANDO SE CONCILIAN LOS DATOS
work_conn.execute(text("DROP TABLE IF EXISTS #af4_33_6"))
sql_af4_33_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       CASE WHEN T1.C_CAGENTE='33901/351' THEN 33901 ELSE TRY_CAST(T1.C_CAGENTE AS float) END AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0233' AS PROC
INTO #af4_33_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN IN('AF.42','AF.41') AND T1.SECTOR=6
  AND T1.C_CAGENTE IN ('36','37','33212','322','33221','351','352','36904','33901/351','33211','3323','3324','33222','361','33231','411')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af4_33_6))


In [ ]:
# SELECCIONA DATO DE SECTORES. CIERRE 2021: NO SE CAMBIA CUENTA REC PRECIO A REC PRECIO REAJ CUANDO SE CONCILIAN LOS DATOS
work_conn.execute(text("DROP TABLE IF EXISTS #af4_sectores_6"))
sql_af4_sectores_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0233' AS PROC
INTO #af4_sectores_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN IN('AF.42','AF.41') AND T1.C_CAGENTE='6'
  AND T1.SECTOR IN (36,37,33212,3321,322,33221,33222,3322,351,36904,33901,352,353,35,33,33211,33231,33232,361,411)
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af4_sectores_6))


In [ ]:
# DATA WORK.AF4_33_6; set WORK.AF4_33_6 WORK.AF4_SECTORES_6; concatenación server-side reemplazando la #tmp original
work_conn.execute(text("DROP TABLE IF EXISTS #af4_33_6_concat"))
sql_af4_33_6_concat = """
SELECT * INTO #af4_33_6_concat FROM #af4_33_6
UNION ALL
SELECT * FROM #af4_sectores_6
"""
work_conn.execute(text(sql_af4_33_6_concat))
work_conn.execute(text("DROP TABLE IF EXISTS #af4_33_6"))
work_conn.execute(text("EXEC sp_rename '#af4_33_6_concat', '#af4_33_6'"))


In [ ]:
# calcula imputación en ptmos pasivo sectores con resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #af4_33_imp"))
sql_af4_33_imp = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       PROC
INTO #af4_33_imp
FROM #af4_33_6
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af4_33_imp))


In [ ]:
# calcula ajuste en ptmos pasivo sectores con bancos, excepto en sector 36 dado que imputación se rebaja de sector resto de empresas con contragente 6. CIERRE 2021: DADO QUE SE INCORPORA RESTO FINANCIERO TODO SE AJUSTA EN BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af4_33_aj"))
sql_af4_33_aj = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO)*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0233b' AS PROC
INTO #af4_33_aj
FROM #af4_33_imp
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af4_33_aj))


In [ ]:
# imputación anterior en ptmos con bancos la rebaja de ptmos de empresas con bancos para equilibrar total activo bancos con su pasivo en ptmos de corto plazo
work_conn.execute(text("DROP TABLE IF EXISTS #af4_51022_aj"))
sql_af4_51022_aj = """
SELECT MONEDA, [AÑO], TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO)*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0233d' AS PROC
INTO #af4_51022_aj
FROM #af4_33_aj
WHERE C_SCN='AF.41' AND C_CAGENTE='321'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af4_51022_aj))


In [ ]:
cols_af4_33 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af4_33}) SELECT {cols_af4_33} FROM #af4_33_imp"))
_log("APPEND TABLAS.dbo.BD_CTSI (af4_33_imp)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af4_33}) SELECT {cols_af4_33} FROM #af4_33_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (af4_33_aj)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af4_33}) SELECT {cols_af4_33} FROM #af4_51022_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (af4_51022_aj)", res.rowcount)
# el SAS también dropea WORK.AF7_36 y WORK.AF7_51022, que este tramo no crea; no se replican por no existir en la sesión
for t in ["#af4_33_6", "#af4_sectores_6", "#af4_33_imp", "#af4_33_aj", "#af4_51022_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA DE PTMOS DE CP entre RM-ACTIVO Y 51021/51022-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 51021 CON CONTRAGENTE 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))
sql_af41_6_51021 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0202a' AS PROC
INTO #af41_6_51021
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('53','51021'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=51021 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=51022 AND T1.MONEDA='D' AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=51022 AND T1.MONEDA='P' AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af41_6_51021))
# el bloque que creaba WORK.AF41_321_51021 desde af41_6_51021 está comentado en el SAS original (anulado); no se traduce


In [ ]:
cols_af41_6_51021 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_51021}) SELECT {cols_af41_6_51021} FROM #af41_6_51021"))
_log("APPEND TABLAS.dbo.BD_CTSI (af41_6_51021)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y EMPRESAS PRIVADAS-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA PTMOS DE LP DEL SECTOR 51022 CON CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. ELIMINA 334 PORQUE AHORA ES DE AUXILIARES. INCORPORA CONTRAGENTE 51021 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_51022"))
sql_af42_6_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0235' AS PROC
INTO #af42_6_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('53','51021'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (51021,51022) AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_6_51022))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_51022"))
sql_af42_321_51022 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0236' AS PROC
INTO #af42_321_51022
FROM #af42_6_51022
"""
work_conn.execute(text(sql_af42_321_51022))


In [ ]:
cols_af42_51022 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_51022}) SELECT {cols_af42_51022} FROM #af42_6_51022"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_6_51022)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_51022}) SELECT {cols_af42_51022} FROM #af42_321_51022"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_321_51022)", res.rowcount)
for t in ["#af42_6_51022", "#af42_321_51022"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y BCCH-PASIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 6 CON CA 31
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_31"))
sql_af42_6_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0238' AS PROC
INTO #af42_6_31
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE='31')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_6_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_6_31"))
sql_af71_6_31 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0239' AS PROC
INTO #af71_6_31
FROM #af42_6_31
"""
work_conn.execute(text(sql_af71_6_31))
af71_6_31 = pd.read_sql(text("SELECT * FROM #af71_6_31"), work_conn)
_log("af71_6_31", af71_6_31)


In [ ]:
cols_af42_af71 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af42_6_31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_6_31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af71_6_31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_6_31)", res.rowcount)
for t in ["#af42_6_31", "#af71_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE GOBIERNO CENTRAL-ACTIVO Y BCCH-PASIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 41 CON CA 31.
# CIERRE 2021: NO HACE CAMBIO EN CUENTA DE REC PRECIO A REC PRECIO REAJ PARA PODER CONCILIAR BIEN LOS INTERESES Y QUE QUEDE TODO AJUSTADO A LO QUE DICE BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31"))
sql_af42_41_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0243' AS PROC
INTO #af42_41_31
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE='31')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_41_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_41_31"))
sql_af71_41_31 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE,
       '0244' AS PROC
INTO #af71_41_31
FROM #af42_41_31
"""
work_conn.execute(text(sql_af71_41_31))
af71_41_31 = pd.read_sql(text("SELECT * FROM #af71_41_31"), work_conn)
_log("af71_41_31", af71_41_31)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af71_41_31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_41_31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af42_41_31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_31)", res.rowcount)
for t in ["#af71_41_31", "#af42_41_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE A DE 3 A 31 EN PTMOS DE LP-PASIVO DEL SECTOR 41
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='31', PROC='0245' WHERE C_CAGENTE='3' AND SECTOR=41 AND C_SCN='AF.42' AND C_ENTRADA='H'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector41 3->31)", res.rowcount)


In [ ]:
# ACTUALIZA CONTRAGENTES A 321 EN EL PASIVO DE PTMOS DE LP DE TODOS LOS SECTORES, EXCEPTO ALGUNOS CA
# EXCEPTO PARA EL SECTOR HOGARES POR LOS PTMOS DE SECURITIZADORAS QUE APARECEN EN 2020, YA QUE COMPRARON CARTERA DE MUTUOS HIPOTECARIOS A CIAS DE SEGUROS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321', PROC='0245b'
WHERE C_SCN='AF.42' AND C_ENTRADA='H'
  AND C_CAGENTE NOT IN ('6','321','31','322','331','334','33211','33212','33222','339011','3324','351','41','411','412','42','51021','413','51022')
  AND SECTOR NOT IN (511)
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI (contragentes -> 321)", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTE A DE 3 A 31 EN PTMOS DE LP-PASIVO DEL SECTOR 42
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='31', PROC='0246' WHERE C_CAGENTE='3' AND SECTOR=42 AND C_SCN='AF.42' AND C_ENTRADA='H'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector42 3->31)", res.rowcount)


In [ ]:
# AJUSTA PTMOS DE LP ENTRE GOBIERNO CENTRAL-PASIVO Y BCCH-ACTIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 41 CON CA 31
# CIERRE 2021. NO SE HACE CAMBIO DE REC PRECIO A REC PRECIO REAJ PARA QUE SE CONCILIEN EXACTAMENTE IGUAL LOS DATOS A LOS DEL CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31_2"))
sql_af42_41_31_2 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0250' AS PROC
INTO #af42_41_31_2
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='41')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE='31')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_41_31_2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_41_31_2"))
sql_af71_41_31_2 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0251' AS PROC
INTO #af71_41_31_2
FROM #af42_41_31_2
"""
work_conn.execute(text(sql_af71_41_31_2))
af71_41_31_2 = pd.read_sql(text("SELECT * FROM #af71_41_31_2"), work_conn)
_log("af71_41_31_2", af71_41_31_2)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af42_41_31_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_31_2)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_af71}) SELECT {cols_af42_af71} FROM #af71_41_31_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_41_31_2)", res.rowcount)
for t in ["#af42_41_31_2", "#af71_41_31_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 333 CON CA 31, USANDO LA INFO DEL ACTIVO DEL SECTOR 31 CON CA 333
work_conn.execute(text("DROP TABLE IF EXISTS #af42_333_31"))
sql_af42_333_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0252' AS PROC
INTO #af42_333_31
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='333'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_333_31))


In [ ]:
cols_simple = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_333_31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_333_31)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_333_31"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 322, USANDO LA INFO DEL ACTIVO DEL SECTOR 322 CON CA 51
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_332_51"))
sql_af42_332_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0255' AS PROC
INTO #af42_332_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=322 AND T1.C_CAGENTE='51'
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_332_51))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_332_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_332_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_332_51"))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR CONTRA PASIVO DE PTMOS DE LP DEL SECTOR 51022 CON CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_321"))
sql_af42_51022_321 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0256' AS PROC
INTO #af42_51022_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=322 AND T1.C_CAGENTE='51'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_51022_321))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_321)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_321"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 3324, USANDO LA INFO DEL ACTIVO DEL SECTOR 3324 CON CA 51022. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_3324_51022"))
sql_af42_3324_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0257' AS PROC
INTO #af42_3324_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=3324 AND T1.C_CAGENTE='51022'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_3324_51022))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_321_"))
sql_af42_51022_321_ = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0258' AS PROC
INTO #af42_51022_321_
FROM #af42_3324_51022
"""
work_conn.execute(text(sql_af42_51022_321_))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_3324_51022"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_3324_51022)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_321_"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_321_)", res.rowcount)
for t in ["#af42_3324_51022", "#af42_51022_321_"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 33211/33212/33222/411, USANDO LA INFO DEL ACTIVO DE LOS SECTORES 33211/33212/33222/411 CON CA 9. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. INCORPORA PASIVO DE EMPRESAS CON ESOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_v_51022"))
sql_af42_v_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '33' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0260' AS PROC
INTO #af42_v_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (33211,33212,33222,411,3324,335)
       AND T1.C_CAGENTE IN ('9','53','51021','51022','5102','51'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (51022,51021,5101)
       AND (SUBSTRING(T1.C_CAGENTE,1,2)='33' OR T1.C_CAGENTE='411'))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_v_51022))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_v"))
sql_af42_51022_v = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0260B' AS PROC
INTO #af42_51022_v
FROM #af42_v_51022
"""
work_conn.execute(text(sql_af42_51022_v))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_v_51022"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_v_51022)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_v"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_v)", res.rowcount)
for t in ["#af42_v_51022", "#af42_51022_v"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 33212 CON CA 339011, USANDO LA INFO DEL ACTIVO DEL SECTOR 339011 CON CA 3/3321. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 33212 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_339011"))
sql_af42_339011 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       33212 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0263' AS PROC
INTO #af42_339011
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=339011 AND T1.C_CAGENTE IN ('3','3321')
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_339011))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_33212"))
sql_af42_33212 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0264' AS PROC
INTO #af42_33212
FROM #af42_339011
"""
work_conn.execute(text(sql_af42_33212))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_339011"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_339011)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_33212"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_33212)", res.rowcount)
for t in ["#af42_339011", "#af42_33212"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 351, USANDO LA INFO DEL ACTIVO DEL SECTOR 351/352 CON CA <>511. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. INCORPORA CALCULO NETO CONSIDERANDO LO QUE EMPRESAS DICE HABER RECIBIDO DE SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_351_352"))
sql_af42_351_352 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '351' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0266' AS PROC
INTO #af42_351_352
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (351,352,353) AND T1.C_CAGENTE NOT LIKE '511')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (51021,51022,5101,5102) AND T1.C_CAGENTE IN ('351','352','353','35'))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_351_352))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_351"))
sql_af42_51022_351 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0267' AS PROC
INTO #af42_51022_351
FROM #af42_351_352
"""
work_conn.execute(text(sql_af42_51022_351))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_351_352"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_351_352)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_351"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_351)", res.rowcount)
for t in ["#af42_351_352", "#af42_51022_351"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 37 CON CA 351, USANDO LA INFO DEL ACTIVO DEL SECTOR 351/352 CON CA AUXILIARES. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR AUX CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af42_35_36"))
sql_af42_35_36 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       36 AS SECTOR,
       '35' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0266' AS PROC
INTO #af42_35_36
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (351,352,353) AND SUBSTRING(T1.C_CAGENTE,1,2)='36')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (351,352,353) AND T1.C_CAGENTE='37')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(6))),1,2)='33' AND T1.C_CAGENTE IN ('351','352','353','35'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=37 AND T1.C_CAGENTE IN ('351','352','353','35'))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_35_36))


In [ ]:
# nota: el SAS reutiliza el nombre WORK.AF42_51022_351 (ya insertado antes); se usa una #tmp distinta para no perder el resultado previo aunque el DROP posterior deja el mismo estado
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_351_b"))
sql_af42_51022_351_b = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0267' AS PROC
INTO #af42_51022_351_b
FROM #af42_35_36
"""
work_conn.execute(text(sql_af42_51022_351_b))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_35_36"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_35_36)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_351_b"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_351 [2a instancia])", res.rowcount)
for t in ["#af42_35_36", "#af42_51022_351_b"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO DE AF42 EN GOBIERNO CON CONTRAGENTE 37, USANDO PASIVO DE SECTOR 37 CA GOBIERNO, AJUSTA EN AF42 ACTIVO GOB CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_37"))
sql_af42_41_37 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
INTO #af42_41_37
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=37 AND T1.C_CAGENTE='41'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_41_37))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_53"))
sql_af42_41_53 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       FUENTE,
       '0271' AS PROC
INTO #af42_41_53
FROM #af42_41_37
"""
work_conn.execute(text(sql_af42_41_53))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_41_37"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_37)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_41_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_53)", res.rowcount)
for t in ["#af42_41_53", "#af42_41_37"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO DE AF42 EN EMPRESAS CON CONTRAGENTE EMPRESAS, USANDO PASIVO DE SECTOR 51021 CA 51021, AJUSTA EN AF7 ACTIVO 51021 CON CA 51021
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51_51"))
sql_af42_51_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(5))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
INTO #af42_51_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=51021 AND T1.C_CAGENTE='51021'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_51_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51_51"))
sql_af7_51_51 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       FUENTE,
       '0271' AS PROC
INTO #af7_51_51
FROM #af42_51_51
"""
work_conn.execute(text(sql_af7_51_51))
af7_51_51 = pd.read_sql(text("SELECT * FROM #af7_51_51"), work_conn)
_log("af7_51_51", af7_51_51)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af7_51_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_51_51)", res.rowcount)
for t in ["#af42_51_51", "#af7_51_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 41, USANDO LA INFO DEL ACTIVO DEL SECTOR 41 CON CA 53/51. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321.
# CIERRE 2021. NO SE CAMBIA REC PRECIO A REC PRECIO REAJ EN EMPRESAS PORQUE SE GENERAN REAJUSTES QUE NO CORRESPONDEN
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_51"))
sql_af42_41_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
INTO #af42_41_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE IN ('51','53'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (51021,51022) AND T1.C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_41_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_41"))
sql_af42_51022_41 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0271' AS PROC
INTO #af42_51022_41
FROM #af42_41_51
"""
work_conn.execute(text(sql_af42_51022_41))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_41_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_51022_41"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_41)", res.rowcount)
for t in ["#af42_41_51", "#af42_51022_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 42, USANDO LA INFO DEL ACTIVO DEL SECTOR 42 CON CA 53/51. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. NO SE CAMBIA REC PRECIO A REC PRECIO REAJ EN EMPRESAS PORQUE SE GENERAN REAJUSTES QUE NO CORRESPONDEN
work_conn.execute(text("DROP TABLE IF EXISTS #af42_42_51"))
sql_af42_42_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0273' AS PROC
INTO #af42_42_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=42 AND T1.C_CAGENTE IN ('51','53')
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_42_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_42_321"))
sql_af42_42_321 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0274' AS PROC
INTO #af42_42_321
FROM #af42_42_51
"""
work_conn.execute(text(sql_af42_42_321))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_42_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_42_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_42_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_42_321)", res.rowcount)
for t in ["#af42_42_51", "#af42_42_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 41 CON CA 31 (EN SECTOR 31 CON CA 53/331/41). RESPETA DATO DEL SECTOR 31 Y AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 41 CA 321
# CIERRE 2021. NO SE HACE CAMBIO EN CUENTA DESDE REC PRECIO REAJ Y REC VOLUMEN A REC PRECIO. SE CONCILIA TAL CUAL PARA AJUSTARSE BIEN A DATO DE CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af42_31_41"))
sql_af42_31_41 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0282' AS PROC
INTO #af42_31_41
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE IN ('331','53','41'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE='31')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_31_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_41"))
sql_af42_321_41 = """
SELECT MONEDA, [AÑO], TRIM,
       41 AS SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN,
       N_SCN,
       'PS' AS FUENTE,
       '0282B' AS PROC
INTO #af42_321_41
FROM #af42_31_41
"""
work_conn.execute(text(sql_af42_321_41))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_321_41"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_321_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_31_41"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_31_41)", res.rowcount)
for t in ["#af42_321_41", "#af42_31_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2025Q3: bloque comentado por completo en el SAS original (WORK.AF42_41_41 / WORK.AF7_41_41) — anulado, no se traduce; se documenta en warnings


In [ ]:
# CIERRE 2025Q4: AJUSTA af42 ENTRE GOBIERNO POR PTMO AL FAPP
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_342"))
sql_af42_41_342 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '342' AS C_CAGENTE,
       'D' AS C_ENTRADA,
       T1.C_CUENTA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0282c' AS PROC
INTO #af42_41_342
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE='342')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=342 AND T1.C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_41_342))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af7_41_53"))
sql_af7_41_53 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       'PS' AS FUENTE,
       '0282c' AS PROC
INTO #af7_41_53
FROM #af42_41_342
"""
work_conn.execute(text(sql_af7_41_53))
af7_41_53 = pd.read_sql(text("SELECT * FROM #af7_41_53"), work_conn)
_log("af7_41_53", af7_41_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af42_41_342"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_41_342 / renombrado a AF42_41_53 en el SAS)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_simple}) SELECT {cols_simple} FROM #af7_41_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_41_53)", res.rowcount)
for t in ["#af42_41_342", "#af7_41_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 51021, USANDO INFO DEL ACTIVO DEL SECTOR 51021 CON CA 9. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
# El SAS corta el tramo aquí, sin PROC SQL siguiente completo dentro de este fragmento: la conciliación de este bloque continúa en el tramo 3
raise NotImplementedError("El SAS del tramo termina en el comentario de 'IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 51021' sin el PROC SQL correspondiente incluido en este fragmento — el CREATE TABLE que lo implementa está en el tramo 3")


In [ ]:
# Imputa en sector 51022 el diferencial de préstamos AF.42 detectado con contraparte sector 51021 / C_CAGENTE '9'
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51021_9"))
sql_af42_51021_9 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       51022 AS SECTOR,
       LTRIM(RTRIM(CAST(t1.SECTOR AS char(11)))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0276' AS PROC
INTO #af42_51021_9
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.42' AND t1.SECTOR = 34 AND t1.C_CAGENTE = '51021'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
# nota: la condición del WHERE original filtra SECTOR=51021 y C_CAGENTE='9' sobre BD_CTSI, se preserva literal
sql_af42_51021_9 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       51022 AS SECTOR,
       LTRIM(RTRIM(CAST(t1.SECTOR AS char(11)))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0276' AS PROC
INTO #af42_51021_9
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.42' AND t1.SECTOR = 51021 AND t1.C_CAGENTE = '9'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_51021_9))
af42_51021_9 = pd.read_sql(text("SELECT * FROM #af42_51021_9"), work_conn)
_log("af42_51021_9", af42_51021_9)


In [ ]:
# Contrapartida en sector 51021/321 con signo invertido del ajuste anterior
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_51021"))
sql_af42_51022_51021 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0277' AS PROC
INTO #af42_51022_51021
FROM #af42_51021_9 t1
"""
work_conn.execute(text(sql_af42_51022_51021))
af42_51022_51021 = pd.read_sql(text("SELECT * FROM #af42_51022_51021"), work_conn)
_log("af42_51022_51021", af42_51022_51021)


In [ ]:
# APPEND server-side de ambas tablas a BD_CTSI (PROC DATASETS APPEND FORCE)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_51021_9"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51021_9)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_51022_51021"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_51021)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51021_9"))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_51021"))


In [ ]:
# CAMBIA CA EN PTMOS DE LP DEL PASIVO DEL SECTOR 5111. POR CIERRE 2020 TMB INCORPORA AF41
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET SECTOR = 51022, PROC = '0278'
WHERE SECTOR = 5111 AND C_SCN IN ('AF.42', 'AF.41') AND C_ENTRADA = 'H'
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 5111->51022)", res.rowcount)


In [ ]:
# CIERRE 2022: IMPUTA PTMOS DE LP ACTIVO DE FONDOS DE PENSIONES EN PASIVO DE EMPRESAS PRIVADAS. RECONCILIA CONTRA AF.7 EMPRESAS CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af42_emp_fp"))
sql_af42_emp_fp = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       51021 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0279' AS PROC
INTO #af42_emp_fp
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.42' AND t1.SECTOR = 34 AND t1.C_CAGENTE = '51021'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_emp_fp))
af42_emp_fp = pd.read_sql(text("SELECT * FROM #af42_emp_fp"), work_conn)
_log("af42_emp_fp", af42_emp_fp)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51021 CON CA 53. REGLA NUEVA CIERRE 2022
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste_emp_fp"))
sql_af7_ajuste_emp_fp = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       t1.FUENTE,
       '0279a' AS PROC
INTO #af7_ajuste_emp_fp
FROM #af42_emp_fp t1
"""
work_conn.execute(text(sql_af7_ajuste_emp_fp))
af7_ajuste_emp_fp = pd.read_sql(text("SELECT * FROM #af7_ajuste_emp_fp"), work_conn)
_log("af7_ajuste_emp_fp", af7_ajuste_emp_fp)


In [ ]:
# APPEND server-side de AF42_EMP_FP y AF7_AJUSTE_EMP_FP
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_emp_fp"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_emp_fp)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_ajuste_emp_fp"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_ajuste_emp_fp)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_emp_fp"))
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste_emp_fp"))


In [ ]:
# CONCILIACIÓN TOTAL PTMOS DE LP - COMPARA ACTIVO VS PASIVO, MANDA DATO DEL ACTIVO, IMPUTA DIF EN PASIVO DE SECTOR 51022 CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af42_total"))
sql_af42_total = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'H' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0175' AS PROC
INTO #af42_total
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.42') OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.42')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_af42_total))
af42_total = pd.read_sql(text("SELECT * FROM #af42_total"), work_conn)
_log("af42_total", af42_total)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51022 CON CA 53. REGLA NUEVA CIERRE 2020
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
sql_af7_ajuste = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       t1.FUENTE,
       t1.PROC
INTO #af7_ajuste
FROM #af42_total t1
"""
work_conn.execute(text(sql_af7_ajuste))
af7_ajuste = pd.read_sql(text("SELECT * FROM #af7_ajuste"), work_conn)
_log("af7_ajuste", af7_ajuste)


In [ ]:
# APPEND server-side de AF42_TOTAL y AF7_AJUSTE (cierra la conciliación de préstamos)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_total"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_total)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_ajuste"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_ajuste)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_total"))
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
# el DELETE FROM TABLAS.BD_CTSI WHERE PROC='0260B' quedó comentado en el SAS original: no se traduce, solo se deja registrado


## S2_06_Bonos

Ajusta y concilia posiciones de bonos (AF.31/AF.32) entre sectores contrapartida, imputando cartera y valor de mercado desde fuentes DCV cuando están disponibles / Ajusta bonos (AF.31, AF.32) a valor de mercado y por imputaciones cruzadas emisor/tenedor (holdings, OFIS, patrimonio separado, corredores), usando el DCV como fuente de verdad y acumulando los ajustes en la tabla de series TABLAS.BD_CTSI / Ajusta y reclasifica tenencias de bonos de largo plazo (AF.31/AF.32) entre sectores y contragentes (ofis, bancos, RM, hogares, empresas) imputando diferencias activo-pasivo y acumulando los ajustes en la tabla consolidada BD_CTSI; cierra con la lectura de la tabla externa DCV pendiente

*confianza: low · verificador: revise · SAS: Secuencia PROC SQL CREATE TABLE / UPDATE / PROC DATASETS APPEND sobre TABLAS.BD_CTSI vía tablas temporales de sesión + PROC SQL CREATE TABLE / UPDATE / DELETE + PROC DATASETS APPEND, encadenados sobre tablas temporales de sesión (#tmp) y TABLAS.BD_CTSI + PROC SQL CREATE TABLE / PROC DATASETS APPEND (múltiples bloques) sobre TABLAS.BD_CTSI vía tablas temporales de sesión*

In [ ]:
# ========= S2_06_Bonos =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con un
# borrado + reinserción deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# ELIMINA TÍTULOS DE CP Y LP CON CA DISTINTOS DE 53 O 6 PARA DEJARLOS TODOS CON CA 53 EN EL PASIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_ca"))
sql_af3_aj_ca = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0424' AS PROC
INTO #af3_aj_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN IN('AF.31','AF.32') AND T1.SECTOR<>6 AND T1.C_CAGENTE NOT IN('53','6','9','54'))
GROUP BY 'P', t1.AÑO, T1.TRIM,
         T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.DATO, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af3_aj_ca))
af3_aj_ca = pd.read_sql(text("SELECT * FROM #af3_aj_ca"), work_conn)
_log("af3_aj_ca", af3_aj_ca)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af3_im_ca"))
sql_af3_im_ca = """
SELECT t1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0425' AS PROC
INTO #af3_im_ca
FROM #af3_aj_ca t1
"""
work_conn.execute(text(sql_af3_im_ca))
af3_im_ca = pd.read_sql(text("SELECT * FROM #af3_im_ca"), work_conn)
_log("af3_im_ca", af3_im_ca)


In [ ]:
cols_af3 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af3}) SELECT {cols_af3} FROM #af3_aj_ca"))
_log("APPEND TABLAS.dbo.BD_CTSI (af3_aj_ca)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af3}) SELECT {cols_af3} FROM #af3_im_ca"))
_log("APPEND TABLAS.dbo.BD_CTSI (af3_im_ca)", res.rowcount)
for t in ["#af3_aj_ca", "#af3_im_ca"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTE BONOS EMITIDOS DE CORTO PLAZO USANDO INFO DCV. para luego llevar a mercado los bonos de cp usando los precios (cambio por cierre 2019)
# INFO AF31 EN CI, EXCEPTO EL DEL RM. CIERRE 2021 EXCEPTO SI CONTRAGENTE EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af31_emisor"))
sql_af31_emisor = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA IN('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0430' AS PROC
INTO #af31_emisor
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN IN('AF.31') AND T1.SECTOR<>6 AND T1.C_CAGENTE NOT IN ('6'))
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         (CASE WHEN t1.C_CUENTA IN('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE t1.C_CUENTA END),
         T1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af31_emisor))


In [ ]:
# INFO DESDE DCV A VALOR PAR
# NOTA DEL ANALISTA: tabla externa DCV (base_af3_total_v2.sas7bdat) no incluida en esta versión del proyecto — no hay ruta relativa ni conexión que la reemplace
raise NotImplementedError("WORK.AF31_DCV_PAS depende del archivo externo DCVRES/base_af3_total_v2.sas7bdat, no disponible en este entorno/proyecto")


In [ ]:
# INFO AF31_EMISOR + AF31_DCV_PAS: depende de la celda anterior (no resuelta)
raise NotImplementedError("AF31_DCV_PAS no se pudo construir (archivo externo no disponible); no se puede aplicar el UPDATE de sectores 3321/3322/36901 ni el APPEND a AF31_EMISOR")


In [ ]:
# CREA DIF ENTRE DCV Y CI DEL AF31 -- depende de AF31_EMISOR ya ajustado con AF31_DCV_PAS (no disponible)
raise NotImplementedError("WORK.AF31_AJUSTE depende de WORK.AF31_EMISOR ya combinado con AF31_DCV_PAS, que no se pudo materializar por falta del archivo DCV externo")


In [ ]:
# AJUSTA DIFERENCIA EN AF32 PARA TODOS LOS SECTORES (ANTES SE HACÍA EN AF7 PARA SECTORES 33 Y 36. SE CAMBIA EN CIERRE 2019)
raise NotImplementedError("WORK.AF31_AJUSTE_2 depende de WORK.AF31_AJUSTE, no disponible por la dependencia rota en AF31_DCV_PAS")


In [ ]:
# APPEND de AF31_AJUSTE y AF31_AJUSTE_2 a TABLAS.BD_CTSI: ambas tablas no se pudieron materializar
raise NotImplementedError("No se pueden hacer los APPEND de AF31_AJUSTE / AF31_AJUSTE_2 a TABLAS.BD_CTSI: las tablas fuente no se materializaron por la dependencia externa faltante (DCV)")


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE CORTO PLAZO EN TODOS LOS SECTORES, EXCEPTO RM
# tabla externa t_aj_mdo_af31.sas7bdat no incluida en esta versión — riesgo señalado por el analista: precio de mercado puede ser nulo/desactualizado
raise NotImplementedError("t_aj_mdo_af31 depende del archivo externo DCVRES/t_aj_mdo_af31.sas7bdat, no disponible en este entorno/proyecto")


In [ ]:
# UPDATE de sectores en t_aj_mdo_af31 (3321->33212, 3322->33222, 36901->369011): depende de la celda anterior
raise NotImplementedError("t_aj_mdo_af31 no está disponible; no se pueden aplicar los UPDATE de SECTOR")


In [ ]:
# VM EN BCE FINAL (AF31_VM_BF) -- depende de t_aj_mdo_af31, no disponible
raise NotImplementedError("WORK.AF31_VM_BF depende del cruce con t_aj_mdo_af31 (archivo externo no disponible)")


In [ ]:
# VM EN BCE INICIO (AF31_VM_BI) -- depende de t_aj_mdo_af31, no disponible
raise NotImplementedError("WORK.AF31_VM_BI depende del cruce con t_aj_mdo_af31 (archivo externo no disponible)")


In [ ]:
# APPEND de AF31_VM_BF y AF31_VM_BI a TABLAS.BD_CTSI: ambas tablas no se pudieron materializar
raise NotImplementedError("No se pueden hacer los APPEND de AF31_VM_BF / AF31_VM_BI a TABLAS.BD_CTSI: dependen de t_aj_mdo_af31 (archivo externo faltante)")


In [ ]:
# CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af31_rp"))
sql_af31_rp = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       (CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0426a' AS PROC
INTO #af31_rp
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN IN ('AF.31')
GROUP BY t1.AÑO, T1.TRIM, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.SECTOR,
         (CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END), T1.C_CUENTA
"""
work_conn.execute(text(sql_af31_rp))
af31_rp = pd.read_sql(text("SELECT * FROM #af31_rp"), work_conn)
_log("af31_rp", af31_rp)


In [ ]:
# AGRUPA ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af31_rp_2"))
sql_af31_rp_2 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af31_rp_2
FROM #af31_rp T1
WHERE T1.DATO<>0
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af31_rp_2))
af31_rp_2 = pd.read_sql(text("SELECT * FROM #af31_rp_2"), work_conn)
_log("af31_rp_2", af31_rp_2)


In [ ]:
cols_af31_rp_2 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_rp_2}) SELECT {cols_af31_rp_2} FROM #af31_rp_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_rp_2)", res.rowcount)
for t in ["#af31_rp_2", "#af31_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERA DE BONOS DE CORTO PLAZO EN TODOS LOS SECTORES DE LA ECON NACIONAL USANDO INFO DCV
# ELIMINA CARTERA AF31 D INFORMADA POR SECTORES, INCLUYENDO SECTOR 3390101. TAMPOCO CONSIDERA CONTRAGENTE 6. cierre 2021: tampoco elimina cartera de resto del mundo con contragente 321 porque no aparece en el DCV
work_conn.execute(text("DROP TABLE IF EXISTS #af31_d_ci"))
sql_af31_d_ci = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0433' AS PROC
INTO #af31_d_ci
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.31' AND T1.C_CAGENTE<>'6'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af31_d_ci))
# elimina cartera de resto del mundo con contragente 321 (no aparece en DCV)
work_conn.execute(text("DELETE FROM #af31_d_ci WHERE SECTOR=6 AND C_CAGENTE='321'"))
af31_d_ci = pd.read_sql(text("SELECT * FROM #af31_d_ci"), work_conn)
_log("af31_d_ci", af31_d_ci)


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN AF32 D
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ajuste"))
sql_af32_ajuste = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE, '0434' AS PROC
INTO #af32_ajuste
FROM #af31_d_ci t1
"""
work_conn.execute(text(sql_af32_ajuste))


In [ ]:
# DCV AF31 TENENCIA. AGREGA TENEDOR 6 PARA IMPUTAR LO DE CORTO DADO QUE NO ESTÁ EN LA CTA DEL RM
# tabla externa t_aj_dcv_af31_act_c21.sas7bdat no incluida en esta versión — riesgo señalado por el analista
raise NotImplementedError("WORK.AF31_D_DCV depende del archivo externo DCVRES/t_aj_dcv_af31_act_c21.sas7bdat, no disponible en este entorno/proyecto")


In [ ]:
# AGRUPA IMPUTACIÓN DEFINITIVA DESDE DCV -- depende de AF31_D_DCV, no disponible
raise NotImplementedError("WORK.AF31_DCV depende de WORK.AF31_D_DCV, no materializado por falta del archivo DCV externo")


In [ ]:
# CIERRE 2021: AJUSTE DEL SECTOR 33 DE DCV SE HACE EN 33901 -- depende de AF31_DCV, no disponible
raise NotImplementedError("UPDATE sobre AF31_DCV no aplicable: AF31_DCV no se materializó (falta AF31_D_DCV)")


In [ ]:
# AJUSTA IMPUTACION DCV DE AF31 CONTRA AF32 EXCEPTO EN TENEDOR 511 -- depende de AF31_DCV, no disponible
raise NotImplementedError("WORK.AF32_D_DCV depende de WORK.AF31_DCV, no materializado por falta del archivo DCV externo")


In [ ]:
# APPEND de AF31_D_CI, AF32_AJUSTE, AF31_DCV, AF32_D_DCV a TABLAS.BD_CTSI
cols_af31_d_ci = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31_d_ci}) SELECT {cols_af31_d_ci} FROM #af31_d_ci"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_d_ci)", res.rowcount)
cols_af32_ajuste = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_ajuste}) SELECT {cols_af32_ajuste} FROM #af32_ajuste"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ajuste)", res.rowcount)
# AF31_DCV y AF32_D_DCV no se materializaron: no hay APPEND posible para ellas
raise NotImplementedError("No se puede hacer APPEND de WORK.AF31_DCV / WORK.AF32_D_DCV a TABLAS.BD_CTSI: no se materializaron por falta del archivo DCV externo t_aj_dcv_af31_act_c21.sas7bdat")


In [ ]:
for t in ["#af31_d_ci", "#af32_ajuste"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACION BONOS DE LP DEL SECTOR 6 Y 31. IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 31 CON CA 6, USANDO INFO DEL ACTIVO DEL SECTOR 6 CON CA 31.
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 31 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_6"))
sql_af32_31_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0451' AS PROC
INTO #af32_31_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='31'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_31_6))
af32_31_6 = pd.read_sql(text("SELECT * FROM #af32_31_6"), work_conn)
_log("af32_31_6", af32_31_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_53"))
sql_af32_31_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0451b' AS PROC
INTO #af32_31_53
FROM #af32_31_6 t1
"""
work_conn.execute(text(sql_af32_31_53))
af32_31_53 = pd.read_sql(text("SELECT * FROM #af32_31_53"), work_conn)
_log("af32_31_53", af32_31_53)


In [ ]:
cols_af32_31 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_31}) SELECT {cols_af32_31} FROM #af32_31_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_31}) SELECT {cols_af32_31} FROM #af32_31_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_31_53)", res.rowcount)
for t in ["#af32_31_53", "#af32_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR 321 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU ACTIVO EL SECTOR 6 CON CA 321. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 321 CON CA 53.
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6"))
sql_af32_321_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0455' AS PROC
INTO #af32_321_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='321') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=321 AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_321_6))
af32_321_6 = pd.read_sql(text("SELECT * FROM #af32_321_6"), work_conn)
_log("af32_321_6", af32_321_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_53"))
sql_af32_321_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0456' AS PROC
INTO #af32_321_53
FROM #af32_321_6 t1
"""
work_conn.execute(text(sql_af32_321_53))
af32_321_53 = pd.read_sql(text("SELECT * FROM #af32_321_53"), work_conn)
_log("af32_321_53", af32_321_53)


In [ ]:
cols_af32_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_321}) SELECT {cols_af32_321} FROM #af32_321_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_321}) SELECT {cols_af32_321} FROM #af32_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_321_53)", res.rowcount)
for t in ["#af32_321_6", "#af32_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP EN SECTOR GOBIERNO, EMPRESAS Y HOLDINGS SIN CA 6. CIERRE 2019, CIERRE 2020
# tabla externa t_aj_mdo_af3.sas7bdat no incluida en esta versión
raise NotImplementedError("p_mcdo_af32 depende del archivo externo DCVRES/t_aj_mdo_af3.sas7bdat, no disponible en este entorno/proyecto")


In [ ]:
# VM EN BCE FINAL SECTOR 41 SIN CA 6 NI 54 -- depende de p_mcdo_af32, no disponible
raise NotImplementedError("WORK.AF32_VM_BF depende del cruce con p_mcdo_af32 (archivo externo no disponible)")


In [ ]:
# VM EN BCE INICIO -- depende de p_mcdo_af32, no disponible
raise NotImplementedError("WORK.AF32_VM_BI depende del cruce con p_mcdo_af32 (archivo externo no disponible)")


In [ ]:
# APPEND de AF32_VM_BF y AF32_VM_BI a TABLAS.BD_CTSI: ambas no se materializaron
raise NotImplementedError("No se pueden hacer los APPEND de AF32_VM_BF / AF32_VM_BI a TABLAS.BD_CTSI: dependen de p_mcdo_af32 (archivo externo faltante)")


In [ ]:
# CIERRE 2020. CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO EN GOBIERNO, EMPRESAS (LO QUE NO ES RECOMPRA) Y HOLDINGS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp"))
sql_af32_rp = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       (CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0460c' AS PROC
INTO #af32_rp
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN IN ('AF.32') AND T1.SECTOR IN (41,37,5101,51021) AND T1.C_CAGENTE NOT IN ('6','54')
GROUP BY t1.AÑO, T1.TRIM, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR,
         (CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END), T1.C_CUENTA
"""
work_conn.execute(text(sql_af32_rp))
af32_rp = pd.read_sql(text("SELECT * FROM #af32_rp"), work_conn)
_log("af32_rp", af32_rp)


In [ ]:
# AGRUPA ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_2"))
sql_af32_rp_2 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_rp_2
FROM #af32_rp T1
WHERE T1.DATO<>0
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af32_rp_2))
af32_rp_2 = pd.read_sql(text("SELECT * FROM #af32_rp_2"), work_conn)
_log("af32_rp_2", af32_rp_2)


In [ ]:
cols_af32_rp_2 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_rp_2}) SELECT {cols_af32_rp_2} FROM #af32_rp_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_rp_2)", res.rowcount)
for t in ["#af32_rp_2", "#af32_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP EN SECTOR GOBIERNO, EMPRESAS Y HOLDINGS CON CA 6 USANDO PRECIO DEL RM. VALORA RECOMPRAS A VALOR DE MERCADO (CA 54). CIERRE 2020
# VM EN SECTOR 41 CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_vm_6"))
sql_af32_vm_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*T2.Precio-T1.DATO) AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'VM' AS FUENTE, '0460a' AS PROC
INTO #af32_vm_6
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.BONOS_EXT_PRECIO T2
    ON t1.AÑO = T2.Año AND T1.TRIM = T2.Trim AND T1.SECTOR = T2.SECTOR AND T1.C_SCN = T2.C_SCN
    AND T1.C_CUENTA = T2.C_CUENTA AND t1.C_CAGENTE = t2.C_CAGENTE
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR IN(41,37,51021,5101,36912) AND T1.C_CAGENTE IN('6','54') AND T1.FUENTE='CI')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_vm_6))
af32_vm_6 = pd.read_sql(text("SELECT * FROM #af32_vm_6"), work_conn)
_log("af32_vm_6", af32_vm_6)


In [ ]:
# REC PRECIO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_6"))
sql_af32_rp_6 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA='Bce Inicio' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0460c' AS PROC
INTO #af32_rp_6
FROM #af32_vm_6 t1
GROUP BY t1.MONEDA, t1.AÑO, T1.TRIM, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_rp_6))
af32_rp_6 = pd.read_sql(text("SELECT * FROM #af32_rp_6"), work_conn)
_log("af32_rp_6", af32_rp_6)


In [ ]:
cols_af32_vm6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_vm6}) SELECT {cols_af32_vm6} FROM #af32_vm_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_vm_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_vm6}) SELECT {cols_af32_vm6} FROM #af32_rp_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_rp_6)", res.rowcount)
for t in ["#af32_vm_6", "#af32_rp_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE DE RECOMPRAS A 9, DESPUES QUE SON LLEVADAS A VALOR DE MERCADO
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='9' WHERE C_CAGENTE='54' AND C_SCN='AF.32' AND C_ENTRADA='H' AND SECTOR IN (5101,51021)"))
_log("UPDATE TABLAS.dbo.BD_CTSI (recompras a 9)", res.rowcount)


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR 41 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU ACTIVO EL SECTOR 6 CON CA 41. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 41 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_6"))
sql_af32_41_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0460' AS PROC
INTO #af32_41_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='41') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=41 AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_41_6))
af32_41_6 = pd.read_sql(text("SELECT * FROM #af32_41_6"), work_conn)
_log("af32_41_6", af32_41_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_53"))
sql_af32_41_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0461' AS PROC
INTO #af32_41_53
FROM #af32_41_6 t1
"""
work_conn.execute(text(sql_af32_41_53))
af32_41_53 = pd.read_sql(text("SELECT * FROM #af32_41_53"), work_conn)
_log("af32_41_53", af32_41_53)


In [ ]:
cols_af32_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_41}) SELECT {cols_af32_41} FROM #af32_41_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_41_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_41}) SELECT {cols_af32_41} FROM #af32_41_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_41_53)", res.rowcount)
for t in ["#af32_41_6", "#af32_41_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 5101 CON CA 6, USANDO INFO DE ACTIVO EL SECTOR 6 CON CA 5101
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 5101 CON CA 53
# CIERRE 2020. AJUSTA DIFERENCIAL PORQUE AHORA EMPRESAS VIENE CON CONTRAGENTE EN LOS BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_5101_6"))
sql_af32_5101_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       5101 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0463' AS PROC
INTO #af32_5101_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='5101') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=5101 AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_5101_6))
af32_5101_6 = pd.read_sql(text("SELECT * FROM #af32_5101_6"), work_conn)
_log("af32_5101_6", af32_5101_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_5101_53"))
sql_af32_5101_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0464' AS PROC
INTO #af32_5101_53
FROM #af32_5101_6 t1
"""
work_conn.execute(text(sql_af32_5101_53))
af32_5101_53 = pd.read_sql(text("SELECT * FROM #af32_5101_53"), work_conn)
_log("af32_5101_53", af32_5101_53)


In [ ]:
cols_af32_5101 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_5101}) SELECT {cols_af32_5101} FROM #af32_5101_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_5101_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_5101}) SELECT {cols_af32_5101} FROM #af32_5101_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_5101_53)", res.rowcount)
for t in ["#af32_5101_6", "#af32_5101_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 6 CON CA 53 CAMBIA CA A SECTOR 51021
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='51021', PROC='0465' WHERE SECTOR=6 AND C_SCN='AF.32' AND C_ENTRADA='D' AND C_CAGENTE='53'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 6 CA 53 -> 51021)", res.rowcount)


In [ ]:
# IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 51021 CON CA 6, USANDO INFO DE ACTIVO EL SECTOR 6 CON CA 51021
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 5101 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_6"))
sql_af32_51021_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0467' AS PROC
INTO #af32_51021_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='51021') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=51021 AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_51021_6))
af32_51021_6 = pd.read_sql(text("SELECT * FROM #af32_51021_6"), work_conn)
_log("af32_51021_6", af32_51021_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_53"))
sql_af32_51021_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0468' AS PROC
INTO #af32_51021_53
FROM #af32_51021_6 t1
"""
work_conn.execute(text(sql_af32_51021_53))
af32_51021_53 = pd.read_sql(text("SELECT * FROM #af32_51021_53"), work_conn)
_log("af32_51021_53", af32_51021_53)


In [ ]:
cols_af32_51021 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_51021}) SELECT {cols_af32_51021} FROM #af32_51021_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51021_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_51021}) SELECT {cols_af32_51021} FROM #af32_51021_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51021_53)", res.rowcount)
for t in ["#af32_51021_6", "#af32_51021_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# RESTA BONOS DE LP EMITIDOS POR LOS HOLDINGS EN EL MERCADO EXTERNO DE LAS EMISIONES DE EMPRESAS EN EL MERCADO EXTERNO.
# SE RESTA LO QUE DICE EL ACTIVO DEL RM QUE LOS SECTORES AUXILIARES DICEN EMITIR FUERA (CIERRE 2020)
# AJUSTA ESTA IMPUTACIÓN EN LOS BONOS DE LP DE EMPRESAS EN EL MERCADO LOCAL (CONTRAGENTE 53). CIERRE 2022Q2: INCORPORA AJUSTE TMB POR EMISIONES EXTERNAS DE SECTOR 36
work_conn.execute(text("DROP TABLE IF EXISTS #af32_37_6"))
sql_af32_37_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0468a' AS PROC
INTO #af32_37_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (37,36912) AND T1.C_CAGENTE='6' AND T1.DATO IS NOT NULL)
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('37','36912') AND T1.DATO IS NOT NULL)
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_37_6))
af32_37_6 = pd.read_sql(text("SELECT * FROM #af32_37_6"), work_conn)
_log("af32_37_6", af32_37_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_53_b"))
sql_af32_51021_53_b = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0468b' AS PROC
INTO #af32_51021_53_b
FROM #af32_37_6 t1
"""
work_conn.execute(text(sql_af32_51021_53_b))
af32_51021_53_ = pd.read_sql(text("SELECT * FROM #af32_51021_53_b"), work_conn)
_log("af32_51021_53_", af32_51021_53_)


In [ ]:
cols_af32_37 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_37}) SELECT {cols_af32_37} FROM #af32_37_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_37_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_37}) SELECT {cols_af32_37} FROM #af32_51021_53_b"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51021_53_)", res.rowcount)
for t in ["#af32_51021_53_b", "#af32_37_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 36904 CON CA 6 CAMBIA CA A SECTOR 53
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='53', PROC='0469' WHERE SECTOR=36904 AND C_SCN='AF.32' AND C_ENTRADA='H' AND C_CAGENTE='6'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 36904 CA 6 -> 53)", res.rowcount)


In [ ]:
# CONCILIACION BONOS DE LP DEL SECTOR 6 Y 31. IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 31 CON CA 6, COMPARANDO CON INFO DEL PASIVO DEL SECTOR 6 CON CA 31.
# RESPETA DATO DEL PASIVO DEL SECTOR 31. AJUSTA IMPUTACIÓN CONTRA AJUSTE DE CONCILIACIÓN EN ACTIVO DE SECTOR 31 CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_6_a"))
sql_af32_31_6_a = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       31 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.SECTOR=6 AND T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio'
             WHEN T1.SECTOR=31 AND T1.C_CUENTA IN('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0473' AS PROC
INTO #af32_31_6_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=31 AND T1.C_CAGENTE='6') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='31')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.SECTOR=6 AND T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio'
               WHEN T1.SECTOR=31 AND T1.C_CUENTA IN('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_31_6_a))
af32_31_6_a = pd.read_sql(text("SELECT * FROM #af32_31_6_a"), work_conn)
_log("af32_31_6_a", af32_31_6_a)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0474' AS PROC
INTO #af71_31_6
FROM #af32_31_6_a t1
"""
work_conn.execute(text(sql_af71_31_6))
af71_31_6 = pd.read_sql(text("SELECT * FROM #af71_31_6"), work_conn)
_log("af71_31_6", af71_31_6)


In [ ]:
cols_af32_31_6a = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_31_6a}) SELECT {cols_af32_31_6a} FROM #af32_31_6_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_31_6_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_31_6a}) SELECT {cols_af32_31_6a} FROM #af71_31_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_31_6)", res.rowcount)
for t in ["#af32_31_6_a", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: AJUSTA TENENCIAS DE AF32 EN BANCOS CON CONTRAGENTE GOBIERNO Y BANCOS USANDO DCV. AJUSTA EN AF32 ACTIVO CONTRAGENTE RESTO DEL MUNDO
# DCV AF32 TENENCIA DE BONOS DE BANCOS CON CA GOB, BCOS Y CENTRAL
# tabla externa base_af3_total_v2.sas7bdat no incluida en esta versión
raise NotImplementedError("WORK.AF32_BCOS_DCV depende del archivo externo DCVRES/base_af3_total_v2.sas7bdat, no disponible en este entorno/proyecto")


In [ ]:
# SELECCIONA ACTIVO AF32 INFORMADA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_bancos"))
sql_af32_bancos = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0478a' AS PROC
INTO #af32_bancos
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (321,322) AND T1.C_CAGENTE IN ('321','322','4','41','31')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_bancos))
af32_bancos = pd.read_sql(text("SELECT * FROM #af32_bancos"), work_conn)
_log("af32_bancos", af32_bancos)


In [ ]:
# JUNTA TABLAS PARA HACER RESTA: APPEND de AF32_BANCOS dentro de AF32_BCOS_DCV -- no aplicable porque AF32_BCOS_DCV no se materializó
raise NotImplementedError("No se puede hacer el APPEND server-side de WORK.AF32_BANCOS dentro de WORK.AF32_BCOS_DCV: AF32_BCOS_DCV no se materializó por falta del archivo DCV externo")


In [ ]:
# CALCULA DELTA A IMPUTAR -- depende de AF32_BCOS_DCV (con AF32_BANCOS ya anexado), no disponible
raise NotImplementedError("WORK.AF32_BANCOS_DELTA depende de WORK.AF32_BCOS_DCV (con AF32_BANCOS anexado), no materializado por falta del archivo DCV externo")


In [ ]:
# AJUSTA CONTRA AF32 EN BANCOS CONTRAGENTE RESTO DEL MUNDO -- depende de AF32_BANCOS_DELTA, no disponible
raise NotImplementedError("WORK.AF32_BANCOS_AJ depende de WORK.AF32_BANCOS_DELTA, no materializado por la dependencia externa faltante (DCV)")


In [ ]:
# AJUSTA CONTRA AF7 EN EMPRESAS CONTRAGENTE 53 -- depende de AF32_BANCOS_DELTA, no disponible
raise NotImplementedError("WORK.AF7_EMP depende de WORK.AF32_BANCOS_DELTA, no materializado por la dependencia externa faltante (DCV)")


In [ ]:
# APPEND de AF32_BANCOS_DELTA, AF32_BANCOS_AJ, AF7_EMP a TABLAS.BD_CTSI: ninguna se materializó
raise NotImplementedError("No se pueden hacer los APPEND de AF32_BANCOS_DELTA / AF32_BANCOS_AJ / AF7_EMP a TABLAS.BD_CTSI: dependen del archivo externo DCV faltante")


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_bancos"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 321 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 321. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE AJUSTE Y DISCREPANCIA DEL SECTOR 321 CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6_a"))
sql_af32_321_6_a = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0478' AS PROC
INTO #af32_321_6_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('321','322')) OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (321,322) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_321_6_a))
af32_321_6_a = pd.read_sql(text("SELECT * FROM #af32_321_6_a"), work_conn)
_log("af32_321_6_a", af32_321_6_a)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_321_6_a"))
sql_af71_321_6_a = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       '0479' AS PROC
INTO #af71_321_6_a
FROM #af32_321_6_a t1
"""
work_conn.execute(text(sql_af71_321_6_a))
af71_321_6_a = pd.read_sql(text("SELECT * FROM #af71_321_6_a"), work_conn)
_log("af71_321_6_a", af71_321_6_a)


In [ ]:
cols_af32_321_6a = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_321_6a}) SELECT {cols_af32_321_6a} FROM #af32_321_6_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_321_6_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_321_6a}) SELECT {cols_af32_321_6a} FROM #af71_321_6_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_321_6_a)", res.rowcount)
for t in ["#af32_321_6_a", "#af71_321_6_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 339011 CON CA 53 CAMBIA CA A SECTOR 6
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='6', PROC='0481' WHERE SECTOR=339011 AND C_SCN='AF.32' AND C_ENTRADA='D' AND C_CAGENTE='53'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 339011 CA 53 -> 6)", res.rowcount)


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 351 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 351 CON ACTIVO DE LOS SECTORES 351/352 CON CA 6. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE BONOS DE LP DEL SECTOR 351 CON CA 53. CIERRE 2021: SE CAMBIA AJUSTE A AF34 CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_6_a"))
sql_af32_351_6_a = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0484' AS PROC
INTO #af32_351_6_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='351') OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN(351,352) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA='Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_351_6_a))
af32_351_6_a = pd.read_sql(text("SELECT * FROM #af32_351_6_a"), work_conn)
_log("af32_351_6_a", af32_351_6_a)


In [ ]:
# imputa derivados en activo de seguros con ca resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_53_a"))
sql_af32_351_53_a = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       'AF.34' AS C_SCN,
       'Derivados financieros' AS N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af32_351_53_a
FROM #af32_351_6_a t1
"""
work_conn.execute(text(sql_af32_351_53_a))
af32_351_53_a = pd.read_sql(text("SELECT * FROM #af32_351_53_a"), work_conn)
_log("af32_351_53_a", af32_351_53_a)


In [ ]:
# imputa derivados en pasivo de resto del mundo con contragente seguros
work_conn.execute(text("DROP TABLE IF EXISTS #af34_6_35"))
sql_af34_6_35 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '35' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af34_6_35
FROM #af32_351_53_a t1
"""
work_conn.execute(text(sql_af34_6_35))
af34_6_35 = pd.read_sql(text("SELECT * FROM #af34_6_35"), work_conn)
_log("af34_6_35", af34_6_35)


In [ ]:
# ajusta en derivados en pasivo de resto del mundo con contragente resto
work_conn.execute(text("DROP TABLE IF EXISTS #af34_6_53"))
sql_af34_6_53 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af34_6_53
FROM #af34_6_35 t1
"""
work_conn.execute(text(sql_af34_6_53))
af34_6_53 = pd.read_sql(text("SELECT * FROM #af34_6_53"), work_conn)
_log("af34_6_53", af34_6_53)


In [ ]:
cols_af32_351 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_351}) SELECT {cols_af32_351} FROM #af32_351_6_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_351_6_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_351}) SELECT {cols_af32_351} FROM #af32_351_53_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_351_53_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_351}) SELECT {cols_af32_351} FROM #af34_6_35"))
_log("APPEND TABLAS.dbo.BD_CTSI (af34_6_35)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_351}) SELECT {cols_af32_351} FROM #af34_6_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af34_6_53)", res.rowcount)
for t in ["#af32_351_6_a", "#af32_351_53_a", "#af34_6_35", "#af34_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 3390101/3390102/339011 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 33901/351 y 33901 CON ACTIVO DE LOS SECTORES 33901/339011 CON CA 6. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE BONOS DE LP DEL SECTOR 33 CON CA 53. CIERRE 2021: PARA AJUSTAR LA IMPUTACIÓN SE CAMBIA AL SECTOR 33901 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_6_a"))
sql_af32_33_6_a = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       33901 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0484b' AS PROC
INTO #af32_33_6_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE IN('33901/351','33901')) OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN(3390101,3390102,339011,33222,33212) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA='Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_33_6_a))
af32_33_6_a = pd.read_sql(text("SELECT * FROM #af32_33_6_a"), work_conn)
_log("af32_33_6_a", af32_33_6_a)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_53_a"))
sql_af32_33_53_a = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485b' AS PROC
INTO #af32_33_53_a
FROM #af32_33_6_a t1
"""
work_conn.execute(text(sql_af32_33_53_a))
af32_33_53_a = pd.read_sql(text("SELECT * FROM #af32_33_53_a"), work_conn)
_log("af32_33_53_a", af32_33_53_a)


In [ ]:
cols_af32_33 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_33}) SELECT {cols_af32_33} FROM #af32_33_6_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_33_6_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_33}) SELECT {cols_af32_33} FROM #af32_33_53_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_33_53_a)", res.rowcount)
for t in ["#af32_33_6_a", "#af32_33_53_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 34 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 34 CON ACTIVO DE LOS SECTORES 34/341 CON CA 6. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE BONOS DE LP DEL SECTOR 34 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_34_6"))
sql_af32_34_6 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       34 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0489' AS PROC
INTO #af32_34_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='34') OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN(34,341) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_34_6))
af32_34_6 = pd.read_sql(text("SELECT * FROM #af32_34_6"), work_conn)
_log("af32_34_6", af32_34_6)


In [ ]:
# CREATE TABLE WORK.AF32_34_53 AS SELECT ... DATO*-1 ... FROM WORK.AF32_34_6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_34_53"))
sql_af32_34_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0490' AS PROC
INTO #af32_34_53
FROM #af32_34_6 T1
"""
work_conn.execute(text(sql_af32_34_53))


In [ ]:
# PROC DATASETS APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AF32_34_6 FORCE / DATA=WORK.AF32_34_53 FORCE (server-side, la fuente es una #tmp)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_34_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_34_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_34_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_34_53)", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE WORK.AF32_34_6; DROP TABLE WORK.AF32_34_53;
for t in ["#af32_34_6", "#af32_34_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE LP EN ACTIVO DEL SECTOR 51022 CON CA 6 USANDO INFO DEL PASIVO DE BONOS DE LP DE SECTOR 6 CON CA 53
# CIERRE 2021: IMPUTA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51022_6"))
sql_af32_51022_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0491' AS PROC
INTO #af32_51022_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='53')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (51021,51022,51,5101) AND T1.C_CAGENTE='6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA,
         CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END,
         T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_ENTRADA='D' THEN 'D' ELSE 'D' END
"""
# NOTA: el GROUP BY del SAS agrupa por CALCULATED MONEDA/C_ENTRADA (constantes) — se colapsa a las columnas reales sin cambiar el resultado
sql_af32_51022_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0491' AS PROC
INTO #af32_51022_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='53')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (51021,51022,51,5101) AND T1.C_CAGENTE='6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_51022_6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51022_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51022_6)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51022_6"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR OFIS+AUX, COMPARANDO LO QUE DICE EL DCV DEL PASIVO DE OFIS+AUX. RESPETA DATO DCV.
# CIERRE 2019 HACE AJUSTE PRIMERO DEL VALOR PAR
# AJUSTA IMPUTACIÓN CONTRA CREDITOS COMERCIALES/PTMOS CA 53 EN EL PASIVO.
# cierre 2021: elimina a holdings de este calculo porque ya se valoro a mercado anteriormente y esta consulta revierte ese efecto
work_conn.execute(text("DROP TABLE IF EXISTS #af32_oa_emision"))
sql_af32_oa_emision = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0493' AS PROC
INTO #af32_oa_emision
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.FUENTE IN ('CI','VM','PS')
  AND T2.C_SI_SCN IN ('S.123','S.124','S.128') AND T1.C_CAGENTE<>'6' AND T1.SECTOR<>37
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR
"""
work_conn.execute(text(sql_af32_oa_emision))


In [ ]:
# INFO DESDE DCV A VALOR PAR — fuente: archivo externo t_aj_dcv_af31_act_c21 / base_af3_total_v2.sas7bdat, no incluido en la BD del proyecto
# NOTA DEL ANALISTA: tabla DCV (t_aj_dcv_af31_act_c21.sas7bdat / base_af3_total_v2.sas7bdat) no incluida en la version actual
raise NotImplementedError("WORK.AF32_OA_DCV depende del archivo externo base_af3_total_v2.sas7bdat (DCV) no incluido en esta migracion; se requiere la ruta/fuente real para completar la traduccion")


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_OA_EMISION DATA=WORK.AF32_OA_DCV FORCE
# depende de af32_oa_dcv, no resuelto arriba
raise NotImplementedError("APPEND de WORK.AF32_OA_DCV sobre WORK.AF32_OA_EMISION no se puede completar: AF32_OA_DCV depende del archivo DCV externo no disponible")


In [ ]:
# CALCULA DIF DCV-CI
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dcv_ci_dif"))
sql_af32_dcv_ci_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0493' AS PROC
INTO #af32_dcv_ci_dif
FROM #af32_oa_emision T1
WHERE T1.SECTOR NOT IN (3324)
-- elimina emisiones de patrimonio separado porque se imputan en otra parte del proceso
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_dcv_ci_dif))


In [ ]:
# AJUSTE EN CRED COM o ptmos según correponda en el sector
work_conn.execute(text("DROP TABLE IF EXISTS #af71_dcv_ci_dif"))
sql_af71_dcv_ci_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       CASE WHEN T1.SECTOR IN (411,33212,33222,339011) THEN '321' WHEN T1.SECTOR=37 THEN '6' ELSE '53' END AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       CASE WHEN T1.SECTOR IN (411,33212,33222,339011,37) THEN 'AF.42' ELSE 'AF.7' END AS C_SCN,
       CASE WHEN T1.SECTOR IN (411,33212,33222,339011,37) THEN N'Préstamos a largo plazo' ELSE N'Créditos comerciales' END AS N_SCN,
       T1.FUENTE, '0494' AS PROC
INTO #af71_dcv_ci_dif
FROM #af32_dcv_ci_dif T1
"""
work_conn.execute(text(sql_af71_dcv_ci_dif))


In [ ]:
# AJUSTE EN PTMOS DE LP PASIVO DE RESTO DE EMPRESAS CON BCOS DEBIDO A AJUSTE 0494 EN LA PARTE QUE SE IMPUTA EN PTMOS BANCARIOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_bcos_dcv"))
sql_af42_bcos_dcv = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494a' AS PROC
INTO #af42_bcos_dcv
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN='AF.42' AND T1.C_CAGENTE='321'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_bcos_dcv))


In [ ]:
# AJUSTE EN PTMOS DE LP ACTIVO DE RESTO DEL MUNDO CON CONTRAGENTE 37 DEBIDO A AJUSTE 0494 EN LA PARTE QUE SE IMPUTA EN PTMOS CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_rm_dcv"))
sql_af42_rm_dcv = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494b' AS PROC
INTO #af42_rm_dcv
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN='AF.42' AND T1.C_CAGENTE='6' AND T1.SECTOR=37
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_rm_dcv))


In [ ]:
# AJUSTE EN PTMOS DE LP ACTIVO DE RESTO DEL MUNDO CON CONTRAGENTE 53 DEBIDO A AJUSTE 0494b
work_conn.execute(text("DROP TABLE IF EXISTS #af42_rm_resto"))
sql_af42_rm_resto = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494c' AS PROC
INTO #af42_rm_resto
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN='AF.42' AND T1.C_CAGENTE='6' AND T1.SECTOR=37
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_rm_resto))


In [ ]:
# AJUSTE EN PTMOS DE LP PASIVO DE RESTO DE EMPRESAS CON EL RESTO DEL MUNDO A AJUSTE 0494
work_conn.execute(text("DROP TABLE IF EXISTS #af42_resto_rm"))
sql_af42_resto_rm = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494d' AS PROC
INTO #af42_resto_rm
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN='AF.42' AND T1.C_CAGENTE='6' AND T1.SECTOR=37
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_resto_rm))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_dcv_ci_dif"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_dcv_ci_dif)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_dcv_ci_dif"))
_log("APPEND TABLAS.dbo.BD_CTSI (af71_dcv_ci_dif)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_bcos_dcv"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_bcos_dcv)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_rm_dcv"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_rm_dcv)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_rm_resto"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_rm_resto)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_resto_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_resto_rm)", res.rowcount)


In [ ]:
for t in ["#af32_dcv_ci_dif", "#af71_dcv_ci_dif", "#af42_bcos_dcv", "#af42_rm_dcv", "#af42_rm_resto", "#af42_resto_rm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA EMISIONES DE OFIS EN PODER DE NO RESIDENTES COMPARANDO CON LO QUE DICE EL RM. RESPETA DATO DEL RM. AJUSTA EN PTMOS DE LP CA 321 EN PASIVO DE OFIS
# CALCULA DIFERENCIA A IMPUTAR EN BONOS DE LP - CIERRE 2021: SE DEJA EN EL SECTOR 33 POR APERTURA DE LOS FONDOS NO MONEY MARKET
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_6"))
sql_af32_33_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       33 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0495' AS PROC
INTO #af32_33_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE LIKE '33%')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND TRY_CAST(SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(9))),1,2) AS int)=33 AND T1.C_CAGENTE='6' AND T1.SECTOR NOT IN (334,331))
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR
"""
work_conn.execute(text(sql_af32_33_6))


In [ ]:
# AJUSTA EN PTMOS DE LP CA BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_33_321"))
sql_af42_33_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       'AF.42' AS C_SCN,
       N'Préstamos a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0495a' AS PROC
INTO #af42_33_321
FROM #af32_33_6 T1
"""
work_conn.execute(text(sql_af42_33_321))


In [ ]:
# AJUSTA EN PTMOS DE LP DE RESTO DE EMPRESAS CA BANCOS PARA EQUILIBRAR LOS PTMOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51_321"))
sql_af42_51_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO AS DATO,
       'AF.42' AS C_SCN,
       N'Préstamos a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0495b' AS PROC
INTO #af42_51_321
FROM #af32_33_6 T1
"""
work_conn.execute(text(sql_af42_51_321))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_33_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_33_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_33_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_33_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_51_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51_321)", res.rowcount)
for t in ["#af32_33_6", "#af42_33_321", "#af42_51_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO AF31 DEL RESTO CON CONTRAGENTES BANCOS EN PASIVO DE BANCOS CON RM. AJUSTA EN PASIVO AF32 DE BANCOS CON 53
# selecciona activo rm con bcos
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_321"))
sql_af31_6_321 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0435a' AS PROC
INTO #af31_6_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.31' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('321','322','32'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.31' AND T1.SECTOR IN (321,322) AND T1.C_CAGENTE IN ('6'))
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_321))


In [ ]:
# ajusta en af32 pasivo de bancos con ca 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6"))
sql_af32_321_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO*-1 AS DATO,
       'AF.32' AS C_SCN,
       N'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_321_6
FROM #af31_6_321 T1
"""
work_conn.execute(text(sql_af32_321_6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_6_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_6_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_321_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_321_6)", res.rowcount)
for t in ["#af31_6_321", "#af32_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERA DE BONOS DE LARGO PLAZO EN EL SECTOR AUXILIARES FINANCIEROS USANDO INFO DCV
# SELECCIONA ACTIVO AF32 INFORMADA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_aux"))
sql_af32_aux = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436d' AS PROC
INTO #af32_aux
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_SCN='S.124'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_aux))


In [ ]:
# DCV AF32 TENENCIA DE AUXILIARES SIN EMISOR RM — fuente: archivo externo t_aj_dcv_af32_act_ao.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_D_DCV depende del archivo externo t_aj_dcv_af32_act_ao.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_AUX DATA=WORK.AF32_D_DCV FORCE — depende de af32_d_dcv no resuelto
raise NotImplementedError("APPEND de WORK.AF32_D_DCV sobre WORK.AF32_AUX no se puede completar: AF32_D_DCV depende del archivo DCV externo no disponible")


In [ ]:
# AGRUPA DIF DCV-CI (depende del resultado combinado af32_aux + af32_d_dcv, no disponible)
raise NotImplementedError("WORK.AF32_DIF (primera ocurrencia) depende de WORK.AF32_AUX ya combinada con AF32_D_DCV, que no se pudo materializar por falta del archivo DCV externo")


In [ ]:
# AJUSTE EN CREDITOS COMERCIALES (depende de af32_dif no disponible)
raise NotImplementedError("WORK.AF7_AJ_OA depende de WORK.AF32_DIF (bloque DCV no disponible)")


In [ ]:
# ANEXA AJUSTE BONOS LP EN AUX / ANEXA AJUSTE cred com EN AUX — dependen de los bloques anteriores no resueltos
raise NotImplementedError("APPEND de WORK.AF32_DIF y WORK.AF7_AJ_OA a TABLAS.BD_CTSI no se puede completar: dependen del archivo DCV externo no disponible")


In [ ]:
# IMPUTA DIFERENCIAL DE BONOS SECURITIZADOS EN RESTO DE EMPRESAS USANDO DCV MENOS TODO LO QUE SE ENCUENTRA EN CARTERA DE LOS INVERSIONISTAS
# SELECCIONA ACTIVO DE SECTORES EN PATRIMONIO SEPARADO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_pat_tot"))
sql_af32_pat_tot = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '3324' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436j' AS PROC
INTO #af32_pat_tot
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.C_CAGENTE IN ('332','3323','3324')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_pat_tot))


In [ ]:
# DCV AF32 EMISION TOTAL PAT SEPARADO — fuente: archivo externo base_af3_total_v2.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_D_DCV (segunda ocurrencia, pat separado) depende del archivo externo base_af3_total_v2.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_PAT_TOT DATA=WORK.AF32_D_DCV FORCE — depende del bloque anterior no resuelto
raise NotImplementedError("APPEND de WORK.AF32_D_DCV sobre WORK.AF32_PAT_TOT no se puede completar: falta el archivo DCV externo")


In [ ]:
# AGRUPA DIF DCV-CI (segunda ocurrencia)
raise NotImplementedError("WORK.AF32_DIF (segunda ocurrencia, pat separado) depende de WORK.AF32_PAT_TOT combinada con AF32_D_DCV, no disponible")


In [ ]:
# AJUSTE EN cred com DE EMPRESAS
raise NotImplementedError("WORK.AF7_EMP depende de WORK.AF32_DIF (bloque DCV pat separado no disponible)")


In [ ]:
# ANEXA AJUSTE BONOS LP EN AUX / ANEXA AJUSTE cred com EN AUX (segunda ocurrencia)
raise NotImplementedError("APPEND de WORK.AF32_DIF y WORK.AF7_EMP a TABLAS.BD_CTSI no se puede completar: dependen del archivo DCV externo no disponible")


In [ ]:
# IMPUTA BONOS DE LARGO PLAZO EMITIDOS POR PATRIMONIO SEPARADO, USANDO INFO DE LA CARTERA DE LOS AUX FINANCIEROS Y EMPRESAS
# TAMBIÉN GENERA ACTIVO DE PRESTAMOS DE LP (2/3) Y CRED COMERCIALES (1/3) USANDO IMPUTACIÓN DE BONOS
# SELECCIONA AF32 DE PAT SEPARADO DESDE AUX FIN
work_conn.execute(text("DROP TABLE IF EXISTS #af32_pat_aux"))
sql_af32_pat_aux = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       3324 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436f' AS PROC
INTO #af32_pat_aux
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (36,51022) AND T1.C_CAGENTE='3324'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_pat_aux))


In [ ]:
# IMPUTA ACTIVO DE PRÉSTAMOS DE LP
work_conn.execute(text("DROP TABLE IF EXISTS #af42_pat"))
sql_af42_pat = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO*2.0/3) AS DATO,
       'AF.42' AS C_SCN,
       N'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0436g' AS PROC
INTO #af42_pat
FROM #af32_pat_aux T1
"""
work_conn.execute(text(sql_af42_pat))


In [ ]:
# IMPUTA ACTIVO DE CRÉDITOS COMERCIALES
work_conn.execute(text("DROP TABLE IF EXISTS #af7_pat"))
sql_af7_pat = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO*1.0/3) AS DATO,
       'AF.7' AS C_SCN,
       N'Créditos comerciales' AS N_SCN,
       'PS' AS FUENTE, '0436h' AS PROC
INTO #af7_pat
FROM #af32_pat_aux T1
"""
work_conn.execute(text(sql_af7_pat))


In [ ]:
# IMPUTA PASIVO DE PRÉSTAMOS DE LP EN SECTOR 51022 CON CA 3324
work_conn.execute(text("DROP TABLE IF EXISTS #af42_resto"))
sql_af42_resto = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '3324' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0436i' AS PROC
INTO #af42_resto
FROM #af42_pat T1
"""
work_conn.execute(text(sql_af42_resto))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_pat_aux"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_pat_aux)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_pat"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_pat)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_pat"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_pat)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_resto"))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_resto)", res.rowcount)
for t in ["#af32_pat_aux", "#af42_pat", "#af7_pat", "#af42_resto"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP, EXCEPTO EN SECTORES 6 Y 3324 Y CA 6
# El SAS deja comentado (anulado) un UPDATE manual de precios 2019 sobre t_aj_mdo_af3 — no se traduce, queda documentado
# fuente: archivo externo t_aj_mdo_af3.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.P_MCDO_AF32 depende del archivo externo t_aj_mdo_af3.sas7bdat no incluido en esta migracion")


In [ ]:
# UPDATE p_mcdo_af32 SET SECTOR=339011/369011 — depende de p_mcdo_af32 no resuelto
raise NotImplementedError("UPDATE de WORK.P_MCDO_AF32 (sector 339011/369011) no se puede completar: falta el archivo t_aj_mdo_af3.sas7bdat")


In [ ]:
# VM EN BCE FINAL, YA NO SE HACE EN SECTOR 41, 5101, 51021 y 37 PORQUE SE HACE ANTES — depende de p_mcdo_af32 no resuelto
raise NotImplementedError("WORK.AF32_VM_BF depende de WORK.P_MCDO_AF32, que no se pudo materializar por falta del archivo externo t_aj_mdo_af3.sas7bdat")


In [ ]:
# VM EN BCE INICIO — depende de p_mcdo_af32 no resuelto
raise NotImplementedError("WORK.AF32_VM_BI depende de WORK.P_MCDO_AF32, que no se pudo materializar por falta del archivo externo t_aj_mdo_af3.sas7bdat")


In [ ]:
# APPEND de WORK.AF32_VM_BF y WORK.AF32_VM_BI — dependen de los bloques anteriores no resueltos
raise NotImplementedError("APPEND de WORK.AF32_VM_BF y WORK.AF32_VM_BI a TABLAS.BD_CTSI no se puede completar: dependen del archivo t_aj_mdo_af3.sas7bdat no disponible")


In [ ]:
# CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp"))
sql_af32_rp = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       N'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0403' AS PROC
INTO #af32_rp
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN IN ('AF.31','AF.32')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR,
         CASE WHEN T1.C_CUENTA<>'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END
"""
work_conn.execute(text(sql_af32_rp))


In [ ]:
# AGRUPA ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_2"))
sql_af32_rp_2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_rp_2
FROM #af32_rp T1
WHERE T1.DATO<>0
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af32_rp_2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_rp_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_rp_2)", res.rowcount)
for t in ["#af32_rp_2", "#af32_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTE TOTAL BONOS COMPARANDO TOTAL EMITIDO CON TOTAL ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_tot"))
sql_af3_aj_tot = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END AS DATO,
       T1.C_SCN,
       N'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0309' AS PROC
INTO #af3_aj_tot
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.32')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN,
         CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END
"""
work_conn.execute(text(sql_af3_aj_tot))


In [ ]:
# agrupa af3_aj_tot
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_tot2"))
sql_af3_aj_tot2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af3_aj_tot2
FROM #af3_aj_tot T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.SECTOR, T1.C_CAGENTE, T1.PROC
"""
work_conn.execute(text(sql_af3_aj_tot2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af3_aj_tot2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af3_aj_tot2)", res.rowcount)
for t in ["#af3_aj_tot", "#af3_aj_tot2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN BONOS DE LP DEL ACTIVO DEL SECTOR 51 CON CA DISTINTOS DE 6 Y 53 PARA CAMBIAR CA A TODOS A 53
# IMPUTA DATA DE SECTOR 51 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51"))
sql_af32_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       '53' AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_03' AS PROC, 'PS' AS FUENTE
INTO #af32_51
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_publ=51 AND (T3.C_SI_publ <> 6 AND T3.C_SI_publ <> 53)
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ,
         CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_51))


In [ ]:
# ELIMINA DATA DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_del"))
sql_af32_51_del = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(T3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_02' AS PROC
INTO #af32_51_del
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_publ=51 AND (T3.C_SI_publ <> 6 AND T3.C_SI_publ <> 53)
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ,
         CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_51_del))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_del"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_del)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51)", res.rowcount)
for t in ["#af32_51_del", "#af32_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LARGO PLAZO DE BONOS EMITIDOS POR HOLDINGS Y CASAS MATRICES EN TODOS LOS SECTORES, EXCEPTO AUXILIARES FINANCIEROS PORQUE YA SE IMPUTO
# ESTA IMPUTACION SE RESTA DE LAS TENENCIAS DE LOS SECTORES RESPECTIVOS DE BONOS DE EMPRESAS
# fuente: archivo externo t_aj_dcv_af32_act_hc.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_D_HC depende del archivo externo t_aj_dcv_af32_act_hc.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# UPDATE WORK.AF32_D_HC SET SECTOR=51 — depende de af32_d_hc no resuelto
raise NotImplementedError("UPDATE de WORK.AF32_D_HC no se puede completar: falta el archivo t_aj_dcv_af32_act_hc.sas7bdat")


In [ ]:
# SELECCIONA TENENCIAS DE BONOS DE LP DE EMISIONES DE AUX FIN
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca36"))
sql_af32_ca36 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_010' AS PROC
INTO #af32_ca36
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND (T1.C_CAGENTE LIKE '%36%' OR T1.C_CAGENTE LIKE '%37%') AND T1.SECTOR NOT IN (36,36904,36906,6) AND T1.[AÑO]>2002
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE,
         CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_ca36))


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_D_HC DATA=WORK.AF32_CA36 FORCE — depende de af32_d_hc no resuelto
raise NotImplementedError("APPEND de WORK.AF32_CA36 sobre WORK.AF32_D_HC no se puede completar: falta el archivo t_aj_dcv_af32_act_hc.sas7bdat")


In [ ]:
# AGRUPA TABLA PARA CALCULAR DIFERENCIA A IMPUTAR. MANDA DCV — depende de la union af32_d_hc+af32_ca36 no disponible
raise NotImplementedError("WORK.AF32_D_AUX depende de WORK.AF32_D_HC (union con AF32_CA36), no disponible por falta del archivo DCV externo")


In [ ]:
# UPDATE AF32_D_AUX SET SECTOR=33901 WHERE SECTOR=33 — depende de af32_d_aux no resuelto
raise NotImplementedError("UPDATE de WORK.AF32_D_AUX no se puede completar: depende del bloque anterior no disponible")


In [ ]:
# AJUSTA EN EMISIONES DE EMPRESAS — depende de af32_d_aux no resuelto
raise NotImplementedError("WORK.AF32_AUX_AJUSTE depende de WORK.AF32_D_AUX, no disponible por falta del archivo DCV externo")


In [ ]:
# ANEXA AJUSTE bonos LP y AJUSTE EN EMISIONES — dependen de bloques anteriores no resueltos
raise NotImplementedError("APPEND de WORK.AF32_D_AUX y WORK.AF32_AUX_AJUSTE a TABLAS.BD_CTSI no se puede completar: dependen del archivo t_aj_dcv_af32_act_hc.sas7bdat no disponible")


In [ ]:
# IMPUTA CARTERA DE BONOS EMITIDOS POR CORREDORES DE BOLSA EN EL SECTOR OFIS USANDO INFO DCV
# SELECCIONA ACTIVO AF32 INFORMADA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofis"))
sql_af32_ofis = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_011a' AS PROC
INTO #af32_ofis
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_SCN='S.123' AND T1.C_CAGENTE='36904'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_ofis))


In [ ]:
# DCV AF32 TENENCIA DE OFIS DE CORREDORES — fuente: archivo externo t_aj_dcv_af32_act_ao.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_D_DCV_ depende del archivo externo t_aj_dcv_af32_act_ao.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_OFIS DATA=WORK.AF32_D_DCV_ FORCE — depende del bloque anterior no resuelto
raise NotImplementedError("APPEND de WORK.AF32_D_DCV_ sobre WORK.AF32_OFIS no se puede completar: falta el archivo t_aj_dcv_af32_act_ao.sas7bdat")


In [ ]:
# AGRUPA DIF DCV-CI (WORK.AF32_DIF, tercera ocurrencia — corredores/OFIS)
raise NotImplementedError("WORK.AF32_DIF (tercera ocurrencia, corredores/OFIS) depende de WORK.AF32_OFIS combinada con AF32_D_DCV_, no disponible")


In [ ]:
# AJUSTE EN AF32 CON CA 51 — depende de af32_dif no resuelto
raise NotImplementedError("WORK.AF32_AJ_OFIS depende de WORK.AF32_DIF (bloque DCV corredores no disponible)")


In [ ]:
# ANEXA AJUSTE — dependen de bloques anteriores no resueltos
raise NotImplementedError("APPEND de WORK.AF32_DIF y WORK.AF32_AJ_OFIS a TABLAS.BD_CTSI no se puede completar: dependen del archivo t_aj_dcv_af32_act_ao.sas7bdat no disponible")


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LARGO PLAZO DE BONOS EMITIDOS POR OFIS EN BANCOS, SEGUROS Y PENSIONES
# ESTA IMPUTACION SE RESTA DE LAS TENENCIAS DE LOS SECTORES RESPECTIVOS DE BONOS DE EMPRESAS
# fuente: archivo externo t_aj_dcv_af32_act_hc.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_OFIS_DCV depende del archivo externo t_aj_dcv_af32_act_hc.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# DCV AF32 TENENCIA DE BONOS DE FONDOS DE INVERSION EN ACTIVOS DE BANCOS, SEGUROS Y PENSIONES — fuente: base_af3_total_v2.sas7bdat no incluido
raise NotImplementedError("WORK.AF32_OFIS_DCV_2 depende del archivo externo base_af3_total_v2.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# DATA AF32_OFIS_DCV; SET AF32_OFIS_DCV AF32_OFIS_DCV_2; — dependen de los dos bloques anteriores no resueltos
raise NotImplementedError("Concatenacion de WORK.AF32_OFIS_DCV con WORK.AF32_OFIS_DCV_2 no se puede completar: ambos dependen de archivos DCV externos no disponibles")


In [ ]:
# UPDATE WORK.AF32_OFIS_DCV SET SECTOR=51 — depende del bloque anterior no resuelto
raise NotImplementedError("UPDATE de WORK.AF32_OFIS_DCV no se puede completar: depende de los archivos DCV externos no disponibles")


In [ ]:
# SELECCIONA TENENCIAS DE BONOS DE LP DE EMISIONES DE OFIS EN BANCOS, SEGUROS Y PENSIONES, EXCEPTO TENENCIAS DE PAT SEPARADO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca33"))
sql_af32_ca33 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_014' AS PROC
INTO #af32_ca33
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND (T1.C_CAGENTE LIKE '%33%' OR T1.C_CAGENTE LIKE '%411%')
  AND T1.SECTOR IN (32,321,351,352,35,353,34,341,322) AND T1.[AÑO]>2002
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE,
         CASE WHEN T1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_ca33))


In [ ]:
# PROC SQL; DELETE FROM WORK.AF32_CA33 WHERE C_CAGENTE IN ('332','3324','3323');
res = work_conn.execute(text("DELETE FROM #af32_ca33 WHERE C_CAGENTE IN ('332','3324','3323')"))
_log("DELETE #af32_ca33", res.rowcount)


In [ ]:
# PROC DATASETS APPEND BASE=WORK.AF32_OFIS_DCV DATA=WORK.AF32_CA33 FORCE — depende de af32_ofis_dcv no resuelto
raise NotImplementedError("APPEND de WORK.AF32_CA33 sobre WORK.AF32_OFIS_DCV no se puede completar: falta materializar AF32_OFIS_DCV (archivos DCV externos)")


In [ ]:
# AGRUPA TABLA PARA CALCULAR DIFERENCIA A IMPUTAR. MANDA DCV — WORK.AF32_D_OFI
raise NotImplementedError("WORK.AF32_D_OFI depende de WORK.AF32_OFIS_DCV combinada con AF32_CA33, no disponible por falta de archivos DCV externos")


In [ ]:
# AJUSTA EN EMISIONES DE EMPRESAS — depende de af32_d_ofi no resuelto
raise NotImplementedError("WORK.AF32_OFI_AJUSTE depende de WORK.AF32_D_OFI, no disponible")


In [ ]:
# ANEXA — dependen de bloques anteriores no resueltos
raise NotImplementedError("APPEND de WORK.AF32_D_OFI y WORK.AF32_OFI_AJUSTE a TABLAS.BD_CTSI no se puede completar: dependen de archivos DCV externos no disponibles")


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LP DE OFIS CON CONTRAGENTE BANCOS/GOBIERNO/BCENTRAL/OFIS USANDO INFO DCV. AJUSTA IMPUTACION EN LAS TENENCIAS DE OFIS CON EMPRESAS
# CIERRE 2021: EL AJUSTE SE IMPUTA EN SECTOR FMNM (SECTOR 33901)PORQUE AHORA SE PUBLICA SEPARADO DE OFIS.
# fuente: archivo externo t_aj_dcv_af32_act_ofis.sas7bdat no incluido en la migracion
raise NotImplementedError("WORK.AF32_OFIS_BCOS_DCV depende del archivo externo t_aj_dcv_af32_act_ofis.sas7bdat (DCV) no incluido en esta migracion")


In [ ]:
# DELETE FROM WORK.AF32_OFIS_BCOS_DCV WHERE C_CAGENTE IN ('3324') — depende del bloque anterior no resuelto
raise NotImplementedError("DELETE sobre WORK.AF32_OFIS_BCOS_DCV no se puede completar: depende del archivo t_aj_dcv_af32_act_ofis.sas7bdat no disponible")


In [ ]:
# SELECCIONA TENENCIAS DE BONOS DE LP DE OFIS CON BANCOS/GOBIERNO/BCENTRAL/OFIS
# CIERRE 2021: AGREGA CONTRAGENTE 411 PORQUE ESTABA QUEDANDO FUERA DE LA REGLA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33ca321"))
sql_af32_33ca321 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       33901 AS SECTOR,
       T1.C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_016' AS PROC
INTO #af32_33ca321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32'
  AND (T1.C_CAGENTE IN ('321','3') OR T1.C_CAGENTE LIKE '31' OR T1.C_CAGENTE LIKE '33%' OR T1.C_CAGENTE = '411')
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))), 1, 2) = '33'
  AND T1.SECTOR NOT IN (334, 331, 3324)
  AND T1.[AÑO] > 2002
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_33ca321))


In [ ]:
work_conn.execute(text("DELETE FROM #af32_33ca321 WHERE C_CAGENTE = :cagente"), {"cagente": "3324"})


In [ ]:
# PROC DATASETS APPEND: JUNTA TABLAS PARA SACAR DIFERENCIA (sobre #af32_ofis_bcos_dcv creado en tramo anterior)
cols_af32_ofis_bcos_dcv = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_ofis_bcos_dcv = f"""
INSERT INTO #af32_ofis_bcos_dcv ({cols_af32_ofis_bcos_dcv})
SELECT {cols_af32_ofis_bcos_dcv}
FROM #af32_33ca321
"""
res = work_conn.execute(text(sql_append_ofis_bcos_dcv))
_log("APPEND #af32_ofis_bcos_dcv", res.rowcount)


In [ ]:
# CALCULA DIF A IMPUTAN EN OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofis_bcos_dif"))
sql_af32_ofis_bcos_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_ofis_bcos_dif
FROM #af32_ofis_bcos_dcv T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af32_ofis_bcos_dif))


In [ ]:
# AJUSTA EN EMISIONES DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofi_aj"))
sql_af32_ofi_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '51' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0498_016' AS PROC
INTO #af32_ofi_aj
FROM #af32_ofis_bcos_dif t1
"""
work_conn.execute(text(sql_af32_ofi_aj))


In [ ]:
# APPEND BD_CTSI de AF32_OFIS_BCOS_DIF y AF32_OFI_AJ (server-side desde #tmp)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ofis_bcos_dif"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ofis_bcos_dif)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ofi_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ofi_aj)", res.rowcount)


In [ ]:
for t in ["#af32_ofis_bcos_dif", "#af32_ofi_aj", "#af32_ofis_bcos_dcv", "#af32_33ca321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CALCULA DIFERENCIA ENTRE BONOS DE LP EMITIDOS POR SECTOR 37 CON CA 6 Y TENENCIA DE SECTOR 6 CON CA 37.
# RESPETA DATO DEL SECTOR 37, POR LO TANTO IMPUTA DIF EN RM CON CA 37
# CIERRE 2022Q2 INCORPORA EN AJUSTE LAS EMISIONES DEL SECTOR 36912 EN EL EXTERIOR
# AJUSTA IMPUTACIÓN DE SECTOR RM CON CA 51021
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_37"))
sql_af32_6_37 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR, '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_6_37
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('37','36912'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (37, 36912) AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_6_37))


In [ ]:
# AJUSTA IMPUTACIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_51021"))
sql_af32_6_51021 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, '51021' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_13' AS PROC, T1.FUENTE
INTO #af32_6_51021
FROM #af32_6_37 t1
"""
work_conn.execute(text(sql_af32_6_51021))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_6_37"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_6_37)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_6_51021"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_6_51021)", res.rowcount)
for t in ["#af32_6_51021", "#af32_6_37"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2020. IMPUTA ACTIVO DEL RM CON CA 36 DE BONOS DE LP EN PASIVO DEL SECTOR 36 CON CA 6.
# ESTE MONTO SE AJUSTA CREANDO UN ACTIVO AF7 EN SECTOR 36 CON CA 51022
# IMPUTA PASIVO AF32 36 CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af32_36_6"))
sql_af32_36_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       36 AS SECTOR, '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_36_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '36' AND T1.FUENTE = 'CI')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_36_6))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR CREANDO ACTIVO AF7 36 CA 51022, PORQUE ES SECTOR NO MEDIDO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_36_51"))
sql_af7_36_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       36 AS SECTOR, '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af7_36_51
FROM #af32_36_6 t1
"""
work_conn.execute(text(sql_af7_36_51))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF32 EMPRESAS CON CA EMPRESAS, PORQUE ES SECTOR NO MEDIDO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_2"))
sql_af32_51_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR, '2' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_51_2
FROM #af32_36_6 t1
"""
work_conn.execute(text(sql_af32_51_2))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_36_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_36_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_36_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_36_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_2)", res.rowcount)
for t in ["#af32_36_6", "#af7_36_51", "#af32_51_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA TOTAL EMITIDO CON LAS TENENCIAS DE LOS SECTORES EMISORES 31/32/33/36/4/51
# SELECCIONA CARTERAS CON CA 31/32/33/36/4/51 - CIERRE 2021: INCORPORA CARTERA DE SECTOR 38
work_conn.execute(text("DROP TABLE IF EXISTS #af32_d"))
sql_af32_d = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T3.C_SI_publ AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_17' AS PROC, 'PS' AS FUENTE
INTO #af32_d
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32'
  AND (T3.C_SI_publ = 31 OR T3.C_SI_publ = 32 OR T3.C_SI_publ = 36 OR T3.C_SI_publ = 4 OR T3.C_SI_publ = 51 OR T3.C_SI_publ = 33 OR T3.C_SI_publ = 38)
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T3.C_SI_publ
"""
work_conn.execute(text(sql_af32_d))


In [ ]:
# SELECCIONA PASIVO DE SECTORES 31/32/33/36/4/51
work_conn.execute(text("DROP TABLE IF EXISTS #af32_h"))
sql_af32_h = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0498_17' AS PROC, 'PS' AS FUENTE
INTO #af32_h
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32'
  AND (T2.C_SI_publ = 31 OR T2.C_SI_publ = 32 OR T2.C_SI_publ = 36 OR T2.C_SI_publ = 4 OR T2.C_SI_publ = 51 OR T2.C_SI_publ = 33 OR T2.C_SI_publ = 38)
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_h))


In [ ]:
# PROC DATASETS APPEND: junta AF32_H dentro de AF32_D
cols_af32_d = "MONEDA, [AÑO], TRIM, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
work_conn.execute(text(f"INSERT INTO #af32_d ({cols_af32_d}) SELECT {cols_af32_d} FROM #af32_h"))


In [ ]:
# CALCULA DIF PASIVO-ACTIVO PARA IMPUTAR EN ACTIVO DE LOS SECTORES 33/36/51 USANDO INFO DE ESTRUCTURA DE CONTRAGENTES DEL DCV...TODO SE VA A EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
sql_af32_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SECTOR AS SECTOR,
       LTRIM(CAST(T1.C_CAGENTE AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO * T2.DATO) AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, T1.PROC, T1.FUENTE
INTO #af32_dif
FROM #af32_d t1
INNER JOIN TABLAS.dbo.T_EST_DCV_AF32_PROMEDIOS T2 ON T1.C_CAGENTE = T2.C_CAGENTE AND T1.C_SCN = T2.C_SCN AND T2.DATO <> 0
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T2.C_SECTOR, T1.C_SCN, T1.PROC, T1.FUENTE, LTRIM(CAST(T1.C_CAGENTE AS varchar(11)))
"""
work_conn.execute(text(sql_af32_dif))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF32 CON CA 53 DEL RESPECTIVO SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca53"))
sql_af32_ca53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_18' AS PROC, T1.FUENTE
INTO #af32_ca53
FROM #af32_dif T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_ca53))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_dif"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_dif)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ca53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ca53)", res.rowcount)
for t in ["#af32_dif", "#af32_ca53", "#af32_d", "#af32_h"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE DE 53 A 51 EN TÍTULOS DE LP EN EL ACTIVO
# SELECCIONA DATA DE ACTIVO PARA CAMBIAR CONTRAGENTE
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51"))
sql_af32_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       '51' AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, 'PS' AS FUENTE, '0498_20' AS PROC
INTO #af32_51
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T3.C_SI_publ = 53 AND T1.DATO <> 0
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_51))


In [ ]:
# ELIMINA DATA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_53"))
sql_af32_53 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(T3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_21' AS PROC
INTO #af32_53
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T3.C_SI_publ = 53 AND T1.DATO <> 0
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ
"""
work_conn.execute(text(sql_af32_53))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_53)", res.rowcount)
for t in ["#af32_53", "#af32_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA EN SECTORES 51 Y 8 (HOGARES) DE ACTIVO AF.32 Y AF.31 CON TODOS LOS CA DEJANDO EL 70% EN CP Y 30% EN LP CON LOS MISMOS CA
# SELECCIONA DATA DE SECTOR 51 Y 8
work_conn.execute(text("DROP TABLE IF EXISTS #af31_51_8"))
sql_af31_51_8 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(T3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN
INTO #af31_51_8
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31' AND T2.C_SI_publ = 8)
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_publ = 51)
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ
"""
work_conn.execute(text(sql_af31_51_8))


In [ ]:
# IMPUTA 70% DEL CP DE SECTOR 8 EN SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #af31_51_70"))
sql_af31_51_70 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       CASE WHEN T1.C_CAGENTE IN ('4','41') THEN SUM(T1.DATO) ELSE SUM(T1.DATO * 0.7) END AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_003' AS PROC, 'PS' AS FUENTE
INTO #af31_51_70
FROM #af31_51_8 T1
WHERE T1.C_SCN = 'AF.31' AND T1.SECTOR = 8 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af31_51_70))


In [ ]:
# ANULA 70% DEL CP PARA CONTRARRESTAR AJUSTE ANTERIOR EN SECTOR 8
work_conn.execute(text("DROP TABLE IF EXISTS #af31_8_70"))
sql_af31_8_70 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -0.7) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_004' AS PROC, 'PS' AS FUENTE
INTO #af31_8_70
FROM #af31_51_8 T1
WHERE T1.C_SCN = 'AF.31' AND T1.SECTOR = 8 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af31_8_70))


In [ ]:
# IMPUTA 30% DEL LP EN SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_30"))
sql_af32_51_30 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * 0.3) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_005' AS PROC, 'PS' AS FUENTE
INTO #af32_51_30
FROM #af31_51_8 T1
WHERE T1.C_SCN = 'AF.32' AND T1.SECTOR = 51 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_51_30))


In [ ]:
# IMPUTA 30% DEL LP EN SECTOR 51 (anulación)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_30_"))
sql_af32_51_30_ = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -0.3) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_006' AS PROC, 'PS' AS FUENTE
INTO #af32_51_30_
FROM #af31_51_8 T1
WHERE T1.C_SCN = 'AF.32' AND T1.SECTOR = 51 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_51_30_))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_51_70"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_51_70)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_8_70"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_8_70)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_30"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_30)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_30_"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_30_)", res.rowcount)
for t in ["#af31_51_8", "#af31_51_70", "#af31_8_70", "#af32_51_30", "#af32_51_30_"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA TITULOS DE CP DE SECTORES 8/511 CON CA 31 Y LO CAMBIA A LP. LO DE CP ADEMÁS LO IMPUTA A SECTOR 51 AJUSTANDOLO CONTRA TITULOS DE LP
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA ELIMINARLO DEL CP
work_conn.execute(text("DROP TABLE IF EXISTS #af31_ca31"))
sql_af31_ca31 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_011' AS PROC, 'PS' AS FUENTE
INTO #af31_ca31
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31' AND T2.C_SI_SCN_N1 = 'S.14'
  AND T1.C_CAGENTE IN ('31','41') AND T1.DATO <> 0
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af31_ca31))


In [ ]:
# IMPUTA DATA DE SECTORES 8/511 CON CA 31 EN LP
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca31"))
sql_af32_ca31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0500_014' AS PROC, T1.FUENTE
INTO #af32_ca31
FROM #af31_ca31 t1
"""
work_conn.execute(text(sql_af32_ca31))


In [ ]:
# IMPUTA DATA DE SECTORES 8/511 CON CA 31 EN CP DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #af31_41_ca31"))
sql_af31_41_ca31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_012' AS PROC, T1.FUENTE
INTO #af31_41_ca31
FROM #af31_ca31 t1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE
"""
work_conn.execute(text(sql_af31_41_ca31))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN SECTOR 51 CON CA 31 EN BONOS DE LP
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_ca31"))
sql_af32_41_ca31 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0500_013' AS PROC, T1.FUENTE
INTO #af32_41_ca31
FROM #af31_41_ca31 t1
"""
work_conn.execute(text(sql_af32_41_ca31))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_41_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_41_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_41_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_41_ca31)", res.rowcount)
for t in ["#af31_ca31", "#af32_ca31", "#af31_41_ca31", "#af32_41_ca31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# el SAS dropea af32_vm_bi (creada en tramo anterior) sin generar tabla nueva
work_conn.execute(text("DROP TABLE IF EXISTS #af32_vm_bi"))


In [ ]:
# RECLASIFICA BONOS DE LARGO PLAZO DEL ACTIVO DE HOGARES CON CA 31 A BONOS DE LP DEL SECTOR 51 CON CA 31
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA ELIMINARLO DEL LP
work_conn.execute(text("DROP TABLE IF EXISTS #af32_hh"))
sql_af32_hh = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_015' AS PROC, 'PS' AS FUENTE
INTO #af32_hh
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_SCN_N1 = 'S.14'
  AND T1.C_CAGENTE IN ('31','41') AND T1.DATO <> 0
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_hh))


In [ ]:
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA REASIGNARLO A SECTOR 51 (LP)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_hh_aj"))
sql_af32_hh_aj = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_016' AS PROC, 'PS' AS FUENTE
INTO #af32_hh_aj
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_SCN_N1 = 'S.14'
  AND T1.C_CAGENTE IN ('31','41') AND T1.DATO <> 0
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_hh_aj))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_hh"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_hh)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_hh_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_hh_aj)", res.rowcount)
for t in ["#af32_hh", "#af32_hh_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# M-001: ruta absoluta del SAS (/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/base_af3_total_v2.sas7bdat)
# reemplazada por ruta relativa al workspace del framework de configuración
raise NotImplementedError(
    "Falta el archivo fuente de DCV (base_af3_total_v2.sas7bdat, léase con pyreadstat) bajo una ruta relativa del workspace; "
    "el analista de fase 2 advirtió que esta tabla no está incluida en la versión actual del proyecto"
)


## S2_07_Cuotas_Fondos

Reclasifica contrapartes de cuotas de fondos entre money market y no money market por sector económico, y concilia activo y pasivo de fondos mutuos/inversión imputando diferencias residuales al sector 51022 y al patrimonio del Banco Central en gobierno central

*confianza: medium · verificador: approve · SAS: PROC SQL: ~20 UPDATE condicionales de reclasificación + múltiples CREATE TABLE/APPEND para conciliación activo-pasivo de cuotas de fondos (AF.521/AF.522)*

In [ ]:
# ========= S2_07_Cuotas_Fondos =========
# COMPRIME TABLA PRINCIPAL (compresión física sin equivalente en SQL Server: se conserva el contenido, se documenta la limitación)
# El SET sin transformación no altera datos; en destino no hay operación equivalente a COMPRESS=YES


In [ ]:
# ACTUALIZA COD DE INSTRUMENTOS Y CONTRAGENTES PARA SEPARAR MONEY MARKET DE NO MONEY MARKET Y PARA DEJAR DENTRO DE LO NO MONEY MARKET LO QUE ES CA 3390102 DE 339011
# MUNICIPALIDADES ES TODO MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=42 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='51'"))
    _log("UPDATE BD_CTSI municipalidades", res.rowcount)


In [ ]:
# BANCOS ES TODO MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=321 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI bancos", res.rowcount)


In [ ]:
# TCC ES TODO MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=334 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='339'"))
    _log("UPDATE BD_CTSI tcc", res.rowcount)


In [ ]:
# ISAPRES ES TODO MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=353 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE IN ('33901','53')"))
    _log("UPDATE BD_CTSI isapres", res.rowcount)


In [ ]:
# AFP ES TODO MONEY MARKET. SE DECIDE ASÍ PORQUE SON MONTOS BAJOS Y SE ASUME QUE PARA HACER CAJA Y REQUIERE LIQUIDEZ
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', C_SCN='AF.521', PROC='0503' WHERE SECTOR=361 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI afp", res.rowcount)


In [ ]:
# MUTUALIDADES ES TODO MONEY MARKET IGUAL QUE GOB CENTRAL
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=412 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='339'"))
    _log("UPDATE BD_CTSI mutualidades", res.rowcount)


In [ ]:
# GOBIERNO ES TODO MONEY MARKET LO NACIONAL
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=41 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI gobierno", res.rowcount)


In [ ]:
# FONDOS DE INVERSIÓN SE SEPARA LO QUE ES FI DE FM. EN FM SE ASUME QUE ES NO MONEY MARKET. CIERRE 2021 AJUSTA ESTE CAMBIO DE CONTRAGENTE
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='339011', C_SCN='AF.522', PROC='0503' WHERE SECTOR=339011 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE<>'3390101'"))
    _log("UPDATE BD_CTSI fi vs fm 1", res.rowcount)


In [ ]:
# FONDOS DE INVERSIÓN SE SEPARA LO QUE ES FI DE FM. EN FM SE ASUME QUE ES NO MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390102', C_SCN='AF.522', PROC='0503' WHERE SECTOR=339011 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI fi vs fm 2", res.rowcount)


In [ ]:
# CORREDORES DE BOLSA SE DEJA EN FM MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=36904 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI corredores", res.rowcount)


In [ ]:
# AFI, BOLSA DE VALORES, SECURITIZADORAS, LEASING, FACTORING SE DEJA EN FM MONEY MARKET. CIERRE 2021: NCORPORA SECTOR 36909
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR IN (36905,36906,33232,33231,33222,33221,33212,33211,36909) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE IN ('339','33901')"))
    _log("UPDATE BD_CTSI afi/bolsa/leasing", res.rowcount)


In [ ]:
# LEASING BANCARIO
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR IN (33211) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI leasing bancario", res.rowcount)


In [ ]:
# clasificadoras, casas matrices, afis, holdings
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR IN (36910,36907,36905,37) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI clasificadoras", res.rowcount)


In [ ]:
# FSCTORING BANCARIO SE DEJA EN FM MONEY MARKET
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR IN (33221,369011) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI factoring bancario", res.rowcount)


In [ ]:
# LO NO CLASIFICADO EN FONDOS DE PENSIONES SE DEJA EN AF522 CON CA 339011
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='339011', PROC='0503' WHERE SECTOR=34 AND C_ENTRADA='D' AND C_SCN='AF.522' AND C_CAGENTE='339'"))
    _log("UPDATE BD_CTSI fondos pensiones no clasif", res.rowcount)


In [ ]:
# LO CLASIFICADO EN CA 339 EN EMPRESAS PUBLICAS Y PRIVADAS SE DEJA EN AF522 CON CA 339011
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.522', C_CAGENTE='339011', PROC='0503' WHERE SECTOR IN (51021,5101) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='339'"))
    _log("UPDATE BD_CTSI empresas pub/priv ca339", res.rowcount)


In [ ]:
# LO CLASIFICADO EN CA 339 EN EMPRESAS PUBLICAS Y PRIVADAS SE DEJA EN AF522 CON CA 339011
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR IN (51021,5101) AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='33901'"))
    _log("UPDATE BD_CTSI empresas pub/priv ca33901", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA INSTRUMENTO DESDE AF.5 A AF.522 EN EL ACTIVO DE FONDOS DE INVERSIÓN CON CONTRAGENTE 339011
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.522', N_SCN='Participaciones emitidas por fondos  de inversión del mercado monetario', PROC='0503' WHERE SECTOR=339011 AND C_ENTRADA='D' AND C_SCN='AF.5' AND C_CAGENTE='339011'"))
    _log("UPDATE BD_CTSI af5 a af522", res.rowcount)


In [ ]:
# CIERRE 2021: INCORPORA CAMBIO DE CONTRAGENTE EN CUOTAS DE FONDOS DE UNIVERSIDADES
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='3390101', PROC='0503' WHERE SECTOR=413 AND C_ENTRADA='D' AND C_SCN='AF.521' AND C_CAGENTE='S129'"))
    _log("UPDATE BD_CTSI universidades", res.rowcount)


In [ ]:
# IMPUTA CARTERA DE FONDOS DE SECTORES CON CA RESTO DEL MUNDO EN EL PASIVO DEL RM Y AJUSTA CONTRA EL PASIVO AF.5 DEL RM
# IMPUTA EN RM EXCEPTO LA CARTERA DE GOB CON RM
sql_fondos_ca6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0504' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CAGENTE='6' AND T1.SECTOR<>41)
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN, CAST(T1.C_CAGENTE AS float), LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
fondos_ca6 = pd.read_sql(text(sql_fondos_ca6), engine)
_log("fondos_ca6", fondos_ca6)


In [ ]:
# AJUSTA IMPUTACIÓN EN AF5. CIERRE 2021: DADO QUE RESTO DEL MUNDO NO TIENE AF.5 CON AUXILIARES, SE RESTA EN CA 53
ajuste_af5_6 = fondos_ca6.copy()
ajuste_af5_6["DATO"] = ajuste_af5_6["DATO"] * -1
ajuste_af5_6["C_SCN"] = "AF.5"
ajuste_af5_6["N_SCN"] = "Acciones y otras participaciones de capital"
ajuste_af5_6["PROC"] = "0505"
ajuste_af5_6.loc[ajuste_af5_6["C_CAGENTE"].str[:2] == "36", "C_CAGENTE"] = "53"
_log("ajuste_af5_6", ajuste_af5_6)


In [ ]:
# APPEND server-side desde dataframes en memoria (subida temporal + insert)
fondos_ca6.to_sql("#fondos_ca6", work_conn, if_exists="replace", index=False)
cols_fondos_ca6 = ", ".join(fondos_ca6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fondos_ca6}) SELECT {cols_fondos_ca6} FROM #fondos_ca6"))
_log("APPEND BD_CTSI fondos_ca6", res.rowcount)
ajuste_af5_6.to_sql("#ajuste_af5_6", work_conn, if_exists="replace", index=False)
cols_ajuste_af5_6 = ", ".join(ajuste_af5_6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ajuste_af5_6}) SELECT {cols_ajuste_af5_6} FROM #ajuste_af5_6"))
_log("APPEND BD_CTSI ajuste_af5_6", res.rowcount)
for t in ["#fondos_ca6", "#ajuste_af5_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA ACTIVO DE AF521 DE SECTOR GOBIERNO CON CA RM CON EL PASIVO DEL RM CA 41. RESPETA DATO DE RM, POR LO TANTO AJUSTA DATO DE GOBIERNO Y EL CONTRAAJUSTE ES AL ACTIVO AF521 DE GOBIERNO CON CA 3390101
# CALCULA DIF ENTRE RM Y GOB
sql_af521_gob_ca6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.SECTOR=6 AND t1.C_CUENTA IN ('Rec Volumen','Rec Precio') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0523' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.521' AND T1.SECTOR=6 AND T1.C_CAGENTE='41') OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.521' AND T1.SECTOR=41 AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN T1.SECTOR=6 AND t1.C_CUENTA IN ('Rec Volumen','Rec Precio') THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
af521_gob_ca6 = pd.read_sql(text(sql_af521_gob_ca6), engine)
_log("af521_gob_ca6", af521_gob_ca6)


In [ ]:
af521_gob_cafm = af521_gob_ca6.copy()
af521_gob_cafm["C_CAGENTE"] = "3390101"
af521_gob_cafm["DATO"] = af521_gob_cafm["DATO"] * -1
af521_gob_cafm["PROC"] = "0524"
_log("af521_gob_cafm", af521_gob_cafm)


In [ ]:
af521_gob_ca6.to_sql("#af521_gob_ca6", work_conn, if_exists="replace", index=False)
cols_af521_gob_ca6 = ", ".join(af521_gob_ca6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af521_gob_ca6}) SELECT {cols_af521_gob_ca6} FROM #af521_gob_ca6"))
_log("APPEND BD_CTSI af521_gob_ca6", res.rowcount)
af521_gob_cafm.to_sql("#af521_gob_cafm", work_conn, if_exists="replace", index=False)
cols_af521_gob_cafm = ", ".join(af521_gob_cafm.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af521_gob_cafm}) SELECT {cols_af521_gob_cafm} FROM #af521_gob_cafm"))
_log("APPEND BD_CTSI af521_gob_cafm", res.rowcount)
for t in ["#af521_gob_ca6", "#af521_gob_cafm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE A 33901 EN ACTIVO AF.521 Y AF.522 DE TODOS LOS SECTORES PARA LOS CA NO ASIGNADOS
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='33901', PROC='0506' WHERE C_ENTRADA='D' AND C_SCN IN ('AF.521','AF.522') AND C_CAGENTE='53'"))
    _log("UPDATE BD_CTSI ca no asignados", res.rowcount)


In [ ]:
# IMPUTA PASIVO DE FONDOS MUTUOS E INVERSIÓN CON CA 6 EN ACTIVO DE RESTO DEL MUNDO DEL INST AF.521 Y AF.522. DIFERENCIAL CON LO INFORMADO POR EL RM SE VA A AF.5 CON CA 33
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.521' WHERE C_ENTRADA='D' AND SECTOR=6 AND C_SCN='AF.522'"))
    _log("UPDATE BD_CTSI sector6 af522 a af521", res.rowcount)


In [ ]:
# IMPUTA FFMM MONEY MARKET EN ACTIVO RM
sql_ffmm_ca6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       6 AS SECTOR,
       '3390101' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0518' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.521' AND T1.SECTOR=3390101 AND T1.C_CAGENTE='6'
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
ffmm_ca6 = pd.read_sql(text(sql_ffmm_ca6), engine)
_log("ffmm_ca6", ffmm_ca6)


In [ ]:
af521_6aj = ffmm_ca6.copy()
af521_6aj["C_CAGENTE"] = "33901"
af521_6aj["DATO"] = af521_6aj["DATO"] * -1
af521_6aj["PROC"] = "0519"
_log("af521_6aj", af521_6aj)


In [ ]:
ffmm_ca6.to_sql("#ffmm_ca6", work_conn, if_exists="replace", index=False)
cols_ffmm_ca6 = ", ".join(ffmm_ca6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmm_ca6}) SELECT {cols_ffmm_ca6} FROM #ffmm_ca6"))
_log("APPEND BD_CTSI ffmm_ca6", res.rowcount)
af521_6aj.to_sql("#af521_6aj", work_conn, if_exists="replace", index=False)
cols_af521_6aj = ", ".join(af521_6aj.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af521_6aj}) SELECT {cols_af521_6aj} FROM #af521_6aj"))
_log("APPEND BD_CTSI af521_6aj", res.rowcount)
for t in ["#ffmm_ca6", "#af521_6aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA FFMM NO MONEY MARKET EN ACTIVO RM
sql_ffmmnm_ca6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       6 AS SECTOR,
       '3390102' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0518a' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.522' AND T1.SECTOR=3390102 AND T1.C_CAGENTE='6'
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
ffmmnm_ca6 = pd.read_sql(text(sql_ffmmnm_ca6), engine)
_log("ffmmnm_ca6", ffmmnm_ca6)


In [ ]:
af522_6aj = ffmmnm_ca6.copy()
af522_6aj["C_CAGENTE"] = "33901"
af522_6aj["DATO"] = af522_6aj["DATO"] * -1
af522_6aj["C_SCN"] = "AF.521"
af522_6aj["PROC"] = "0519a"
_log("af522_6aj", af522_6aj)


In [ ]:
ffmmnm_ca6.to_sql("#ffmmnm_ca6", work_conn, if_exists="replace", index=False)
cols_ffmmnm_ca6 = ", ".join(ffmmnm_ca6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmmnm_ca6}) SELECT {cols_ffmmnm_ca6} FROM #ffmmnm_ca6"))
_log("APPEND BD_CTSI ffmmnm_ca6", res.rowcount)
af522_6aj.to_sql("#af522_6aj", work_conn, if_exists="replace", index=False)
cols_af522_6aj = ", ".join(af522_6aj.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_6aj}) SELECT {cols_af522_6aj} FROM #af522_6aj"))
_log("APPEND BD_CTSI af522_6aj", res.rowcount)
for t in ["#ffmmnm_ca6", "#af522_6aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA FI EN ACTIVO RM
sql_fi_ca6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       6 AS SECTOR,
       '339011' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0518b' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.522' AND T1.SECTOR=339011 AND T1.C_CAGENTE='6'
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
fi_ca6 = pd.read_sql(text(sql_fi_ca6), engine)
_log("fi_ca6", fi_ca6)


In [ ]:
af522_6aj_2 = fi_ca6.copy()
af522_6aj_2["C_CAGENTE"] = "33901"
af522_6aj_2["DATO"] = af522_6aj_2["DATO"] * -1
af522_6aj_2["C_SCN"] = "AF.521"
af522_6aj_2["PROC"] = "0519b"
_log("af522_6aj_2", af522_6aj_2)


In [ ]:
fi_ca6.to_sql("#fi_ca6", work_conn, if_exists="replace", index=False)
cols_fi_ca6 = ", ".join(fi_ca6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fi_ca6}) SELECT {cols_fi_ca6} FROM #fi_ca6"))
_log("APPEND BD_CTSI fi_ca6", res.rowcount)
af522_6aj_2.to_sql("#af522_6aj_2", work_conn, if_exists="replace", index=False)
cols_af522_6aj_2 = ", ".join(af522_6aj_2.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_6aj_2}) SELECT {cols_af522_6aj_2} FROM #af522_6aj_2"))
_log("APPEND BD_CTSI af522_6aj_2", res.rowcount)
for t in ["#fi_ca6", "#af522_6aj_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA LO RESIDUAL QUE QUEDA EN AF521 CON CA 33901 A AF5 CON CA 33
sql_fondos_s6 = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0518c' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.521' AND T1.SECTOR=6 AND T1.C_CAGENTE='33901')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.SECTOR, t1.C_CAGENTE
"""
fondos_s6 = pd.read_sql(text(sql_fondos_s6), engine)
_log("fondos_s6", fondos_s6)


In [ ]:
# AJUSTA EN AF5 CON CA 33
af5_s6_ca33 = fondos_s6.copy()
af5_s6_ca33["C_CAGENTE"] = "33"
af5_s6_ca33["DATO"] = af5_s6_ca33["DATO"] * -1
af5_s6_ca33["C_SCN"] = "AF.5"
af5_s6_ca33["N_SCN"] = "Acciones y otras participaciones de capital"
af5_s6_ca33["FUENTE"] = "PS"
af5_s6_ca33["PROC"] = "0519c"
_log("af5_s6_ca33", af5_s6_ca33)


In [ ]:
fondos_s6.to_sql("#fondos_s6", work_conn, if_exists="replace", index=False)
cols_fondos_s6 = ", ".join(fondos_s6.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fondos_s6}) SELECT {cols_fondos_s6} FROM #fondos_s6"))
_log("APPEND BD_CTSI fondos_s6", res.rowcount)
af5_s6_ca33.to_sql("#af5_s6_ca33", work_conn, if_exists="replace", index=False)
cols_af5_s6_ca33 = ", ".join(af5_s6_ca33.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af5_s6_ca33}) SELECT {cols_af5_s6_ca33} FROM #af5_s6_ca33"))
_log("APPEND BD_CTSI af5_s6_ca33", res.rowcount)
for t in ["#fondos_s6", "#af5_s6_ca33"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE SECTORES NACIONALES (EXCEPTO HOGARES) DE INST AF.521 CON CA 3390101 EN PASIVO DE SECTOR 3390101 CA 321. AJUSTA EN PASIVO AF521 SECTOR 3390101 CA 53
# IMPUTA CARTERAS DE INST FFMM MONEY MARKET EN PASIVO DE SECTOR 3390101
sql_af521_invnac = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0510' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE ((T1.C_ENTRADA='D' AND T1.C_SCN='AF.521' AND T1.SECTOR<>6 AND T1.C_CAGENTE='3390101') AND (T1.C_ENTRADA='D' AND T1.C_SCN='AF.521' AND T1.SECTOR<>511 AND T1.C_CAGENTE='3390101'))
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS float), LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af521_invnac = pd.read_sql(text(sql_af521_invnac), engine)
_log("af521_invnac", af521_invnac)


In [ ]:
af521_ca53 = af521_invnac.groupby(["MONEDA", "AÑO", "TRIM", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN", "SECTOR", "FUENTE"], as_index=False, dropna=False)["DATO"].sum()
af521_ca53["DATO"] = af521_ca53["DATO"] * -1
af521_ca53["C_CAGENTE"] = "53"
af521_ca53["PROC"] = "0510"
cols_order = ["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]
af521_ca53 = af521_ca53[cols_order]
_log("af521_ca53", af521_ca53)


In [ ]:
af521_invnac.to_sql("#af521_invnac", work_conn, if_exists="replace", index=False)
cols_af521_invnac = ", ".join(af521_invnac.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af521_invnac}) SELECT {cols_af521_invnac} FROM #af521_invnac"))
_log("APPEND BD_CTSI af521_invnac", res.rowcount)
af521_ca53.to_sql("#af521_ca53", work_conn, if_exists="replace", index=False)
cols_af521_ca53 = ", ".join(af521_ca53.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af521_ca53}) SELECT {cols_af521_ca53} FROM #af521_ca53"))
_log("APPEND BD_CTSI af521_ca53", res.rowcount)
for t in ["#af521_invnac", "#af521_ca53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ANULA CONTRAGENTES DISTINTOS DE HOGARES Y RM DEL PATRIMONIO DE LOS FI, Y LOS DEJA EN CONTRAGENTE 53
sql_af522_fi_p = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0510x' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE ((T1.C_ENTRADA='H' AND T1.C_SCN='AF.522' AND T1.SECTOR=339011 AND T1.C_CAGENTE<>'511') AND (T1.C_ENTRADA='H' AND T1.C_SCN='AF.522' AND T1.SECTOR=339011 AND T1.C_CAGENTE<>'6'))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
af522_fi_p = pd.read_sql(text(sql_af522_fi_p), engine)
_log("af522_fi_p", af522_fi_p)


In [ ]:
# IMPUTA AJUSTE ANTERIOR EN CONTRAGENTE 53
af522_fi_53 = af522_fi_p.groupby(["MONEDA", "AÑO", "TRIM", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN", "SECTOR", "FUENTE"], as_index=False, dropna=False)["DATO"].sum()
af522_fi_53["DATO"] = af522_fi_53["DATO"] * -1
af522_fi_53["C_CAGENTE"] = "53"
af522_fi_53["PROC"] = "0510z"
af522_fi_53 = af522_fi_53[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af522_fi_53", af522_fi_53)


In [ ]:
af522_fi_p.to_sql("#af522_fi_p", work_conn, if_exists="replace", index=False)
cols_af522_fi_p = ", ".join(af522_fi_p.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_fi_p}) SELECT {cols_af522_fi_p} FROM #af522_fi_p"))
_log("APPEND BD_CTSI af522_fi_p", res.rowcount)
af522_fi_53.to_sql("#af522_fi_53", work_conn, if_exists="replace", index=False)
cols_af522_fi_53 = ", ".join(af522_fi_53.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_fi_53}) SELECT {cols_af522_fi_53} FROM #af522_fi_53"))
_log("APPEND BD_CTSI af522_fi_53", res.rowcount)
for t in ["#af522_fi_p", "#af522_fi_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERAS DE INST FONDOS NO MONEY MARKET EN PASIVO DE SECTORES 3390102 Y 339011. AJUSTA IMPUTACIÓN EN CA 53
sql_af522_invnac = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       (CASE WHEN CAST(T1.C_CAGENTE AS float)=3390102 THEN 3390102 ELSE 339011 END) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0510a' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE ((T1.C_ENTRADA='D' AND T1.C_SCN='AF.522' AND T1.SECTOR<>6 AND T1.C_CAGENTE IN ('3390102','339011','33901','339')) AND (T1.C_ENTRADA='D' AND T1.C_SCN='AF.522' AND T1.SECTOR<>511 AND T1.C_CAGENTE IN ('3390102','339011','33901','339')))
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, (CASE WHEN CAST(T1.C_CAGENTE AS float)=3390102 THEN 3390102 ELSE 339011 END), LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af522_invnac = pd.read_sql(text(sql_af522_invnac), engine)
_log("af522_invnac", af522_invnac)


In [ ]:
af522_ca53 = af522_invnac.groupby(["MONEDA", "AÑO", "TRIM", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN", "SECTOR", "FUENTE"], as_index=False, dropna=False)["DATO"].sum()
af522_ca53["DATO"] = af522_ca53["DATO"] * -1
af522_ca53["C_CAGENTE"] = "53"
af522_ca53["PROC"] = "0510a"
af522_ca53 = af522_ca53[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af522_ca53", af522_ca53)


In [ ]:
af522_invnac.to_sql("#af522_invnac", work_conn, if_exists="replace", index=False)
cols_af522_invnac = ", ".join(af522_invnac.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_invnac}) SELECT {cols_af522_invnac} FROM #af522_invnac"))
_log("APPEND BD_CTSI af522_invnac", res.rowcount)
af522_ca53.to_sql("#af522_ca53", work_conn, if_exists="replace", index=False)
cols_af522_ca53 = ", ".join(af522_ca53.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af522_ca53}) SELECT {cols_af522_ca53} FROM #af522_ca53"))
_log("APPEND BD_CTSI af522_ca53", res.rowcount)
for t in ["#af522_invnac", "#af522_ca53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACIÓN FINAL DE FM Y FI. COMPARA PASIVO CON ACTIVO, LO QUE FALTA EN ACTIVO LO IMPUTA AL ACTIVO DEL SECTOR 51022
sql_fondos_cierre = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       (CASE WHEN T1.C_SCN='AF.521' THEN '3390101' ELSE '339011' END) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0534' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.SECTOR<>6) OR (T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CAGENTE<>'6')
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN T1.C_SCN='AF.521' THEN '3390101' ELSE '339011' END), T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
fondos_cierre = pd.read_sql(text(sql_fondos_cierre), engine)
_log("fondos_cierre", fondos_cierre)


In [ ]:
fondos_cierre.to_sql("#fondos_cierre", work_conn, if_exists="replace", index=False)
cols_fondos_cierre = ", ".join(fondos_cierre.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fondos_cierre}) SELECT {cols_fondos_cierre} FROM #fondos_cierre"))
_log("APPEND BD_CTSI fondos_cierre", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #fondos_cierre"))


In [ ]:
# IMPUTA PATRIMONIO DEL BANCO CENTRAL EN GOBIERNO CENTRAL
sql_pat_bc = """
SELECT T1.MONEDA,
       t1.[AÑO],
       T1.TRIM,
       41 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0536' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.5' AND T1.C_CAGENTE='53' AND T1.SECTOR=31)
"""
pat_bc = pd.read_sql(text(sql_pat_bc), engine)
_log("pat_bc", pat_bc)


In [ ]:
pat_bc.to_sql("#pat_bc", work_conn, if_exists="replace", index=False)
cols_pat_bc = ", ".join(pat_bc.columns)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_pat_bc}) SELECT {cols_pat_bc} FROM #pat_bc"))
_log("APPEND BD_CTSI pat_bc", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #pat_bc"))


In [ ]:
# ACTUALIZA EN ACTIVO AF5 DEL SECTOR 41 CONTRAGENTES RM Y EMPRESAS A CONTRAGENTE 5101
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='5101', PROC='0538' WHERE C_CAGENTE IN ('6','51') AND SECTOR=41 AND C_SCN='AF.5'"))
    _log("UPDATE BD_CTSI af5 sector41 ca RM/empresas", res.rowcount)


## S2_08_Derivados

Neteo e imputación de posiciones de derivados (AF.34) entre sectores institucionales (RM, FP, bancos, OFIS/AUX/pensiones), generando activos netos que eliminan el pasivo y ajustando contrapartidas por diferencias de cuadre, acumulando cada resultado en la base de cuentas del sistema

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con GROUP BY/CALCULATED + PROC DATASETS APPEND FORCE (repetido en cascada) sobre TABLAS.BD_CTSI*

In [ ]:
# ========= S2_08_Derivados =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES no tiene equivalente en SQL Server; no-op funcional)
with engine.begin() as conn:
    pass  # sin acción: COMPRESS es un atributo de almacenamiento SAS, no aplica en la BD destino


In [ ]:
# IMPUTA PASIVO DE DERIVADOS EN SECTOR 321 CON CA 6, USANDO INFO DEL ACTIVO DEL RM CON CA 321. AJUSTA IMPUTACIÓN EN PASIVO DE SECTOR 321 CON CA 53
sql_af34_6_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0408' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 6
GROUP BY t1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         CAST(T1.C_CAGENTE AS int),
         LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af34_6_321 = pd.read_sql(text(sql_af34_6_321), engine)
_log("af34_6_321", af34_6_321)


In [ ]:
# Ajuste espejo en pasivo de sector 321 con CA 53 (signo invertido)
af34_321_53 = af34_6_321.copy()
af34_321_53["C_CAGENTE"] = "53"
af34_321_53["DATO"] = af34_321_53["DATO"] * -1
af34_321_53 = af34_321_53[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_321_53", af34_321_53)


In [ ]:
# APPEND de ambos resultados a TABLAS.BD_CTSI (acumulativo, igual que el SAS original)
af34_6_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_321_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_6_321 + af34_321_53)", len(af34_6_321) + len(af34_321_53))


In [ ]:
# IMPUTA PASIVO DE DERIVADOS EN SECTOR 321 CON CA 31, USANDO INFO DEL ACTIVO DEL 31 CON CA 321. AJUSTA IMPUTACIÓN EN PASIVO DE SECTOR 321 CON CA 53
sql_af34_31_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0409' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 31
GROUP BY t1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         CAST(T1.C_CAGENTE AS int),
         LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af34_31_321 = pd.read_sql(text(sql_af34_31_321), engine)
_log("af34_31_321", af34_31_321)


In [ ]:
# Ajuste espejo en pasivo de sector 321 con CA 53 (signo invertido), PROC 0410
af34_321_53 = af34_31_321.copy()
af34_321_53["C_CAGENTE"] = "53"
af34_321_53["DATO"] = af34_321_53["DATO"] * -1
af34_321_53["PROC"] = "0410"
af34_321_53 = af34_321_53[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_321_53", af34_321_53)


In [ ]:
af34_31_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_321_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_31_321 + af34_321_53)", len(af34_31_321) + len(af34_321_53))


In [ ]:
# IMPUTA ACTIVO DE DERIVADOS EN SECTOR 321 CON CA 6, USANDO INFO DEL PASIVO DEL 6 CON CA 321. AJUSTA IMPUTACIÓN EN ACTIVO DE SECTOR 321 CON CA 53
sql_af34_321_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0411' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 6
GROUP BY t1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         CAST(T1.C_CAGENTE AS int),
         LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af34_321_6 = pd.read_sql(text(sql_af34_321_6), engine)
_log("af34_321_6", af34_321_6)


In [ ]:
# Ajuste espejo en activo de sector 321 con CA 53 (signo invertido), PROC 0412
af34_321_53d = af34_321_6.copy()
af34_321_53d["C_CAGENTE"] = "53"
af34_321_53d["DATO"] = af34_321_53d["DATO"] * -1
af34_321_53d["PROC"] = "0412"
af34_321_53d = af34_321_53d[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_321_53d", af34_321_53d)


In [ ]:
af34_321_6.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_321_53d.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_321_6 + af34_321_53d)", len(af34_321_6) + len(af34_321_53d))


In [ ]:
# IMPUTA ACTIVO DE DERIVADOS EN SECTOR 339011 CON CA 6, USANDO INFO DEL PASIVO DEL 6 CON CA 33901/351. AJUSTA IMPUTACIÓN EN ACTIVO DE AF71 DE SECTOR 339011 CON CA 6
# AJUSTE CIERRE 2024: INCORPORA AJUSTE EN TÉRMINOS DE DELTA, YA QUE POR DESCUADRE EN AF.32 CON RM EN 2024Q4, SE DECIDE GENERAR AF.34 ACTIVO EN FI
sql_af34_339011_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        339011 AS SECTOR,
        '6' AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0413' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '33901/351' AND T1.SECTOR = 6)
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '6' AND T1.SECTOR = 339011)
GROUP BY t1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
af34_339011_6 = pd.read_sql(text(sql_af34_339011_6), engine)
_log("af34_339011_6", af34_339011_6)


In [ ]:
# Ajuste en activo de AF.71 sector 339011 con CA 6 (signo invertido), reclasificado como AF.71 'Ajuste conciliación'
af71_339011_6 = af34_339011_6.copy()
af71_339011_6["DATO"] = af71_339011_6["DATO"] * -1
af71_339011_6["C_SCN"] = "AF.71"
af71_339011_6["N_SCN"] = "Ajuste conciliación"
af71_339011_6["PROC"] = "0414"
af71_339011_6 = af71_339011_6[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af71_339011_6", af71_339011_6)


In [ ]:
af71_339011_6.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_339011_6.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af71_339011_6 + af34_339011_6)", len(af71_339011_6) + len(af34_339011_6))


In [ ]:
# COMPARA ACTIVO DE FP CON RM CON PASIVO DE RM CON FP. LA DIFERENCIA LA IMPUTA EN ACTIVO DE FP CON RM, Y AJUSTA ACTIVO DE AF34 DE FP CON CA 3
sql_af34_34_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        34 AS SECTOR,
        '6' AS C_CAGENTE,
        t1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0418' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.SECTOR IN (34, 341) AND T1.C_CAGENTE = '6')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '34')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
af34_34_6 = pd.read_sql(text(sql_af34_34_6), engine)
_log("af34_34_6", af34_34_6)


In [ ]:
# Ajuste espejo en activo de FP con CA 3 (signo invertido), PROC 0419
af34_34_3 = af34_34_6.copy()
af34_34_3["C_CAGENTE"] = "3"
af34_34_3["DATO"] = af34_34_3["DATO"] * -1
af34_34_3["PROC"] = "0419"
af34_34_3 = af34_34_3[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_34_3", af34_34_3)


In [ ]:
af34_34_6.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_34_3.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_34_6 + af34_34_3)", len(af34_34_6) + len(af34_34_3))


In [ ]:
# CIERRE 2021: COMPARA ACTIVO DE DERIVADOS DE SECTORES OFIS-AUX-PENSIONES CON CONTRAGENTE BANCOS. IMPUTA EN PASIVO DE BANCOS Y AJUSTA EN BANCOS CONTRAGENTE 53
sql_af34_sect_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0419a' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321'
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(6))), 1, 2) IN ('34', '33', '36', '37')
  AND T1.SECTOR NOT IN (339011, 3390101, 3390102, 33901, 331)
GROUP BY t1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         CAST(T1.C_CAGENTE AS int),
         LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
af34_sect_321 = pd.read_sql(text(sql_af34_sect_321), engine)
_log("af34_sect_321", af34_sect_321)


In [ ]:
# Ajuste espejo en bancos contragente 53 (signo invertido), PROC 0419b
af34_321_53 = af34_sect_321.copy()
af34_321_53["C_CAGENTE"] = "53"
af34_321_53["DATO"] = af34_321_53["DATO"] * -1
af34_321_53["PROC"] = "0419b"
af34_321_53 = af34_321_53[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_321_53", af34_321_53)


In [ ]:
af34_sect_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_321_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_sect_321 + af34_321_53)", len(af34_sect_321) + len(af34_321_53))


In [ ]:
# CIERRA AJUSTE EN DERIVADOS COMPARANDO ACTIVO CON PASIVO, E IMPUTANDO DIFERENCIA EN ACTIVO DE SECTOR 51022 CON CA 53
sql_af34_cierre = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        '53' AS C_CAGENTE,
        t1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0422' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34') OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
af34_cierre = pd.read_sql(text(sql_af34_cierre), engine)
af34_cierre.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_cierre)", len(af34_cierre))


In [ ]:
# GENERA ACTIVOS NETOS DE DERIVADOS ELIMINANDO EL PASIVO E IMPUTANDO ESE PASIVO NEGATIVO EN EL ACTIVO DE LOS SECTORES
sql_af34_elimina = """
SELECT  T1.MONEDA,
        t1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        t1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0651' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34'
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, t1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
af34_elimina = pd.read_sql(text(sql_af34_elimina), engine)
_log("af34_elimina", af34_elimina)


In [ ]:
# Reclasifica el neteo como entrada de activo (C_ENTRADA='D'), PROC 0652
af34_netea = af34_elimina.copy()
af34_netea["C_ENTRADA"] = "D"
af34_netea["PROC"] = "0652"
af34_netea = af34_netea[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
_log("af34_netea", af34_netea)


In [ ]:
af34_elimina.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
af34_netea.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (af34_elimina + af34_netea)", len(af34_elimina) + len(af34_netea))


## S2_09_Capital_Cierre

Imputa y concilia el capital/patrimonio (AF.5) entre sectores institucionales (bancos, seguros, auxiliares, ofis, RM, gobierno, empresas) distribuyendo la inversión cruzada según ratios y estructura de contrapartes, y acumula los ajustes resultantes en TABLAS.BD_CTSI (tramo 1 de 2: hasta la conciliación de patrimonio de empresas) / Calcula el cierre de capital (AF.5) del sector 51022 imputando la diferencia entre pasivo y activo por contraparte (respetando el pasivo) y acumula el ajuste (delta) en la base de series

*confianza: medium · verificador: revise · SAS: PROC SQL: UPDATE + CREATE TABLE con GROUP BY/CALCULATED + PROC DATASETS APPEND FORCE (múltiples imputaciones recursivas sobre AF.5), todo server-side vía #tmp + PROC SQL: cierre de AF.5 (pasivo/activo por sector con contrapartida), DATA step de concatenación, agregación de delta y APPEND acumulativo*

In [ ]:
# ========= S2_09_Capital_Cierre =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_dummy"))
# COMPRIME TABLA PRINCIPAL: SET simple, sin efecto en T-SQL más allá de la opción de compresión de SAS (no aplica en la BD destino)
# no-op deliberado: la compresión es un atributo de almacenamiento SAS sin equivalente en SQL Server


In [ ]:
# CIERRE 2021: EN SECTOR 361 INST AF5 PASIVO CAMBIA CONTRAGENTE DESDE 3 A 53
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53' WHERE C_CAGENTE = '3' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 361"))
_log("UPDATE TABLAS.BD_CTSI sector 361 CA 3->53", res.rowcount)

# CIERRE 2021: EN SECTOR 339011 INST AF5 ACTIVO CAMBIA CONTRAGENTE DESDE 3 A 53
# valor faltante de SAS (C_CAGENTE='') equivale a NULL o cadena vacía en SQL Server
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53' WHERE (C_CAGENTE IS NULL OR C_CAGENTE = '') AND C_SCN = 'AF.5' AND C_ENTRADA = 'D' AND SECTOR = 339011"))
_log("UPDATE TABLAS.BD_CTSI sector 339011 CA ''->53", res.rowcount)


In [ ]:
# ELIMINA ACTIVO AF5 DEL SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111"))
sql_af5_5111 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0539' AS PROC
INTO #af5_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE = '53' AND T1.SECTOR = 5111
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_5111))

# APPEND FORCE: columnas explícitas alineadas por nombre
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_5111"))
_log("APPEND TABLAS.BD_CTSI desde AF5_5111 (0539)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111"))


In [ ]:
# IMPUTA ACTIVO DE AF5 EN SECTOR 511 CON CA 53, USANDO INFO DEL PASIVO AF5 DE SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511"))
sql_af5_511 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 511 AS SECTOR, T1.C_CAGENTE, T1.C_CUENTA,
       'D' AS C_ENTRADA, SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0543' AS PROC
INTO #af5_511
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE = '53' AND T1.SECTOR = 5111 AND T1.FUENTE = 'CI'
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_511))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_511"))
_log("APPEND TABLAS.BD_CTSI desde AF5_511 (0543)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_511"))


In [ ]:
# ELIMINA ACTIVO Y PASIVO AF7 SECTOR 5111 CON CA 53. CIERRE 2020: ELIMINA TMB CA 511, 6 Y 4 DE AF7. CIERRE 2021: ELIMINA TODO EL AF5 DEL SECTOR 5111 CON TODOS LOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_7_5111"))
sql_af5_7_5111 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0544' AS PROC
INTO #af5_7_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.7') AND T1.C_CAGENTE IN ('511', '53', '6', '4') AND T1.SECTOR = 5111 AND T1.DATO <> 0
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_7_5111))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_7_5111"))
_log("APPEND TABLAS.BD_CTSI desde AF5_7_5111 (0544 AF.7)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_7_5111"))


In [ ]:
# ELIMINA ACTIVO Y PASIVO AF5 SECTOR 5111 CON CA 53. CIERRE 2021: ELIMINA TODO EL AF5 DEL SECTOR 5111 CON TODOS LOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111_2"))
sql_af5_5111_2 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0544' AS PROC
INTO #af5_5111_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.5') AND T1.SECTOR = 5111 AND T1.DATO <> 0
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_5111_2))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_5111_2"))
_log("APPEND TABLAS.BD_CTSI desde AF5_5111 (0544 AF.5)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111_2"))


In [ ]:
# IMPUTA PASIVO DE AF5 EN SECTOR 51022 CON CA 53, USANDO INFO DEL PASIVO AF5 DE SECTOR 5111 CON CA 53 FUENTE CI.
# CIERRE 2021: ELIMINA CONTRAGENTE PORQUE AHORA VIENEN SECTORIZADO EL PATRIMONIO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022"))
sql_af5_51022 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 51022 AS SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0550' AS PROC
INTO #af5_51022
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 5111 AND T1.FUENTE = 'CI'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_51022))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022"))
_log("APPEND TABLAS.BD_CTSI desde AF5_51022 (0550)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022"))


In [ ]:
# IMPUTA ACTIVO DE AF5 EN SECTOR 511 CON CA 322, USANDO INFO DEL PASIVO AF5 DE SECTOR 322 FUENTE CI
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_322"))
sql_af5_511_322 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 511 AS SECTOR, '322' AS C_CAGENTE, T1.C_CUENTA,
       'D' AS C_ENTRADA, SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0551' AS PROC
INTO #af5_511_322
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 322 AND T1.FUENTE = 'CI'
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_511_322))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_511_322"))
_log("APPEND TABLAS.BD_CTSI desde AF5_511 (0551)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_322"))


In [ ]:
# IMPUTA ACTIVO AF5 EN SECTOR 511 CON CA 51021, USANDO INFO DE PASIVO DE SECTOR 51021 (PERO SOLO EL 0,036280969257607 DE ESE TOTAL)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_saa"))
sql_af5_511_saa = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 511 AS SECTOR, '51021' AS C_CAGENTE, T1.C_CUENTA,
       'D' AS C_ENTRADA, SUM(T1.DATO * 0.036280969257607) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0559' AS PROC
INTO #af5_511_saa
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 51021
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_511_saa))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_511_saa"))
_log("APPEND TABLAS.BD_CTSI desde AF5_511_SAA (0559)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_saa"))


In [ ]:
# CONCILIA PATRIMONIO BANCARIO
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '34', PROC = '0560a' WHERE C_CAGENTE = '361' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 321"))
_log("UPDATE BD_CTSI sector 321 CA 361->34", res.rowcount)

res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51021', PROC = '0560a' WHERE C_CAGENTE = '321' AND C_SCN = 'AF.5' AND C_ENTRADA = 'D' AND SECTOR = 353"))
_log("UPDATE BD_CTSI sector 353 CA 321->51021", res.rowcount)

res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51021', PROC = '0560a' WHERE C_CAGENTE = '321' AND C_SCN = 'AF.5' AND C_ENTRADA = 'D' AND SECTOR = 361"))
_log("UPDATE BD_CTSI sector 361 CA 321->51021", res.rowcount)

res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51022', PROC = '0560a' WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 411"))
_log("UPDATE BD_CTSI sector 411 CA 511->51022", res.rowcount)


In [ ]:
# SELECCIONA PAT BANCOS SECTORIZADO PARA GOB, EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
# CIERRE 2021: EN OFIS NO SE DEBE SELECCIONAR LOS FONDOS MUTUOS NI DE INVERSIÓN, PARA NO ELIMINAR SU TENENCIA DE ACCIONES DE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321"))
sql_af5_321 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(3)), 3) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0560b' AS PROC
INTO #af5_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 321
  AND T1.C_CAGENTE NOT IN ('511','351','352','353','34','341','53','36904','3390101','3390102','339011','33901')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(3)), 3)
"""
work_conn.execute(text(sql_af5_321))

# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE BANCOS EN SECTORES GOB, EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
sql_af5_321_act = """
INSERT INTO #af5_321
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0560b' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE = '321'
  AND T1.SECTOR NOT IN (511,351,352,353,34,341,53,36904,339001,3390102,339011,33901)
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_321_act))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_imp"))
sql_af5_321_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_321_imp
FROM #af5_321
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_321_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN GOBIERNO QUE AJUSTA EN 5101. CIERRE 2021: SI ES SECTOR 36 SE RESTA DEL MISMO SECTOR CON CA 53 Y NO EN 51022
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 41 THEN '5101' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_321_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_321_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_321_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0560b)", res.rowcount)

for t in ['#af5_53_imp', '#af5_321_imp', '#af5_321']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE AF5 DE RESTO DE EMPRESAS CON BANCOS: SELECCIONA PAT BANCOS SECTORIZADO CON RM Y SOC DE INVERSIONES. CIERRE 2021: INCORPORA CONTRAGENTE 36912
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_6"))
sql_af5_321_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 6 AS SECTOR, LEFT(CAST(T1.SECTOR AS varchar(3)), 3) AS C_CAGENTE,
       T1.C_CUENTA, 'D' AS C_ENTRADA, SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0560c' AS PROC
INTO #af5_321_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 321 AND T1.C_CAGENTE IN ('6','36','36912')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, LEFT(CAST(T1.SECTOR AS varchar(3)), 3)
"""
work_conn.execute(text(sql_af5_321_6))

# SELECCIONA ACTIVO DE AF.5 DEL RM CON CONTRAGENTE BANCOS PARA CREAR INV DE SECTOR 53 EN 321
sql_af5_rm_act = """
INSERT INTO #af5_321_6
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0560c' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 6 AND T1.FUENTE = 'CI'
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_rm_act))


In [ ]:
# CALCULA DIFERENCIAL A IMPUTAR EN SECTOR 51022 CON BANCOS: MISMO QUE SE IMPUTARÁ EN PASIVO DE EMPRESAS CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_dif_53"))
sql_af5_dif_53 = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_dif_53
FROM #af5_321_6
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_dif_53))

# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF5 DE RESTO DE EMPRESAS CON CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_dif_aj"))
sql_af5_dif_aj = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_dif_aj
FROM #af5_dif_53
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_dif_aj))

# IMPUTA AF PASIVO EN RESTO DE EMPRESAS CON CA RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pas_r_rm"))
sql_af5_pas_r_rm = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_pas_r_rm
FROM #af5_dif_53
"""
work_conn.execute(text(sql_af5_pas_r_rm))

# IMPUTA AF PASIVO EN RESTO DE EMPRESAS CON CA RESTO DEL MUNDO (ajuste espejo)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pas_r_r"))
sql_af5_pas_r_r = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA, DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_pas_r_r
FROM #af5_pas_r_rm
"""
work_conn.execute(text(sql_af5_pas_r_r))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_dif_53"))
_log("APPEND TABLAS.BD_CTSI desde AF5_DIF_53", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_dif_aj"))
_log("APPEND TABLAS.BD_CTSI desde AF5_DIF_AJ", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_pas_r_rm"))
_log("APPEND TABLAS.BD_CTSI desde AF5_PAS_R_RM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_pas_r_r"))
_log("APPEND TABLAS.BD_CTSI desde AF5_PAS_R_R", res.rowcount)

for t in ['#af5_dif_aj', '#af5_321_6', '#af5_dif_53', '#af5_pas_r_rm', '#af5_pas_r_r']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO AF5 DEL RM CON CONTRAGENTE SOC DE INVERSIONES (36). REBAJA DE SECTOR RM CON CA 53 EN AF5 ACTIVO. ADEMÁS POR ESTE MISMO MONTO IMPUTA PASIVO AF5 DE SECTOR 36 CON CA RM
# SELECCIONA INVERSION AF5 DE SECTOR 36 CON CA BANCOS PARA IMPUTARLO EN SECTOR 6 CON CA 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_321"))
sql_af5_36_321 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 6 AS SECTOR, '36' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0560d' AS PROC
INTO #af5_36_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (36,36912) AND T1.C_CAGENTE = '321'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_36_321))

# AJUSTA ANTERIOR EN SECTOR RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_53"))
sql_af5_6_53 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA, DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_53
FROM #af5_36_321
"""
work_conn.execute(text(sql_af5_6_53))

# AJUSTA IMPUTACION DE PASIVO EN AF5 DEL SECTOR 51022 CON CONTRAGENTE 53. CIERRE 2021: SE MANTIENE ESTE AJUSTE EN PATRIMONIO DE EMPRESAS PORQ SINO AUMENTA MUCHO SU VALOR
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_pas"))
sql_af5_51022_pas = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA, DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_51022_pas
FROM #af5_36_321
"""
work_conn.execute(text(sql_af5_51022_pas))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_321"))
_log("APPEND TABLAS.BD_CTSI desde AF5_36_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_53"))
_log("APPEND TABLAS.BD_CTSI desde AF5_6_53 (0560d)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_pas"))
_log("APPEND TABLAS.BD_CTSI desde AF5_51022_PAS (0560d)", res.rowcount)

# nota: el bloque AF5_36_PAS quedó comentado (anulado) en el SAS original — no se traduce
for t in ['#af5_6_53', '#af5_36_321', '#af5_51022_pas']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO SEGUROS E ISAPRES
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51022', PROC = '0561a' WHERE C_CAGENTE = '53' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 353"))
_log("UPDATE BD_CTSI sector 353 CA 53->51022", res.rowcount)

# SELECCIONA PAT SEGUROS E ISAPRES SECTORIZADO PARA EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS. CIERRE 2021: EN OFIS NO SE INCLUYEN FONDOS MUTUOS NI DE INVERSIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35"))
sql_af5_35 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(3)), 3) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0561b' AS PROC
INTO #af5_35
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (351,352,353)
  AND T1.C_CAGENTE NOT IN ('511','412','41','53','36904','3390101','3390102','339011','33901')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(3)), 3)
"""
work_conn.execute(text(sql_af5_35))

# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE SEGUROS E ISAPRES EN SECTORES EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
sql_af5_35_act = """
INSERT INTO #af5_35
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0561b' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE IN ('35','351','352','353','351/352')
  AND T1.SECTOR NOT IN (511,412,41,53,36904,3390101,3390102,339011,33901)
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_35_act))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35_imp"))
sql_af5_35_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_35_imp
FROM #af5_35
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_35_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR 36 QUE LO CAMBIA A 51022 CON CA 53 (comentado en SAS) Y EN SECTOR 6 QUE LO RESTA DE CA 33, SECTOR 361 A CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '33' WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_35_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_35_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_35_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0561b)", res.rowcount)

for t in ['#af5_53_imp', '#af5_35_imp', '#af5_35']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO AF5 DEL RM CON CONTRAGENTE SOC DE INVERSIONES (36) POR IMPUTACION DE SEGUROS E ISAPRES. REBAJA DE SECTOR RM CON CA 33 EN AF5 ACTIVO. ADEMÁS POR ESTE MISMO MONTO IMPUTA PASIVO AF5 DE SECTOR 36 CON CA RM
# SELECCIONA INVERSION AF5 DE SECTOR 36 CON CA SEGUROS PARA IMPUTARLO EN SECTOR 6 CON CA 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_35"))
sql_af5_36_35 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, 6 AS SECTOR, '36' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0561d' AS PROC
INTO #af5_36_35
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (36,36912) AND T1.C_CAGENTE IN ('351','352')
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_36_35))

# AJUSTA ANTERIOR EN SECTOR RM CA 33
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_53_2"))
sql_af5_6_53_2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '33' AS C_CAGENTE, C_CUENTA, C_ENTRADA, DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_53_2
FROM #af5_36_35
"""
work_conn.execute(text(sql_af5_6_53_2))

# nota: el bloque AF5_36_PAS quedó comentado (anulado) en el SAS original — no se traduce
# AJUSTA IMPUTACION DE PASIVO EN AF5 DEL SECTOR 51022 CON CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_pas_2"))
sql_af5_51022_pas_2 = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA, DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_51022_pas_2
FROM #af5_36_35
"""
work_conn.execute(text(sql_af5_51022_pas_2))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_35"))
_log("APPEND TABLAS.BD_CTSI desde AF5_36_35 (0561d)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_53_2"))
_log("APPEND TABLAS.BD_CTSI desde AF5_6_53 (0561d)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_pas_2"))
_log("APPEND TABLAS.BD_CTSI desde AF5_51022_PAS (0561d)", res.rowcount)

for t in ['#af5_6_53_2', '#af5_36_35', '#af5_51022_pas_2']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO DE AUXILIARES FINANCIEROS
# SELECCIONA PAT AUXILIARES SECTORIZADO EN TODOS MENOS HOGARES, CORREDORAS Y RESTO. CIERRE 2021: TAMPOCO SE CONSIDERA FONDOS MUTUOS E INVERSIÓN, INCORPORA COMO SECTOR EL 334 QUE AHORA ESTA EN AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36"))
sql_af5_36 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(6)), 6) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0562b' AS PROC
INTO #af5_36
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND CAST(T1.SECTOR AS varchar(6)) LIKE '%36%'
       AND T1.C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 37
       AND T1.C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 334
       AND T1.C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(6)), 6)
"""
work_conn.execute(text(sql_af5_36))

# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE AUXILIARES EN SECTORES EXCEPTO HOGARES, CORREDORES DE BOLSA Y RESTO
sql_af5_36_act = """
INSERT INTO #af5_36
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0562b' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE LIKE '%36%' AND T1.SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE LIKE '%37%' AND T1.SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE LIKE '%334%' AND T1.SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_36_act))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_imp"))
sql_af5_36_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_36_imp
FROM #af5_36
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_36_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR 6 QUE LO RESTA DE CA 33 Y SECTOR 361 A CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '33' WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_36_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_36_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0562b)", res.rowcount)

for t in ['#af5_53_imp', '#af5_36_imp', '#af5_36']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO AF5 EN SECTOR 36 CON CA 53, PORQUE SE IMPUTARON INVERSIONES EN ESTE SECTOR POR LOS AUXILIARES. RESTA ESE AF5 PASIVO EN SECTOR 51022 CON CONTRAGENTE 53
# nota: el bloque AF5_36_36 quedó comentado (anulado) en el SAS original — no se traduce
# AJUSTA ANTERIOR EN SECTOR 51022 CA 33
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_53"))
sql_af5_51022_53 = """
SELECT 'p' AS MONEDA, T1.AÑO, T1.TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, T1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0562c' AS PROC
INTO #af5_51022_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (36,36912) AND T1.C_CAGENTE LIKE '%36%')
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (36,36912) AND T1.C_CAGENTE = '37')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_51022_53))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_53"))
_log("APPEND TABLAS.BD_CTSI desde AF5_51022_53 (0562c)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_53"))


In [ ]:
# CONCILIA PATRIMONIO DE OFIS. SELECCIONA PAT OFIS SECTORIZADO EN TODOS MENOS HOGARES, CORREDORAS Y RESTO. CIERRE 2021: TAMPOCO SE INCLUYEN FONDOS MUTUOS E INVERSIÓN, PARA RESPETAR SUS DATOS ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33"))
sql_af5_33 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(6)), 6) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0563b' AS PROC
INTO #af5_33
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND CAST(T1.SECTOR AS varchar(6)) LIKE '%33%'
       AND T1.C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','3390102','339011','33901'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 411
       AND T1.C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','3390102','339011','33901'))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(6)), 6)
"""
work_conn.execute(text(sql_af5_33))

# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE OFIS EN TODOS MENOS HOGARES, CORREDORAS Y RESTO
sql_af5_33_act = """
INSERT INTO #af5_33
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0563b' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE LIKE '%33%' AND T1.SECTOR NOT IN (511,34,53,36904,3390101,3390102,339011,33901))
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.C_CAGENTE LIKE '%411%' AND T1.SECTOR NOT IN (511,34,53,36904,3390101,3390102,339011,33901))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_33_act))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33_imp"))
sql_af5_33_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_33_imp
FROM #af5_33
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_33_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR RM QUE LO CAMBIA A 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '36' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_33_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_33_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_33_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0563b)", res.rowcount)

for t in ['#af5_33', '#af5_33_imp', '#af5_53_imp']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO EN AUXILIARES (SOC DE INV 36) CON CA RM, USANDO INFO DE IMPUTACIÓN REALIZADA EN PROC 0563b: SE ASUME QUE EL RM INVIERTE EN SOC DE INV PARA A TRAVÉS DE ESTAS INV INDIRECTAMENTE EN AUXILIARES
# LA MISMA INV IMPUTADA EN AF ACTIVO 6 CON CA 36 SE IMPUTA EN PASIVO DE SECTOR 36 CON CA RM Y LA REBAJA DE SECTOR 36 CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_6"))
sql_af5_36_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, 36 AS SECTOR, '6' AS C_CAGENTE, T1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0563e' AS PROC
INTO #af5_36_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.PROC = '0563b' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '36'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_36_6))

# REBAJA DE SECTOR 36 CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_53"))
sql_af5_36_53 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, 36 AS SECTOR, '53' AS C_CAGENTE, T1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0563e' AS PROC
INTO #af5_36_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.PROC = '0563b' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '36'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af5_36_53))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_6"))
_log("APPEND TABLAS.BD_CTSI desde AF5_36_6 (0563e)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_53"))
_log("APPEND TABLAS.BD_CTSI desde AF5_36_53 (0563e)", res.rowcount)

for t in ['#af5_36_6', '#af5_36_53']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO RESTO DEL MUNDO
# 1. CONCILIA RM CON BCO CENTRAL. RESPETA DATO DEL BCENTRAL Y AJUSTA EN RM CON CA 53
# SELECCIONA PAT RM CON BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_31"))
sql_af5_6_31 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0564a' AS PROC
INTO #af5_6_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '31'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_6_31))

# SELECCIONA ACTIVO AF5 DEL BC CON RM
sql_af5_31_6 = """
INSERT INTO #af5_6_31
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(6)), 6) AS C_CAGENTE, T1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0564a' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 31 AND T1.C_CAGENTE = '6'
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(6)), 6)
"""
work_conn.execute(text(sql_af5_31_6))


In [ ]:
# CALCULA DIF A IMPUTAR EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_imp"))
sql_af5_6_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_imp
FROM #af5_6_31
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_6_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_6_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_6_IMP (0564a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0564a)", res.rowcount)

for t in ['#af5_6_31', '#af5_6_imp', '#af5_53_imp']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))

# CIERRE 2021: CAMBIA CONTRAGENTE EN PASIVO AF5 DEL RM DESDE 3390101 A 33901
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '33901' WHERE SECTOR = 6 AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND C_CAGENTE = '3390101'"))
_log("UPDATE BD_CTSI sector 6 CA 3390101->33901", res.rowcount)


In [ ]:
# 2. CONCILIA RM CON BCOS, SEGUROS, OFIS Y PENSIONES. RESPETA DATO DEL RM IMPUTANDO DIFERENCIAL EN SECTORES CON CA RM Y AJUSTA EN SECTORES CON CA 53
# SELECCIONA PAT RM CON SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_sect"))
sql_af5_6_sect = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(6)), 6) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0564b' AS PROC
INTO #af5_6_sect
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 6 AND T1.C_CAGENTE NOT IN ('31','511','53','9')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, LEFT(CAST(T1.SECTOR AS varchar(6)), 6), CAST(T1.C_CAGENTE AS int)
"""
work_conn.execute(text(sql_af5_6_sect))

# SELECCIONA ACTIVO AF5 DE SECTORES CON RM
sql_af5_sect_6 = """
INSERT INTO #af5_6_sect
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(8)), 8), 1, 5) AS int) AS SECTOR,
       T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0564b' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR NOT IN (31,511,53,9) AND T1.C_CAGENTE = '6'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(8)), 8), 1, 5) AS int), T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_sect_6))


In [ ]:
# CALCULA DIF A IMPUTAR EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_imp2"))
sql_af5_6_imp2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_imp2
FROM #af5_6_sect
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_6_imp2))

# nota: CIERRE 2021 comenta el UPDATE que cambiaba SECTOR 33->36 en AF5_6_IMP — no se aplica (bloque anulado en el SAS original)
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp2"))
sql_af5_53_imp2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp2
FROM #af5_6_imp2
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp2))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_imp2"))
_log("APPEND TABLAS.BD_CTSI desde AF5_6_IMP (0564b)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp2"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0564b)", res.rowcount)

for t in ['#af5_6_sect', '#af5_6_imp2', '#af5_53_imp2']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO DE EMPRESAS
# SELECCIONA PAT EMPRESAS CON SECTORES A CONCILIAR. CIERRE 2021: EXCEPTO EN SECTORES FONDOS MUTUOS E INVERSIÓN, PARA RESPETAR SUS DATOS ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51_pas"))
sql_af5_51_pas = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(6)), 6) AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0565a' AS PROC
INTO #af5_51_pas
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR IN (51021,5101)
  AND T1.C_CAGENTE NOT IN ('511','35','351','352','353','34','341','36904','53','3390101','3390102','339011','33901')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LEFT(CAST(T1.SECTOR AS varchar(6)), 6)
"""
work_conn.execute(text(sql_af5_51_pas))

# SELECCIONA ACTIVO AF5 DE SECTORES CON EMPRESAS SUPERVISADAS
sql_af5_sect_51 = """
INSERT INTO #af5_51_pas
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0565a' AS PROC
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5' AND T1.SECTOR NOT IN (511,35,351,352,353,34,341,36904,3390101,3390102,339011,33901)
  AND T1.C_CAGENTE IN ('51021','5101','2','51')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af5_sect_51))


In [ ]:
# CALCULA DIF A IMPUTAR EN SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_sect_imp"))
sql_af5_sect_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_sect_imp
FROM #af5_51_pas
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_sect_imp))

# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO SECTOR 361 QUE AJUSTA CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp_final"))
sql_af5_53_imp_final = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA, SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp_final
FROM #af5_sect_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC,
         CASE WHEN SECTOR = 361 THEN '321' ELSE '53' END
"""
work_conn.execute(text(sql_af5_53_imp_final))

cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_sect_imp"))
_log("APPEND TABLAS.BD_CTSI desde AF5_SECT_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp_final"))
_log("APPEND TABLAS.BD_CTSI desde AF5_53_IMP (0565a final)", res.rowcount)

for t in ['#af5_51_pas', '#af5_sect_imp', '#af5_53_imp_final']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pasivo"))
# CALCULA TOTAL PASIVO POR CADA SECTOR
sql_af5_pasivo = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CAST(T2.C_SI_publ AS char(4)), 4) END AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0560' AS PROC
INTO    #af5_pasivo
FROM    TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE   (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5') AND T1.SECTOR = T2.C_SI
GROUP BY t1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CAST(T2.C_SI_publ AS char(4)), 4) END
"""
work_conn.execute(text(sql_af5_pasivo))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_activo"))
# CALCULA TOTAL PASIVO POR CADA SECTOR (activo con signo invertido)
# GROUP BY por CALCULATED C_CAGENTE = LEFT(PUT(T2.C_SI_publ,CHAR4.)), no por C_SI_publ crudo
sql_af5_activo = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        LEFT(CAST(T2.C_SI_publ AS char(4)), 4) AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO) * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0560' AS PROC
INTO    #af5_activo
FROM    TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_CONTRAPARTIDAS T2
WHERE   (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5') AND T1.C_CAGENTE = T2.C_CAGENTE
GROUP BY t1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         LEFT(CAST(T2.C_SI_publ AS char(4)), 4)
"""
work_conn.execute(text(sql_af5_activo))


In [ ]:
# DATA AF5_PASIVO; SET AF5_PASIVO AF5_ACTIVO; RUN; -> concatenación server-side
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pasivo_full"))
cols_af5_pasivo = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_af5_pasivo_full = f"""
SELECT {cols_af5_pasivo}
INTO #af5_pasivo_full
FROM #af5_pasivo
UNION ALL
SELECT {cols_af5_pasivo}
FROM #af5_activo
"""
work_conn.execute(text(sql_af5_pasivo_full))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_delta"))
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
sql_af5_delta = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        T1.PROC
INTO    #af5_delta
FROM    #af5_pasivo_full T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af5_delta))
af5_delta = pd.read_sql(text("SELECT * FROM #af5_delta"), work_conn)
_log("af5_delta", af5_delta)


In [ ]:
# APPEND server-side de AF5_DELTA hacia TABLAS.BD_CTSI
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_delta = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af5_delta
"""
res = work_conn.execute(text(sql_append_delta))
_log("APPEND TABLAS.dbo.BD_CTSI (AF5_DELTA)", res.rowcount)


In [ ]:
# DROP TABLE AF5_PASIVO, AF5_ACTIVO, AF5_DELTA — limpieza de temporales de sesión
for t in ["#af5_pasivo", "#af5_activo", "#af5_pasivo_full", "#af5_delta"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_10_CCom

Concilia créditos comerciales (AF.7/AF.71/AF.9) entre pares de sectores contrapartida (hogares, empresas, bancos, seguros, auxiliares, gobierno, resto del mundo), imputando activo/pasivo faltante con reglas de reparto 55%/45% y ajuste de conciliación, cerrando la diferencia final en el sector 51022

*confianza: medium · verificador: approve · SAS: PROC SQL con más de 20 CREATE TABLE encadenados + PROC DATASETS APPEND FORCE, sobre tabla temporal de sesión BD_CTSI*

In [ ]:
# ========= S2_10_CCom =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES no aplica en SQL Server; se preserva el contenido)
work_conn.execute(text("SELECT * INTO #bd_ctsi_tmp FROM TABLAS.dbo.BD_CTSI"))
_log("BD_CTSI recompresion no-op", None)


In [ ]:
# CIERRE 2021: AJUSTES DE CONTRAGENTE EN EMPRESAS Y HOLDINGS
sql_update_cagente = """
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '53', PROC = '0600a'
WHERE SECTOR IN (51021, 5101, 37) AND C_SCN = 'AF.7' AND (C_CAGENTE = '' OR C_CAGENTE = '71')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_update_cagente))
    _log("UPDATE TABLAS.dbo.BD_CTSI cagente 53", res.rowcount)


In [ ]:
# IMPUTA ACTIVO DE CREDITOS COMERCIALES EN SECTOR HOGARES CON CA 51, USANDO INFO DEL PASIVO DEL SECTOR 51 CON CA 511 (NEGATIVO DE ESTO)
work_conn.execute(text("DROP TABLE IF EXISTS #af7_511"))
sql_af7_511 = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    511 AS SECTOR,
    '51' AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(T1.DATO * -1) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0602' AS PROC
INTO #af7_511
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND T2.C_SI_SCN_N1 = 'S.11' AND T3.C_SI_SCN_N0 = 'S.14'
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA
"""
work_conn.execute(text(sql_af7_511))


In [ ]:
# ELIMINA PASIVO DE CREDITOS COMERCIALES EN SECTOR 51 CON CA 511 (USA CONSULTA ANTERIOR)
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51"))
sql_af7_51 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    51 AS SECTOR,
    '511' AS C_CAGENTE,
    T1.C_CUENTA,
    'H' AS C_ENTRADA,
    T1.DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0603' AS PROC
INTO #af7_51
FROM #af7_511 T1
"""
work_conn.execute(text(sql_af7_51))


In [ ]:
# IMPUTA INFO DE ACTIVO DE CREDITOS COMERCIALES EN SECTOR 51 CON CA 53, USANDO EL NEGATIVO DEL PASIVO DEL SECTOR 51 CON CA 511
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51_53"))
sql_af7_51_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    51 AS SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    T1.DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0604' AS PROC
INTO #af7_51_53
FROM #af7_511 T1
"""
work_conn.execute(text(sql_af7_51_53))


In [ ]:
# APPEND server-side de las tres imputaciones anteriores a la tabla base
cols_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_tbl in ["#af7_511", "#af7_51", "#af7_51_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)

for tmp_tbl in ["#af7_511", "#af7_51", "#af7_51_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DEL SECTOR 352 CON CA 6, USANDO INFO DEL PASIVO DEL SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_6"))
sql_af7_352_6 = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
    LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(T1.DATO) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0614' AS PROC
INTO #af7_352_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND T1.C_CAGENTE = '352' AND T1.SECTOR = 6
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af7_352_6))


In [ ]:
# IMPUTA CRED COMERCIALES EN PASIVO DE SECTOR 352 CON CA 511 USANDO 55% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_511"))
sql_af7_352_511 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '511' AS C_CAGENTE,
    T1.C_CUENTA,
    'H' AS C_ENTRADA,
    T1.DATO * 0.55 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0615' AS PROC
INTO #af7_352_511
FROM #af7_352_6 T1
"""
work_conn.execute(text(sql_af7_352_511))


In [ ]:
# IMPUTA CRED COMERCIALES EN PASIVO DE SECTOR 352 CON CA 51 USANDO 45% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_51"))
sql_af7_352_51 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '51' AS C_CAGENTE,
    T1.C_CUENTA,
    'H' AS C_ENTRADA,
    T1.DATO * 0.45 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0616' AS PROC
INTO #af7_352_51
FROM #af7_352_6 T1
"""
work_conn.execute(text(sql_af7_352_51))


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DE SECTOR 511 CON CA 352, USANDO 55% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_511_352"))
sql_af7_511_352 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    511 AS SECTOR,
    '352' AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    T1.DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0616' AS PROC
INTO #af7_511_352
FROM #af7_352_511 T1
"""
work_conn.execute(text(sql_af7_511_352))


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DE SECTOR 51 CON CA 352, USANDO 45% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51_352"))
sql_af7_51_352 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    51 AS SECTOR,
    '352' AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    T1.DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0617' AS PROC
INTO #af7_51_352
FROM #af7_352_51 T1
"""
work_conn.execute(text(sql_af7_51_352))


In [ ]:
# APPEND server-side de las 5 tablas de créditos comerciales sector 352
for tmp_tbl in ["#af7_352_6", "#af7_352_511", "#af7_352_51", "#af7_511_352", "#af7_51_352"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)

for tmp_tbl in ["#af7_352_6", "#af7_352_511", "#af7_352_51", "#af7_511_352", "#af7_51_352"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DEL RESTO DEL MUNDO CON CA BCO CENTRAL Y GOBIERNO EN EL ACTIVO DE LOS SECTORES.
# REALIZA CONTRA AJUSTE EN CA 53, EXCEPTO EN BCENTRAL QUE AJUSTA EN AF.71 CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af7_rm_p"))
sql_af7_rm_p = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.SECTOR = 6 THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
    '6' AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0618' AS PROC
INTO #af7_rm_p
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND T1.SECTOR IN (31, 41) AND T1.C_CAGENTE = '6')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('31', '41'))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.SECTOR = 6 THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END
"""
work_conn.execute(text(sql_af7_rm_p))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 EN GOBIERNO Y EN AF71 CA RM EN BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af7_53_6"))
sql_af7_53_6 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    CASE WHEN T1.SECTOR = 31 THEN '6' ELSE '53' END AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    T1.DATO * -1 AS DATO,
    CASE WHEN T1.SECTOR = 31 THEN 'AF.71' ELSE T1.C_SCN END AS C_SCN,
    CASE WHEN T1.SECTOR = 31 THEN 'Ajuste conciliación' ELSE T1.N_SCN END AS N_SCN,
    T1.FUENTE,
    '0618b' AS PROC
INTO #af7_53_6
FROM #af7_rm_p T1
"""
work_conn.execute(text(sql_af7_53_6))

for tmp_tbl in ["#af7_rm_p", "#af7_53_6"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_rm_p", "#af7_53_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES EN ACTIVO DE LOS BANCOS, EXCEPTO CRUCE BANCOS CON BANCOS, CA 41 Y 53.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_bcos_a"))
sql_af7_bcos_a = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.C_ENTRADA = 'H' AND T1.C_CAGENTE <> '3' THEN TRY_CAST(T1.C_CAGENTE AS float)
         WHEN T1.C_ENTRADA = 'H' AND T1.C_CAGENTE = '3' THEN 32
         ELSE T1.SECTOR END AS SECTOR,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0619' AS PROC
INTO #af7_bcos_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND T1.SECTOR IN (321, 322, 32) AND T1.C_CAGENTE NOT IN ('321', '322', '32', '41', '53', '9'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND T1.SECTOR NOT IN (321, 41) AND T1.C_CAGENTE IN ('321', '322', '32', '3'))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.C_ENTRADA = 'H' AND T1.C_CAGENTE <> '3' THEN TRY_CAST(T1.C_CAGENTE AS float)
         WHEN T1.C_ENTRADA = 'H' AND T1.C_CAGENTE = '3' THEN 32
         ELSE T1.SECTOR END,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END
"""
work_conn.execute(text(sql_af7_bcos_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_321_53"))
sql_af7_321_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0619b' AS PROC
INTO #af7_321_53
FROM #af7_bcos_a T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af7_321_53))

for tmp_tbl in ["#af7_bcos_a", "#af7_321_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_bcos_a", "#af7_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES Y OFIS EN ACTIVO DE LOS AUXILIARES FINANCIEROS.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_aux_a"))
sql_af7_aux_a = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0620' AS PROC
INTO #af7_aux_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) = '36' AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '33', '37'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '33', '37') AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '37'))
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END
"""
work_conn.execute(text(sql_af7_aux_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #af7_36_53"))
sql_af7_36_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0620b' AS PROC
INTO #af7_36_53
FROM #af7_aux_a T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af7_36_53))

for tmp_tbl in ["#af7_aux_a", "#af7_36_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_aux_a", "#af7_36_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES, OFIS Y SEGUROS EN ACTIVO DE LOS SEGUROS.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_seg_a"))
sql_af7_seg_a = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0621' AS PROC
INTO #af7_seg_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) = '35' AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '33', '37', '35'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '33', '37', '35') AND SUBSTRING(T1.C_CAGENTE, 1, 2) = '35' AND T1.C_CAGENTE <> '351_34')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END
"""
work_conn.execute(text(sql_af7_seg_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_35_53"))
sql_af7_35_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0621b' AS PROC
INTO #af7_35_53
FROM #af7_seg_a T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af7_35_53))

for tmp_tbl in ["#af7_seg_a", "#af7_35_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_seg_a", "#af7_35_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES BCENTRAL, AUXILIARES, OFIS Y SEGUROS EN ACTIVO DE GOBIERNO.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_gob_a"))
sql_af7_gob_a = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0622' AS PROC
INTO #af7_gob_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND T1.SECTOR IN (4, 41, 42, 412) AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '33', '37', '35', '31'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '33', '37', '35', '31') AND SUBSTRING(T1.C_CAGENTE, 1, 1) = '4' AND T1.C_CAGENTE <> '411')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END
"""
work_conn.execute(text(sql_af7_gob_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_4_53"))
sql_af7_4_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0622b' AS PROC
INTO #af7_4_53
FROM #af7_gob_a T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af7_4_53))

for tmp_tbl in ["#af7_gob_a", "#af7_4_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_gob_a", "#af7_4_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES Y SEGUROS EN ACTIVO DE RESTO DEL MUNDO.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_rm_a"))
sql_af7_rm_a = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0623' AS PROC
INTO #af7_rm_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7' AND T1.SECTOR = 6 AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '37', '35'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '37', '35') AND T1.C_CAGENTE = '6')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
    CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END
"""
work_conn.execute(text(sql_af7_rm_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR RM
work_conn.execute(text("DROP TABLE IF EXISTS #af7_6_53"))
sql_af7_6_53 = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0623b' AS PROC
INTO #af7_6_53
FROM #af7_rm_a T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af7_6_53))

for tmp_tbl in ["#af7_rm_a", "#af7_6_53"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af7_rm_a", "#af7_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# CIERRA CREDITOS COMERCIALES IMPUTANDO DIFERENCIA ENTRE PASIVO Y ACTIVO AL SECTOR 51022 CON CA CORRESPONDIENTE
# CALCULA TOTAL PASIVO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af7_pasivo_1"))
sql_af7_pasivo_1 = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    51022 AS SECTOR,
    CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LTRIM(CAST(T2.C_SI_publ AS varchar(4))) END AS C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    SUM(T1.DATO) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0648' AS PROC
INTO #af7_pasivo_1
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.7'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LTRIM(CAST(T2.C_SI_publ AS varchar(4))) END
"""
work_conn.execute(text(sql_af7_pasivo_1))


In [ ]:
# CALCULA TOTAL ACTIVO POR CADA SECTOR
# CORREGIDO: GROUP BY usa el C_CAGENTE ya formateado a varchar(4) (CALCULATED en el SAS original),
# no T2.C_SI_publ crudo, para que dos valores que truncan al mismo string se agrupen igual que en SAS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_activo"))
sql_af7_activo = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    51022 AS SECTOR,
    LTRIM(CAST(T2.C_SI_publ AS varchar(4))) AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) * -1 AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0648' AS PROC
INTO #af7_activo
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T2 ON T1.C_CAGENTE = T2.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.7'
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
    LTRIM(CAST(T2.C_SI_publ AS varchar(4)))
"""
work_conn.execute(text(sql_af7_activo))


In [ ]:
# DATA AF7_PASIVO; SET AF7_PASIVO AF7_ACTIVO; -> UNION ALL server-side en una sola #tmp
work_conn.execute(text("DROP TABLE IF EXISTS #af7_pasivo"))
sql_af7_pasivo_union = f"SELECT {cols_ctsi} INTO #af7_pasivo FROM #af7_pasivo_1 UNION ALL SELECT {cols_ctsi} FROM #af7_activo"
work_conn.execute(text(sql_af7_pasivo_union))


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_delta"))
sql_af7_delta = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    T1.C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    T1.PROC
INTO #af7_delta
FROM #af7_pasivo T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af7_delta))

sql_append_delta = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM #af7_delta"
res = work_conn.execute(text(sql_append_delta))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_delta", res.rowcount)

for tmp_tbl in ["#af7_delta", "#af7_pasivo", "#af7_pasivo_1", "#af7_activo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# GENERA ACTIVO NETO DE ERRORES Y OMISIONES Y DISCREPANCIA ESTADÍSTICAS, ELIMINANDO EL PASIVO E IMPUTANDOLO CON SIGNO CONTRARIO EN EL ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af71_9_elimina"))
sql_af71_9_elimina = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    T1.C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO * -1) AS DATO,
    T1.C_SCN,
    T1.N_SCN,
    'PS' AS FUENTE,
    '0654' AS PROC
INTO #af71_9_elimina
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.71', 'AF.9')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af71_9_elimina))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_9_netea"))
sql_af71_9_netea = """
SELECT
    T1.MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    T1.C_CAGENTE,
    T1.C_CUENTA,
    'D' AS C_ENTRADA,
    T1.DATO,
    T1.C_SCN,
    T1.N_SCN,
    T1.FUENTE,
    '0655' AS PROC
INTO #af71_9_netea
FROM #af71_9_elimina T1
"""
work_conn.execute(text(sql_af71_9_netea))

for tmp_tbl in ["#af71_9_elimina", "#af71_9_netea"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af71_9_elimina", "#af71_9_netea"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# IMPUTA ACTIVO DE AF71 EN SECTOR 51 CON CA 53, USANDO TOTAL ACTIVO DE AF71 MULTIPLICADO POR AJUSTE DE CONCILIACIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #af71_51"))
sql_af71_51 = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    51 AS SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO * T2.Porcentaje * -1) AS DATO,
    T1.C_SCN,
    CASE WHEN T1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE T1.N_SCN END AS N_SCN,
    'PS' AS FUENTE,
    '0656' AS PROC
INTO #af71_51
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_AJUSTE_CONCILIACION T2 ON T1.AÑO = T2.Año AND T1.TRIM = T2.Trim
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.71', 'AF.9') AND T1.DATO <> 0
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN,
    CASE WHEN T1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE T1.N_SCN END
"""
work_conn.execute(text(sql_af71_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_511"))
sql_af71_511 = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    511 AS SECTOR,
    '53' AS C_CAGENTE,
    T1.C_CUENTA,
    T1.C_ENTRADA,
    SUM(T1.DATO * (1 - T2.Porcentaje) * -1) AS DATO,
    T1.C_SCN,
    CASE WHEN T1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE T1.N_SCN END AS N_SCN,
    'PS' AS FUENTE,
    '0657' AS PROC
INTO #af71_511
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_AJUSTE_CONCILIACION T2 ON T1.AÑO = T2.Año AND T1.TRIM = T2.Trim
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.71', 'AF.9') AND T1.DATO <> 0
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN,
    CASE WHEN T1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE T1.N_SCN END
"""
work_conn.execute(text(sql_af71_511))

for tmp_tbl in ["#af71_51", "#af71_511"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM {tmp_tbl}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_tbl}", res.rowcount)
for tmp_tbl in ["#af71_51", "#af71_511"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_tbl}"))


In [ ]:
# IMPUTA SALDOS DE CTA FINANCIERA
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))
sql_saldos = """
SELECT
    'P' AS MONEDA,
    T1.AÑO,
    T1.TRIM,
    T1.SECTOR,
    T1.C_CUENTA,
    'H' AS C_ENTRADA,
    SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
    CASE WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
         WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
         WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3'
         WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2' END AS C_SCN,
    CASE WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
         WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
         WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales'
         WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen' END AS N_SCN,
    'PS' AS FUENTE,
    '0009' AS PROC
INTO #saldos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CUENTA IN ('Financiera', 'Bce Final', 'Bce Inicio', 'Rec Volumen', 'Rec Precio', 'Rec Precio Reaj')
GROUP BY T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA,
    CASE WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
         WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
         WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3'
         WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2' END,
    CASE WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
         WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
         WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales'
         WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen' END
"""
work_conn.execute(text(sql_saldos))

sql_append_saldos = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ctsi}) SELECT {cols_ctsi} FROM #saldos"
res = work_conn.execute(text(sql_append_saldos))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))


## S2_11_CNF

Reclasifica y reconcilia rentas de la propiedad (intereses, dividendos, renta atribuida a fondos/seguros) entre sectores institucionales según reglas de cierre trimestral, imputando contrapartidas donde el dato de origen no las trae / Reclasifica y ajusta rentas de la propiedad (D.42/D.43/D.45) por sector institucional imputando aperturas por contraparte según estructura de AF.5, y acumula los ajustes en la base consolidada BD_CTSI / Reclasifica y cierra rentas de la propiedad (D.5, D.61, D.62, D.71, D.72, D.75, D.8) entre sectores institucionales vía múltiples CREATE/APPEND/UPDATE sobre TABLAS.BD_CTSI, ajustando intereses del Banco Central según estructura de activos que devengan interés / Reclasifica y elimina/reasigna intereses (D.41) entre sectores (Gobierno, Bancos, Resto del Mundo, Empresas) usando proporciones calculadas desde saldos de activos financieros, y ajusta con datos DCV para bonos pagados por Gobierno a Bancos y Empresas / Reclasifica y ajusta rentas de la propiedad (D.41/SIFMI) entre bancos, OFIs, seguros, auxiliares y resto del mundo mediante imputaciones proporcionales y contrapartidas en CA, y cierra intereses del sector 51022 comparando recibido vs pagado / Reclasifica y cierra rentas (D.9, D.41, B.2, B.8) por sector: imputa transferencias de capital, elimina duplicidad en cuentas de producción, calcula excedente de explotación/ingreso mixto ajustado por SIFMI, imputa valor agregado y diferenciales de FBCF, y recalcula el ahorro por sector / Elimina e imputa Ahorro/CCF/B.9 entre cuentas de Capital, YG y Producción, ajusta el sector Gobierno y Resto del Mundo por conciliación macro-CNT, ajusta totales D.42/D.75 y su efecto en Ahorro, imputa pagos del sector 51 al RM del cierre 2019, aplica el ajuste de bonos y arranca el ajuste del sector Seguros/Auxiliares y Banco Central por conciliación de cartera financiera / Reclasifica e imputa saldos y flujos (rec. volumen, cta financiera, balance/saldos, ajustes B.8/B.9, remuneraciones y variaciones de valor neto) por sector en la tabla de series de cuentas nacionales, acumulando cada imputación y depurando cuentas de balance/ceros ya consolidadas

*confianza: low · verificador: revise · SAS: PROC SQL: serie de UPDATE de reclasificación de rentas (D.41/D.42/D.44/D.443) + CREATE TABLE/APPEND server-side de ajustes e imputaciones por sector, sobre tabla temporal de sesión + PROC SQL con CREATE TABLE + GROUP BY/CALCULATED + PROC APPEND FORCE (server-side sobre #tmp), imputación de dividendos y utilidades reinvertidas por sector + PROC SQL CREATE TABLE + PROC DATASETS APPEND FORCE + PROC SQL UPDATE (múltiples bloques encadenados sobre WORK como #tmp de sesión) + PROC SQL CREATE TABLE + PROC DATASETS APPEND + UPDATE encadenados sobre TABLAS.BD_CTSI, con tablas de sesión (#tmp) + PROC SQL: múltiples CREATE TABLE con GROUP BY/CASE + PROC DATASETS APPEND FORCE contra TABLAS.BD_CTSI (tramo 5/8, session_sql) + PROC SQL CREATE TABLE + PROC DATASETS APPEND + UPDATE, encadenados sobre TABLAS.BD_CTSI vía tablas de sesión #tmp + PROC SQL CREATE TABLE (agregaciones/reclasificaciones) + PROC APPEND + PROC DATASETS DROP, todo sobre TABLAS.BD_CTSI vía #tmp de sesión + PROC SQL CREATE TABLE (session #tmp) + PROC DATASETS APPEND FORCE + DELETE, encadenados sobre tabla de BD*

In [ ]:
# ========= S2_11_CNF =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con un
# borrado + reinserción deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# EN SECTOR SEG DE VIDA CAMBIA PRODUCCION A INTERESES RECIBIDOS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.41', N_SCN = 'Intereses', PROC = '0680'
WHERE C_SCN = 'P.11' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 351
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0680", res.rowcount)


In [ ]:
# EN SECTORES 34,341,351 CAMBIA INTERESES Y DIVIDENDOS PAGADOS A D.44
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.44', N_SCN = 'Renta de la propiedad atribuida a los titulares de pólizas de seguros', PROC = '0681'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR = 351 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0681 (sector 351)", res.rowcount)
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.44', N_SCN = 'Renta de la propiedad atribuida a los titulares de pólizas de seguros', PROC = '0681'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (34,341) AND C_CAGENTE = '511'
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0681 (sector 34,341)", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA CONTRAGENTE EN FONDOS DE INVERSION EN INTERESES RECIBIDOS SIN CONTRAGENTE
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '53', PROC = '0681b'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 339011 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0681b", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA INSTRUMENTO EN FONDOS DE INVERSION EN DIVIDENDOS RECIBIDOS DESDE CA 339011 PORQUE DEBE SER D.443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0681c'
WHERE C_SCN = 'D.42' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 339011 AND C_CAGENTE = '339011'
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0681c", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA CONTRAGENTE EN MUTUALIDADES DEL INST D443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390101', PROC = '0681b'
WHERE C_SCN IN ('D.443') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 412 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0681b (sector 412)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA D443 EN GASTO DEL RESTO DEL MUNDO USANDO DATO INGRESO DE FONDOS DE PENSIONES CON RM. AJUSTE EN DIVIDENDOS DEL RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #d443_rm_fp"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '34' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783a' AS PROC
INTO #d443_rm_fp
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CUENTA IN ('YG') AND T1.SECTOR IN (34,341) AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.443' AND T1.C_CAGENTE = '6'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""))


In [ ]:
# D42_RM_53: reversa el D443_RM_FP como D.42 con contragente 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_53"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'D.42' AS C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d42_rm_53
FROM #d443_rm_fp T1
"""))


In [ ]:
# APPEND server-side de D443_RM_FP y D42_RM_53 a la tabla base (columnas explícitas por el FORCE de PROC APPEND)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_rm_fp"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D443_RM_FP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_rm_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D42_RM_53", res.rowcount)
for t in ["#d443_rm_fp", "#d42_rm_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA TRANSACCIONES DE LA CTA DE PRODUCCION, YG Y CAPITAL DE SECTORES 412 Y YG/CAPITAL EN 411.
# SOLO LO HACE EN SECTOR 412...NO EN 411 PORQUE ESO TENIA SENTIDO SOLO CUANDO LAS CAJAS SE INCORPORABAN EN GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #drop_412_411"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783' AS PROC
INTO #drop_412_411
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA IN ('YG','Capital','Producción') AND T1.SECTOR = 412)
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #drop_412_411"))
_log("APPEND TABLAS.dbo.BD_CTSI desde DROP_412_411", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_412_411"))


In [ ]:
# CAMBIA_D443_412: nota SAS trae una coma de más antes de 'PS' AS FUENTE (','N_SCN,0 'PS' AS FUENTE') que en T-SQL es sintácticamente inválida;
# se traduce respetando la intención original (columna N_SCN y FUENTE='PS'), descartando el literal huérfano '0' que no tiene destino de columna
work_conn.execute(text("DROP TABLE IF EXISTS #cambia_d443_412"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR, 'Otras sociedades' AS N_SI, '51' AS C_SI_publ, 'Sociedades no financieras' AS N_SI_publ, '3390101' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783b' AS PROC
INTO #cambia_d443_412
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA IN ('YG') AND T1.SECTOR = 412 AND T1.C_SCN = 'D.443' AND T1.PROC = '0783')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))


In [ ]:
# APPEND de CAMBIA_D443_412: solo columnas que existen en la tabla base (N_SI, C_SI_publ, N_SI_publ son columnas de apoyo que FORCE descarta)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #cambia_d443_412"))
_log("APPEND TABLAS.dbo.BD_CTSI desde CAMBIA_D443_412", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #cambia_d443_412"))


In [ ]:
# CONCILIA TOTAL DEBE Y HABER DEL D44. MANDA EL HABER, DIF LA IMPUTA AL DEBE DEL SECTOR 35 CON CA 511.
# POR CIERRE 2019 SE AJUSTA EL DATO AL HABER, YA QUE FALTA INCORPORAR EN HOGARES LOS REAJUSTES DEL D44 DE LOS FONDOS DE PENSIONES PARA QUE HOGARES SE IGUALE A ESE DATO FINAL DE LOS FP
work_conn.execute(text("DROP TABLE IF EXISTS #d44_cierre"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       (CASE WHEN T1.SECTOR = 511 THEN T1.C_CAGENTE ELSE LTRIM(CAST(T1.SECTOR AS varchar(11))) END) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta de la propiedad atribuida a los titulares de pólizas de seguros' AS N_SCN,
       'PS' AS FUENTE,
       '0684' AS PROC
INTO #d44_cierre
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.44') OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.44')
GROUP BY T1.[AÑO], T1.TRIM, (CASE WHEN T1.SECTOR = 511 THEN T1.C_CAGENTE ELSE LTRIM(CAST(T1.SECTOR AS varchar(11))) END),
         T1.C_CUENTA, T1.C_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d44_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D44_CIERRE", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d44_cierre"))


In [ ]:
# ELIMINA INTERESES RECIBIDOS Y PAGADOS EN EL SECTOR 5111. POR CIERRE ELIMINA TMB D75,D5
work_conn.execute(text("DROP TABLE IF EXISTS #d41_5111"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0700' AS PROC
INTO #d41_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN IN ('D.41','D.75','D.5'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_5111"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D41_5111", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_5111"))


In [ ]:
# EN SECTORES 339011,3390101 y 3390102 CAMBIA INTERESES PAGADOS A D.443. CIERRE 2021: EN FONDOS DE INVERSIÓN NO SE HACE EL CAMBIO EN LOS INTERESES PORQ DEBEN PAGAR POR PTMOS CON BANCOS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0702'
WHERE C_SCN IN ('D.41') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (3390101,3390102)
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0702 (D.41)", res.rowcount)


In [ ]:
# EN SECTORES 339011 CAMBIA DIVIDENDOS PAGADOS A D.443. CIERRE 2019
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', C_CAGENTE = '53', PROC = '0702'
WHERE C_SCN IN ('D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (339011)
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0702 (D.42)", res.rowcount)


In [ ]:
# EN SECTORES QUE RECIBEN INTERESES DE FONDOS DE INV Y FONDOS MUTUOS CAMBIA INTERESES A D.443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0702'
WHERE C_SCN IN ('D.41') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('339011','3390101','3390102','33901','339')
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0702 (H fondos)", res.rowcount)


In [ ]:
# ACTUALIZA CONTRAGENTES EN SECTORES CON RENTAS RECIBIDAS DESDE LOS FONDOS PARA DEJARLO CONSISTENTE CON SU ACTIVO. CIERRE 2019
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390101', PROC = '0702b'
WHERE C_SCN = 'D.443' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('33901','339') AND SECTOR IN (321,3390101,33221,33222)
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0702b (3390101)", res.rowcount)
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390102', PROC = '0702b'
WHERE C_SCN = 'D.443' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('33901','339') AND SECTOR IN (341,351,3390102)
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0702b (3390102)", res.rowcount)


In [ ]:
# CIERRE 2021: CALCULA D443 DE LOS SECTORES CON CONTRAGENTE RM, PARA REBAJARLO DE D443 CON EL MERCADO NACIONAL,
# EN BASE A % DE CUOTAS DE FONDOS MANTENIDAS EN EL EXTERIOR SOBRE EL TOTAL DE CUOTAS APLICADO A D.443
# CUOTAS EN EL EXTERIOR CA RM DE LOS SECTORES, EXCEPTO PENSIONES QUE YA TIENE DATO Y GOBIERNO YA QUE TIENE POCO D443 RECIBIDO EN SU TOTAL
work_conn.execute(text("DROP TABLE IF EXISTS #cf_exterior"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       SUM(T1.DATO) AS DATO
INTO #cf_exterior
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CUENTA = 'Bce Final' AND T1.C_CAGENTE = '6'
  AND T1.SECTOR NOT IN (41,4,412,413,42,34,341)
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE
"""))


In [ ]:
# CUOTAS TOTALES DE LOS SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #cf_total"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       SUM(T1.DATO) AS DATO
INTO #cf_total
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CUENTA = 'Bce Final'
  AND T1.SECTOR NOT IN (41,4,412,413,42,6,34,341)
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR
"""))


In [ ]:
# PORCENTAJE PARA CREAR D443 CON EL RM
work_conn.execute(text("DROP TABLE IF EXISTS #porc_ext"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.DATO / NULLIF(T2.DATO, 0) AS DATO
INTO #porc_ext
FROM #cf_exterior T1, #cf_total T2
WHERE T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""))


In [ ]:
# D443 INGRESO CONTRAGENTE RM
work_conn.execute(text("DROP TABLE IF EXISTS #d443_rm"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, '6' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, 'PS' AS FUENTE, '0702c' AS PROC,
       SUM(T1.DATO * T2.DATO) AS DATO
INTO #d443_rm
FROM TABLAS.dbo.BD_CTSI T1, #porc_ext T2
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('D.443') AND T1.C_CUENTA = 'YG' AND T1.SECTOR NOT IN (41,4,412,413,42,6,34,341)
  AND T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA
"""))


In [ ]:
# D443 AJUSTE EN CONTRAGENTE 33901
work_conn.execute(text("DROP TABLE IF EXISTS #d443_fmnm"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, '33901' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) * -1 AS DATO
INTO #d443_fmnm
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, T1.FUENTE, T1.PROC
"""))


In [ ]:
# D443 IMPUTA EN GASTO DEL RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d443_6"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 6 AS SECTOR, '53' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, 'D' AS C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) AS DATO
INTO #d443_6
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""))


In [ ]:
# AJUSTA EN GASTO D42 DEL RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_6"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 6 AS SECTOR, '53' AS C_CAGENTE, 'YG' AS C_CUENTA, 'D.42' AS C_SCN, 'Renta distribuida de las sociedades' AS N_SCN, 'D' AS C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) * -1 AS DATO
INTO #d42_6
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.FUENTE, T1.PROC
"""))


In [ ]:
# APPEND server-side de D443_RM, D443_FMNM, D443_6, D42_6
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D443_RM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_fmnm"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D443_FMNM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_6"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D443_6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_6"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D42_6", res.rowcount)
for t in ["#d443_fmnm", "#d443_rm", "#porc_ext", "#cf_exterior", "#cf_total", "#d443_6", "#d42_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA TOTAL DEBE Y HABER DEL D443. MANDA EL TOTAL PAGADO (DEBE), DIF LA IMPUTA AL HABER (RECIBIDO) DEL SECTOR 51022 CON CA CORRESPONDIENTE
work_conn.execute(text("DROP TABLE IF EXISTS #d443_cierre"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       (CASE WHEN T1.SECTOR = 3390101 AND T1.C_ENTRADA = 'D' THEN '3390101'
             WHEN T1.SECTOR IN (3390102,339011) AND T1.C_ENTRADA = 'D' THEN '33901'
             WHEN T1.C_CAGENTE NOT IN ('3390101') THEN '33901'
             ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva' AS N_SCN,
       'PS' AS FUENTE,
       '0703' AS PROC
INTO #d443_cierre
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.443' AND T1.SECTOR IN (3390102,339011,3390101,6)) OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.443')
GROUP BY T1.[AÑO], T1.TRIM,
         (CASE WHEN T1.SECTOR = 3390101 AND T1.C_ENTRADA = 'D' THEN '3390101'
               WHEN T1.SECTOR IN (3390102,339011) AND T1.C_ENTRADA = 'D' THEN '33901'
               WHEN T1.C_CAGENTE NOT IN ('3390101') THEN '33901'
               ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, T1.C_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D443_CIERRE", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d443_cierre"))


In [ ]:
# ELIMINA RENTAS DISTRIBUIDAS RECIBIDOS Y PAGADOS EN EL SECTOR 5111 Y 51022
work_conn.execute(text("DROP TABLE IF EXISTS #d42_5111"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0701' AS PROC
INTO #d42_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN = 'D.42') OR (T1.SECTOR = 51022 AND T1.C_SCN = 'D.42' AND T1.MONEDA = 'P')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_5111"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D42_5111", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d42_5111"))


In [ ]:
# CIERRE 2019. CAMBIA CONTRAGENTE EN BANCOS DESDE AFP A FP EN DIVIDENDOS PAGADOS POR LOS BANCOS
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '34', PROC = '0800' WHERE SECTOR = 321 AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '361'"))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0800 (321)", res.rowcount)
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR LOS SEGUROS CAMBIA CONTRAGENTE GOBIERNO A RESTO DE EMPRESAS
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0800' WHERE SECTOR IN (351,352,353,35) AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '41'"))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0800 (seguros)", res.rowcount)
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR LOS EMPRESAS PRIVADAS CAMBIA CONTRAGENTE HOGARES A RESTO DE EMPRESAS, PARA LUEGO RECALCULAR EL PAGADO A HOGARES
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0800' WHERE SECTOR = 51021 AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '511'"))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0800 (51021)", res.rowcount)


In [ ]:
# CIERRE 2019. IMPUTA DIVIDENDOS PAGADOS POR EMPRESAS SUPERVISADAS PRIVADAS A HOGARES USANDO MISMO CRITERIO QUE LA IMPUTACIÓN DEL AF5 DE EMPRESAS EN HOGARES:
# UN PORCENTAJE SOBRE EL TOTAL PAGADO POR EL SECTOR Y RESTADO DE LO PAGADO AL RESTO DE LA ECONOMÍA. IMPUTACIÓN CON CA HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #d42_51021"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '511' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * 0.036281 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0801' AS PROC
INTO #d42_51021
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 51021 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'D'
GROUP BY T1.[AÑO], T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))


In [ ]:
# D42_RESTO: reversa la imputación con contragente 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_resto"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0801b' AS PROC
INTO #d42_resto
FROM #d42_51021 T1
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_51021"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D42_51021", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_resto"))
_log("APPEND TABLAS.dbo.BD_CTSI desde D42_RESTO", res.rowcount)
for t in ["#d42_51021", "#d42_resto"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. CAMBIA CONTRAGENTE vacíos en todos los sectores a 53 en los dividendos
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0802' WHERE C_SCN = 'D.42' AND (C_CAGENTE IS NULL OR C_CAGENTE IN ('','512'))"))
_log("UPDATE TABLAS.dbo.BD_CTSI PROC=0802", res.rowcount)


In [ ]:
# CIERRE 2019. EN DIVIDENDOS RECIBIDOS POR EL RESTO DEL MUNDO REALIZA APERTURA POR CONTRAPARTIDA UTILIZANDO LA ESTRUCTURA DEL AF5 ACTIVO DEL SECTOR GENERADO EN LA SÍNTESIS
# BALANCE FINAL AF5 ACTIVO RESTO DEL MUNDO CON CA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_a_ca"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       (CASE WHEN T1.C_CAGENTE = '351/352' THEN '351' ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_rm_a_ca
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.[AÑO], T1.SECTOR, T1.TRIM, (CASE WHEN T1.C_CAGENTE = '351/352' THEN '351' ELSE T1.C_CAGENTE END), T1.C_CUENTA, T1.C_ENTRADA
"""))
af5_rm_a_ca = pd.read_sql(text("SELECT * FROM #af5_rm_a_ca"), work_conn)
_log("af5_rm_a_ca", af5_rm_a_ca)


In [ ]:
# BALANCE FINAL AF5 ACTIVO RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_a"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_rm_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.[AÑO], T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA
"""))
af5_rm_a = pd.read_sql(text("SELECT * FROM #af5_rm_a"), work_conn)
_log("af5_rm_a", af5_rm_a)


In [ ]:
# ESTRUCTURA (% de cada contragente sobre el total del activo AF5 del RM)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_est"))
work_conn.execute(text("""
SELECT T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / NULLIF(T2.DATO, 0) AS DATO
INTO #af5_rm_est
FROM #af5_rm_a_ca T1, #af5_rm_a T2
WHERE T1.SECTOR = T2.SECTOR AND T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM
"""))
af5_rm_est = pd.read_sql(text("SELECT * FROM #af5_rm_est"), work_conn)
_log("af5_rm_est", af5_rm_est)


In [ ]:
# ESTRUCTURA: proporción de contraparte AF5_RM_P por sector/trim/año
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_est_p"))
sql_af5_rm_est_p = """
SELECT t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
       t1.DATO / t2.DATO AS DATO
INTO #af5_rm_est_p
FROM #af5_rm_p_ca t1
INNER JOIN #af5_rm_p t2
    ON t1.SECTOR = t2.SECTOR AND t1.AÑO = t2.AÑO AND t1.TRIM = t2.TRIM
"""
work_conn.execute(text(sql_af5_rm_est_p))


In [ ]:
# IMPUTACIÓN APERTURA DIVIDENDOS PAGADOS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_pag"))
sql_d42_rm_pag = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t2.C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO * t2.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0804' AS PROC
INTO #d42_rm_pag
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN #af5_rm_est_p t2
    ON t1.SECTOR = t2.SECTOR AND t1.AÑO = t2.AÑO AND t1.TRIM = t2.TRIM
WHERE t1.SECTOR = 6 AND t1.C_SCN = 'D.42' AND t1.C_ENTRADA = 'D'
GROUP BY t1.AÑO, t1.SECTOR, t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t2.C_CAGENTE
"""
work_conn.execute(text(sql_d42_rm_pag))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_p_aj"))
sql_d42_rm_p_aj = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0804b' AS PROC
INTO #d42_rm_p_aj
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.SECTOR = 6 AND t1.C_SCN = 'D.42' AND t1.C_ENTRADA = 'D'
GROUP BY t1.MONEDA, t1.AÑO, t1.SECTOR, t1.TRIM, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d42_rm_p_aj))


In [ ]:
# PROC APPEND: D42_RM_PAG y D42_RM_P_AJ hacia TABLAS.BD_CTSI (server-side, desde #tmp)
cols_bd_ctsi_pag = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_pag = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_pag})
SELECT {cols_bd_ctsi_pag}
FROM #d42_rm_pag
"""
res = work_conn.execute(text(sql_append_pag))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_PAG)", res.rowcount)


In [ ]:
cols_bd_ctsi_aj = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_aj = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_aj})
SELECT {cols_bd_ctsi_aj}
FROM #d42_rm_p_aj
"""
res = work_conn.execute(text(sql_append_aj))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_P_AJ)", res.rowcount)


In [ ]:
# DROP TABLE D42_RM_PAG, D42_RM__PAJ (nombre con typo en el SAS original, no existe -> se ignora), AF5_RM_P_CA, AF5_RM_P, AF5_RM_EST_P
for t in ["#d42_rm_pag", "#af5_rm_p_ca", "#af5_rm_p", "#af5_rm_est_p"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS RECIBIDOS POR BANCOS REALIZA APERTURA POR CONTRAPARTIDA UTILIZANDO LA ESTRUCTURA DEL AF5 ACTIVO DEL SECTOR GENERADO EN LA SÍNTESIS
# BALANCE FINAL AF5 ACTIVO CON CA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_a_ca"))
sql_af5_bcos_a_ca = """
SELECT t1.AÑO, t1.TRIM,
       321 AS SECTOR,
       t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO
INTO #af5_bcos_a_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.SECTOR IN (321, 32) AND t1.C_SCN = 'AF.5' AND t1.C_ENTRADA = 'D' AND t1.C_CUENTA = 'Bce Final'
GROUP BY t1.AÑO, t1.TRIM, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA
"""
work_conn.execute(text(sql_af5_bcos_a_ca))


In [ ]:
# BALANCE FINAL AF5 ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_a"))
sql_af5_bcos_a = """
SELECT t1.AÑO, t1.TRIM,
       321 AS SECTOR,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO
INTO #af5_bcos_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.SECTOR IN (321, 32) AND t1.C_SCN = 'AF.5' AND t1.C_ENTRADA = 'D' AND t1.C_CUENTA = 'Bce Final'
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA
"""
work_conn.execute(text(sql_af5_bcos_a))


In [ ]:
# ESTRUCTURA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_est"))
sql_af5_bcos_est = """
SELECT t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
       t1.DATO / t2.DATO AS DATO
INTO #af5_bcos_est
FROM #af5_bcos_a_ca t1
INNER JOIN #af5_bcos_a t2
    ON t1.SECTOR = t2.SECTOR AND t1.AÑO = t2.AÑO AND t1.TRIM = t2.TRIM
"""
work_conn.execute(text(sql_af5_bcos_est))


In [ ]:
# IMPUTACIÓN APERTURA DIVIDENDOS RECIBIDOS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_rec"))
sql_d42_bcos_rec = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t2.C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO * t2.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0804c' AS PROC
INTO #d42_bcos_rec
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN #af5_bcos_est t2
    ON t1.SECTOR = t2.SECTOR AND t1.AÑO = t2.AÑO AND t1.TRIM = t2.TRIM
WHERE t1.SECTOR = 321 AND t1.C_SCN = 'D.42' AND t1.C_ENTRADA = 'H'
GROUP BY t1.AÑO, t1.SECTOR, t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t2.C_CAGENTE
"""
work_conn.execute(text(sql_d42_bcos_rec))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_aj"))
sql_d42_bcos_aj = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0804d' AS PROC
INTO #d42_bcos_aj
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.SECTOR = 321 AND t1.C_SCN = 'D.42' AND t1.C_ENTRADA = 'H'
GROUP BY t1.MONEDA, t1.AÑO, t1.SECTOR, t1.TRIM, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d42_bcos_aj))


In [ ]:
cols_bd_ctsi_bcos_rec = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_bcos_rec = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_bcos_rec})
SELECT {cols_bd_ctsi_bcos_rec}
FROM #d42_bcos_rec
"""
res = work_conn.execute(text(sql_append_bcos_rec))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_REC)", res.rowcount)


In [ ]:
cols_bd_ctsi_bcos_aj = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_bcos_aj = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_bcos_aj})
SELECT {cols_bd_ctsi_bcos_aj}
FROM #d42_bcos_aj
"""
res = work_conn.execute(text(sql_append_bcos_aj))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_AJ)", res.rowcount)


In [ ]:
for t in ["#d42_bcos_rec", "#d42_bcos_aj", "#af5_bcos_est", "#af5_bcos_a", "#af5_bcos_a_ca"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR EL RESTO DEL MUNDO IMPUTA LO DE BCENTRAL, BCOS, OFIS, SEGUROS y FP. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_p"))
sql_d42_rm_p = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0805' AS PROC
INTO #d42_rm_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42' AND t1.SECTOR = 6 AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('31','32','33','35','34'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))), 1, 2) IN ('31','32','33','35','34') AND t1.C_CAGENTE = '6')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_rm_p))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d42_6_53"))
sql_d42_6_53 = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE,
       '0805b' AS PROC
INTO #d42_6_53
FROM #d42_rm_p t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d42_6_53))


In [ ]:
cols_bd_ctsi_rm_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_rm_p = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_rm_p})
SELECT {cols_bd_ctsi_rm_p}
FROM #d42_rm_p
"""
res = work_conn.execute(text(sql_append_rm_p))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_P)", res.rowcount)


In [ ]:
cols_bd_ctsi_6_53 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_6_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_6_53})
SELECT {cols_bd_ctsi_6_53}
FROM #d42_6_53
"""
res = work_conn.execute(text(sql_append_6_53))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_6_53)", res.rowcount)


In [ ]:
for t in ["#d42_6_53", "#d42_rm_p"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR EL BANCOS IMPUTA LO DE BANCOS,SEGUROS y CORREDORES DE BOLSA (EL RESTO DE AUXILIARES SE MANTIENE SEGÚN LO QUE DICE BANCOS). AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_p"))
sql_d42_bcos_p = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0806' AS PROC
INTO #d42_bcos_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42' AND t1.SECTOR IN (321,322,32) AND t1.C_CAGENTE IN ('321','35','351','352','353','36904'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42' AND t1.SECTOR IN (321,35,351,352,353,36904) AND t1.C_CAGENTE IN ('321','322','32','3'))
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_bcos_p))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_53"))
sql_d42_bcos_53 = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE,
       '0806b' AS PROC
INTO #d42_bcos_53
FROM #d42_bcos_p t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d42_bcos_53))


In [ ]:
cols_bd_ctsi_bcos_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_bcos_p = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_bcos_p})
SELECT {cols_bd_ctsi_bcos_p}
FROM #d42_bcos_p
"""
res = work_conn.execute(text(sql_append_bcos_p))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_P)", res.rowcount)


In [ ]:
cols_bd_ctsi_bcos_53 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_bcos_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_bcos_53})
SELECT {cols_bd_ctsi_bcos_53}
FROM #d42_bcos_53
"""
res = work_conn.execute(text(sql_append_bcos_53))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_53)", res.rowcount)


In [ ]:
for t in ["#d42_bcos_p", "#d42_bcos_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR OFIS IMPUTA LO DE BANCOS. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_ofis_p"))
sql_d42_ofis_p = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0807' AS PROC
INTO #d42_ofis_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42' AND (SUBSTRING(LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))), 1, 2) = '33' OR t1.SECTOR = 411) AND t1.C_CAGENTE = '321')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42' AND t1.SECTOR = 321 AND SUBSTRING(t1.C_CAGENTE, 1, 2) = '33')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_ofis_p))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR BANCOS (OFIS)
work_conn.execute(text("DROP TABLE IF EXISTS #d42_ofis_53"))
sql_d42_ofis_53 = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE,
       '0807b' AS PROC
INTO #d42_ofis_53
FROM #d42_ofis_p t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d42_ofis_53))


In [ ]:
cols_bd_ctsi_ofis_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_ofis_p = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_ofis_p})
SELECT {cols_bd_ctsi_ofis_p}
FROM #d42_ofis_p
"""
res = work_conn.execute(text(sql_append_ofis_p))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_OFIS_P)", res.rowcount)


In [ ]:
cols_bd_ctsi_ofis_53 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_ofis_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_ofis_53})
SELECT {cols_bd_ctsi_ofis_53}
FROM #d42_ofis_53
"""
res = work_conn.execute(text(sql_append_ofis_53))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_OFIS_53)", res.rowcount)


In [ ]:
for t in ["#d42_ofis_p", "#d42_ofis_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR SEGUROS IMPUTA LO DE BANCOS. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_seg_p"))
sql_d42_seg_p = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0808' AS PROC
INTO #d42_seg_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))), 1, 2) = '35' AND t1.C_CAGENTE = '321')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42' AND t1.SECTOR = 321 AND SUBSTRING(t1.C_CAGENTE, 1, 2) = '35')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_seg_p))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_seg_53"))
sql_d42_seg_53 = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE,
       '0808b' AS PROC
INTO #d42_seg_53
FROM #d42_seg_p t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d42_seg_53))


In [ ]:
cols_bd_ctsi_seg_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_seg_p = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_seg_p})
SELECT {cols_bd_ctsi_seg_p}
FROM #d42_seg_p
"""
res = work_conn.execute(text(sql_append_seg_p))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_SEG_P)", res.rowcount)


In [ ]:
cols_bd_ctsi_seg_53 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_seg_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_seg_53})
SELECT {cols_bd_ctsi_seg_53}
FROM #d42_seg_53
"""
res = work_conn.execute(text(sql_append_seg_53))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_SEG_53)", res.rowcount)


In [ ]:
for t in ["#d42_seg_p", "#d42_seg_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR AUXILIARES IMPUTA LO DE BANCOS Y LO DEL RESTO DEL MUNDO (SOLO CA 36).
# AJUSTA EN CA 53 SOLO LO DE BANCOS POR LO DEL RM CON CA 36 SE DEBE ADICIONAR AL PAGADO POR AUXILIARES YA QUE ES EL SECTOR NO MEDIDO
work_conn.execute(text("DROP TABLE IF EXISTS #d42_aux_p"))
sql_d42_aux_p = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0809' AS PROC
INTO #d42_aux_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))), 1, 2) IN ('36','37') AND t1.C_CAGENTE = '321')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42' AND t1.SECTOR = 321 AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36','37'))
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(RTRIM(CONVERT(varchar(7), t1.SECTOR))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_aux_p))


In [ ]:
# NOTA: bloque DELETE FROM D42_AUX_P comentado en el SAS original (código anulado, no se traduce)
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL AUXILIARES..SOLO LA PARTE DE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_aux_53"))
sql_d42_aux_53 = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE,
       '0809b' AS PROC
INTO #d42_aux_53
FROM #d42_aux_p t1
WHERE t1.C_CAGENTE = '321'
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d42_aux_53))


In [ ]:
cols_bd_ctsi_aux_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_aux_p = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_aux_p})
SELECT {cols_bd_ctsi_aux_p}
FROM #d42_aux_p
"""
res = work_conn.execute(text(sql_append_aux_p))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_AUX_P)", res.rowcount)


In [ ]:
cols_bd_ctsi_aux_53 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_aux_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_aux_53})
SELECT {cols_bd_ctsi_aux_53}
FROM #d42_aux_53
"""
res = work_conn.execute(text(sql_append_aux_53))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_AUX_53)", res.rowcount)


In [ ]:
for t in ["#d42_aux_p", "#d42_aux_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
# NOTA: el bloque D42_AUX_R completo está comentado en el SAS original (código anulado, no se traduce)


In [ ]:
# CONCILIA TOTAL DEBE Y HABER DEL D42. MANDA EL HABER, DIF LA IMPUTA AL DEBE DEL SECTOR 51022 CON CA RESPECTIVO
# CALCULA TOTAL RECIBIDO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #d42_recibido"))
sql_d42_recibido = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51022 AS SECTOR,
       CASE WHEN t1.SECTOR = 51022 THEN '53' ELSE LTRIM(RTRIM(CONVERT(varchar(4), t2.C_SI_publ))) END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0704' AS PROC
INTO #d42_recibido
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION t2 ON t1.SECTOR = t2.C_SI
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.42'
GROUP BY t1.AÑO, t1.TRIM, t1.C_ENTRADA, t1.C_CUENTA, t1.C_SCN, t1.N_SCN,
         CASE WHEN t1.SECTOR = 51022 THEN '53' ELSE LTRIM(RTRIM(CONVERT(varchar(4), t2.C_SI_publ))) END
"""
work_conn.execute(text(sql_d42_recibido))


In [ ]:
# UPDATE C_CAGENTE 8->511 en #d42_recibido
res = work_conn.execute(text("UPDATE #d42_recibido SET C_CAGENTE = '511' WHERE C_CAGENTE = '8'"))
_log("UPDATE #d42_recibido C_CAGENTE 8->511", res.rowcount)


In [ ]:
# CALCULA TOTAL PAGADO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #d42_pagado"))
sql_d42_pagado = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51022 AS SECTOR,
       LTRIM(RTRIM(CONVERT(varchar(4), t2.C_SI_publ))) AS C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0704' AS PROC
INTO #d42_pagado
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS t2 ON t1.C_CAGENTE = t2.C_CAGENTE
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.42'
GROUP BY t1.AÑO, t1.TRIM, t1.C_ENTRADA, t1.C_CUENTA, t1.C_SCN, t1.N_SCN,
         LTRIM(RTRIM(CONVERT(varchar(4), t2.C_SI_publ)))
"""
work_conn.execute(text(sql_d42_pagado))


In [ ]:
res = work_conn.execute(text("UPDATE #d42_pagado SET C_CAGENTE = '511' WHERE C_CAGENTE = '8'"))
_log("UPDATE #d42_pagado C_CAGENTE 8->511", res.rowcount)


In [ ]:
# DATA D42_RECIBIDO; SET D42_RECIBIDO D42_PAGADO; (concatenación server-side)
work_conn.execute(text("DROP TABLE IF EXISTS #d42_recibido_2"))
sql_concat_recibido = """
SELECT * INTO #d42_recibido_2 FROM #d42_recibido
UNION ALL
SELECT * FROM #d42_pagado
"""
work_conn.execute(text(sql_concat_recibido))
work_conn.execute(text("DROP TABLE IF EXISTS #d42_recibido"))
work_conn.execute(text("EXEC sp_rename '#d42_recibido_2', '#d42_recibido'"))


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #d42_delta"))
sql_d42_delta = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
       t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
INTO #d42_delta
FROM #d42_recibido t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.C_ENTRADA, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.SECTOR, t1.C_CAGENTE, t1.FUENTE, t1.PROC
"""
work_conn.execute(text(sql_d42_delta))


In [ ]:
cols_bd_ctsi_delta = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_delta = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_delta})
SELECT {cols_bd_ctsi_delta}
FROM #d42_delta
"""
res = work_conn.execute(text(sql_append_delta))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_DELTA)", res.rowcount)


In [ ]:
for t in ["#d42_delta", "#d42_pagado", "#d42_recibido"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS RECIBIDAS POR SECTOR 51021 CON CA 53, USANDO INFO DE LO PAGADO POR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d43_h_snf"))
sql_d43_h_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51021 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       t1.DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0705' AS PROC
INTO #d43_h_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.43' AND t1.SECTOR = 6
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.DATO
"""
work_conn.execute(text(sql_d43_h_snf))


In [ ]:
cols_bd_ctsi_d43h = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d43h = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d43h})
SELECT {cols_bd_ctsi_d43h}
FROM #d43_h_snf
"""
res = work_conn.execute(text(sql_append_d43h))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_H_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d43_h_snf"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS PAGADAS POR SECTOR 51021 CON CA 53, USANDO INFO DE LO RECIBIDO POR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_snf"))
sql_d43_d_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51021 AS SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       t1.DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0706' AS PROC
INTO #d43_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.43' AND t1.SECTOR = 6
GROUP BY t1.AÑO, t1.TRIM, t1.C_CAGENTE, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.DATO
"""
work_conn.execute(text(sql_d43_d_snf))


In [ ]:
cols_bd_ctsi_d43d = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d43d = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d43d})
SELECT {cols_bd_ctsi_d43d}
FROM #d43_d_snf
"""
res = work_conn.execute(text(sql_append_d43d))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_snf"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS PAGADAS POR SECTOR 32 CON CA 6. AJUSTA EN SECTOR 51 CON CA 6. CIERRE 2021: INCORPORA APERTURA ENTRE BANCOS Y SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_sf"))
sql_d43_d_sf = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       t1.SECTOR,
       '6' AS C_CAGENTE,
       'YG' AS C_CUENTA,
       'D' AS C_ENTRADA,
       t1.DATO,
       'D.43' AS C_SCN,
       'Utilidades reinvertidas de la inversión extranjera directa' AS N_SCN,
       'PS' AS FUENTE,
       '0706b' AS PROC
INTO #d43_d_sf
FROM TABLAS.dbo.UR_SF_CR18 t1
"""
work_conn.execute(text(sql_d43_d_sf))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_aj"))
sql_d43_d_aj = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'YG' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       'D.43' AS C_SCN,
       'Utilidades reinvertidas de la inversión extranjera directa' AS N_SCN,
       'PS' AS FUENTE,
       '0706c' AS PROC
INTO #d43_d_aj
FROM TABLAS.dbo.UR_SF_CR18 t1
GROUP BY t1.AÑO, t1.TRIM
"""
work_conn.execute(text(sql_d43_d_aj))


In [ ]:
cols_bd_ctsi_d43sf = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d43sf = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d43sf})
SELECT {cols_bd_ctsi_d43sf}
FROM #d43_d_sf
"""
res = work_conn.execute(text(sql_append_d43sf))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_SF)", res.rowcount)


In [ ]:
cols_bd_ctsi_d43aj = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d43aj = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d43aj})
SELECT {cols_bd_ctsi_d43aj}
FROM #d43_d_aj
"""
res = work_conn.execute(text(sql_append_d43aj))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_AJ)", res.rowcount)


In [ ]:
for t in ["#d43_d_sf", "#d43_d_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA D45 PAGADO EN SECTOR 51022, USANDO INFO DE LO RECIBIDO POR SECTOR 41
work_conn.execute(text("DROP TABLE IF EXISTS #d45_d_snf"))
sql_d45_d_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, t1.TRIM,
       51022 AS SECTOR,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       t1.DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE,
       '0707' AS PROC
INTO #d45_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.45' AND t1.SECTOR = 41
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.DATO
"""
work_conn.execute(text(sql_d45_d_snf))


In [ ]:
cols_bd_ctsi_d45 = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d45 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d45})
SELECT {cols_bd_ctsi_d45}
FROM #d45_d_snf
"""
res = work_conn.execute(text(sql_append_d45))
_log("APPEND TABLAS.dbo.BD_CTSI (D45_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d45_d_snf"))
# AJUSTA D5: continúa en el tramo siguiente


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d5_d_snf"))
sql_d5_d_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0711' AS PROC
INTO #d5_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='D.5') OR (T1.C_ENTRADA='D' AND T1.C_SCN='D.5' AND T1.SECTOR NOT IN (412))
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d5_d_snf))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.D5_D_SNF FORCE (server-side, la #tmp se dropea después)
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d5_d_snf = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d5_d_snf
"""
res = work_conn.execute(text(sql_append_d5_d_snf))
_log("APPEND TABLAS.dbo.BD_CTSI (D5_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d5_d_snf"))


In [ ]:
# AJUSTA D61
work_conn.execute(text("DROP TABLE IF EXISTS #d61_d"))
sql_d61_d = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       511 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0715' AS PROC
INTO #d61_d
FROM TABLAS.dbo.BD_CTSI t1
-- SECTOR NOT IN (/*411,*/412): el 411 quedó comentado en el SAS original, no se traduce
WHERE (T1.C_SCN='D.61' AND T1.SECTOR NOT IN (412))
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d61_d))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d61_d"))
_log("APPEND TABLAS.dbo.BD_CTSI (D61_D)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d61_d"))


In [ ]:
# AJUSTA D62
work_conn.execute(text("DROP TABLE IF EXISTS #d62_d"))
sql_d62_d = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       511 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0719' AS PROC
INTO #d62_d
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.62' AND T1.SECTOR NOT IN (412))
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d62_d))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d62_d"))
_log("APPEND TABLAS.dbo.BD_CTSI (D62_D)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d62_d"))


In [ ]:
# AJUSTA D8
work_conn.execute(text("DROP TABLE IF EXISTS #d8"))
sql_d8 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       34 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0722' AS PROC
INTO #d8
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.8')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN
"""
work_conn.execute(text(sql_d8))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d8"))
_log("APPEND TABLAS.dbo.BD_CTSI (D8)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d8"))


In [ ]:
# CALCULO DEL D8 EN BASE A D61 Y D62 DE SECTORES 34 Y 341. AJUSTA DIFERENCIAS EN D8
work_conn.execute(text("DROP TABLE IF EXISTS #d8_calculo"))
sql_d8_calculo = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       34 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       'D.8' AS C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0729' AS PROC
INTO #d8_calculo
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.61' AND T1.C_ENTRADA='H' AND T1.SECTOR IN (34,341)) OR (T1.C_SCN='D.62' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (34,341))
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_d8_calculo))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_inicial"))
sql_d8_inicial = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       34 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0729' AS PROC
INTO #d8_inicial
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.8' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (34,341))
GROUP BY t1.AÑO, T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_SCN
"""
work_conn.execute(text(sql_d8_inicial))


In [ ]:
# PROC APPEND BASE=WORK.D8_CALCULO DATA=WORK.D8_INICIAL FORCE (ambas #tmp de sesión)
cols_d8_calculo = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO #d8_calculo ({cols_d8_calculo}) SELECT {cols_d8_calculo} FROM #d8_inicial"))
_log("APPEND #d8_calculo (D8_INICIAL)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_ajuste"))
sql_d8_ajuste = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d8_ajuste
FROM #d8_calculo t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_d8_ajuste))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_511"))
sql_d8_511 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       511 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0731' AS PROC
INTO #d8_511
FROM #d8_ajuste t1
"""
work_conn.execute(text(sql_d8_511))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d8_ajuste"))
_log("APPEND TABLAS.dbo.BD_CTSI (D8_AJUSTE)", res.rowcount)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d8_511"))
_log("APPEND TABLAS.dbo.BD_CTSI (D8_511)", res.rowcount)


In [ ]:
for t in ["#d8_ajuste", "#d8_511", "#d8_inicial", "#d8_calculo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA D71 Y D72 CON CA 53 EN SECTORES 352 Y 353
# ADAPTADO PARA QUE SOLO LO HAGA EN 353 PORQUE EN 352 YA NO CORRESPONDE POR NUEVO MARCO
# FROM '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat': ruta absoluta hardcodeada (M-001) — no hay tabla de BD ni ruta relativa provista para este archivo SAS externo al catálogo del proyecto
raise NotImplementedError("WORK.D71_53 depende de la fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat' (dataset SAS externo, no incluido en input_datasets ni en el catálogo de conexiones del proyecto): se requiere definir cómo se accede a este archivo (ruta relativa del workspace o tabla de BD equivalente) antes de traducir este bloque")


In [ ]:
# WORK.D72_53 se deriva de WORK.D71_53 (bloque anterior no resuelto)
raise NotImplementedError("WORK.D72_53 depende de WORK.D71_53, que no se pudo materializar por falta de la fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat'")


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.D71_53 / WORK.D72_53 FORCE — bloqueado por el hueco anterior
raise NotImplementedError("APPEND de WORK.D71_53 y WORK.D72_53 a TABLAS.BD_CTSI bloqueado: ambas tablas dependen de la fuente externa no resuelta '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat'")


In [ ]:
# IMPUTA D71 Y D72 CON RM EN SECTORES 351 Y 352
# ADAPTADO PARA QUE SOLO LO HAGA EN 351 PORQUE EN 352 YA NO CORRESPONDE POR NUEVO MARCO
# FROM '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat': misma fuente externa sin resolver que WORK.D71_53
raise NotImplementedError("WORK.D71_6 depende de la fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat' (dataset SAS externo no mapeado a BD ni a ruta relativa del workspace)")


In [ ]:
raise NotImplementedError("WORK.D72_6 depende de WORK.D71_6, no materializado por la fuente externa no resuelta")


In [ ]:
raise NotImplementedError("APPEND de WORK.D71_6 y WORK.D72_6 a TABLAS.BD_CTSI bloqueado por la misma fuente externa no resuelta")


In [ ]:
# IMPUTA D71 Y D72 EN SECTOR RM CON CA 351 Y 352 USANDO INFO DE CONTRAPARTIDA. AJUSTA IMPUTACIONES CONTRA D75 DEL SECTOR RM CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d71_rm"))
sql_d71_rm = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0739' AS PROC
INTO #d71_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.71' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (351,352) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CAGENTE, T1.SECTOR, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d71_rm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d75_rm_h"))
sql_d75_rm_h = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       'D.75' AS C_SCN,
       'Transferencias corrientes diversas' AS N_SCN,
       T1.FUENTE,
       '0748' AS PROC
INTO #d75_rm_h
FROM #d71_rm t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.FUENTE
"""
work_conn.execute(text(sql_d75_rm_h))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d72_rm"))
sql_d72_rm = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0740' AS PROC
INTO #d72_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.72' AND T1.C_ENTRADA='H' AND T1.SECTOR IN (351,352) AND T1.C_CAGENTE='6')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CAGENTE, T1.SECTOR, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d72_rm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d75_rm_d"))
sql_d75_rm_d = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       'D.75' AS C_SCN,
       'Transferencias corrientes diversas' AS N_SCN,
       T1.FUENTE,
       '0749' AS PROC
INTO #d75_rm_d
FROM #d72_rm t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.FUENTE
"""
work_conn.execute(text(sql_d75_rm_d))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d71_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (D71_RM)", res.rowcount)


In [ ]:
# El SAS referencia DATA=D75_RM_H sin prefijo WORK: mismo dataset que WORK.D75_RM_H (#d75_rm_h)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_rm_h"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_RM_H)", res.rowcount)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d72_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (D72_RM)", res.rowcount)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_rm_d"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_RM_d)", res.rowcount)
for t in ["#d71_rm", "#d72_rm", "#d75_rm_h", "#d75_rm_d"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE D71. RESPETA EL TOTAL RECIBIDO, DIFERENCIA LA IMPUTA EN PAGADO DEL SECTOR 51022 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #d71_cierre"))
sql_d71_cierre = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       '352' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0744' AS PROC
INTO #d71_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.71')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d71_cierre))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d71_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI (D71_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d71_cierre"))


In [ ]:
# CIERRE D72. RESPETA EL TOTAL PAGADO, DIFERENCIA LA IMPUTA EN RECIBIDO DEL SECTOR 51022 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #d72_cierre"))
sql_d72_cierre = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       '352' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0747' AS PROC
INTO #d72_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.72')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d72_cierre))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d72_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI (D72_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d72_cierre"))


In [ ]:
# ELIMINA D75 EN SEGUROS GENERALES (>2009). YA NO APLICA CON MARCO NUEVO DE SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #d75_352"))
sql_d75_352 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0750' AS PROC
INTO #d75_352
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.SECTOR=352 AND t1.AÑO>2009)
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_352))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_352"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_352)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_352"))


In [ ]:
# IMPUTA 14% APROX DE TOTAL DE D75 RECIBIDAS EN LAS RECIBIDAS DEL SECTOR 51022 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_51022"))
sql_d75_51022 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*0.146847233207581 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0751' AS PROC
INTO #d75_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.C_ENTRADA='H' AND T1.SECTOR NOT IN (411,412,6))
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_51022))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_51022"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_51022)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_51022"))


In [ ]:
# CIERRE DE D75. COMPARA RECIBIDO Y PAGADO. RESPETA LO RECIBIDO E IMPUTA EL 65% DEL DIFERENCIAL EN LO PAGADO DEL SECTOR 51022 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre"))
sql_d75_cierre = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END)*0.65 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0755' AS PROC
INTO #d75_cierre
FROM TABLAS.dbo.BD_CTSI t1
-- SECTOR NOT IN (/*411,*/412): el 411 quedó comentado en el SAS original, no se traduce
WHERE (T1.C_SCN='D.75' AND T1.SECTOR NOT IN (412))
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_cierre))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre"))


In [ ]:
# IMPUTA AJUSTE EN TRANSF CORRIENTES PAGADAS DE EMPRESAS PARA QUE NO SEAN NEGATIVAS EN 2018 Y 2019. ESTO HARÁ QUE SE AJUSTEN LAS RECIBIDAS TMB POR DEFECTO
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=TABLAS.AJ_TCORR: fuente ya es tabla permanente de BD, append server-side directo (no requiere #tmp)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM TABLAS.dbo.AJ_TCORR"))
_log("APPEND TABLAS.dbo.BD_CTSI (AJ_TCORR)", res.rowcount)


In [ ]:
# IMPUTA LA DIFERENCIA ENTRE DEBE Y HABER DEL D75 EN EL RECIBIDO DE SECTOR 51 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre_2"))
sql_d75_cierre_2 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0755b' AS PROC
INTO #d75_cierre_2
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.SECTOR NOT IN (412))
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_cierre_2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d75_cierre_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (D75_CIERRE_2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre_2"))


In [ ]:
# CIERRE 2019. ACTUALIZA CONTRAGENTE VACÍOS EN INTERESES A 53
# C_CAGENTE='' de SAS: blanco real, se traduce literal (no viene de un JOIN, es dato propio de la tabla)
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='53', PROC='0755c' WHERE (C_CAGENTE IS NULL OR C_CAGENTE = '') AND C_SCN='D.41' AND C_CUENTA='YG'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (contragente vacío -> 53)", res.rowcount)


In [ ]:
# CIERRE 2019. ACTUALIZA CONTRAGENTE DE INTERESES RECIBIDOS DE GOBIERNO CON CA OFIS (3321) A CA BANCOS. DADO QUE FINALMENTE EL GRUESO DE LOS PRÉSTAMOS DE LP SE VA A BCOS EN LA SÍNTESIS.
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321', PROC='0755d' WHERE C_CAGENTE='3321' AND C_SCN='D.41' AND C_CUENTA='YG' AND C_ENTRADA='H' AND SECTOR=41"))
_log("UPDATE TABLAS.dbo.BD_CTSI (CA 3321->321, sector 41)", res.rowcount)


In [ ]:
# CIERRE 2021. ACTUALIZA CONTRAGENTE DE INTERESES PAGADOS POR OFIS CON CA RESTO Y EMPRESAS A CA BANCOS. DADO QUE EN LA SÍNTESIS SE GENERAN LAS CONTRAPARTIDAS Y CASI TODO INICIALMENTE VIENE CON CA BANCOS.
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321', PROC='0755d' WHERE C_CAGENTE NOT IN ('321') AND C_SCN='D.41' AND C_CUENTA='YG' AND C_ENTRADA='D' AND SECTOR IN (411,33211,33212,33221,33222,33231,33232,339011,3390102)"))
_log("UPDATE TABLAS.dbo.BD_CTSI (CA != 321 -> 321, sectores OFIS)", res.rowcount)


In [ ]:
# CIERRE 2021. AJUSTA INTERESES RECIBIDOS POR BANCO CENTRAL UTILIZANDO ESTRUCTURA DE ACTIVOS QUE DEVENGAN INTERESES EN TODOS SUS CONTRAGENTES EXCEPTO RESTO DEL MUNDO PORQUE ESO VIENEN DE LAS RESERVAS
# INTERESES TOTALES RECIBIDOS POR BCCH DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_h_ci"))
sql_d41_bcch_h_ci = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN
INTO #d41_bcch_h_ci
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR=31 AND T1.C_ENTRADA='H' AND T1.C_SCN='D.41'
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, t1.SECTOR
"""
work_conn.execute(text(sql_d41_bcch_h_ci))


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch"))
sql_act_bcch = """
SELECT t1.AÑO, T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #act_bcch
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=31 AND T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31','AF.29','AF.22') AND T1.C_CUENTA IN ('Bce Final','Bce Inicio') AND T1.C_CAGENTE<>'6'
GROUP BY t1.AÑO, T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, t1.SECTOR
"""
work_conn.execute(text(sql_act_bcch))
act_bcch = pd.read_sql(text("SELECT * FROM #act_bcch"), work_conn)
_log("act_bcch", act_bcch)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, promedio saldo final e inicial
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_2"))
sql_act_bcch_2 = """
SELECT t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_ENTRADA, AVG(T1.DATO) AS DATO
INTO #act_bcch_2
FROM #act_bcch t1
GROUP BY t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_ENTRADA
"""
work_conn.execute(text(sql_act_bcch_2))
act_bcch_2 = pd.read_sql(text("SELECT * FROM #act_bcch_2"), work_conn)
_log("act_bcch_2", act_bcch_2)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, con detalle contragente
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_c"))
sql_act_bcch_c = """
SELECT t1.AÑO, T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #act_bcch_c
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=31 AND T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31','AF.29','AF.22') AND T1.C_CUENTA IN ('Bce Final','Bce Inicio') AND T1.C_CAGENTE<>'6'
GROUP BY t1.AÑO, T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, T1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_act_bcch_c))
act_bcch_c = pd.read_sql(text("SELECT * FROM #act_bcch_c"), work_conn)
_log("act_bcch_c", act_bcch_c)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, con detalle contragente, promedio saldo final e inicial
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_c2"))
sql_act_bcch_c2 = """
SELECT t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CAGENTE, T1.C_ENTRADA, AVG(T1.DATO) AS DATO
INTO #act_bcch_c2
FROM #act_bcch_c t1
GROUP BY t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_ENTRADA, T1.C_CAGENTE
"""
work_conn.execute(text(sql_act_bcch_c2))
act_bcch_c2 = pd.read_sql(text("SELECT * FROM #act_bcch_c2"), work_conn)
_log("act_bcch_c2", act_bcch_c2)


In [ ]:
# ESTRUCTURA DE ACTIVOS POR CONTRAGENTE
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_est"))
sql_act_bcch_est = """
SELECT t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CAGENTE, T1.DATO / NULLIF(T2.DATO, 0) AS EST
INTO #act_bcch_est
FROM #act_bcch_c2 t1, #act_bcch_2 T2
WHERE T1.AÑO=T2.AÑO AND T1.TRIM=T2.TRIM
"""
work_conn.execute(text(sql_act_bcch_est))
act_bcch_est = pd.read_sql(text("SELECT * FROM #act_bcch_est"), work_conn)
_log("act_bcch_est", act_bcch_est)


In [ ]:
# INTERESES SECTORIZADOS MODIFICADOS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_def"))
sql_d41_bcch_def = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, T2.C_CAGENTE,
       T1.DATO * T2.EST AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755d1' AS PROC
INTO #d41_bcch_def
FROM #d41_bcch_h_ci T1, #act_bcch_est T2
WHERE T1.AÑO=T2.AÑO AND T1.TRIM=T2.TRIM
"""
work_conn.execute(text(sql_d41_bcch_def))
d41_bcch_def = pd.read_sql(text("SELECT * FROM #d41_bcch_def"), work_conn)
_log("d41_bcch_def", d41_bcch_def)


In [ ]:
# INTERESES SECTORIZADOS INICIALES A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_drop"))
sql_d41_bcch_drop = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755d1' AS PROC
INTO #d41_bcch_drop
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR=31 AND T1.C_ENTRADA='H' AND T1.C_SCN='D.41'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_d41_bcch_drop))


In [ ]:
# APPEND server-side de D41_BCCH_DROP y D41_BCCH_DEF (este último viene de un tramo anterior como #d41_bcch_def)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_bcch_drop"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_BCCH_DROP)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_bcch_def"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_BCCH_DEF)", res.rowcount)


In [ ]:
# DROP de las #tmp de este bloque (equivalente a PROC SQL DROP TABLE D41_BCCH_DEF, D41_BCCH_DROP)
for t in ["#d41_bcch_def", "#d41_bcch_drop"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. Ajusta intereses pagados por Gobierno a Bancos, usando préstamos totales con bancos y total de intereses pagados en la CI
# INTERESES TOTALES PAGADOS POR GOB DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_d_ci"))
sql_d41_gob_d_ci = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN
INTO #d41_gob_d_ci
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR=41 AND T1.C_ENTRADA='D' AND T1.C_SCN='D.41'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_gob_d_ci))


In [ ]:
# PASIVOS TOTALES QUE DEVENGAN INTERESES DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_h"))
sql_pas_gob_h = """
SELECT T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #pas_gob_h
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR IN (41,412,413,42) AND T1.C_ENTRADA='H'
  AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31') AND T1.C_CUENTA='Bce Final'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA
"""
work_conn.execute(text(sql_pas_gob_h))


In [ ]:
# PASIVOS TOTALES QUE DEVENGAN INTERESES DESDE CI CON BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_bcos"))
sql_pas_gob_bcos = """
SELECT T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #pas_gob_bcos
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR IN (41,412,413,42) AND T1.C_CAGENTE IN ('321','32') AND T1.C_ENTRADA='H'
  AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31') AND T1.C_CUENTA='Bce Final'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_CAGENTE
"""
work_conn.execute(text(sql_pas_gob_bcos))


In [ ]:
# % QUE REPRESENTA EL PASIVO CON BANCOS DEL TOTAL
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_est"))
sql_pas_gob_est = """
SELECT T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, CAST(T1.DATO AS float)/T2.DATO AS EST
INTO #pas_gob_est
FROM #pas_gob_bcos T1, #pas_gob_h T2
WHERE T1.[AÑO]=T2.[AÑO] AND T1.TRIM=T2.TRIM
"""
work_conn.execute(text(sql_pas_gob_est))


In [ ]:
# CALCULA INTERESES PAGADOS A BANCOS CORREGIDOS PARA REEMPLAZAR DATO CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_bcos"))
sql_d41_gob_bcos = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T2.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO*T2.EST AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755e' AS PROC
INTO #d41_gob_bcos
FROM #d41_gob_d_ci T1, #pas_gob_est T2
WHERE T1.[AÑO]=T2.[AÑO] AND T1.TRIM=T2.TRIM AND T1.SECTOR=T2.SECTOR
"""
work_conn.execute(text(sql_d41_gob_bcos))


In [ ]:
# INTERESES A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_bcos_drop"))
sql_d41_gob_bcos_drop = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO*-1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755e' AS PROC
INTO #d41_gob_bcos_drop
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR=41 AND T1.C_ENTRADA='D' AND T1.C_SCN='D.41' AND T1.C_CAGENTE='321'
"""
work_conn.execute(text(sql_d41_gob_bcos_drop))


In [ ]:
# APPEND server-side de D41_GOB_BCOS y D41_GOB_BCOS_DROP (primera pasada, antes del ajuste contra Resto del Mundo)
cols_bd_ctsi_e = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_bcos"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_BCOS)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_bcos_drop"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_BCOS_DROP)", res.rowcount)


In [ ]:
# PARA AJUSTARLO CONTRA RESTO DEL MUNDO
res = work_conn.execute(text("UPDATE #d41_gob_bcos SET DATO=DATO*-1, C_CAGENTE='6'"))
_log("UPDATE #d41_gob_bcos (ajuste RM)", res.rowcount)


In [ ]:
# PARA AJUSTARLO CONTRA RESTO DEL MUNDO
res = work_conn.execute(text("UPDATE #d41_gob_bcos_drop SET DATO=DATO*-1, C_CAGENTE='6'"))
_log("UPDATE #d41_gob_bcos_drop (ajuste RM)", res.rowcount)


In [ ]:
# Segunda pasada del APPEND, ya con el ajuste contra Resto del Mundo aplicado
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_bcos"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_BCOS ajustado)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_bcos_drop"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_BCOS_DROP ajustado)", res.rowcount)


In [ ]:
for t in ["#d41_gob_d_ci", "#pas_gob_h", "#pas_gob_bcos", "#pas_gob_est", "#d41_gob_bcos", "#d41_gob_bcos_drop"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. Ajusta intereses pagados por Gobierno a Bancos y Empresas por concepto de bonos usando DCV
# DATOS DCV INTERESES PAGADOS POR GOBIERNO A BANCOS Y EMPRESAS
# El origen es un archivo SAS externo (BASE_D41_FINAL.sas7bdat) referenciado por ruta absoluta en el SAS original.
# M-001: no hay una tabla equivalente en el catálogo de conexiones ni una ruta relativa provista para este dataset externo.
raise NotImplementedError(
    "GOB_BR requiere leer '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "(fuente DCV externa a la BD, no está en el catálogo de conexiones ni se proveyó ruta relativa de reemplazo — M-001 exige eliminar rutas hardcodeadas pero no se definió el reemplazo para este archivo)"
)


In [ ]:
# INTERESES PAGADOS POR GOB A EMPRESAS A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_53"))
sql_d41_gob_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755f' AS PROC
INTO #d41_gob_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=41 AND T1.C_ENTRADA='D' AND T1.C_SCN='D.41' AND T1.C_CAGENTE='53'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_d41_gob_53))


In [ ]:
# APPEND server-side de D41_GOB_53 y GOB_BR (primera pasada)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_53)", res.rowcount)
# GOB_BR depende del bloque NotImplementedError anterior — no se puede ejecutar el APPEND sin esa tabla
raise NotImplementedError("APPEND de GOB_BR a TABLAS.BD_CTSI depende de la tabla GOB_BR no resuelta (ver bloque DCV externo anterior)")


In [ ]:
# AJUSTA INT ELIMINADOS DE EMPRESAS Y LO LLEVA A RESTO DEL MUNDO
res = work_conn.execute(text("UPDATE #d41_gob_53 SET DATO=DATO*-1, C_CAGENTE='6'"))
_log("UPDATE #d41_gob_53 (ajuste RM)", res.rowcount)


In [ ]:
# AJUSTA INT DCV EN BCOS Y EMPRESAS CONTRA RESTO DEL MUNDO — depende de GOB_BR, no resuelta
raise NotImplementedError("UPDATE de GOB_BR depende de la tabla GOB_BR no resuelta (fuente DCV externa)")


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_e}) SELECT {cols_bd_ctsi_e} FROM #d41_gob_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_GOB_53 ajustado)", res.rowcount)
raise NotImplementedError("APPEND final de GOB_BR depende de la tabla GOB_BR no resuelta (fuente DCV externa)")


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_53"))
# #gob_br no se creó (bloque NotImplementedError) — no hay nada que dropear ahí


In [ ]:
# SIFMI de bancos con contragente 321/322/32 (fuente DI_Aj_SIFMI) reetiquetado a moneda 'P' y sector 321
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_bcos_h"))
sql_sifmi_bcos_h = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       321 AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(7))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758c' AS PROC
INTO #sifmi_bcos_h
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41' AND T1.SECTOR IN (511, 41, 36, 35)
      AND T1.C_CAGENTE IN ('321', '322', '32') AND T1.FUENTE = 'DI_Aj_SIFMI'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.SECTOR
"""
work_conn.execute(text(sql_sifmi_bcos_h))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN CA 53 DEL SECTOR BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_321_53"))
sql_sifmi_321_53 = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758d' AS PROC
INTO #sifmi_321_53
FROM #sifmi_bcos_h T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_sifmi_321_53))


In [ ]:
# APPEND server-side de ambas tablas a TABLAS.BD_CTSI (mismas columnas que la tabla base)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #sifmi_bcos_h"))
_log("APPEND TABLAS.dbo.BD_CTSI (sifmi_bcos_h)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #sifmi_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (sifmi_321_53)", res.rowcount)
for t in ["#sifmi_bcos_h", "#sifmi_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES GOB,HH, BCOS, OFIS, FP, SEGUROS, AUX, FM MM EN GASTO DE BCOS.
# REALIZA CONTRA AJUSTE EN CA 53. CIERRE 2021: AGREGA IMPUTACIÒN DE DATO BANCO CENTRAL Y 34 PORQUE AHOR ES AUXILIAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcos_a"))
sql_d41_bcos_a = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758c' AS PROC
INTO #d41_bcos_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR IN (321, 322, 32)
       AND T1.C_CAGENTE NOT IN ('6', '53', '9', '51', ''))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41' AND T1.SECTOR NOT IN (5101, 51021, 51022, 6, 51)
       AND T1.C_CAGENTE IN ('321', '322', '32', '3'))
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_bcos_a))


In [ ]:
work_conn.execute(text("UPDATE #d41_bcos_a SET SECTOR = 32 WHERE SECTOR = 3"))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 53 DEL SECTOR BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_321_53"))
sql_d41_321_53 = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758d' AS PROC
INTO #d41_321_53
FROM #d41_bcos_a T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_321_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_bcos_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_bcos_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_321_53)", res.rowcount)
for t in ["#d41_bcos_a", "#d41_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES PAGADOS POR BANCOS A RESTO DEL MUNDO, DADO QUE LA CI NO CONTIENE LOS PAGADOS POR BONOS Y DEPÒSITOS. SE AJUSTA CONTRA EMPRESAS
# ACTIVO DEL RM CON BANCOS POR CONCEPTO DE TÌTULOS Y DEPÒSITOS
work_conn.execute(text("DROP TABLE IF EXISTS #bcos_rm"))
sql_bcos_rm = """
SELECT [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, SUM(DATO) AS DATO
INTO #bcos_rm
FROM TABLAS.dbo.BD_CTSI
WHERE SECTOR = 6 AND C_CAGENTE IN ('321') AND C_SCN IN ('AF.22', 'AF.29', 'AF.31', 'AF.32')
      AND C_ENTRADA = 'D' AND C_CUENTA IN ('Bce Final', 'Bce Inicio')
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_bcos_rm))


In [ ]:
# SALDOS PROMEDIOS
work_conn.execute(text("DROP TABLE IF EXISTS #bcos_rm_avg"))
sql_bcos_rm_avg = """
SELECT [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, AVG(DATO) AS DATO
INTO #bcos_rm_avg
FROM #bcos_rm
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_bcos_rm_avg))


In [ ]:
raise NotImplementedError("TASAS se lee de un archivo SAS7BDAT en ruta absoluta ('/sasdata/BCCH/GEM_DCNI/02_CNSI/09_SI_BCOS/321_BCOS/tasas.sas7bdat'), prohibida por regla dura (M-001: rutas hardcodeadas). No hay información en el contexto del proyecto sobre dónde vive este archivo en el nuevo entorno (no es tabla del catálogo GOBGENER/TABLAS ni file_import declarado) -- se requiere definir la fuente real (ruta relativa del workspace o tabla de BD) antes de traducir")


In [ ]:
raise NotImplementedError("INT_BCOS_RM depende de TASAS (celda anterior sin resolver: archivo tasas.sas7bdat) -- no se puede calcular hasta que se defina la fuente de TASAS")


In [ ]:
raise NotImplementedError("APPEND de INT_BCOS_RM a TABLAS.BD_CTSI depende de INT_BCOS_RM, que no se pudo calcular por falta de la fuente TASAS")


In [ ]:
raise NotImplementedError("UPDATE de INT_BCOS_RM (C_CAGENTE='53', DATO=DATO*-1) depende de INT_BCOS_RM, que no se pudo calcular por falta de la fuente TASAS")


In [ ]:
raise NotImplementedError("Segundo APPEND de INT_BCOS_RM a TABLAS.BD_CTSI depende de INT_BCOS_RM, que no se pudo calcular por falta de la fuente TASAS")


In [ ]:
# limpieza de temporales de este bloque (BCOS_RM, BCOS_RM_AVG, TASAS, INT_BCOS_RM)
# TASAS e INT_BCOS_RM nunca se crearon (ver NotImplementedError arriba); se dropean solo las que existen
for t in ["#bcos_rm", "#bcos_rm_avg"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES BCENTRAL, FP Y SEGUROS, y OFIS EN GASTO DE OFIS.
# REALIZA CONTRA AJUSTE EN CA 321
# CIERRE 2021. SE INCORPORA IMPUTACION DE INGRESOS DE AUXILIARES EN OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofis_a"))
sql_d41_ofis_a = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758e' AS PROC
INTO #d41_ofis_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41'
       AND SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('33')
       AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('33', '34', '31', '35', '36'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
       AND SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('34', '35', '31', '33', '36')
       AND (SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('33') OR T1.C_CAGENTE = '411')
       AND T1.C_CAGENTE NOT IN ('331'))
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_ofis_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 321 DEL SECTOR OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_33_53"))
sql_d41_33_53 = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758f' AS PROC
INTO #d41_33_53
FROM #d41_ofis_a T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_33_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_ofis_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_ofis_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_33_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_33_53)", res.rowcount)
for t in ["#d41_ofis_a", "#d41_33_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
raise NotImplementedError("OFIS_INT se lee de un archivo SAS7BDAT en ruta absoluta ('/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat'), prohibida por regla dura (M-001). No hay información en el contexto del proyecto sobre esta fuente en el nuevo entorno (no es tabla del catálogo TABLAS/GOBGENER) -- se requiere definir la fuente real antes de traducir")


In [ ]:
# BONOS EMITIDOS OFIS MERCADO LOCAL
work_conn.execute(text("DROP TABLE IF EXISTS #ofis_bonos"))
sql_ofis_bonos = """
SELECT [AÑO], TRIM, 33 AS SECTOR, SUM(DATO) AS DATO
INTO #ofis_bonos
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_CUENTA = 'Bce Final' AND SUBSTRING(C_SCN, 1, 4) = 'AF.3'
      AND (SUBSTRING(LEFT(CAST(SECTOR AS varchar(7))), 1, 2) IN ('33') OR SECTOR = 411)
      AND SECTOR NOT IN (334) AND C_CAGENTE NOT IN ('6')
GROUP BY [AÑO], TRIM
"""
work_conn.execute(text(sql_ofis_bonos))


In [ ]:
raise NotImplementedError("TASA_B_OFIS depende de OFIS_INT (bloque anterior sin resolver: archivo BASE_D41_FINAL.sas7bdat) -- no se puede calcular la tasa implícita hasta definir esa fuente")


In [ ]:
# BONOS EMITIDOS OFIS EN PODER DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #ofis_bonos_emp"))
sql_ofis_bonos_emp = """
SELECT [AÑO], TRIM, 51 AS SECTOR, SUM(DATO) AS DATO
INTO #ofis_bonos_emp
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_CUENTA = 'Bce Final' AND SUBSTRING(C_SCN, 1, 4) = 'AF.3'
      AND (SUBSTRING(LEFT(CAST(SECTOR AS varchar(7))), 1, 2) IN ('51') OR SECTOR = 334 OR SECTOR = 53)
      AND SECTOR NOT IN (511)
      AND (SUBSTRING(C_CAGENTE, 1, 2) = '33' OR C_CAGENTE = '411') AND [AÑO] > 2002
GROUP BY [AÑO], TRIM
"""
work_conn.execute(text(sql_ofis_bonos_emp))


In [ ]:
raise NotImplementedError("INT_OFIS_EMP depende de TASA_B_OFIS, no calculada por falta de la fuente OFIS_INT (archivo BASE_D41_FINAL.sas7bdat)")


In [ ]:
raise NotImplementedError("INT_OFIS_bcos depende de INT_OFIS_EMP, no calculada por falta de la fuente OFIS_INT")


In [ ]:
raise NotImplementedError("APPEND de INT_OFIS_EMP a TABLAS.BD_CTSI depende de INT_OFIS_EMP, no calculada por falta de la fuente OFIS_INT")


In [ ]:
raise NotImplementedError("APPEND de INT_OFIS_bcos a TABLAS.BD_CTSI depende de INT_OFIS_bcos, no calculada por falta de la fuente OFIS_INT")


In [ ]:
# limpieza de temporales de este bloque; OFIS_INT, TASA_B_OFIS, INT_OFIS_EMP, INT_OFIS_bcos nunca se crearon (ver NotImplementedError arriba)
for t in ["#ofis_bonos", "#ofis_bonos_emp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INTERESES PAGADOS POR AUXILIARES (SECTOR 37) A BCENTRAL USANDO INFO DEL BANCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #d41_aux_bc"))
sql_d41_aux_bc = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758g' AS PROC
INTO #d41_aux_bc
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41'
       AND (SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36') OR T1.SECTOR = 37)
       AND T1.C_CAGENTE = '31')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41' AND T1.SECTOR = 31
       AND (SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36') OR T1.C_CAGENTE = '37'))
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LEFT(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_aux_bc))


In [ ]:
# intereses pagados sector 37 con contragente (SIN CA 31)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc_ca"))
sql_d41_sc_ca = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN
INTO #d41_sc_ca
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR = 37
      AND T1.C_CAGENTE NOT IN ('31') AND T1.FUENTE NOT IN ('DI_Aj_REAJ')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_sc_ca))


In [ ]:
# intereses pagados sector 37 sin contragente
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc"))
sql_d41_sc = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       SUM(T1.DATO) AS DATO
INTO #d41_sc
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR = 37 AND T1.FUENTE NOT IN ('DI_Aj_REAJ')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_d41_sc))


In [ ]:
# estructura contragente
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc_est"))
sql_d41_sc_est = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS EST
INTO #d41_sc_est
FROM #d41_sc_ca T1, #d41_sc T2
WHERE T1.[AÑO] = T2.[AÑO] AND T1.SECTOR = T2.SECTOR AND T1.MONEDA = 'P' AND T1.TRIM = T2.TRIM
"""
work_conn.execute(text(sql_d41_sc_est))


In [ ]:
# AJUSTA IMPUTACIÓN EN D41 CA POR ESTRUCTURA DEL SECTOR AUXILIARES (SECTOR 37)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_aux_aj"))
sql_d41_aux_aj = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * T2.EST) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758h' AS PROC
INTO #d41_aux_aj
FROM #d41_aux_bc T1, #d41_sc_est T2
WHERE T1.[AÑO] = T2.[AÑO] AND T1.SECTOR = T2.SECTOR AND T1.MONEDA = T2.MONEDA AND T1.TRIM = T2.TRIM
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_aux_aj))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_aux_bc"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_aux_bc)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_aux_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_aux_aj)", res.rowcount)
for t in ["#d41_aux_bc", "#d41_aux_aj", "#d41_sc_ca", "#d41_sc", "#d41_sc_est"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. ASIGNA INTERESES CON EL RM EN SECTOR SEGUROS DE VIDA UTILIZANDO EL BALANCE DE PTMOS CON CA RM Y BCOS COMO ESTRUCTURA
# balances con contragente
work_conn.execute(text("DROP TABLE IF EXISTS #b_h_seg"))
sql_b_h_seg = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       SUM(T1.DATO) AS DATO
INTO #b_h_seg
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.41', 'AF.42') AND T1.C_CUENTA = 'Bce Final' AND T1.SECTOR = 351
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_b_h_seg))


In [ ]:
# balances sin contragente
work_conn.execute(text("DROP TABLE IF EXISTS #b_h_seg_2"))
sql_b_h_seg_2 = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       SUM(T1.DATO) AS DATO
INTO #b_h_seg_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.41', 'AF.42') AND T1.C_CUENTA = 'Bce Final' AND T1.SECTOR = 351
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_b_h_seg_2))


In [ ]:
# estructura de balances
work_conn.execute(text("DROP TABLE IF EXISTS #b_est_seg"))
sql_b_est_seg = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS DATO
INTO #b_est_seg
FROM #b_h_seg T1, #b_h_seg_2 T2
WHERE T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""
work_conn.execute(text(sql_b_est_seg))


In [ ]:
# intereses pagados por seguros de vida al resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #d41_seg"))
sql_d41_seg = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       SUM(T1.DATO) * T2.DATO AS DATO,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758i' AS PROC
INTO #d41_seg
FROM TABLAS.dbo.BD_CTSI T1, #b_est_seg T2
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.C_CUENTA = 'YG' AND T1.FUENTE NOT IN ('DI_Aj_REAJ')
      AND T2.C_CAGENTE NOT IN ('321') AND T1.SECTOR = T2.SECTOR AND T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T2.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_seg))


In [ ]:
# ajuste en intereses pagados por seguros de vida a bancos
work_conn.execute(text("DROP TABLE IF EXISTS #d41_seg_bcos"))
sql_d41_seg_bcos = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.DATO * -1 AS DATO,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758j' AS PROC
INTO #d41_seg_bcos
FROM #d41_seg T1
"""
work_conn.execute(text(sql_d41_seg_bcos))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_seg"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_seg)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_seg_bcos"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_seg_bcos)", res.rowcount)
for t in ["#d41_seg_bcos", "#d41_seg", "#b_h_seg", "#b_h_seg_2", "#b_est_seg"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. ASIGNA INTERESES CON EL RM EN SECTOR OFIS UTILIZANDO EL BALANCE DE PTMOS CON CA RM Y BCOS COMO ESTRUCTURA
# CIERRE 2021. INCORPORA BONOS DE LP EN EL CÁLCULO DADO QUE TMB EL RM COMPRA BONOS DE OFIS
# balances con contragente
work_conn.execute(text("DROP TABLE IF EXISTS #b_h_ofi"))
sql_b_h_ofi = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       SUM(T1.DATO) AS DATO
INTO #b_h_ofi
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.41', 'AF.42', 'AF.32') AND T1.C_CUENTA = 'Bce Final'
      AND (SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('33') OR T1.SECTOR = 411)
      AND T1.SECTOR NOT IN (334) AND T1.C_CAGENTE IN ('6', '321')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_b_h_ofi))


In [ ]:
# balances sin contragente
work_conn.execute(text("DROP TABLE IF EXISTS #b_h_ofi_2"))
sql_b_h_ofi_2 = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       SUM(T1.DATO) AS DATO
INTO #b_h_ofi_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.41', 'AF.42', 'AF.32') AND T1.C_CUENTA = 'Bce Final'
      AND (SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('33') OR T1.SECTOR = 411)
      AND T1.SECTOR NOT IN (334) AND T1.C_CAGENTE IN ('6', '321')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_b_h_ofi_2))


In [ ]:
# estructura de balances
work_conn.execute(text("DROP TABLE IF EXISTS #b_est_ofi"))
sql_b_est_ofi = """
SELECT T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS DATO
INTO #b_est_ofi
FROM #b_h_ofi T1, #b_h_ofi_2 T2
WHERE T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""
work_conn.execute(text(sql_b_est_ofi))


In [ ]:
# intereses pagados por ofis con ca bancos para hacer la apertura entre bcos y rm
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofi_bcos"))
sql_d41_ofi_bcos = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       SUM(T1.DATO) AS DATO,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.C_SCN,
       T1.N_SCN
INTO #d41_ofi_bcos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.C_CUENTA = 'YG' AND T1.FUENTE NOT IN ('DI_Aj_REAJ')
      AND T1.C_CAGENTE = '321'
      AND (SUBSTRING(LEFT(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('33') OR T1.SECTOR = 411) AND T1.SECTOR NOT IN (334)
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_d41_ofi_bcos))


In [ ]:
# intereses pagados por ofis al resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofi_rm"))
sql_d41_ofi_rm = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       SUM(T1.DATO) * T2.DATO AS DATO,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758k' AS PROC
INTO #d41_ofi_rm
FROM #d41_ofi_bcos T1, #b_est_ofi T2
WHERE T1.SECTOR = T2.SECTOR AND T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T2.C_CAGENTE NOT IN ('321')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T2.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_ofi_rm))


In [ ]:
res = work_conn.execute(text("DELETE FROM #d41_ofi_rm WHERE SECTOR IN (33221, 3324)"))
_log("DELETE #d41_ofi_rm", res.rowcount)


In [ ]:
# ajuste en intereses pagados por ofis de vida a bancos
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofi_bcos_f"))
sql_d41_ofi_bcos_f = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.DATO * -1 AS DATO,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0758l' AS PROC
INTO #d41_ofi_bcos_f
FROM #d41_ofi_rm T1
"""
work_conn.execute(text(sql_d41_ofi_bcos_f))
res = work_conn.execute(text("DELETE FROM #d41_ofi_bcos_f WHERE SECTOR IN (33221, 3324)"))
_log("DELETE #d41_ofi_bcos_f", res.rowcount)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_ofi_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_ofi_rm)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_ofi_bcos_f"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_ofi_bcos_f)", res.rowcount)
for t in ["#d41_ofi_rm", "#d41_ofi_bcos_f", "#d41_ofi_bcos", "#b_est_ofi", "#b_h_ofi_2", "#b_h_ofi"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE DE INTERESES. COMPARA RECIBIDO Y PAGADO. RESPETA LO RECIBIDO E IMPUTA EL DIFERENCIAL EN LO PAGADO DEL SECTOR 51022 CON CA RESPECTIVO
# CALCULA TOTAL RECIBIDO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_recibido"))
sql_d41_recibido = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CAST(T2.C_SI_publ AS varchar(4))) END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0759' AS PROC
INTO #d41_recibido
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('D.41', 'D.41r')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR,
         CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CAST(T2.C_SI_publ AS varchar(4))) END
"""
work_conn.execute(text(sql_d41_recibido))


In [ ]:
# CALCULA TOTAL PAGADO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_pagado"))
sql_d41_pagado = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       LEFT(CAST(T2.C_SI_publ AS varchar(4))) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0759' AS PROC
INTO #d41_pagado
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T2 ON T1.C_CAGENTE = T2.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('D.41', 'D.41r')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         LEFT(CAST(T2.C_SI_publ AS varchar(4)))
"""
work_conn.execute(text(sql_d41_pagado))


In [ ]:
# DATA D41_RECIBIDO;SET D41_RECIBIDO D41_PAGADO;RUN; -- concatenación server-side (append) en la misma #tmp
res = work_conn.execute(text("INSERT INTO #d41_recibido SELECT * FROM #d41_pagado"))
_log("APPEND #d41_recibido += #d41_pagado", res.rowcount)


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #d41_delta"))
sql_d41_delta = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
INTO #d41_delta
FROM #d41_recibido T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_d41_delta))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_DELTA FORCE -- server-side, fuente en #tmp
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_delta = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d41_delta
"""
res = work_conn.execute(text(sql_append_d41_delta))
_log("APPEND TABLAS.dbo.BD_CTSI (D41_DELTA)", res.rowcount)


In [ ]:
# DROP TABLE D41_DELTA, D41_PAGADO, D41_RECIBIDO
for t in ["#d41_delta", "#d41_pagado", "#d41_recibido"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA TRANSFERENCIAS DE CAPITAL EN SECTOR 5101 CON CA 41 USANDO INFO DE GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #d9_5101"))
sql_d9_5101 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.C_SI AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO,
       T1.C_INSTRUMENTO_SCN AS C_SCN,
       'Transferencias de capital' AS N_SCN,
       'PS' AS FUENTE,
       '0760' AS PROC
INTO #d9_5101
FROM TABLAS.dbo.T_TK_GG_EPU T1
"""
work_conn.execute(text(sql_d9_5101))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d9_5101
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (D9_5101)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d9_5101"))


In [ ]:
# CAMBIA D9 RECIBIDAS POR SECTOR 41 A PAGADAS MULTIPLICANDO POR -1
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET DATO = DATO * -1, C_ENTRADA = 'D', PROC = '0761'
WHERE SECTOR = 41 AND C_CUENTA = 'Capital' AND C_SCN = 'D.9' AND C_ENTRADA = 'H'
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI (D9 sector 41)", res.rowcount)


In [ ]:
# CIERRE DE TRANSFERENCIAS DE CAPITAL. IMPUTA 51 (85% del diferencial pagado/recibido)
work_conn.execute(text("DROP TABLE IF EXISTS #d9_51"))
sql_d9_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) * 1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0764a' AS PROC
INTO #d9_51
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN = 'D.9'
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
# nota: MONEDA se fija en el SELECT ('P'), la referencia a CALCULATED se resuelve con literal directo en GROUP BY
work_conn.execute(text(sql_d9_51))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d9_51
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (D9_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d9_51"))


In [ ]:
# IMPUTA K21 EN SECTOR 51022 CON CA 41, USANDO INFO DE SECTOR 41
work_conn.execute(text("DROP TABLE IF EXISTS #k21"))
sql_k21 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0765' AS PROC
INTO #k21
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN = 'K.21' AND T1.SECTOR = 41
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_k21))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #k21
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (K21)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #k21"))


In [ ]:
# ELIMINA TRANSACCIONES DE SECTOR 5111
work_conn.execute(text("DROP TABLE IF EXISTS #elimina5111"))
sql_elimina5111 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0766' AS PROC
INTO #elimina5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.11','P.2','D.1','B.2','K.1'))
   OR (T1.SECTOR = 5111 AND T1.C_CUENTA = 'YG' AND T1.C_SCN IN ('B.2','B.8'))
   OR (T1.SECTOR = 5111 AND T1.C_CUENTA = 'Capital' AND T1.C_SCN IN ('K.1','B.8','B.9','P.51','P.52'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina5111))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina5111
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (ELIMINA5111)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina5111"))


In [ ]:
# ELIMINA TRANSACCIONES DE CUENTAS DE PRODUCCIÓN DE TODOS LOS SECTORES. OK
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod"))
sql_elimina_prod = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0769' AS PROC
INTO #elimina_prod
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.11','P.13') AND T1.C_ENTRADA = 'H')
   OR (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.2') AND T1.C_ENTRADA = 'D')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_prod))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina_prod
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (ELIMINA_PROD)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod"))


In [ ]:
# ELIMINA TRANSACCIONES DE CUENTAS DE PRODUCCIÓN EXCEPTO SECTOR GOBIERNO E IMPUTACIONES BANCARIAS
# CIERRE 2021: EXCEPTO EN SECTOR FINANCIERO DONDE SE IMPUTARÁN LAS DIFERENCIAS
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod2"))
sql_elimina_prod2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       (T1.DATO) * -1 AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0770' AS PROC
INTO #elimina_prod2
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI)
  AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('B.2','D.1','D.21','D.29','K.1') AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 NOT IN ('S.13','S.9','S.12'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_prod2))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina_prod2
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (ELIMINA_PROD2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod2"))


In [ ]:
# IMPUTA CCAS. CIERRE 2021: EXCEPTO EN SECTOR FINANCIERO DONDE SE IMPUTARÁN LAS DIFERENCIAS
work_conn.execute(text("DROP TABLE IF EXISTS #ccas"))
sql_ccas = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #ccas
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN)
  AND ((T1.C_SI IN (511) AND T1.C_SCN <> 'B.1') OR (T1.C_SI = 4 AND T1.C_SCN NOT IN ('B.1','K.1')))
"""
work_conn.execute(text(sql_ccas))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (CCAS)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #ccas"))


In [ ]:
# CIERRE 2021: CALCULA DIFERENCIA A IMPUTAR EN VARIABLES DE CTA DE PRODUCCION DEL SECTOR FINANCIERO
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_sf"))
sql_elimina_sf = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       3 AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #elimina_sf
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI)
  AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('B.2','D.1','D.21','D.29','K.1') AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 IN ('S.12'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_sf))


In [ ]:
res = work_conn.execute(text("UPDATE #elimina_sf SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso Mixto' WHERE C_SCN = 'B.2'"))
_log("UPDATE #elimina_sf (B.2 -> B.2/B.3)", res.rowcount)


In [ ]:
# OBTIENE DATA DEL SF DESDE CCAS
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_sf"))
sql_ccas_sf = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #ccas_sf
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE T1.C_SCN = T2.C_SCN AND T1.C_SI = 3 AND T1.C_SCN <> 'B.1'
"""
work_conn.execute(text(sql_ccas_sf))


In [ ]:
# DATA ELIMINA_SF; SET ELIMINA_SF CCAS_SF -- concatenación server-side (misma estructura de columnas)
res = work_conn.execute(text(f"""
INSERT INTO #elimina_sf ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas_sf
"""))
_log("CONCAT #elimina_sf += #ccas_sf", res.rowcount)


In [ ]:
# CALCULA DIFERENCIAS A IMPUTAR EN SF
work_conn.execute(text("DROP TABLE IF EXISTS #dif_prod_sf"))
sql_dif_prod_sf = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #dif_prod_sf
FROM #elimina_sf T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_dif_prod_sf))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #dif_prod_sf
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (DIF_PROD_SF)", res.rowcount)
for t in ["#dif_prod_sf", "#elimina_sf", "#ccas_sf"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR AGREGADO
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_2"))
sql_ccas_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0772' AS PROC
INTO #ccas_2
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN) AND (T1.C_SI IN (3,511) AND T1.C_SCN = 'B.1')
"""
work_conn.execute(text(sql_ccas_2))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas_2
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (CCAS_2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_2"))


In [ ]:
# CAMBIA EN SECTOR GOBIERNO EL EXCEDENTE DE EXPLOTACIÓN A EXCEDENTE DE EXPLOTACIÓN E INGRESO MIXTO
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso Mixto'
WHERE C_ENTRADA = 'D' AND SECTOR = 41 AND C_SCN = 'B.2'
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 41 B.2)", res.rowcount)


In [ ]:
# CALCULA DIFERENCIAL ENTRE ECONOMIA NACIONAL Y LOS SECTORES PARA LA CUENTA DE PRODUCCIÓN (SUMA DE SECTORES)
work_conn.execute(text("DROP TABLE IF EXISTS #cierre_prod"))
sql_cierre_prod = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T3.N_SCN,
       'PS' AS FUENTE,
       '0777' AS PROC
INTO #cierre_prod
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2, TABLAS.dbo.T_INSTRUMENTOS T3
WHERE (T1.SECTOR = T2.C_SI AND T1.C_SCN = T3.C_SCN)
  AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T1.SECTOR NOT IN (6,412) AND T2.C_SI_SCN_N1 IN ('S.12','S.13','S.14') AND T1.C_SCN NOT IN ('P.2'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T3.N_SCN
"""
work_conn.execute(text(sql_cierre_prod))


In [ ]:
# DATA ECONOMIA NACIONAL
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_en"))
sql_ccas_en = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0777' AS PROC
INTO #ccas_en
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN) AND (T1.C_SI = 1 AND T1.C_SCN <> 'B.1')
"""
work_conn.execute(text(sql_ccas_en))


In [ ]:
# PROC APPEND BASE=WORK.CIERRE_PROD DATA=WORK.CCAS_EN -- ambas son #tmp de sesión
res = work_conn.execute(text(f"""
INSERT INTO #cierre_prod ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas_en
"""))
_log("CONCAT #cierre_prod += #ccas_en", res.rowcount)


In [ ]:
# IMPUTACION
work_conn.execute(text("DROP TABLE IF EXISTS #prod_dif"))
sql_prod_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #prod_dif
FROM #cierre_prod T1 WHERE T1.DATO <> 0
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.PROC, T1.FUENTE
"""
work_conn.execute(text(sql_prod_dif))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #prod_dif
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (PROD_DIF)", res.rowcount)
for t in ["#prod_dif", "#cierre_prod", "#ccas_en"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR AGREGADO EN SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #va_51"))
sql_va_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.1' AS C_SCN,
       'Valor agregado bruto' AS N_SCN,
       'PS' AS FUENTE,
       '0778' AS PROC
INTO #va_51
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 = 'S.11')
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_va_51))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #va_51
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (VA_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #va_51"))


In [ ]:
# IMPUTA VALOR AGREGADO EN SECTOR 4
work_conn.execute(text("DROP TABLE IF EXISTS #va_4"))
sql_va_4 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       4 AS SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.1' AS C_SCN,
       'Valor agregado bruto' AS N_SCN,
       'PS' AS FUENTE,
       '0778b' AS PROC
INTO #va_4
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 = 'S.13' AND T1.C_SCN NOT IN ('B.2','B.2/B.3'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_va_4))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #va_4
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (VA_4)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #va_4"))


In [ ]:
# IMPUTA EXCEDENTE DE EXPLOTACION EN LA CTA DE PRODUCCION DEL RM, USANDO LA INFO DEL EXCEDENTE DE LA CTA YG
work_conn.execute(text("DROP TABLE IF EXISTS #exc_rm"))
sql_exc_rm = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0779' AS PROC
INTO #exc_rm
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'B.2' AND T1.SECTOR = 6)
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_exc_rm))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #exc_rm
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (EXC_RM)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #exc_rm"))


In [ ]:
# ACTUALIZA CODIGO DE EXCEDENTE DE EXPLOTACION EN TODOS LOS SECTORES EXCEPTO RM
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso mixto', PROC = '0780'
WHERE C_SCN = 'B.2' AND C_ENTRADA = 'H' AND C_CUENTA = 'YG' AND SECTOR <> 6
"""))
_log("UPDATE TABLAS.dbo.BD_CTSI (B.2 excepto RM)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA EFECTO NETO DE SIFMI DE SEGUROS, AUXILIARES Y BANCOS EN EXCEDENTE
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_sf"))
sql_sifmi_sf = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.2/B.3' AS C_SCN,
       'Excedente de explotación/ Ingreso mixto' AS N_SCN,
       'PS' AS FUENTE,
       '0780b' AS PROC
INTO #sifmi_sf
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (35,36,321) AND T1.C_CUENTA = 'YG' AND T1.C_SCN = 'D.41' AND T1.FUENTE = 'DI_Aj_SIFMI'
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_sifmi_sf))


In [ ]:
# AJUSTA EN EXCEDENTE TOTAL DEL SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #aj_exc"))
sql_aj_exc = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       3 AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0780c' AS PROC
INTO #aj_exc
FROM #sifmi_sf T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_exc))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #sifmi_sf
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (SIFMI_SF)", res.rowcount)
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #aj_exc
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (AJ_EXC)", res.rowcount)
for t in ["#sifmi_sf", "#aj_exc"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA EXCEDENTE/ING MIXTO DE LA CUENTA DE YG EN SECTORES SNF Y SF
work_conn.execute(text("DROP TABLE IF EXISTS #elimin_exc"))
sql_elimin_exc = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0781' AS PROC
INTO #elimin_exc
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI)
  AND (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H' AND T2.C_SI_SCN_N1 IN ('S.11','S.12') AND T1.C_SCN IN ('B.2/B.3','B.2'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimin_exc))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimin_exc
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (ELIMIN_EXC)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimin_exc"))


In [ ]:
# IMPUTA EXCEDENTE EN LA CTA YG DE LOS SECTORES SNF Y SF, USANDO INFO DE LA CTA DE PRODUCCION
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_exc"))
sql_imputa_exc = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       'YG' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0782' AS PROC
INTO #imputa_exc
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI)
  AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 IN ('S.11','S.12') AND T1.C_SCN IN ('B.2/B.3','B.2'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_exc))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #imputa_exc
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (IMPUTA_EXC)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_exc"))


In [ ]:
# IMPUTA DIFERENCIAL DE FBCF Y VAR DE EXISTENCIAS EN SECTOR 51022
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_sectores"))
sql_fbc_sectores = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0768' AS PROC
INTO #fbc_sectores
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_SCN IN ('P.51','P.52') AND T1.C_ENTRADA = 'D')
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_fbc_sectores))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_total"))
sql_fbc_total = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE,
       '0768' AS PROC
INTO #fbc_total
FROM TABLAS.dbo.CNT T1
WHERE (T1.C_SCN IN ('P.51','P.52') AND T1.C_ENTRADA = 'D')
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_fbc_total))


In [ ]:
# PROC APPEND BASE=WORK.FBC_SECTORES DATA=WORK.FBC_TOTAL -- ambas #tmp de sesión
res = work_conn.execute(text(f"""
INSERT INTO #fbc_sectores ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #fbc_total
"""))
_log("CONCAT #fbc_sectores += #fbc_total", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_dif"))
sql_fbc_dif = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #fbc_dif
FROM #fbc_sectores T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_fbc_dif))


In [ ]:
res = work_conn.execute(text(f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #fbc_dif
"""))
_log("APPEND TABLAS.dbo.BD_CTSI (FBC_DIF)", res.rowcount)
for t in ["#fbc_dif", "#fbc_sectores", "#fbc_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# RECALCULA EL AHORRO PARA TODOS LOS SECTORES
# CIERRE 2021: CAMBIA NIVEL DEL SECTOR AL CUAL SE HACE EL CÁLCULO DEL AHORRO
work_conn.execute(text("DROP TABLE IF EXISTS #ahorro"))
sql_ahorro = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       'PS' AS FUENTE,
       '0786' AS PROC
INTO #ahorro
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI)
  AND ((T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'D' AND T1.C_SCN <> 'P.31') OR (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H'))
GROUP BY MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_CUENTA
"""
work_conn.execute(text(sql_ahorro))
ahorro = pd.read_sql(text("SELECT * FROM #ahorro"), work_conn)
_log("ahorro", ahorro)


In [ ]:
# Tramo 7 de 8. Continúa el flujo de reclasificación de rentas D.4/D.7/B.8/B.9 sobre TABLAS.BD_CTSI.
# APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AHORRO FORCE (WORK.AHORRO ya existe como #ahorro de un tramo anterior)
cols_ahorro = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_ahorro = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_ahorro})
SELECT {cols_ahorro}
FROM #ahorro
"""
res = work_conn.execute(text(sql_append_ahorro))
_log("APPEND TABLAS.BD_CTSI (AHORRO)", res.rowcount)


In [ ]:
# DROP TABLE WORK.AHORRO (WORK.AHORRO_2 estaba comentado en el SAS: no se traduce)
work_conn.execute(text("DROP TABLE IF EXISTS #ahorro"))


In [ ]:
# ELIMINA AHORRO DE LA CTA DE CAPITAL E IMPUTA EL AHORRO DE LA CUENTA YG
# ELIMINA: reversa (signo negado) los movimientos de Capital/B.8
work_conn.execute(text("DROP TABLE IF EXISTS #drop_b8_ck"))
sql_drop_b8_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0787' AS PROC
INTO #drop_b8_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'Capital' AND T1.C_SCN = 'B.8')
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_drop_b8_ck))
drop_b8_ck = pd.read_sql(text("SELECT * FROM #drop_b8_ck"), work_conn)
_log("drop_b8_ck", drop_b8_ck)


In [ ]:
cols_drop_b8_ck = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_drop_b8_ck = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_drop_b8_ck})
SELECT {cols_drop_b8_ck}
FROM #drop_b8_ck
"""
res = work_conn.execute(text(sql_append_drop_b8_ck))
_log("APPEND TABLAS.BD_CTSI (DROP_B8_CK)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_b8_ck"))


In [ ]:
# IMPUTA: traspasa el Ahorro (B.8, Debe) de la cuenta YG a Capital (Haber)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_b8_ck"))
sql_imputa_b8_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0788' AS PROC
INTO #imputa_b8_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'YG' AND T1.C_SCN = 'B.8' AND T1.C_ENTRADA = 'D')
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_b8_ck))
imputa_b8_ck = pd.read_sql(text("SELECT * FROM #imputa_b8_ck"), work_conn)
_log("imputa_b8_ck", imputa_b8_ck)


In [ ]:
cols_imputa_b8_ck = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_imputa_b8_ck = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_imputa_b8_ck})
SELECT {cols_imputa_b8_ck}
FROM #imputa_b8_ck
"""
res = work_conn.execute(text(sql_append_imputa_b8_ck))
_log("APPEND TABLAS.BD_CTSI (IMPUTA_B8_CK)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_b8_ck"))


In [ ]:
# ELIMINA CCF DE LA CTA DE CAPITAL E IMPUTA EL CCF DE LA CUENTA DE PRODUCCION
# ELIMINA: reversa los movimientos de Capital/K.1
work_conn.execute(text("DROP TABLE IF EXISTS #drop_ccf_ck"))
sql_drop_ccf_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0789' AS PROC
INTO #drop_ccf_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'Capital' AND T1.C_SCN = 'K.1')
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_drop_ccf_ck))
drop_ccf_ck = pd.read_sql(text("SELECT * FROM #drop_ccf_ck"), work_conn)
_log("drop_ccf_ck", drop_ccf_ck)


In [ ]:
cols_drop_ccf_ck = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_drop_ccf_ck = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_drop_ccf_ck})
SELECT {cols_drop_ccf_ck}
FROM #drop_ccf_ck
"""
res = work_conn.execute(text(sql_append_drop_ccf_ck))
_log("APPEND TABLAS.BD_CTSI (DROP_CCF_CK)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_ccf_ck"))


In [ ]:
# IMPUTA: traspasa el CCF (K.1, Debe, sector Producción, dato<>0) a Capital (Haber) por sector de publicación
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_ccf_ck"))
sql_imputa_ccf_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0790' AS PROC
INTO #imputa_ccf_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN = 'K.1' AND T1.C_ENTRADA = 'D' AND T1.DATO <> 0)
GROUP BY t1.AÑO, T1.TRIM, T2.C_SI_publ, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_ccf_ck))
imputa_ccf_ck = pd.read_sql(text("SELECT * FROM #imputa_ccf_ck"), work_conn)
_log("imputa_ccf_ck", imputa_ccf_ck)


In [ ]:
cols_imputa_ccf_ck = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_imputa_ccf_ck = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_imputa_ccf_ck})
SELECT {cols_imputa_ccf_ck}
FROM #imputa_ccf_ck
"""
res = work_conn.execute(text(sql_append_imputa_ccf_ck))
_log("APPEND TABLAS.BD_CTSI (IMPUTA_CCF_CK)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_ccf_ck"))


In [ ]:
# CIERRE 2021: RECALCULA PTMO NETO DE LA CUENTA DE CAPITAL (bloque CAP_NEC_FMNM quedó comentado en el SAS original: no se traduce)
# RECALCULA PTMO NETO DE LA CUENTA DE CAPITAL a nivel de código de sector de publicación de cuentas financieras
work_conn.execute(text("DROP TABLE IF EXISTS #cap_nec"))
sql_cap_nec = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       'PS' AS FUENTE,
       '0793' AS PROC
INTO #cap_nec
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Capital')
GROUP BY t1.AÑO, T1.TRIM, T2.C_SI_publ, T1.C_CUENTA
"""
work_conn.execute(text(sql_cap_nec))
cap_nec = pd.read_sql(text("SELECT * FROM #cap_nec"), work_conn)
_log("cap_nec", cap_nec)
# Bloque WORK.CAP_NEC_2 (contra T_SECTORIZACION_N1) quedó comentado en el SAS original: no se traduce


In [ ]:
cols_cap_nec = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_cap_nec = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_cap_nec})
SELECT {cols_cap_nec}
FROM #cap_nec
"""
res = work_conn.execute(text(sql_append_cap_nec))
_log("APPEND TABLAS.BD_CTSI (CAP_NEC)", res.rowcount)
# el SAS repite DROP TABLE WORK.CAP_NEC dos veces: un solo DROP alcanza
work_conn.execute(text("DROP TABLE IF EXISTS #cap_nec"))


In [ ]:
# AJUSTA PTMO NETO EN SECTOR GOBIERNO AL DE LA CTA DE CAPITAL. AJUSTA LA CF CONTRA AJUSTES DE CONCILIACION
# IMPUTA AF71 EN CF: reversa Financiera de sector Gobierno (S.13) contra B.9
work_conn.execute(text("DROP TABLE IF EXISTS #gob_ck"))
sql_gob_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA = 'Financiera' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0902' AS PROC
INTO #gob_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.13')
GROUP BY t1.AÑO, T1.TRIM
"""
work_conn.execute(text(sql_gob_ck))
gob_ck = pd.read_sql(text("SELECT * FROM #gob_ck"), work_conn)
_log("gob_ck", gob_ck)


In [ ]:
# IMPUTA AF71 EN REC VOLUMEN: mismo valor de GOB_CK, negado, como Rec Volumen
work_conn.execute(text("DROP TABLE IF EXISTS #gob_rv"))
sql_gob_rv = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0903' AS PROC
INTO #gob_rv
FROM #gob_ck T1
"""
work_conn.execute(text(sql_gob_rv))
gob_rv = pd.read_sql(text("SELECT * FROM #gob_rv"), work_conn)
_log("gob_rv", gob_rv)


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51: mismo valor de GOB_CK negado, sector 51 con CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #cf_51"))
sql_cf_51 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0904' AS PROC
INTO #cf_51
FROM #gob_ck T1
"""
work_conn.execute(text(sql_cf_51))
cf_51 = pd.read_sql(text("SELECT * FROM #cf_51"), work_conn)
_log("cf_51", cf_51)


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51 (RV_51): valor de GOB_CK sin negar, como Rec Volumen sector 51/CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #rv_51"))
sql_rv_51 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0905' AS PROC
INTO #rv_51
FROM #gob_ck T1
"""
work_conn.execute(text(sql_rv_51))
rv_51 = pd.read_sql(text("SELECT * FROM #rv_51"), work_conn)
_log("rv_51", rv_51)


In [ ]:
cols_gob = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp in ["#gob_ck", "#gob_rv", "#cf_51", "#rv_51"]:
    sql_append_gob = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_gob})
SELECT {cols_gob}
FROM {tmp}
"""
    res = work_conn.execute(text(sql_append_gob))
    _log(f"APPEND TABLAS.BD_CTSI ({tmp})", res.rowcount)
for tmp in ["#gob_ck", "#gob_rv", "#cf_51", "#rv_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))


In [ ]:
# AJUSTA CUENTA DEL RESTO DEL MUNDO CONSIDERANDO VARIABLES MACRO DADAS POR LAS CNT
# RM DESDE SINTESIS: agrupa D.4x en D.4 y D.7x en D.7 para el sector 6, reversando el signo
work_conn.execute(text("DROP TABLE IF EXISTS #rm_cnf"))
sql_rm_cnf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       (CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'D.4'
             WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'D.7' ELSE T1.C_SCN END) AS C_SCN,
       (CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'Renta distribuida de las sociedades'
             WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'Transferencias corrientes diversas' ELSE T1.N_SCN END) AS N_SCN,
       'PS' AS FUENTE,
       '0906' AS PROC
INTO #rm_cnf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.SECTOR = 6 AND T1.C_CUENTA IN ('Producción','YG','Capital')
       AND T1.C_SCN IN ('P.7','P.6','D.1','D.41','D.42','D.43','D.443','D.72','D.71','D.75','B.8'))
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
         (CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'D.4'
               WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'D.7' ELSE T1.C_SCN END),
         (CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'Renta distribuida de las sociedades'
               WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'Transferencias corrientes diversas' ELSE T1.N_SCN END)
"""
work_conn.execute(text(sql_rm_cnf))
rm_cnf = pd.read_sql(text("SELECT * FROM #rm_cnf"), work_conn)
_log("rm_cnf", rm_cnf)


In [ ]:
# RM DESDE CNT: mismo recorte pero desde TABLAS.CNT, sector 6, renombrando N_SCN por código agregado
work_conn.execute(text("DROP TABLE IF EXISTS #rm_cnt"))
sql_rm_cnt = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO AS DATO,
       T1.C_SCN,
       (CASE WHEN T1.C_SCN = 'D.4' THEN 'Renta distribuida de las sociedades'
             WHEN T1.C_SCN = 'D.7' THEN 'Transferencias corrientes diversas'
             WHEN T1.C_SCN = 'B.8' THEN 'Ahorro' ELSE T1.N_SCN END) AS N_SCN,
       'PS' AS FUENTE,
       '0906' AS PROC
INTO #rm_cnt
FROM TABLAS.dbo.CNT t1
WHERE T1.SECTOR = 6
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN,
         (CASE WHEN T1.C_SCN = 'D.4' THEN 'Renta distribuida de las sociedades'
               WHEN T1.C_SCN = 'D.7' THEN 'Transferencias corrientes diversas'
               WHEN T1.C_SCN = 'B.8' THEN 'Ahorro' ELSE T1.N_SCN END)
"""
work_conn.execute(text(sql_rm_cnt))
rm_cnt = pd.read_sql(text("SELECT * FROM #rm_cnt"), work_conn)
_log("rm_cnt", rm_cnt)


In [ ]:
# PROC APPEND BASE=WORK.RM_CNF DATA=WORK.RM_CNT: acumula RM_CNT dentro de la propia #rm_cnf (WORK como base)
cols_rm = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_rm_cnf = f"""
INSERT INTO #rm_cnf ({cols_rm})
SELECT {cols_rm}
FROM #rm_cnt
"""
res = work_conn.execute(text(sql_append_rm_cnf))
_log("APPEND #rm_cnf (RM_CNT)", res.rowcount)


In [ ]:
# calcula diferencias: excluye Capital, recodifica C_SCN agregado a su componente original (D.4->D.42, D.7->D.75)
work_conn.execute(text("DROP TABLE IF EXISTS #rm_dif"))
sql_rm_dif = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       (CASE WHEN T1.C_SCN = 'D.4' THEN 'D.42'
             WHEN T1.C_SCN = 'D.7' THEN 'D.75' ELSE T1.C_SCN END) AS C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_dif
FROM #rm_cnf t1
WHERE (T1.C_CUENTA <> 'Capital')
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
         (CASE WHEN T1.C_SCN = 'D.4' THEN 'D.42' WHEN T1.C_SCN = 'D.7' THEN 'D.75' ELSE T1.C_SCN END),
         T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_rm_dif))
rm_dif = pd.read_sql(text("SELECT * FROM #rm_dif"), work_conn)
_log("rm_dif", rm_dif)


In [ ]:
# IMPUTA AHORRO EN CTA DE CAPITAL AL HABER (B.8)
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b8_ck"))
sql_rm_b8_ck = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_b8_ck
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_b8_ck))
rm_b8_ck = pd.read_sql(text("SELECT * FROM #rm_b8_ck"), work_conn)
_log("rm_b8_ck", rm_b8_ck)


In [ ]:
# IMPUTA DIF EN PTMO NETO CTA DE CAPITAL AL DEBE (B.9)
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b9_ck"))
sql_rm_b9_ck = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_b9_ck
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_b9_ck))
rm_b9_ck = pd.read_sql(text("SELECT * FROM #rm_b9_ck"), work_conn)
_log("rm_b9_ck", rm_b9_ck)


In [ ]:
# IMPUTA DIF EN CTA FINANCIERA INST AF71 CON CA 51 AL HABER
work_conn.execute(text("DROP TABLE IF EXISTS #rm_af71_cf"))
sql_rm_af71_cf = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_af71_cf
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_af71_cf))
rm_af71_cf = pd.read_sql(text("SELECT * FROM #rm_af71_cf"), work_conn)
_log("rm_af71_cf", rm_af71_cf)


In [ ]:
# IMPUTA DIF EN REC VOLUMEN INST AF71 CON CA 51 AL HABER
work_conn.execute(text("DROP TABLE IF EXISTS #rm_af71_rv"))
sql_rm_af71_rv = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '51' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_af71_rv
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_af71_rv))
rm_af71_rv = pd.read_sql(text("SELECT * FROM #rm_af71_rv"), work_conn)
_log("rm_af71_rv", rm_af71_rv)


In [ ]:
# IMPUTA DIF EN CTA FINANCIERA SECTOR 51 INST AF71 CON CA 6 AL HABER
work_conn.execute(text("DROP TABLE IF EXISTS #af71_cf_51"))
sql_af71_cf_51 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af71_cf_51
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_af71_cf_51))
af71_cf_51 = pd.read_sql(text("SELECT * FROM #af71_cf_51"), work_conn)
_log("af71_cf_51", af71_cf_51)


In [ ]:
# IMPUTA DIF EN RV SECTOR 51 INST AF71 CON CA 6 AL HABER
work_conn.execute(text("DROP TABLE IF EXISTS #af71_rv_51"))
sql_af71_rv_51 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af71_rv_51
FROM #rm_dif t1
WHERE (T1.C_SCN = 'B.8')
"""
work_conn.execute(text(sql_af71_rv_51))
af71_rv_51 = pd.read_sql(text("SELECT * FROM #af71_rv_51"), work_conn)
_log("af71_rv_51", af71_rv_51)


In [ ]:
# CALCULA AJUSTE EN B2 A CTA DE PRODUCCION
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b2"))
sql_rm_b2 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.2' AS C_SCN,
       'Excedente de explotación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_b2
FROM #rm_dif t1
WHERE (T1.C_CUENTA = 'Producción')
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.PROC, T1.FUENTE
"""
work_conn.execute(text(sql_rm_b2))
rm_b2 = pd.read_sql(text("SELECT * FROM #rm_b2"), work_conn)
_log("rm_b2", rm_b2)


In [ ]:
# CALCULA AJUSTE EN B2 A CTA DE YG
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b2_yg"))
sql_rm_b2_yg = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'YG' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.2' AS C_SCN,
       'Excedente de explotación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #rm_b2_yg
FROM #rm_dif t1
WHERE (T1.C_CUENTA = 'Producción')
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.PROC, T1.FUENTE
"""
work_conn.execute(text(sql_rm_b2_yg))
rm_b2_yg = pd.read_sql(text("SELECT * FROM #rm_b2_yg"), work_conn)
_log("rm_b2_yg", rm_b2_yg)


In [ ]:
cols_rm_dif = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
tmps_rm = ["#rm_dif", "#rm_b8_ck", "#rm_b9_ck", "#rm_af71_cf", "#rm_af71_rv", "#af71_cf_51", "#af71_rv_51", "#rm_b2", "#rm_b2_yg"]
for tmp in tmps_rm:
    sql_append_rm = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_rm_dif})
SELECT {cols_rm_dif}
FROM {tmp}
"""
    res = work_conn.execute(text(sql_append_rm))
    _log(f"APPEND TABLAS.BD_CTSI ({tmp})", res.rowcount)


In [ ]:
for tmp in ["#rm_dif", "#rm_cnf", "#rm_cnt", "#rm_b8_ck", "#rm_b9_ck", "#rm_af71_cf", "#rm_af71_rv", "#af71_cf_51", "#af71_rv_51", "#rm_b2", "#rm_b2_yg"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))


In [ ]:
# AJUSTA TOTAL D75 Y D42. DIFERENCIAL LO IMPUTA EN EL HABER DEL SECTOR 51 CON CA 53. AJUSTA AHORRO POR ESTA DIFERENCIA EN CTA YG Y CAPITAL
# CALCULA DIFERENCIA: reversa el Haber de D.42/D.75 y lo imputa como Haber en sector 51/CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d4d7"))
sql_aj_d4d7 = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0909' AS PROC
INTO #aj_d4d7
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42', 'D.75')
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d4d7))
aj_d4d7 = pd.read_sql(text("SELECT * FROM #aj_d4d7"), work_conn)
_log("aj_d4d7", aj_d4d7)


In [ ]:
# AJUSTA AHORRO CTA YG (B.8, Debe) por la diferencia calculada
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8"))
sql_aj_b8 = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       '0910' AS PROC
INTO #aj_b8
FROM #aj_d4d7 t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.FUENTE
"""
work_conn.execute(text(sql_aj_b8))
aj_b8 = pd.read_sql(text("SELECT * FROM #aj_b8"), work_conn)
_log("aj_b8", aj_b8)


In [ ]:
# AJUSTA AHORRO CTA CK (B.8, Haber, cuenta Capital)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_ck"))
sql_aj_b8_ck = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       '0911' AS PROC
INTO #aj_b8_ck
FROM #aj_d4d7 t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE
"""
work_conn.execute(text(sql_aj_b8_ck))
aj_b8_ck = pd.read_sql(text("SELECT * FROM #aj_b8_ck"), work_conn)
_log("aj_b8_ck", aj_b8_ck)


In [ ]:
# AJUSTA PTMO NETO CTA CK (B.9, Debe, cuenta Capital)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b9_ck"))
sql_aj_b9_ck = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       T1.FUENTE,
       '0912' AS PROC
INTO #aj_b9_ck
FROM #aj_d4d7 t1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE
"""
work_conn.execute(text(sql_aj_b9_ck))
aj_b9_ck = pd.read_sql(text("SELECT * FROM #aj_b9_ck"), work_conn)
_log("aj_b9_ck", aj_b9_ck)


In [ ]:
cols_aj_d4d7 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp in ["#aj_d4d7", "#aj_b8", "#aj_b8_ck", "#aj_b9_ck"]:
    sql_append_aj = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_aj_d4d7})
SELECT {cols_aj_d4d7}
FROM {tmp}
"""
    res = work_conn.execute(text(sql_append_aj))
    _log(f"APPEND TABLAS.BD_CTSI ({tmp})", res.rowcount)
for tmp in ["#aj_d4d7", "#aj_b8", "#aj_b8_ck", "#aj_b9_ck"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))


In [ ]:
# CIERRE 2019. Los recibidos del RM imputados en PROC 0906 se imputan como pagados por el sector 51 al RM,
# y este mismo valor se rebaja del sector 51 con CA 53.
# PAGADOS 51 AL RM
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d42_rm"))
sql_aj_d42_rm = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0919' AS PROC
INTO #aj_d42_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.PROC = '0906'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d42_rm))
aj_d42_rm = pd.read_sql(text("SELECT * FROM #aj_d42_rm"), work_conn)
_log("aj_d42_rm", aj_d42_rm)


In [ ]:
# PAGADOS AJUSTE EN 51 CON 53 (negado)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d42_resto"))
sql_aj_d42_resto = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0919' AS PROC
INTO #aj_d42_resto
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.PROC = '0906'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d42_resto))
aj_d42_resto = pd.read_sql(text("SELECT * FROM #aj_d42_resto"), work_conn)
_log("aj_d42_resto", aj_d42_resto)


In [ ]:
cols_aj_d42 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp in ["#aj_d42_rm", "#aj_d42_resto"]:
    sql_append_aj_d42 = f"""
INSERT INTO TABLAS.BD_CTSI ({cols_aj_d42})
SELECT {cols_aj_d42}
FROM {tmp}
"""
    res = work_conn.execute(text(sql_append_aj_d42))
    _log(f"APPEND TABLAS.BD_CTSI ({tmp})", res.rowcount)
for tmp in ["#aj_d42_resto", "#aj_d42_rm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))


In [ ]:
# IMPUTA AJUSTE EN BONOS PARA ELIMINAR SALDOS NEGATIVOS EN AF32
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=TABLAS.AJUSTE_BONOS: fuente y destino son tablas permanentes de la BD
sql_append_ajuste_bonos = """
INSERT INTO TABLAS.BD_CTSI
SELECT *
FROM TABLAS.dbo.AJUSTE_BONOS
"""
res = work_conn.execute(text(sql_append_ajuste_bonos))
_log("APPEND TABLAS.BD_CTSI (AJUSTE_BONOS)", res.rowcount)


In [ ]:
# CIERRE 2021: AJUSTA PTMO NETO EN SECTOR SEGUROS Y AUXILIARES AL DE LA CTA DE CAPITAL. AJUSTA LA CF CONTRA AJUSTES DE CONCILIACION. TMB EN BCO CENTRAL
# IMPUTA AF71 EN CF: sector 35 (S.125) o 36 (resto de seguros/auxiliares), reversando Financiera contra B.9
# El tramo corta aquí: WORK.SEG_CK queda para el siguiente tramo, que continúa el bloque PROC SQL
work_conn.execute(text("DROP TABLE IF EXISTS #seg_ck"))
sql_seg_ck = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       (CASE WHEN T2.C_SI_SCN = 'S.125' THEN 35 ELSE 36 END) AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA = 'Financiera' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0920' AS PROC
INTO #seg_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_SCN = 'B.9' AND T2.C_SI_SCN IN ('S.125', 'S.124', 'S.121', 'S.123'))
GROUP BY t1.AÑO, T1.TRIM, (CASE WHEN T2.C_SI_SCN = 'S.125' THEN 35 ELSE 36 END)
"""
work_conn.execute(text(sql_seg_ck))
seg_ck = pd.read_sql(text("SELECT * FROM #seg_ck"), work_conn)
_log("seg_ck", seg_ck)


In [ ]:
# IMPUTA AF71 EN REC VOLUMEN
work_conn.execute(text("DROP TABLE IF EXISTS #seg_rv"))
sql_seg_rv = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0903' AS PROC
INTO #seg_rv
FROM #seg_ck T1
"""
work_conn.execute(text(sql_seg_rv))


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #cf_51"))
sql_cf_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0904' AS PROC
INTO #cf_51
FROM #seg_ck T1
"""
work_conn.execute(text(sql_cf_51))


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #rv_51"))
sql_rv_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0905' AS PROC
INTO #rv_51
FROM #seg_ck T1
"""
work_conn.execute(text(sql_rv_51))


In [ ]:
# PROC APPEND (FORCE) de SEG_CK, SEG_RV, CF_51, RV_51 hacia TABLAS.BD_CTSI: server-side desde las #tmp
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_name in ["#seg_ck", "#seg_rv", "#cf_51", "#rv_51"]:
    sql_append = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM {tmp_name}
"""
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
# limpieza de #tmp de este bloque (re-ejecutable)
for t in ["#seg_ck", "#seg_rv", "#cf_51", "#rv_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA CTAS DE BALANCE, FINANCIERA, REC PRECIO Y VOLUMEN DE CUENTAS YG
sql_delete_cuentas = """
DELETE FROM TABLAS.dbo.BD_CTSI
WHERE C_CUENTA IN ('Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen','Bce Final')
  AND C_SCN IN ('D.41','K.1','P.2','P.51','P.11','D.42','D.5','B.9','B.90','B.10.2','B.10.3','P.52','D.62','D.75','D.1')
"""
res = work_conn.execute(text(sql_delete_cuentas))
_log("DELETE TABLAS.dbo.BD_CTSI (ctas balance/financiera/rec precio/volumen)", res.rowcount)


In [ ]:
# CALCULA E IMPUTA PTMO NETO, VALOR NETO, Y VARIACIONES DEL VALOR NETO DEBIDO A VOLUMENES Y REC PRECIOS
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))
sql_saldos = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
            WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
            WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END AS C_SCN,
       CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
            WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
            WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END AS N_SCN,
       'PS' AS FUENTE,
       '0009' AS PROC
INTO #saldos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CUENTA IN ('Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen','Bce Final')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA,
         CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
              WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
              WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END,
         CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
              WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
              WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END
"""
work_conn.execute(text(sql_saldos))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_saldos = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #saldos
"""
res = work_conn.execute(text(sql_append_saldos))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))


In [ ]:
# ELIMA CEROS
sql_delete_ceros = "DELETE FROM TABLAS.dbo.BD_CTSI WHERE DATO = 0"
res = work_conn.execute(text(sql_delete_ceros))
_log("DELETE TABLAS.dbo.BD_CTSI (ceros)", res.rowcount)


In [ ]:
# AJUSTA REMUNERACIONES EN HOGARES
# AJUSTE REMU RECIBIDAS POR HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_remu"))
sql_aj_remu = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '53' AS C_CAGENTE,
       'H' AS C_ENTRADA,
       'YG' AS C_CUENTA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'DI_Aj_REM' AS FUENTE,
       '0913' AS PROC
INTO #aj_remu
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN = 'D.1'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_remu))


In [ ]:
# AJUSTE AHORRO CTA YG HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_yg"))
sql_aj_b8_yg = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b8_yg
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b8_yg))


In [ ]:
# AJUSTE AHORRO CTA K HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_k"))
sql_aj_b8_k = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b8_k
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b8_k))


In [ ]:
# AJUSTE PTMO NETO CTA K HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b9_k"))
sql_aj_b9_k = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b9_k
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b9_k))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_CUENTA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_name in ["#aj_remu", "#aj_b8_yg", "#aj_b8_k", "#aj_b9_k"]:
    sql_append = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM {tmp_name}
"""
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
for t in ["#aj_remu", "#aj_b8_yg", "#aj_b8_k", "#aj_b9_k"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO NETO DE LAS SNF. AJUSTA PTMO NETO DE LA CF E IMPUTA NEGATIVO DE ESE AJUSTE EN EL PTMO NETO DE LA CF DE HOGARES
# TMB GENERA AJUSTE Y DISCREPANCIA EN LA CTA DE CADA SECTOR DEBIDO A ESTE AJUSTE
# CALCULA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_snf"))
sql_aj_d9_snf = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0918' AS PROC
INTO #aj_d9_snf
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.11'
GROUP BY T1.[AÑO], T1.TRIM, T2.C_SI_SCN_N1, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d9_snf))


In [ ]:
# IMPUTA NEGATIVO EN CTA FIN HH
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_hh"))
sql_aj_d9_hh = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_d9_hh
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_d9_hh))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_51"))
sql_aj_af71_cf_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '511' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_51
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_cf_51))


In [ ]:
# IMPUTA AJUSTE EN CTA REC VOLUMEN INST AF71 EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_51"))
sql_aj_af71_rv_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '511' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_51
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_rv_51))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_511"))
sql_aj_af71_cf_511 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_511
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_cf_511))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES (rec volumen)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_511"))
sql_aj_af71_rv_511 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '51' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_511
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_rv_511))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
cols_bd_ctsi_snf = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_snf = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_snf})
SELECT {cols_bd_ctsi_snf}
FROM #aj_d9_snf
"""
res = work_conn.execute(text(sql_append_snf))
_log("APPEND TABLAS.dbo.BD_CTSI desde #aj_d9_snf", res.rowcount)
sql_append_hh = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_snf})
SELECT {cols_bd_ctsi_snf}
FROM #aj_d9_hh
"""
res = work_conn.execute(text(sql_append_hh))
_log("APPEND TABLAS.dbo.BD_CTSI desde #aj_d9_hh", res.rowcount)
for tmp_name in ["#aj_af71_cf_51", "#aj_af71_rv_51", "#aj_af71_cf_511", "#aj_af71_rv_511"]:
    sql_append = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM {tmp_name}
"""
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
for t in ["#aj_d9_snf", "#aj_d9_hh", "#aj_af71_cf_51", "#aj_af71_rv_51", "#aj_af71_cf_511", "#aj_af71_rv_511"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO NETO DE HOGARES. AJUSTA PTMO NETO DE LA CF E IMPUTA NEGATIVO DE ESE AJUSTE EN EL PTMO NETO DE LA CF DE BANCOS
# TMB GENERA AJUSTE Y DISCREPANCIA EN LA CTA DE CADA SECTOR DEBIDO A ESTE AJUSTE
# CALCULA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_hh"))
sql_aj_d9_hh_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0933' AS PROC
INTO #aj_d9_hh
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.14'
GROUP BY T1.[AÑO], T1.TRIM, T2.C_SI_SCN_N1, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d9_hh_2))


In [ ]:
# IMPUTA NEGATIVO EN CTA FIN DE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_bcos"))
sql_aj_d9_bcos = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_d9_bcos
FROM #aj_d9_hh T1
"""
work_conn.execute(text(sql_aj_d9_bcos))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES (sector 511, contraparte 321)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_511"))
sql_aj_af71_cf_511_b = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '321' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_511
FROM #aj_d9_hh T1
"""
work_conn.execute(text(sql_aj_af71_cf_511_b))


In [ ]:
# IMPUTA AJUSTE EN CTA REC VOLUMEN INST AF71 HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_511"))
sql_aj_af71_rv_511_b = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '321' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_511
FROM #aj_d9_hh T1
"""
work_conn.execute(text(sql_aj_af71_rv_511_b))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_321"))
sql_aj_af71_cf_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '511' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_321
FROM #aj_d9_hh T1
"""
work_conn.execute(text(sql_aj_af71_cf_321))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 BANCOS (rec volumen)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_321"))
sql_aj_af71_rv_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '511' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_321
FROM #aj_d9_hh T1
"""
work_conn.execute(text(sql_aj_af71_rv_321))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
cols_bd_ctsi_hh = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_hh2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_hh})
SELECT {cols_bd_ctsi_hh}
FROM #aj_d9_hh
"""
res = work_conn.execute(text(sql_append_hh2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #aj_d9_hh (2)", res.rowcount)
sql_append_bcos = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_hh})
SELECT {cols_bd_ctsi_hh}
FROM #aj_d9_bcos
"""
res = work_conn.execute(text(sql_append_bcos))
_log("APPEND TABLAS.dbo.BD_CTSI desde #aj_d9_bcos", res.rowcount)
for tmp_name in ["#aj_af71_cf_511", "#aj_af71_rv_511", "#aj_af71_cf_321", "#aj_af71_rv_321"]:
    sql_append = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM {tmp_name}
"""
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
for t in ["#aj_d9_hh", "#aj_d9_bcos", "#aj_af71_cf_511", "#aj_af71_rv_511", "#aj_af71_cf_321", "#aj_af71_rv_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CALCULA E IMPUTA VARIACIONES DEL VALOR NETO DEBIDO A VOLUMENES Y REC PRECIOS
work_conn.execute(text("DROP TABLE IF EXISTS #saldos_2"))
sql_saldos_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END AS C_SCN,
       CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END AS N_SCN,
       'PS' AS FUENTE,
       '0010' AS PROC
INTO #saldos_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CUENTA IN ('Rec Precio','Rec Precio Reaj','Rec Volumen')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA,
         CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END,
         CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END
"""
work_conn.execute(text(sql_saldos_2))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_saldos_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #saldos_2
"""
res = work_conn.execute(text(sql_append_saldos_2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos_2", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #saldos_2"))


## S2_12_Reajustes

Calcula los reajustes (D.41/D.443/D.44) por sector desde AF.29/AF.42 y los sectoriza entre fondos mutuos money market y no money market, con y sin hogares, acumulando todo en TABLAS.REAJUSTES y reclasificando el sector 53 a 51022 al final

*confianza: low · verificador: revise · SAS: PROC SQL CREATE TABLE (agregaciones y joins) + PROC DATASETS APPEND FORCE + UPDATE, sobre TABLAS.BD_CTSI/TABLAS.REAJUSTES*

In [ ]:
# ========= S2_12_Reajustes =========
# COMPRIME TABLA PRINCIPAL (SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;)
# COMPRESS=YES es una opción de almacenamiento física de SAS sin equivalente en SQL Server: no hay nada que replicar salvo dejar la tabla intacta.
bd_ctsi = pd.read_sql(text("SELECT * FROM TABLAS.dbo.BD_CTSI"), engine)
_log("bd_ctsi", bd_ctsi)


In [ ]:
# CALCULO DE REAJUSTES DE INST AF.29 Y AF.42 DE TODOS LOS SECTORES QUE SE IMPUTAN EN CTA YG COMO INTERESES
# INCOPORA SECTOR DE CONTRAPARTIDA Y SECTOR DETALLADO
# CREATE TABLE TABLAS.REAJUSTES AS ... reemplaza la tabla física completa (base inicial, antes de los APPEND posteriores)
sql_reajustes = """
SELECT	'P' AS MONEDA,
	t1.[AÑO],
	T1.TRIM,
	T1.SECTOR,
	LEFT(CAST(t3.C_SI AS varchar(8)), 8) AS C_CAGENTE,
	'YG' AS C_CUENTA,
	CASE WHEN T1.C_ENTRADA = 'D' THEN 'H' ELSE 'D' END AS C_ENTRADA,
	SUM(T1.DATO) AS DATO,
	CAST('D.41' AS varchar(5)) AS C_SCN,
	CAST('Intereses' AS varchar(70)) AS N_SCN,
	'DI_Aj_REAJ' AS FUENTE
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION t2, TABLAS.dbo.T_CONTRAPARTIDAS t3
WHERE (T1.SECTOR = t2.C_SI) AND (T1.C_CAGENTE = t3.C_CAGENTE)
	AND (T1.C_SCN IN ('AF.29', 'AF.42') AND T1.C_CUENTA = 'Rec Precio Reaj'
	AND t2.C_SI_Publ <> 6)
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR,
	CASE WHEN T1.C_ENTRADA = 'D' THEN 'H' ELSE 'D' END,
	t3.C_SI
"""
reajustes = pd.read_sql(text(sql_reajustes), engine)
_log("reajustes", reajustes)


In [ ]:
# Reemplaza la tabla física TABLAS.REAJUSTES con la base inicial (D.41 Intereses de todos los sectores)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.REAJUSTES"))
    _log("DELETE TABLAS.dbo.REAJUSTES", res.rowcount)
reajustes.to_sql("REAJUSTES", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# CALCULA ESTRUCTURA SIN HOGARES PARA SECTORIZAR REAJUSTES A PAGAR POR FONDOS MUTUOS MONEY MARKET Y NO MONEY MARKET
# BALANCE DEPÓSITOS FFMM POR SECTOR SIN HOGARES
sql_bce_dep_fm = """
SELECT	'P' AS MONEDA,
	t1.[AÑO],
	T1.TRIM,
	t1.SECTOR,
	t1.C_CAGENTE,
	T1.C_CUENTA,
	T1.C_ENTRADA,
	SUM(T1.DATO) AS DATO,
	T1.C_SCN
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION t2
WHERE (T1.SECTOR = t2.C_SI)
	AND (T1.C_SCN IN ('AF.521', 'AF.522') AND T1.C_CUENTA = 'Bce Final' AND T1.SECTOR IN (3390102, 3390101)
	AND T1.C_ENTRADA = 'H' AND T1.C_CAGENTE NOT IN ('511'))
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, t1.C_CAGENTE
"""
bce_dep_fm = pd.read_sql(text(sql_bce_dep_fm), engine)
_log("bce_dep_fm", bce_dep_fm)


In [ ]:
# BALANCE DEPÓSITOS FFMM TOTAL SIN HOGARES
bce_dep_tot = (
    bce_dep_fm.groupby(["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_ENTRADA", "C_SCN"], as_index=False)["DATO"]
    .sum()
)
_log("bce_dep_tot", bce_dep_tot)


In [ ]:
# CALCULA ESTRUCTURA PARA SECTORIZAR REAJUSTES SIN HOGARES
# NOTA: TABLAS.T_CONTRAPARTIDAS sí está en el catálogo de conexiones (alias TABLAS); se lee vía engine
t_contrapartidas = pd.read_sql(text("SELECT C_CAGENTE, C_SI FROM TABLAS.dbo.T_CONTRAPARTIDAS"), engine)
cruce_est = bce_dep_fm.merge(
    bce_dep_tot, on=["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_ENTRADA", "C_SCN"],
    suffixes=("_fm", "_tot"),
).merge(t_contrapartidas, on="C_CAGENTE")
cruce_est["EST_calc"] = cruce_est["DATO_fm"] / cruce_est["DATO_tot"]
est_reaj = (
    cruce_est.groupby(["AÑO", "TRIM", "SECTOR", "C_SI"], as_index=False)["EST_calc"]
    .sum()
    .rename(columns={"EST_calc": "EST"})
)
est_reaj["C_CAGENTE"] = est_reaj["C_SI"].astype("Int64").astype(str).str.strip()
est_reaj = est_reaj.drop(columns=["C_SI"])
_log("est_reaj", est_reaj)


In [ ]:
# AJUSTE INTERESES PAGADOS EN FFMM MONEY MARKET, TOTAL...A SECTORIZAR
# TABLAS.DEP_HH_FM no está en el catálogo de conexiones ni en input_datasets: no se puede resolver de dónde sale
raise NotImplementedError("TABLAS.DEP_HH_FM: tabla externa no encontrada en el catálogo de conexiones ni en input_datasets del nodo; no se puede materializar REAJ_FMMM sin su origen")


In [ ]:
# REAJUSTES EN FFMM NO MONEY MARKET, TOTAL
sql_reaj_fmnm = """
SELECT	'P' AS MONEDA,
	t1.[AÑO],
	T1.TRIM,
	T1.SECTOR,
	t1.C_CAGENTE,
	'YG' AS C_CUENTA,
	CASE WHEN T1.C_ENTRADA = 'D' THEN 'H' ELSE 'D' END AS C_ENTRADA,
	SUM(T1.DATO) AS DATO,
	'D.443' AS C_SCN,
	'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva' AS N_SCN,
	'DI_Aj_REAJ' AS FUENTE
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION t2
WHERE (T1.SECTOR = t2.C_SI) AND (T1.C_SCN IN ('AF.29', 'AF.42') AND T1.C_CUENTA = 'Rec Precio Reaj') AND T1.SECTOR = 3390102
GROUP BY t1.[AÑO], T1.TRIM, t1.SECTOR,
	CASE WHEN T1.C_ENTRADA = 'D' THEN 'H' ELSE 'D' END,
	t1.C_CAGENTE
"""
reaj_fmnm = pd.read_sql(text(sql_reaj_fmnm), engine)
_log("reaj_fmnm", reaj_fmnm)


In [ ]:
# AJUSTE INTERESES PAGADOS EN FFMM NO MONEY MARKET, TOTAL...A SECTORIZAR
d41_fmnm = (
    reaj_fmnm.assign(C_CAGENTE="53", C_ENTRADA="D", C_SCN="D.443",
                      N_SCN="Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva")
    .groupby(["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE"], as_index=False)["DATO"]
    .sum()
)
_log("d41_fmnm", d41_fmnm)


In [ ]:
# UNE DATOS DE FONDOS MUTUOS CON REAJUSTES QUE PAGAN PARA SECTORIZARLOS (PROC DATASETS APPEND FORCE, acumula en memoria)
reaj_fmmm = pd.concat([reaj_fmmm, d41_fmnm], ignore_index=True)
_log("reaj_fmmm", reaj_fmmm)


In [ ]:
# SECTORIZA REAJUSTES PAGADOS POR FFMM DE HOGARES
# depende de TABLAS.DEP_HH_FM, cuyo origen no se pudo resolver (ver celda anterior con NotImplementedError)
raise NotImplementedError("FM_SECT_HH depende de TABLAS.DEP_HH_FM, tabla externa no encontrada en el catálogo de conexiones ni en input_datasets")


In [ ]:
# APPEND server-side: acumula FM_SECT_HH en la tabla física TABLAS.REAJUSTES (PROC DATASETS APPEND FORCE)
cols_reajustes = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE"
fm_sect_hh.to_sql("REAJUSTES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.REAJUSTES (FM_SECT_HH)", len(fm_sect_hh))


In [ ]:
# RESTA PORCION DE HOGARES DEL TOTAL
fm_hh_resta = fm_sect_hh.assign(C_CAGENTE="53", DATO=lambda d: d["DATO"] * -1)
_log("fm_hh_resta", fm_hh_resta)


In [ ]:
# Acumula la resta de hogares en REAJ_FMMM (PROC DATASETS APPEND FORCE)
reaj_fmmm = pd.concat([reaj_fmmm, fm_hh_resta], ignore_index=True)
_log("reaj_fmmm", reaj_fmmm)


In [ ]:
# SECTORIZA REAJUSTES PAGADOS POR FFMM SIN HOGARES
fm_sect = (
    reaj_fmmm.merge(est_reaj, on=["AÑO", "TRIM", "SECTOR"], suffixes=("", "_est"))
    .assign(DATO_calc=lambda d: d["DATO"] * d["EST"])
    .groupby(["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE_est", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE"], as_index=False)["DATO_calc"]
    .sum()
    .rename(columns={"C_CAGENTE_est": "C_CAGENTE", "DATO_calc": "DATO"})
)
_log("fm_sect", fm_sect)


In [ ]:
# APPEND server-side: acumula FM_SECT en la tabla física TABLAS.REAJUSTES (PROC DATASETS APPEND FORCE)
fm_sect.to_sql("REAJUSTES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.REAJUSTES (FM_SECT)", len(fm_sect))


In [ ]:
# GENERA INTERESES POR REAJUSTES RECIBIDOS POR LOS SECTORES DESDE LOS FONDOS MUTUOS PARA SER IMPUTADOS EN LA CTA YG
sql_reaj_fm_h = """
SELECT	MONEDA,
	[AÑO],
	TRIM,
	TRY_CAST(C_CAGENTE AS float) AS SECTOR,
	LEFT(CAST(SECTOR AS varchar(8)), 8) AS C_CAGENTE,
	C_CUENTA,
	'H' AS C_ENTRADA,
	DATO,
	C_SCN,
	N_SCN,
	FUENTE
FROM TABLAS.dbo.REAJUSTES
WHERE SECTOR IN (3390101, 3390102) AND C_ENTRADA = 'D' AND C_CAGENTE NOT IN ('6', '3390101', '3390102')
"""
reaj_fm_h = pd.read_sql(text(sql_reaj_fm_h), engine)
_log("reaj_fm_h", reaj_fm_h)


In [ ]:
# APPEND server-side: acumula REAJ_FM_H en la tabla física TABLAS.REAJUSTES (PROC DATASETS APPEND FORCE)
reaj_fm_h.to_sql("REAJUSTES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.REAJUSTES (REAJ_FM_H)", len(reaj_fm_h))


In [ ]:
# AJUSTE EN FONDOS DE PENSIONES POR TEMA DE REAJUSTES...SE DEBEN CALZAR PARA NO GENERAR PTMO NETO
sql_d44 = """
SELECT	MONEDA,
	[AÑO],
	TRIM,
	SECTOR,
	'511' AS C_CAGENTE,
	C_CUENTA,
	'D' AS C_ENTRADA,
	SUM(DATO) AS DATO,
	CASE WHEN C_SCN IN ('D.41', 'D.443') THEN 'D.44' ELSE C_SCN END AS C_SCN,
	CASE WHEN C_SCN IN ('D.41', 'D.443') THEN 'Renta de la propiedad atribuida a los titulares de pólizas de seguros' ELSE N_SCN END AS N_SCN,
	'DI_Aj_REAJ' AS FUENTE
FROM TABLAS.dbo.REAJUSTES
WHERE SECTOR IN (34, 341, 342) AND C_ENTRADA = 'H'
GROUP BY [AÑO], TRIM, SECTOR,
	CASE WHEN C_SCN IN ('D.41', 'D.443') THEN 'D.44' ELSE C_SCN END,
	CASE WHEN C_SCN IN ('D.41', 'D.443') THEN 'Renta de la propiedad atribuida a los titulares de pólizas de seguros' ELSE N_SCN END,
	MONEDA, C_CUENTA
"""
d44 = pd.read_sql(text(sql_d44), engine)
_log("d44", d44)


In [ ]:
# APPEND server-side: acumula D44 en la tabla física TABLAS.REAJUSTES (PROC DATASETS APPEND FORCE)
d44.to_sql("REAJUSTES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.REAJUSTES (D44)", len(d44))


In [ ]:
# DROP TABLE de los WORK intermedios: en Python no hay nada físico que borrar (son DataFrames en memoria); no requiere traducción

# UPDATE final: reclasifica sector 53 -> 51022 en la tabla física TABLAS.REAJUSTES
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.REAJUSTES SET SECTOR = :nuevo WHERE SECTOR = :viejo"), {"nuevo": 51022, "viejo": 53})
    _log("UPDATE TABLAS.dbo.REAJUSTES (sector 53->51022)", res.rowcount)
